# Publication Model — Stage 2: Spatial Super-Resolution

**Stage-2 copy of `GAT_LSTM_NN_Publication.ipynb` (frozen at sha256 a36d99ae0b233836).**
That notebook stays untouched so the shipped v5 run can be reproduced; all Stage-2
edits are applied here via `_nb_patch_sr_NN_*.py`.

- **Task:** co-temporal spatial reconstruction — given the x,z motion of the mooring-line nodes over a
  `window_len = 1000`-step (100 s, 10 Hz) window, predict the **tension at every node over the SAME window**.
  No forecasting. Tension is NOT an input.
- **Stage 1 (DONE, job 10532625):** matched resolution `N_out = N_in`, one weight set for all node
  counts {4,5,6,7,8,10,12,15,18,21}. Results in `Results/checkpoints_P1_matched_v5/`.
- **Stage 2 (THIS NOTEBOOK):** super-resolution — a query-based cross-attention decoder placed
  between the GAT encoder and the LSTM decouples `N_out` from `N_in`, so the dense 21-node tension
  field is reconstructed from as few as 4 motion sensors. `N_out = 21` is the ceiling (native FE
  resolution = the finest ground truth that exists).
- **Loss:** 8-term Kendall uncertainty weighting (MSE, L1, temporal-gradient, spectral, constitutive
  EA·strain, catenary equilibrium, nodal momentum balance, τ=0.9 peak pinball). The three physics
  terms act at the query stations, on the input geometry interpolated up to them — inference-available
  information only.
- **Baseline that must be beaten:** Stage-1 v5 evaluated at `N_in`, its predicted tension pushed to
  `N_out` by `resample_fe_output`. Setting `use_query_decoder=False` reduces this model exactly to
  Stage 1, so the baseline runs in this same notebook.
- **Design plan:** `Publication_Model_Plan.md` §10 in this folder.


In [1]:
import copy
import math
import os
import random

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import RandomSampler
from torch.utils.data import Dataset, DataLoader, Subset, Sampler
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv


# TF32: ~2× faster matmul on Ampere+ GPUs; silently ignored on CPU.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

In [2]:
# ===================================================================
# MOORING LINE PROPERTIES  (Santjer et al. 2025 – Table 2)
# ===================================================================
MOORING_PROPERTIES = {
    "diameter":         0.068,        # [m]
    "area":             0.003619,     # [m2]
    "mass_per_length":  28.23,        # [kg/m]  (rhoA)
    "EA":               232.7e6,      # [N]     axial stiffness
    "Cd_n":             2.6,          # normal drag coefficient
    "Cd_t":             1.4,          # axial drag coefficient
    "k_b":              3.0e6,        # [Pa/m]  seabed spring stiffness
    "c_b":              3.0e5,        # [Pa*s/m] seabed damping
}


def build_mooring_graph(
    x0_nodes: torch.Tensor,
    z0_nodes: torch.Tensor,
    tension0_nodes: torch.Tensor,
    s0_nodes: torch.Tensor,
    water_depth: float,
    depths_current=None,
) -> Data:
    """
    Build a static PyG graph from the first time-step (t=0.1 s) of gnl_data1.

    Node positions (x0, z0) and initial tension come directly from the FE
    output at t=0.1 s, resampled to the desired node count.

    Static node features [N x 12]:
        0  s_over_L_line    normalised arc-length position [0..1]
        1  is_anchor        1 for r0
        2  is_fairlead      1 for rN
        3  is_intermediate  1 for interior nodes
        4  x0               initial x-coordinate [m]  (from gnl_data1 t=0)
        5  z0               initial z-coordinate [m]  (from gnl_data1 t=0)
        6  nodal_mass       lumped mass [kg]
        7  diameter         line diameter [m]
        8  area             cross-sectional area [m2]
        9  submerged_weight submerged weight contribution [N]
        10 z_bed            seabed elevation [m]  (= anchor z from gnl_data1)
        11 tension0         initial tension [N]  (from gnl_data1 t=0)

    Cd_n / Cd_t are constant line properties, NOT per-node features:
    stored globally as data.Cd_n / data.Cd_t (used by the drag force
    features computed in the dataset).

    Static edge features [E x 10]:
        0  segment_length             reference arc length [m]
        1  segment_length_over_L_line normalised segment length
        2  EA                         axial stiffness [N]
        3  mass_per_length            linear density [kg/m]
        4  diameter                   line diameter [m]
        5  tangent_x                  x-component of reference tangent
        6  tangent_z                  z-component of reference tangent
        7  edge_is_anchor               1.0 if anchor-side edge  else 0.0
        8  edge_is_intermediate         1.0 if internal edge     else 0.0
        9  edge_is_fairlead             1.0 if fairlead-side edge else 0.0

    Parameters
    ----------
    x0_nodes : torch.Tensor  shape [N]
        Initial x-coordinates from gnl_data1 row 0, resampled to N nodes.
    z0_nodes : torch.Tensor  shape [N]
        Initial z-coordinates from gnl_data1 row 0, resampled to N nodes.
    tension0_nodes : torch.Tensor  shape [N]
        Initial tension from gnl_data1 row 0, resampled to N nodes.
    s0_nodes : torch.Tensor  shape [N]
        Arc-length coordinate s [m] per node from gnl_data1 row 0, resampled to N nodes.
        s0_nodes[0] == 0 (anchor), s0_nodes[-1] == L_line (fairlead).
    water_depth : float
        Total water depth h0 [m] read from the input CSV for this case.
    depths_current : sequence, optional
        Current-profile depths d1..d5 [m] read from the input CSV for this case.
    """
    props = MOORING_PROPERTIES

    h = float(water_depth)
    if h <= 0.0:
        raise ValueError(f"water_depth must be positive, got {water_depth}.")


    # ------------------------------------------------------------------
    # 1. NODE POSITIONS  (from gnl_data1 first time step, t=0.1 s)
    # ------------------------------------------------------------------
    x0        = x0_nodes
    z0        = z0_nodes
    num_nodes = x0.shape[0]
    L_line    = float(s0_nodes[-1])   # actual arc length from gnl_data1

    # ------------------------------------------------------------------
    # 2. ARC-LENGTH COORDINATE along the mooring line
    # ------------------------------------------------------------------
    s_over_L = s0_nodes / L_line

    # ------------------------------------------------------------------
    # 3. NODE TYPE FLAGS
    # ------------------------------------------------------------------
    is_anchor       = torch.zeros(num_nodes)
    is_fairlead     = torch.zeros(num_nodes)
    is_intermediate = torch.ones(num_nodes)
    is_anchor[0]        = 1.0
    is_fairlead[-1]     = 1.0
    is_intermediate[0]  = 0.0
    is_intermediate[-1] = 0.0

    # ------------------------------------------------------------------
    # 4. PHYSICAL NODE ATTRIBUTES
    # ------------------------------------------------------------------
    seg_len_nom = L_line / (num_nodes - 1)

    nodal_mass = torch.full((num_nodes,), props["mass_per_length"] * seg_len_nom)
    nodal_mass[0]  *= 0.5
    nodal_mass[-1] *= 0.5

    diameter_node = torch.full((num_nodes,), props["diameter"])
    area_node     = torch.full((num_nodes,), props["area"])

    rho_water = 1025.0
    g         = 9.81
    submerged_weight_per_length = (
        props["mass_per_length"] - rho_water * props["area"]
    ) * g
    submerged_weight_node = torch.full(
        (num_nodes,), submerged_weight_per_length * seg_len_nom
    )
    submerged_weight_node[0]  *= 0.5
    submerged_weight_node[-1] *= 0.5

    z_bed      = float(z0[0])       # seabed level = anchor elevation
    z_bed_node = torch.full((num_nodes,), z_bed)

    # ------------------------------------------------------------------
    # 5. STATIC NODE FEATURE MATRIX  [N x 12]
    # ------------------------------------------------------------------
    x = torch.stack([
        s_over_L,              # 0
        is_anchor,             # 1
        is_fairlead,           # 2
        is_intermediate,       # 3
        x0,                    # 4
        z0,                    # 5
        nodal_mass,            # 6
        diameter_node,         # 7
        area_node,             # 8
        submerged_weight_node, # 9
        z_bed_node,            # 10
        tension0_nodes,        # 11
    ], dim=1)

    # ------------------------------------------------------------------
    # 6. GRAPH CONNECTIVITY  –  bidirectional chain
    # ------------------------------------------------------------------
    senders, receivers = [], []
    for i in range(num_nodes - 1):
        senders   += [i,     i + 1]
        receivers += [i + 1, i    ]
    edge_index = torch.tensor([senders, receivers], dtype=torch.long)

    # ------------------------------------------------------------------
    # 7. STATIC EDGE FEATURES  [E x 10]  (same for forward and backward)
    # ------------------------------------------------------------------
    edge_features = []
    for i in range(num_nodes - 1):
        dx = float(x0[i + 1] - x0[i])
        dz = float(z0[i + 1] - z0[i])
        seg_len    = math.sqrt(dx ** 2 + dz ** 2)
        seg_normed = seg_len / L_line
        tx = dx / seg_len if seg_len > 0 else 0.0
        tz = dz / seg_len if seg_len > 0 else 0.0
        edge_is_anchor       = 1.0 if i == 0             else 0.0
        edge_is_fairlead     = 1.0 if i == num_nodes - 2 else 0.0
        edge_is_intermediate = 1.0 if edge_is_anchor == 0.0 and edge_is_fairlead == 0.0 else 0.0
        feat = [
            seg_len,                   # 0
            seg_normed,                # 1
            props["EA"],               # 2
            props["mass_per_length"],  # 3
            props["diameter"],         # 4
            tx,                        # 5
            tz,                        # 6
            edge_is_anchor,            # 7
            edge_is_intermediate,      # 8
            edge_is_fairlead,          # 9
        ]
        edge_features.extend([feat, feat])    # forward + backward

    edge_attr = torch.tensor(edge_features, dtype=torch.float32)

    # ------------------------------------------------------------------
    # 8. GRAPH OBJECT
    # ------------------------------------------------------------------
    pos  = torch.stack([x0, z0], dim=1)
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, pos=pos)

    data.num_nodes_total  = num_nodes
    data.water_depth      = h
    data.line_length = L_line
    data.ea               = props["EA"]
    data.mass_per_length  = props["mass_per_length"]
    data.diameter         = props["diameter"]
    data.area             = props["area"]
    data.k_b              = props["k_b"]
    data.c_b              = props["c_b"]
    data.Cd_n             = props["Cd_n"]     # global, no longer a node feature
    data.Cd_t             = props["Cd_t"]     # global, no longer a node feature
    data.L0               = seg_len_nom        # uniform reference segment length [m]
    data.z_bed            = z_bed
    data.z_bed_node       = z_bed_node
    rope_positions        = [f"r{i}" for i in range(num_nodes)]
    data.rope_positions   = rope_positions
    data.depths_current   = None if depths_current is None else [float(d) for d in depths_current]

    return data


def print_graph_summary(data: Data):
    print("----- GRAPH SUMMARY -----")
    print(f"Water depth [m] : {data.water_depth:.1f}")
    print(f"L_line [m]      : {data.line_length:.2f}")
    print(f"EA [N]          : {data.ea:.3e}")
    print(f"Nodes           : {data.num_nodes_total}  ({', '.join(data.rope_positions)})")
    print()
    print(f"x shape       : {tuple(data.x.shape)}")
    print(f"edge_index    : {tuple(data.edge_index.shape)}")
    print(f"edge_attr     : {tuple(data.edge_attr.shape)}")
    print()
    print("Static node feature columns (12):")
    for i, c in enumerate([
        "s_over_L_line", "is_anchor", "is_fairlead", "is_intermediate",
        "x0", "z0", "nodal_mass", "diameter", "area",
        "submerged_weight", "z_bed", "tension0",
    ]):
        print(f"  {i:2d}  {c}")
    print()
    print("Static edge feature columns (10):")
    for i, c in enumerate([
        "segment_length", "segment_length_over_L_line", "EA",
        "mass_per_length", "diameter", "tangent_x", "tangent_z",
        "edge_is_anchor", "edge_is_intermediate", "edge_is_fairlead",
    ]):
        print(f"  {i}  {c}")
    print()
    print("Node positions (x, z) [m]:")
    print(data.pos)
    
# build_mooring_graph now requires x0_nodes, z0_nodes, tension0_nodes from gnl_data1.
# Call build_real_load_case_datasets() to construct graphs with real initial conditions.

In [3]:
# -------------------------------------------------------------
# resampling utility
# -------------------------------------------------------------



def resample_fe_output(data_tN: torch.Tensor, target_N: int) -> torch.Tensor:
    """
    Resample FE output from N_orig nodes to target_N nodes along the
    normalised arc-length axis using linear interpolation (np.interp).

    The anchor (node 0) and fairlead (node -1) are always preserved
    exactly because they are the endpoints of the np.linspace grid.

    Parameters
    ----------
    data_tN : torch.Tensor  shape [T, N_orig]
        Full FE output for a single signal (e.g. x_abs, z_abs, tension)
        at N_orig nodes.
    target_N : int
        Desired number of output nodes (includes anchor and fairlead).

    Returns
    -------
    torch.Tensor  shape [T, target_N], dtype=torch.float32
    """
    T, N_orig = data_tN.shape
    if target_N == N_orig:
        return data_tN.to(torch.float32)

    src_pos = np.linspace(0.0, 1.0, N_orig, dtype=np.float32)
    tgt_pos = np.linspace(0.0, 1.0, target_N, dtype=np.float32)

    # Find left-neighbour index for each target position
    idx = np.searchsorted(src_pos, tgt_pos, side="right") - 1
    idx = np.clip(idx, 0, N_orig - 2)                      # [target_N]

    # Linear interpolation weight in [0, 1]
    weight = (tgt_pos - src_pos[idx]) / (src_pos[idx + 1] - src_pos[idx])
    weight = np.clip(weight, 0.0, 1.0).astype(np.float32)  # [target_N]

    arr   = data_tN.numpy()         # zero-copy view of float32 tensor
    left  = arr[:, idx]             # [T, target_N]
    right = arr[:, idx + 1]         # [T, target_N]
    out   = left + weight * (right - left)  # [T, target_N]

    return torch.from_numpy(out.astype(np.float32))


In [ ]:
ENV_COL_NAMES = [
    "h0", "Hs", "Tp",
    "d1", "v1",
    "d2", "v2",
    "d3", "v3",
    "d4", "v4",
    "d5", "v5",
]

FEATURE_DT = 0.1     # [s] cache sampling interval (10 Hz)
RHO_WATER  = 1025.0  # [kg/m3] sea-water density (drag force features)


def _time_gradient(sig: torch.Tensor, dt: float = FEATURE_DT) -> torch.Tensor:
    """Finite-difference time derivative along dim 0.

    Central differences in the interior, one-sided at the window edges.
    sig : [T, N]  ->  [T, N]
    """
    grad = torch.empty_like(sig)
    grad[1:-1] = (sig[2:] - sig[:-2]) / (2.0 * dt)
    grad[0]    = (sig[1] - sig[0]) / dt
    grad[-1]   = (sig[-1] - sig[-2]) / dt
    return grad


def _unit_tangents(x_abs: torch.Tensor, z_abs: torch.Tensor):
    """Per-node unit tangent of the line from instantaneous positions.

    Central differences along the node dimension (dim 1), one-sided at
    the anchor/fairlead ends. x_abs, z_abs : [T, N] -> (tx, tz) [T, N].
    """
    def _node_diff(sig):
        d = torch.empty_like(sig)
        d[:, 1:-1] = sig[:, 2:] - sig[:, :-2]
        d[:, 0]    = sig[:, 1] - sig[:, 0]
        d[:, -1]   = sig[:, -1] - sig[:, -2]
        return d
    dx = _node_diff(x_abs)
    dz = _node_diff(z_abs)
    mag = torch.sqrt(dx * dx + dz * dz).clamp_min(1e-8)
    return dx / mag, dz / mag


# --- patch sr-17: motion-sensor noise (EVAL ONLY; OFF unless a sweep sets it) --
# A spec is a plain dict so it pickles cleanly into DataLoader workers:
#   {"id": int, "name": str,
#    "sigma_x", "sigma_z":   [m]    white noise on the MEASURED positions
#    "sigma_vx", "sigma_vz": [m/s]  white noise on the REPORTED velocities
#    "sigma_bx", "sigma_bz": [m]    constant per-station installation bias}
# None, or an all-zero spec, means clean.
SENSOR_NOISE_KEYS = ("sigma_x", "sigma_z", "sigma_vx", "sigma_vz",
                     "sigma_bx", "sigma_bz")


def sensor_noise_is_off(spec) -> bool:
    """True when the spec cannot perturb anything -- the fast path.

    Checked BEFORE a generator is constructed, so a clean pass draws no random
    numbers at all and is bit-identical to an unpatched run. That property is
    what makes the sigma = 0 reproduction gate meaningful.
    """
    if not spec:
        return True
    return all(float(spec.get(k, 0.0) or 0.0) == 0.0 for k in SENSOR_NOISE_KEYS)


def sensor_noise_generator(spec, lc_id, case_id, target_N, idx):
    """Deterministic per-(level, case, N_in, window) CPU generator.

    out_N is deliberately EXCLUDED from the seed. The perturbation is on the
    INPUT, which depends on N_in only, while evaluate_superres_grid sweeps
    out_N over ten values and run_interpolation_baseline repeats every one of
    them. All of those passes must see the SAME realisation, or the
    decoder-minus-baseline delta surface does not subtract.

    The mix is injective over the real ranges (lc < 16, case < 512, N_in < 32,
    window < 32768).

    WARNING -- do NOT simplify the finaliser away. torch's CPU generator
    IGNORES seed bits above bit 32: two seeds differing only above bit 32
    produce an IDENTICAL stream (verified on torch 2.11). The injective mix
    puts the discriminating bits high, so without the splitmix64 avalanche
    distinct LEVELS silently share a realisation and the entire sweep is void
    while looking perfectly healthy. Measured after the fix: 9600 distinct
    (level, N_in, loc, case, window) combinations -> 9600 distinct streams,
    0 collisions.
    """
    raw = ((((int(spec.get("id", 0)) * 32 + int(target_N)) * 512
             + int(case_id)) * 16 + int(lc_id)) * 32768 + int(idx))
    m = (1 << 64) - 1
    z = (raw + 0x9E3779B97F4A7C15) & m
    z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & m
    z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & m
    z = (z ^ (z >> 31)) & m
    g = torch.Generator()
    g.manual_seed(int(z & 0x7FFFFFFF))
    return g


def sensor_noise_apply(spec, gen, sig, sigma_key, bias_key, n_nodes):
    """Additive white noise (+ a constant per-station bias) in PHYSICAL units.

    sig : [W, N]. The bias is drawn once per station and held over the window,
    which is what an installation or calibration offset looks like.
    """
    sd = float(spec.get(sigma_key, 0.0) or 0.0)
    if sd > 0.0:
        sig = sig + torch.randn(sig.shape, generator=gen, dtype=sig.dtype) * sd
    if bias_key:
        sb = float(spec.get(bias_key, 0.0) or 0.0)
        if sb > 0.0:
            sig = sig + torch.randn((1, int(n_nodes)), generator=gen,
                                    dtype=sig.dtype) * sb
    return sig


class MooringSequenceDatasetPositionTension(Dataset):
    """
    Lazy sliding-window dataset for the PUBLICATION reconstruction task.

    Co-temporal motion -> tension: the input window and the target window are
    the SAME window_len rows of the case time series. There is no forecast
    horizon, and tension NEVER enters the inputs.

    Data is stored on disk as uncompressed .npy files and accessed via mmap.
    File handles are opened lazily inside each DataLoader worker on the first
    __getitem__ call, so ~30,000 dataset objects consume negligible RAM.

    Dynamic node features (9 base + 13 env = 22 total; env optional):
    ---------------------------------------------------------------
    0  x_abs             absolute horizontal node position [m]
    1  z_abs             absolute vertical node position [m]
    2  x_vel             horizontal node velocity [m/s] (finite difference)
    3  z_vel             vertical node velocity [m/s] (finite difference)
    4  contact_flag      binary: 1 if node touches seabed
    5  penetration_depth vertical penetration below seabed [m]
    6  bed_reaction_z    seabed reaction [N]: submerged weight +
                         (k_b*penetration - c_b*v_z) * contact area
    7  F_drag_t          axial quadratic drag force [N] (opposes v_t)
    8  F_drag_n          normal quadratic drag force [N] (opposes v_n)
    9+ h0, Hs, Tp, d1, v1, ..., d5, v5  env features (if use_env_features)

    NOTE: disabling use_velocity_features, use_drag_features or
    use_env_features changes the
    feature layout — cfg.node_continuous_idx / node_binary_idx must then be
    overridden to match in the run cell.

    Target features (1):
    --------------------
    0  tension  [N]  co-temporal with the input window
    """

    def __init__(
        self,
        graph_data,
        case_dir: Path,
        target_N: int,
        window_len: int,
        contact_tol: float = 1e-6,
        max_time_steps: int = None,
        use_env_features: bool = True,
        use_velocity_features: bool = True,
        use_drag_features: bool = True,
        out_N: Optional[int] = None,
    ):
        """
        Parameters
        ----------
        graph_data : torch_geometric.data.Data
            Static graph built by build_mooring_graph() for target_N nodes.
        case_dir : Path
            Directory containing x_abs.npy, z_abs.npy, tension.npy,
            reference.npy, env.npy for this (loc, case).
        target_N : int
            Number of nodes to resample to.
        window_len : int
            Co-temporal input/output window length in time steps (10 Hz).
        contact_tol : float
            Tolerance for seabed contact detection [m].
        max_time_steps : int, optional
            If set, expose only the first max_time_steps rows (debug mode).
        use_env_features : bool
            Include the 13 constant-per-case env columns in the inputs.
        use_velocity_features : bool
            Include finite-difference x/z velocities in the inputs.
        use_drag_features : bool
            Include quadratic axial/normal drag force features in the inputs.
        out_N : int, optional
            Stage-2 super-resolution: number of TARGET stations. None (default)
            means matched resolution, i.e. out_N == target_N, reproducing the
            Stage-1 behaviour exactly. When set, y_seq is resampled to out_N
            while the inputs stay at target_N = N_in. Mutable at runtime so the
            N_in x N_out evaluation grid can sweep it without rebuilding.
        """
        self.graph_data   = graph_data
        self._case_dir    = Path(case_dir)
        self._contact_tol = contact_tol
        self.target_N     = target_N
        self.window_len   = window_len
        self.use_env_features      = use_env_features
        self.use_velocity_features = use_velocity_features
        self.use_drag_features     = use_drag_features
# --- patch sr-01: Stage-2 target resolution (None = matched) ---
        self.out_N = out_N
# --- patch sr-17: motion-sensor noise spec + the identity used to seed it.
        # Both are set by MultiLoadCaseWindowSubset.__getitem__, exactly as
        # out_N is, so a sweep can change them between passes without
        # rebuilding (persistent_workers is False everywhere).
        self.sensor_noise = None
        self._sn_key = (0, 0)

        # Per-node effective segment length: end nodes carry HALF a segment,
        # matching the lumped convention in build_mooring_graph (nodal_mass
        # and submerged_weight are halved at anchor/fairlead). Used for the
        # contact and drag reference areas.
        node_len = torch.full((target_N,), float(graph_data.L0))
        node_len[0]  *= 0.5
        node_len[-1] *= 0.5
        self._node_len = node_len.unsqueeze(0)   # [1, N]

        if window_len < 2:
            raise ValueError(f"window_len must be >= 2, got {window_len}.")

        # Read only the array shape — numpy reads just the .npy header
        T_full = np.load(self._case_dir / "x_abs.npy", mmap_mode="r").shape[0]
        self.num_steps = T_full if max_time_steps is None else min(T_full, max_time_steps)
        self.num_nodes = target_N
        self.num_windows = self.num_steps - window_len + 1

        if self.num_windows <= 0:
            raise ValueError(
                f"Not enough time steps ({self.num_steps}) for window_len={window_len}."
            )

        # Load the small env row eagerly (13 floats — negligible memory)
        self._env_row = torch.tensor(
            np.load(self._case_dir / "env.npy"), dtype=torch.float32
        )   # [13]

        # Feature-count metadata (used by build_model_from_dataset_example)
        n_base = (5 + (2 if use_velocity_features else 0)
                    + (2 if use_drag_features else 0))
        self.n_dynamic_node_features = n_base + (len(ENV_COL_NAMES) if use_env_features else 0)
        self.n_dynamic_edge_features = 4
        self.n_targets               = 1

        # Mmap handles — None until first __getitem__ in this worker process
        self._mmap = None

    # ------------------------------------------------------------------
    # Feature-name metadata (used by print_dataset_summary and leak checks)
    # ------------------------------------------------------------------
    @property
    def dynamic_feature_names(self):
        names = ["x_abs", "z_abs"]
        if self.use_velocity_features:
            names += ["x_vel", "z_vel"]
        names += ["contact_flag", "penetration_depth", "bed_reaction_z"]
        if self.use_drag_features:
            names += ["F_drag_t", "F_drag_n"]
        if self.use_env_features:
            names += list(ENV_COL_NAMES)
        return names

    @property
    def dynamic_edge_feature_names(self):
        return [
            "edge_contact_fraction",
            "edge_penetration_mean",
            "edge_bed_reaction_mean",
            "edge_touchdown_flag",
        ]

    @property
    def target_feature_names(self):
        return ["tension"]

    def __len__(self):
        return self.num_windows

    def _open_mmap(self):
        """Open mmap handles for this worker process (called once per worker)."""
        self._mmap = {
            k: np.load(self._case_dir / f"{k}.npy", mmap_mode="r")
            for k in ("x_abs", "z_abs", "tension", "reference")
        }

    def __getitem__(self, idx):
        if idx < 0 or idx >= self.num_windows:
            raise IndexError(f"Window index {idx} out of range [0, {self.num_windows}).")

        # Open mmap lazily — each DataLoader worker opens its own file handles
        if self._mmap is None:
            self._open_mmap()

        start = idx
        end   = idx + self.window_len

        # 1. Read the required rows from mmap — OS page cache backed
        x_abs_w = torch.tensor(self._mmap["x_abs"][start:end],   dtype=torch.float32)
        z_abs_w = torch.tensor(self._mmap["z_abs"][start:end],   dtype=torch.float32)
        ten_w   = torch.tensor(self._mmap["tension"][start:end], dtype=torch.float32)

        # 2. Resample from N_FULL=21 nodes to target_N
        x_abs   = resample_fe_output(x_abs_w, self.target_N)    # [W, target_N]
        z_abs   = resample_fe_output(z_abs_w, self.target_N)
        _out_N = self.target_N if self.out_N is None else int(self.out_N)
        tension = resample_fe_output(ten_w,   _out_N)          # TARGET ONLY — never an input

        # --- patch sr-17: motion-sensor noise. Applied to the MEASURED
        #     positions, before anything is derived from them. Steps 3-7 below
        #     compute velocity, contact flag, penetration, bed reaction, both
        #     drag features and all four dynamic EDGE features from x_abs/z_abs,
        #     and the Kendall physics terms re-derive geometry from
        #     x_seq[..., 0:2], so a perturbation here propagates everywhere it
        #     should and nowhere it should not. `tension` above came from the
        #     CLEAN mmap, so the TARGET is never perturbed.
        _sn   = getattr(self, "sensor_noise", None)
        _sn_g = None
        _sn_clean = None
        if not sensor_noise_is_off(_sn):
            _sn_lc, _sn_cs = getattr(self, "_sn_key", (0, 0))
            _sn_g = sensor_noise_generator(_sn, _sn_lc, _sn_cs,
                                           self.target_N, idx)
            _sn_clean = (x_abs, z_abs)     # kept: see the velocity block
            x_abs = sensor_noise_apply(_sn, _sn_g, x_abs, "sigma_x",
                                       "sigma_bx", self.target_N)
            z_abs = sensor_noise_apply(_sn, _sn_g, z_abs, "sigma_z",
                                       "sigma_bz", self.target_N)

        # 3. Node velocities (always computed: bed damping + drag need them)
        _vx_src, _vz_src = x_abs, z_abs
        if _sn_g is not None and (float(_sn.get("sigma_vx", 0.0) or 0.0) > 0.0
                                  or float(_sn.get("sigma_vz", 0.0) or 0.0) > 0.0):
            # ARM P -- a fused GNSS-IMU reports velocity DIRECTLY (IMU +
            # Doppler), so its velocity error is INDEPENDENT of its position
            # error. Differentiate the CLEAN signal and give velocity its own
            # noise below, otherwise the channel would carry the position
            # noise derivative PLUS its own and would not be independent at all.
            # ARM C (sigma_v = 0) skips this: velocity is DERIVED from the noisy
            # position and inherits ~7x it at dt = 0.1 s -- the pessimistic bound.
            _vx_src, _vz_src = _sn_clean
        x_vel = _time_gradient(_vx_src)      # [W, N]
        z_vel = _time_gradient(_vz_src)
        if _sn_g is not None:
            x_vel = sensor_noise_apply(_sn, _sn_g, x_vel, "sigma_vx",
                                       None, self.target_N)
            z_vel = sensor_noise_apply(_sn, _sn_g, z_vel, "sigma_vz",
                                       None, self.target_N)

        # 4. Seabed interaction. Unified reaction for contacted nodes:
        #    submerged weight + (spring stiffness - vertical damping) x area.
        #    k_b [Pa/m] and c_b [Pa*s/m] are DISTRIBUTED quantities, so the
        #    projected contact area (diameter x L_eff; L_eff = L0, halved at
        #    the anchor/fairlead end nodes) restores Newtons; the
        #    -c_b*v_z damping term resists vertical motion while in contact.
        z_bed          = self.graph_data.z_bed_node.detach().cpu().unsqueeze(0)   # [1, N]
        contact_flag   = (z_abs <= z_bed + self._contact_tol).float()
        penetration    = torch.clamp(z_bed - z_abs, min=0.0)
        submerged_w    = self.graph_data.x[:, 9].detach().cpu().unsqueeze(0)      # [1, N]
        contact_area   = self.graph_data.diameter * self._node_len                # [1,N] m2
        v_z            = z_vel
        bed_reaction   = torch.where(
            contact_flag > 0.0,
            submerged_w + (self.graph_data.k_b * penetration
                           - self.graph_data.c_b * v_z) * contact_area,
            torch.zeros_like(submerged_w),
        )

        # 5. Quadratic drag force features: node velocity projected on the
        #    instantaneous line tangent/normal; negative sign = drag opposes
        #    the motion (v * |v| keeps the quadratic law sign-correct).
        if self.use_drag_features:
            tx, tz = _unit_tangents(x_abs, z_abs)
            v_t = x_vel * tx + z_vel * tz                    # axial component
            v_n = -x_vel * tz + z_vel * tx                   # normal component
            A_n = self.graph_data.diameter * self._node_len                # [1,N] projected
            A_t = math.pi * self.graph_data.diameter * self._node_len      # [1,N] wetted
            F_drag_t = -0.5 * RHO_WATER * self.graph_data.Cd_t * A_t * v_t * v_t.abs()
            F_drag_n = -0.5 * RHO_WATER * self.graph_data.Cd_n * A_n * v_n * v_n.abs()

        # 6. Dynamic node features [W, N, 22] (motion + derived + env; NO tension)
        feats = [x_abs, z_abs]
        if self.use_velocity_features:
            feats += [x_vel, z_vel]
        feats += [contact_flag, penetration, bed_reaction]
        if self.use_drag_features:
            feats += [F_drag_t, F_drag_n]
        dyn = torch.stack(feats, dim=-1)   # [W, N, 9]
        if self.use_env_features:
            env_exp = self._env_row.unsqueeze(0).unsqueeze(0).expand(
                self.window_len, self.target_N, -1
            )
            dyn = torch.cat([dyn, env_exp], dim=-1)   # [W, N, 22]

        # 7. Dynamic edge features [W, E, 4]
        src_nodes, tgt_nodes = self.graph_data.edge_index
        edge_dyn = torch.stack(
            [
                0.5 * (contact_flag[:, src_nodes] + contact_flag[:, tgt_nodes]),
                0.5 * (penetration[:, src_nodes]  + penetration[:, tgt_nodes]),
                0.5 * (bed_reaction[:, src_nodes]  + bed_reaction[:, tgt_nodes]),
                (contact_flag[:, src_nodes] != contact_flag[:, tgt_nodes]).float(),
            ],
            dim=-1,
        )   # [W, E, 4]

        return {
            "graph":     self.graph_data.clone(),
            "x_seq":     dyn,                        # [W, N, 22]
            "edge_seq":  edge_dyn,                   # [W, E, 4]
            "y_seq":     tension.unsqueeze(-1),      # [W, N, 1] — co-temporal tension
            "start_idx": idx,
            "out_N":     _out_N,
        }


def print_dataset_summary(dataset: MooringSequenceDatasetPositionTension):
    print("----- DATASET SUMMARY (reconstruction task) -----")
    print(f"Time steps : {dataset.num_steps}")
    print(f"Nodes      : {dataset.num_nodes}")
    print(f"Window len : {dataset.window_len} (co-temporal input/output)")
    print(f"Windows    : {len(dataset)}")
    print("Dynamic input features :", dataset.dynamic_feature_names)
    print("Dynamic edge  features :", dataset.dynamic_edge_feature_names)
    print("Target features        :", dataset.target_feature_names)
    sample = dataset[0]
    print("x_seq   :", tuple(sample["x_seq"].shape))
    print("edge_seq:", tuple(sample["edge_seq"].shape))
    print("y_seq   :", tuple(sample["y_seq"].shape))


In [5]:
class MooringGATEncoder(nn.Module):
    """
    Spatial graph encoder for one history time step.

    It takes:
      - static node features from graph.x
      - static edge features from graph.edge_attr
      - dynamic node features for one time step x_t
      - dynamic edge features for one time step edge_t

    and returns:
      - node embeddings h_t for that time step

    Expected shapes for one sample:
      graph.x         : [N, n_static_node_features]
      graph.edge_index: [2, E]
      graph.edge_attr : [E, n_static_edge_features]
      x_t             : [N, n_dynamic_node_features]
      edge_t          : [E, n_dynamic_edge_features]

    Output:
      h_t             : [N, gat_hidden_dim]
    """

    def __init__(
        self,
        n_static_node_features: int = 12,
        n_dynamic_node_features: int = 6,
        n_static_edge_features: int = 10,
        n_dynamic_edge_features: int = 4,
        gat_hidden_dim: int = 64,
        gat_out_dim: int = 64,
        num_heads: int = 4,
        dropout: float = 0.1,
        use_layernorm: bool = True,
        add_residual_projection: bool = True,
        use_global_context: bool = False,
    ):
        super().__init__()

        self.n_static_node_features = n_static_node_features
        self.n_dynamic_node_features = n_dynamic_node_features
        self.n_static_edge_features = n_static_edge_features
        self.n_dynamic_edge_features = n_dynamic_edge_features

        self.node_input_dim = n_static_node_features + n_dynamic_node_features
        self.edge_input_dim = n_static_edge_features + n_dynamic_edge_features

        self.gat_hidden_dim = gat_hidden_dim
        self.gat_out_dim = gat_out_dim
        self.num_heads = num_heads
        self.dropout = dropout
        self.use_layernorm = use_layernorm

        # -------------------------------------------------------------
        # GAT layer 1
        # concat=True => output dim = gat_hidden_dim * num_heads
        # -------------------------------------------------------------
        self.gat1 = GATv2Conv(
            in_channels=self.node_input_dim,
            out_channels=gat_hidden_dim,
            heads=num_heads,
            concat=True,
            dropout=dropout,
            edge_dim=self.edge_input_dim,
            add_self_loops=False,
            bias=True,
        )

        self.norm1 = nn.LayerNorm(gat_hidden_dim * num_heads) if use_layernorm else nn.Identity()

        # -------------------------------------------------------------
        # GAT layer 2
        # concat=False => output dim = gat_out_dim
        # -------------------------------------------------------------
        self.gat2 = GATv2Conv(
            in_channels=gat_hidden_dim * num_heads,
            out_channels=gat_out_dim,
            heads=1,
            concat=False,
            dropout=dropout,
            edge_dim=self.edge_input_dim,
            add_self_loops=False,
            bias=True,
        )

        self.norm2 = nn.LayerNorm(gat_out_dim) if use_layernorm else nn.Identity()

        self.act = nn.ELU()
        self.dropout_layer = nn.Dropout(dropout)

        # Optional residual projection from raw concatenated node input
        if add_residual_projection:
            self.residual_proj = nn.Linear(self.node_input_dim, gat_out_dim)
        else:
            self.residual_proj = None

        # patch 17: optional line-global context (virtual-node broadcast).
        # Reduces the effective graph diameter to ~2 so 2 GAT layers reach
        # the touchdown-point signal from far grounded nodes (Finding 4).
        self.use_global_context = use_global_context
        if use_global_context:
            _gc_dim = gat_hidden_dim * num_heads
            self.global_mlp = nn.Sequential(
                nn.Linear(_gc_dim, _gc_dim),
                nn.ELU(),
            )
        else:
            self.global_mlp = None

    def forward(self, graph, x_t: torch.Tensor, edge_t: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        graph : torch_geometric.data.Data
            Static graph object containing graph.x, graph.edge_index, graph.edge_attr.
        x_t : torch.Tensor
            Dynamic node features at one time step, shape [N, n_dynamic_node_features].
        edge_t : torch.Tensor
            Dynamic edge features at one time step, shape [E, n_dynamic_edge_features].

        Returns
        -------
        h_t : torch.Tensor
            Encoded node embeddings for this time step, shape [N, gat_out_dim].
        """

        # -------------------------------------------------------------
        # Basic shape checks
        # -------------------------------------------------------------
        if x_t.dim() != 2:
            raise ValueError(f"x_t must have shape [N, F_dyn_node], got {tuple(x_t.shape)}")

        if edge_t.dim() != 2:
            raise ValueError(f"edge_t must have shape [E, F_dyn_edge], got {tuple(edge_t.shape)}")

        x_static = graph.x
        edge_index = graph.edge_index
        edge_static = graph.edge_attr

        if x_static.size(0) != x_t.size(0):
            raise ValueError(
                f"Node count mismatch: graph.x has {x_static.size(0)} nodes, "
                f"but x_t has {x_t.size(0)} nodes."
            )

        if edge_static.size(0) != edge_t.size(0):
            raise ValueError(
                f"Edge count mismatch: graph.edge_attr has {edge_static.size(0)} edges, "
                f"but edge_t has {edge_t.size(0)} edges."
            )

        if x_static.size(1) != self.n_static_node_features:
            raise ValueError(
                f"Expected {self.n_static_node_features} static node features, "
                f"got {x_static.size(1)}."
            )

        if x_t.size(1) != self.n_dynamic_node_features:
            raise ValueError(
                f"Expected {self.n_dynamic_node_features} dynamic node features, "
                f"got {x_t.size(1)}."
            )

        if edge_static.size(1) != self.n_static_edge_features:
            raise ValueError(
                f"Expected {self.n_static_edge_features} static edge features, "
                f"got {edge_static.size(1)}."
            )

        if edge_t.size(1) != self.n_dynamic_edge_features:
            raise ValueError(
                f"Expected {self.n_dynamic_edge_features} dynamic edge features, "
                f"got {edge_t.size(1)}."
            )

        # -------------------------------------------------------------
        # Concatenate static + dynamic features for this time step
        # -------------------------------------------------------------
        x_in = torch.cat([x_static, x_t], dim=-1)           
        edge_in = torch.cat([edge_static, edge_t], dim=-1) 

        # -------------------------------------------------------------
        # GAT block 1
        # -------------------------------------------------------------
        h = self.gat1(x_in, edge_index, edge_in)            # [N, gat_hidden_dim * num_heads]
        h = self.norm1(h)
        h = self.act(h)
        h = self.dropout_layer(h)

        # patch 17: inject line-global context between message-passing layers
        if self.global_mlp is not None:
            g = self.global_mlp(h.mean(dim=0, keepdim=True))   # [1, hid*heads]
            h = h + g

        # -------------------------------------------------------------
        # GAT block 2
        # -------------------------------------------------------------
        h = self.gat2(h, edge_index, edge_in)               # [N, gat_out_dim]
        h = self.norm2(h)

        # -------------------------------------------------------------
        # Optional residual connection from raw input
        # -------------------------------------------------------------
        if self.residual_proj is not None:
            h = h + self.residual_proj(x_in)

        h = self.act(h)
        h = self.dropout_layer(h)

        return h

    def forward_batch(
        self,
        x_static: torch.Tensor,
        x_dynamic: torch.Tensor,
        edge_static: torch.Tensor,
        edge_dynamic: torch.Tensor,
        edge_index: torch.Tensor,
    ) -> torch.Tensor:
        """
        Vectorised forward over M independent graphs with shared topology.
        Builds one block-diagonal super-graph (M*N nodes, M*E edges) and
        runs GATv2Conv once instead of M serial kernel launches.

        Parameters
        ----------
        x_static    : [M, N, F_static_node]
        x_dynamic   : [M, N, F_dyn_node]
        edge_static : [M, E, F_static_edge]
        edge_dynamic: [M, E, F_dyn_edge]
        edge_index  : [2, E]  shared topology for all M graphs
        """
        M, N, _ = x_dynamic.shape
        E = edge_index.size(1)
        device = x_dynamic.device

        x_in    = torch.cat([x_static,    x_dynamic],   dim=-1)  # [M, N, F_node_total]
        edge_in = torch.cat([edge_static, edge_dynamic], dim=-1)  # [M, E, F_edge_total]

        x_in_flat    = x_in.view(M * N, -1)    # [M*N, F_node_total]
        edge_in_flat = edge_in.view(M * E, -1) # [M*E, F_edge_total]

        # Tile edge_index with per-graph node offsets -> block-diagonal adjacency
        offsets  = torch.arange(M, device=device) * N               # [M]
        ei_tiled = edge_index.unsqueeze(1) + offsets.view(1, M, 1)  # [2, M, E]
        ei_flat  = ei_tiled.reshape(2, M * E)                       # [2, M*E]

        h = self.gat1(x_in_flat, ei_flat, edge_in_flat)
        h = self.norm1(h)
        h = self.act(h)
        h = self.dropout_layer(h)

        # patch 17: per-graph line-global context between message-passing layers
        if self.global_mlp is not None:
            hg = h.view(M, N, -1)
            g  = self.global_mlp(hg.mean(dim=1, keepdim=True))   # [M, 1, hid*heads]
            h  = (hg + g).reshape(M * N, -1)

        h = self.gat2(h, ei_flat, edge_in_flat)
        h = self.norm2(h)

        if self.residual_proj is not None:
            h = h + self.residual_proj(x_in_flat)

        h = self.act(h)
        h = self.dropout_layer(h)

        return h.view(M, N, self.gat_out_dim)  # [M, N, gat_out_dim]


In [ ]:
class NodeTemporalLSTM(nn.Module):
    """
    Temporal encoder that processes the sequence of spatial node embeddings
    produced by the MooringGATEncoder.

    Reconstruction task: the model must emit one output PER TIME STEP, so the
    canonical path is encode_sequence(), which returns the full LSTM output
    sequence (LayerNorm + dropout applied per step). Both the single-sample
    and the batched forward paths of MooringGATLSTM call encode_sequence(),
    so there is a single source of truth for the readout behaviour.

    Input:
        H : [window_len, N, gat_out_dim]

    Output (forward, return_sequences=True):
        node_temporal : [N, window_len, output_dim]

    Notes:
        - This block is node-wise in time.
        - It does NOT mix nodes with each other.
        - Spatial coupling has already been handled by the GAT encoder.
    """

    def __init__(
        self,
        input_dim: int = 64,
        lstm_hidden_dim: int = 128,
        num_lstm_layers: int = 1,
        dropout: float = 0.1,
        bidirectional: bool = False,
        use_layernorm: bool = True,
        return_sequences: bool = True,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.lstm_hidden_dim = lstm_hidden_dim
        self.num_lstm_layers = num_lstm_layers
        self.bidirectional = bidirectional
        self.use_layernorm = use_layernorm
        self.return_sequences = return_sequences

        # PyTorch LSTM only uses dropout internally when num_layers > 1
        lstm_dropout = dropout if num_lstm_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            dropout=lstm_dropout,
            bidirectional=bidirectional,
        )

        self.output_dim = lstm_hidden_dim * (2 if bidirectional else 1)

        self.norm = nn.LayerNorm(self.output_dim) if use_layernorm else nn.Identity()
        self.dropout_layer = nn.Dropout(dropout)

    def encode_sequence(self, seqs: torch.Tensor) -> torch.Tensor:
        """
        Run the LSTM over a batch of per-node sequences and return the FULL
        per-step output (norm + dropout applied at every step).

        Parameters
        ----------
        seqs : torch.Tensor
            [S, window_len, input_dim] — S independent node sequences
            (S = N for the single path, S = B*N for the batched path).

        Returns
        -------
        torch.Tensor
            [S, window_len, output_dim]
        """
        lstm_out, _ = self.lstm(seqs)            # [S, W, output_dim]
        lstm_out = self.norm(lstm_out)           # LayerNorm over last dim, per step
        lstm_out = self.dropout_layer(lstm_out)
        return lstm_out

    def forward(self, H: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        H : torch.Tensor
            Sequence of node embeddings from the spatial encoder.
            Expected shape: [window_len, N, input_dim]

        Returns
        -------
        node_temporal : torch.Tensor
            [N, window_len, output_dim] if return_sequences (reconstruction),
            [N, output_dim] otherwise (legacy last-step readout).
        """
        if H.dim() != 3:
            raise ValueError(
                f"H must have shape [window_len, N, input_dim], got {tuple(H.shape)}"
            )

        window_len, N, F = H.shape

        if F != self.input_dim:
            raise ValueError(
                f"Expected input_dim={self.input_dim}, but got last dimension {F}."
            )

        # Rearrange so each node becomes one sequence sample:
        # [window_len, N, input_dim] -> [N, window_len, input_dim]
        H_nodes = H.permute(1, 0, 2).contiguous()

        seq_out = self.encode_sequence(H_nodes)   # [N, W, output_dim]

        if self.return_sequences:
            return seq_out
        return seq_out[:, -1, :]                  # legacy: [N, output_dim]


In [ ]:
# -------------------------------------------------------------
# Stage 2 — query-based cross-attention decoder (super-resolution)
# -------------------------------------------------------------
#
# Sits BETWEEN the GAT encoder and the LSTM:
#
#   x_seq [B,W,N_in,F] --GAT--> H [B*W, N_in, d]
#                      --QueryTensionDecoder--> [B*W, N_out, d]
#                      --LSTM (per query station)--> --head--> [B,W,N_out,1]
#
# so temporal_encoder and prediction_head keep their Stage-1 shapes and a
# Stage-1 checkpoint transfers whole. Every weight remains node-count
# agnostic, which is why one weight set already spans N = 4..21.


def interp_nodes(t: torch.Tensor, n_out: int) -> torch.Tensor:
    """Linear arc-length interpolation along the NODE axis (dim -2).

    [..., N_in, F] -> [..., N_out, F]; identity when n_out == N_in.

    Uses the same uniform normalised-arc-length grid and the same
    searchsorted/weight arithmetic as resample_fe_output(), so interpolating
    features here is consistent with how the FE record was resampled to build
    them in the first place.
    """
    n_in = t.shape[-2]
    if n_out == n_in:
        return t
    # float64 grid arithmetic: resample_fe_output() builds its grid in float32,
    # where linspace rounding perturbs the interpolation weights by ~8e-7. The
    # indices are identical either way; float64 simply removes that jitter.
    src = torch.linspace(0.0, 1.0, n_in, device=t.device, dtype=torch.float64)
    tgt = torch.linspace(0.0, 1.0, n_out, device=t.device, dtype=torch.float64)
    idx = torch.clamp(torch.searchsorted(src, tgt, right=True) - 1, 0, n_in - 2)
    w = ((tgt - src[idx]) / (src[idx + 1] - src[idx])).clamp(0.0, 1.0)
    left = t.index_select(-2, idx)
    right = t.index_select(-2, idx + 1)
    shape = [1] * (t.dim() - 2) + [n_out, 1]
    return left + w.to(t.dtype).view(shape) * (right - left)


def arclength_fourier(s: torch.Tensor, n_bands: int) -> torch.Tensor:
    """Fourier positional encoding of normalised arc length.

    s : [..., M] with values in [0, 1]  ->  [..., M, 2 * n_bands]
    """
    freqs = (2.0 ** torch.arange(n_bands, device=s.device, dtype=torch.float32)) * math.pi
    ang = s.float().unsqueeze(-1) * freqs
    return torch.cat([torch.sin(ang), torch.cos(ang)], dim=-1).to(s.dtype)


class QueryTensionDecoder(nn.Module):
    """Cross-attention from N_out query stations onto N_in encoded nodes.

    Queries are built from the static node features of the graph at the OUTPUT
    resolution (already standardised by static_node_norm) concatenated with a
    Fourier encoding of their normalised arc length. Keys and values are the
    encoder output plus a learned embedding of the INPUT stations' arc length,
    so attention is geometry-aware rather than index-aware — that is what lets
    one weight set serve every (N_in, N_out) pair.

    residual_interp
        Adds interp_nodes(h, N_out) to the attention output, making the module
        a learned CORRECTION on top of arc-length interpolation of the
        embeddings.
    zero_init
        Zeroes the final output projection, so at initialisation the correction
        is exactly zero and the module degenerates to interp_nodes(h, N_out).
        Combined with a Stage-1 warm start that makes the untrained Stage-2
        model reproduce Stage 1 BIT-EXACTLY on the diagonal (N_in == N_out,
        where the interpolation is the identity). OFF the diagonal it starts
        from interpolation of the EMBEDDINGS, which is not the same as the
        interpolation baseline's interpolation of the predicted TENSION -- the
        LSTM and head in between are nonlinear. Requires residual_interp=True:
        without the skip the module would emit an identically zero embedding.
    """

    def __init__(
        self,
        d_model: int = 128,
        n_static_node_features: int = 12,
        num_heads: int = 4,
        ffn_dim: int = 256,
        fourier_bands: int = 8,
        dropout: float = 0.1,
        residual_interp: bool = True,
        zero_init: bool = True,
    ):
        super().__init__()

        if zero_init and not residual_interp:
            raise ValueError(
                "query_decoder_zero_init=True requires "
                "query_decoder_residual_interp=True: without the interpolation "
                "skip a zeroed output projection emits an identically zero "
                "embedding and the model cannot train."
            )

        self.d_model = d_model
        self.residual_interp = residual_interp
        pe_dim = 2 * fourier_bands

        self.query_proj = nn.Linear(n_static_node_features + pe_dim, d_model)
        self.key_pos_proj = nn.Linear(pe_dim, d_model)

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True,
        )

        self.norm_q = nn.LayerNorm(d_model)
        self.norm_attn = nn.LayerNorm(d_model)
        self.norm_ffn = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )

        self.out_proj = nn.Linear(d_model, d_model)
        if zero_init:
            nn.init.zeros_(self.out_proj.weight)
            nn.init.zeros_(self.out_proj.bias)

    def forward(
        self,
        h: torch.Tensor,
        q_static: torch.Tensor,
        s_in: torch.Tensor,
        s_out: torch.Tensor,
    ) -> torch.Tensor:
        """
        h        : [M, N_in,  d]   encoder output (M = B*W independent graphs)
        q_static : [M, N_out, F]   standardised static features at the queries
        s_in     : [M, N_in]       normalised arc length of the input stations
        s_out    : [M, N_out]      normalised arc length of the query stations

        returns  : [M, N_out, d]
        """
        n_out = q_static.shape[-2]

        kv = h + self.key_pos_proj(arclength_fourier(s_in, self._bands()))
        q = self.norm_q(
            self.query_proj(
                torch.cat([q_static, arclength_fourier(s_out, self._bands())], dim=-1)
            )
        )

        a, _ = self.attn(q, kv, kv, need_weights=False)
        z = self.norm_attn(q + a)
        z = self.norm_ffn(z + self.ffn(z))
        delta = self.out_proj(z)

        if self.residual_interp:
            return interp_nodes(h, n_out) + delta
        return delta

    def _bands(self) -> int:
        return self.key_pos_proj.in_features // 2


def model_forward(model, sample):
    """Single call site for the model forward pass.

    Keeps the query-graph plumbing in one place: train_one_epoch, evaluate,
    calibrate_log_vars and dump_test_timeseries all go through here, so
    Stage-1 and Stage-2 samples are handled identically.
    """
    graph_or_graphs = sample.get("graphs", sample.get("graph"))
    query = sample.get("query_graphs", sample.get("query_graph"))
    return model(graph_or_graphs, sample["x_seq"], sample["edge_seq"], query)


In [ ]:
class MooringGATLSTM(nn.Module):
    """
    Spatiotemporal model for the PUBLICATION reconstruction task.

    Co-temporal motion -> tension: for a window of W steps the model receives
    the motion-derived node/edge features at every step and predicts the
    tension at every node at every one of the SAME W steps. There is no
    forecast horizon and no last-step compression.

    Pipeline:
        1) For each window time step t:
              - combine static + dynamic graph information
              - run MooringGATEncoder
              - get node embeddings h_t                  [N, gat_out_dim]

        2) Stack all h_t over the window:
              H = [window_len, N, gat_out_dim]

        3) Run NodeTemporalLSTM over each node's sequence, keeping the FULL
           per-step output:
              seq = [N, window_len, temporal_out_dim]

        4) Per-step prediction head (applied at every time step):
              y_hat = [window_len, N, output_dim]

    Default output_dim=1 corresponds to [tension].
    """

    def __init__(
        self,
        # ----- graph feature sizes -----
        n_static_node_features: int = 12,
        n_dynamic_node_features: int = 20,
        n_static_edge_features: int = 10,
        n_dynamic_edge_features: int = 4,

        # ----- spatial encoder -----
        gat_hidden_dim: int = 64,
        gat_out_dim: int = 64,
        num_heads: int = 4,
        gat_dropout: float = 0.1,
        gat_use_layernorm: bool = True,
        add_residual_projection: bool = True,
        use_global_context: bool = False,

        # ----- temporal encoder -----
        lstm_hidden_dim: int = 128,
        num_lstm_layers: int = 1,
        lstm_dropout: float = 0.1,
        bidirectional: bool = False,
        lstm_use_layernorm: bool = True,

        # ----- prediction head -----
        output_dim: int = 1,
        head_hidden_dim: int = 128,
        head_dropout: float = 0.1,

        # ----- patch sr-03: Stage-2 query decoder (N_out decoupled from N_in) -----
        use_query_decoder: bool = False,
        query_decoder_heads: int = 4,
        query_decoder_ffn_dim: int = 256,
        query_decoder_fourier_bands: int = 8,
        query_decoder_dropout: float = 0.1,
        query_decoder_residual_interp: bool = True,
        query_decoder_zero_init: bool = True,
    ):
        super().__init__()

        self.output_dim = output_dim

        # -------------------------------------------------------------
        # Spatial encoder: one time step -> node embeddings
        # -------------------------------------------------------------
        self.spatial_encoder = MooringGATEncoder(
            n_static_node_features=n_static_node_features,
            n_dynamic_node_features=n_dynamic_node_features,
            n_static_edge_features=n_static_edge_features,
            n_dynamic_edge_features=n_dynamic_edge_features,
            gat_hidden_dim=gat_hidden_dim,
            gat_out_dim=gat_out_dim,
            num_heads=num_heads,
            dropout=gat_dropout,
            use_layernorm=gat_use_layernorm,
            add_residual_projection=add_residual_projection,
            use_global_context=use_global_context,
        )

        # -------------------------------------------------------------
        # Temporal encoder: per-node sequence -> per-node per-step features
        # -------------------------------------------------------------
        self.temporal_encoder = NodeTemporalLSTM(
            input_dim=gat_out_dim,
            lstm_hidden_dim=lstm_hidden_dim,
            num_lstm_layers=num_lstm_layers,
            dropout=lstm_dropout,
            bidirectional=bidirectional,
            use_layernorm=lstm_use_layernorm,
            return_sequences=True,
        )

        temporal_out_dim = self.temporal_encoder.output_dim

        # -------------------------------------------------------------
        # patch sr-03: optional query decoder between GAT and LSTM.
        # When None the forward path is bit-identical to Stage 1 and no
        # query_decoder.* keys exist in the state_dict.
        # -------------------------------------------------------------
        self.use_query_decoder = use_query_decoder
        if use_query_decoder:
            self.query_decoder = QueryTensionDecoder(
                d_model=gat_out_dim,
                n_static_node_features=n_static_node_features,
                num_heads=query_decoder_heads,
                ffn_dim=query_decoder_ffn_dim,
                fourier_bands=query_decoder_fourier_bands,
                dropout=query_decoder_dropout,
                residual_interp=query_decoder_residual_interp,
                zero_init=query_decoder_zero_init,
            )
        else:
            self.query_decoder = None

        # -------------------------------------------------------------
        # Per-step prediction head:
        # [..., W, temporal_out_dim] -> [..., W, output_dim]
        # (Linear layers act on the last dim, so the same head applies at
        #  every time step — non-autoregressive, one evaluation per window.)
        # -------------------------------------------------------------
        self.prediction_head = nn.Sequential(
            nn.Linear(temporal_out_dim, head_hidden_dim),
            nn.ELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden_dim, output_dim),
        )

    def forward(self, graph_or_graphs, x_seq: torch.Tensor, edge_seq: torch.Tensor,
                query_graph_or_graphs=None) -> torch.Tensor:
        """
        Dispatches on input dimensionality.

        Unbatched  — graph_or_graphs: Data,
                     x_seq: [W, N_in, F_node], edge_seq: [W, E, F_edge]
                     returns [W, N_out, output_dim]

        Batched    — graph_or_graphs: List[Data] (all share same N_in, E),
                     x_seq: [B, W, N_in, F_node], edge_seq: [B, W, E, F_edge]
                     returns [B, W, N_out, output_dim]

        query_graph_or_graphs is required when use_query_decoder is True and
        supplies the OUTPUT stations; N_out == N_in and it is ignored otherwise.
        Bucketed batching guarantees all B samples share the same N_in and E.
        """
        if x_seq.dim() == 4:
            return self._forward_batched(graph_or_graphs, x_seq, edge_seq,
                                         query_graph_or_graphs)
        return self._forward_single(graph_or_graphs, x_seq, edge_seq,
                                    query_graph_or_graphs)

    def _decode_queries(self, H, graphs, query_graphs, B, W, single):
        """patch sr-03: apply the query decoder to [M, N_in, d] -> [M, N_out, d].

        `single` selects between one graph (M = W) and a bucketed batch of B
        graphs each expanded over W (M = B*W). Arc-length positions come from
        x_raw column 0 (s_over_L_line); graph.x is standardised, so its column 0
        is a z-score and must NOT be used here.
        """
        if self.query_decoder is None:
            return H
        if query_graphs is None:
            raise ValueError(
                "use_query_decoder=True but no query graph was passed. "
                "Set out_N on the dataset subset so samples carry query_graph."
            )
        if single:
            qg, g = query_graphs, graphs
            q_static = qg.x.unsqueeze(0).expand(W, -1, -1)
            s_out = qg.x_raw[:, 0].unsqueeze(0).expand(W, -1)
            s_in = g.x_raw[:, 0].unsqueeze(0).expand(W, -1)
        else:
            n_in = H.shape[1]
            q_static_B = torch.stack([q.x for q in query_graphs])      # [B, N_out, F]
            n_out = q_static_B.shape[1]
            q_static = q_static_B.unsqueeze(1).expand(B, W, n_out, -1).reshape(B * W, n_out, -1)
            s_out = (torch.stack([q.x_raw[:, 0] for q in query_graphs])
                     .unsqueeze(1).expand(B, W, n_out).reshape(B * W, n_out))
            s_in = (torch.stack([g.x_raw[:, 0] for g in graphs])
                    .unsqueeze(1).expand(B, W, n_in).reshape(B * W, n_in))
        return self.query_decoder(H, q_static.to(H.dtype), s_in.to(H.dtype),
                                  s_out.to(H.dtype))


    def _forward_single(self, graph, x_seq: torch.Tensor, edge_seq: torch.Tensor,
                        query_graph=None) -> torch.Tensor:
        """
        Single-sample forward pass.

        Returns
        -------
        y_hat : torch.Tensor
            Co-temporal tension prediction, shape [window_len, N_out, output_dim].
        """
        if x_seq.dim() != 3:
            raise ValueError(
                f"x_seq must have shape [window_len, N, F_dyn_node], got {tuple(x_seq.shape)}"
            )

        if edge_seq.dim() != 3:
            raise ValueError(
                f"edge_seq must have shape [window_len, E, F_dyn_edge], got {tuple(edge_seq.shape)}"
            )

        window_len_x, N_x, _ = x_seq.shape
        window_len_e, E_x, _ = edge_seq.shape

        if window_len_x != window_len_e:
            raise ValueError(
                f"Window length mismatch: x_seq has {window_len_x}, edge_seq has {window_len_e}."
            )

        if graph.x.size(0) != N_x:
            raise ValueError(
                f"Node count mismatch: graph.x has {graph.x.size(0)} nodes, but x_seq has {N_x}."
            )

        if graph.edge_attr.size(0) != E_x:
            raise ValueError(
                f"Edge count mismatch: graph.edge_attr has {graph.edge_attr.size(0)} edges, "
                f"but edge_seq has {E_x}."
            )

        # Vectorised: one GATv2Conv call over W*N nodes (block-diagonal super-graph)
        x_static  = graph.x.unsqueeze(0).expand(window_len_x, N_x, -1)
        ea_static = graph.edge_attr.unsqueeze(0).expand(window_len_x, E_x, -1)
        H = self.spatial_encoder.forward_batch(
            x_static, x_seq, ea_static, edge_seq, graph.edge_index
        )  # [W, N_in, gat_out_dim]

        # patch sr-03: N_in -> N_out at the query stations
        H = self._decode_queries(H, graph, query_graph, 1, window_len_x, single=True)

        # [W, N_out, F] -> [N_out, W, F] -> LSTM full sequence
        seq = self.temporal_encoder.encode_sequence(
            H.permute(1, 0, 2).contiguous()
        )                                                    # [N_out, W, temporal_out]
        y_hat = self.prediction_head(seq)                    # [N_out, W, output_dim]
        return y_hat.permute(1, 0, 2).contiguous()           # [W, N_out, output_dim]

    def _forward_batched(
        self,
        graphs,
        x_seq: torch.Tensor,
        edge_seq: torch.Tensor,
        query_graphs=None,
    ) -> torch.Tensor:
        """
        Batched forward, vectorised over B*W graphs in one GATv2Conv call and
        over B*N_out node sequences in one LSTM call. Uses the SAME
        encode_sequence() readout as the single path (no path divergence).
        """
        B, W, N, _ = x_seq.shape
        E = edge_seq.size(2)

        # Stack per-sample static features, expand over W, flatten to [B*W, ...]
        x_static_B  = torch.stack([g.x        for g in graphs], dim=0)  # [B, N, F_static]
        ea_static_B = torch.stack([g.edge_attr for g in graphs], dim=0) # [B, E, F_static_edge]
        x_static_flat  = x_static_B.unsqueeze(1).expand(B, W, N, -1).contiguous().view(B * W, N, -1)
        ea_static_flat = ea_static_B.unsqueeze(1).expand(B, W, E, -1).contiguous().view(B * W, E, -1)

        # Single GATv2Conv call for all B*W graphs
        H_enc_flat = self.spatial_encoder.forward_batch(
            x_static_flat,
            x_seq.view(B * W, N, -1),
            ea_static_flat,
            edge_seq.view(B * W, E, -1),
            graphs[0].edge_index,  # shared topology for same-N chain graphs
        )  # [B*W, N_in, gat_out_dim]

        # patch sr-03: N_in -> N_out at the query stations
        H_enc_flat = self._decode_queries(H_enc_flat, graphs, query_graphs, B, W,
                                          single=False)
        N_out = H_enc_flat.shape[1]

        H_enc = H_enc_flat.view(B, W, N_out, -1)  # [B, W, N_out, gat_out_dim]

        # [B, W, N_out, F] -> [B, N_out, W, F] -> [B*N_out, W, F]; the LSTM has no
        # cross-sample interactions so results are identical to a per-sample loop.
        gat_out_dim = H_enc.shape[-1]
        H_lstm = H_enc.permute(0, 2, 1, 3).reshape(B * N_out, W, gat_out_dim)

        seq = self.temporal_encoder.encode_sequence(H_lstm)   # [B*N_out, W, temporal_out]
        y_hat = self.prediction_head(seq)                     # [B*N_out, W, output_dim]
        y_hat = y_hat.view(B, N_out, W, self.output_dim)
        return y_hat.permute(0, 2, 1, 3).contiguous()         # [B, W, N_out, output_dim]


In [ ]:
# -------------------------------------------------------------
# training config
# -------------------------------------------------------------

@dataclass
class TrainingConfig:
    split_seed: int = 42
    seed_list: Tuple[int, ...] = (42,)

    # Locations available from batchRieke dataset (loc02 and loc12 not used)
    # Locs 1,3-9 used for training (time-based train/val/test split per case)
    # Locs 10,11 used as held-out location-level test set
    train_lc_ids:      Tuple[int, ...] = (1, 3, 4, 5, 6, 7, 8, 9)
    test_extra_lc_ids: Tuple[int, ...] = (10, 11)
    node_counts: Tuple[int, ...] = (4, 5, 6, 7, 8, 10, 12, 15, 18, 21)

    test_time_fraction: float = 0.30
    train_fraction_within_dev_blocks: float = 0.70
    raw_block_len: int = 2000   # must exceed window_len; ~4-5 dev blocks per case

    # --- Publication reconstruction task ---------------------------------
    window_len: int = 1000            # co-temporal input/output window [steps @ 10 Hz]
    use_env_features: bool = True     # ablation flag: env metadata in dynamic node feats
    use_velocity_features: bool = True
    use_drag_features: bool = True     # quadratic drag force features F_drag_t/F_drag_n
    output_dim: int = 1               # tension only

    # optimization
    num_epochs: int = 2
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 1
    grad_clip_max_norm: float = 1.0

    # scheduler / stopping
    use_scheduler: bool = False
    scheduler_factor: float = 0.5
    scheduler_patience: int = 5
    # --- patch 26: sign-safe plateau threshold ---------------------------
    # The scheduler is fed -R2 (NEGATIVE). PyTorch's DEFAULT threshold_mode
    # "rel" tests `a < best*(1-threshold)`, which for a negative `best` moves
    # the bar the WRONG WAY (less negative = easier), so the patience counter
    # resets almost every epoch and the LR never decays. "abs" tests
    # `a < best - threshold`, correct for a metric of ANY sign.
    # Set to "rel" ONLY to reproduce the old buggy behaviour (bake-off control).
    #
    # scheduler_min_lr is part of the fix, not a separate tweak: with the sign
    # corrected and no floor, late R2 gains (~5e-5) sit under the 1e-4 bar so the
    # scheduler fires every ~6 epochs forever (12 decays -> lr 2.4e-7 = frozen).
    # Measured on v4's real trajectory: min_lr 5e-5 gives 5 decays with a 5e-5
    # floor, closest to v2 (4 decays, floor 6.25e-5) -- the last healthy schedule.
    scheduler_threshold: float = 1e-4
    scheduler_threshold_mode: str = "abs"
    scheduler_min_lr: float = 5e-5
    early_stopping_patience: int = 2
    min_delta: float = 1e-5

    checkpoint_dir: str = "./checkpoints_gat_lstm_debug"
    eps: float = 1e-8

    # Optional caps for first real-data debug runs. Set to None for full experiments.
    max_train_windows: Optional[int] = 256
    max_val_windows: Optional[int] = 128
    max_test_windows: Optional[int] = 128

    # Dynamic node feature indices (9 base + 13 env = 22 total):
    #   0  x_abs             continuous
    #   1  z_abs             continuous
    #   2  x_vel             continuous  (finite-difference velocity)
    #   3  z_vel             continuous
    #   4  contact_flag      binary  (excluded from normalisation)
    #   5  penetration_depth continuous
    #   6  bed_reaction_z    continuous  (weight + (k_b*pen - c_b*v_z)*area)
    #   7  F_drag_t          continuous  (axial quadratic drag force [N])
    #   8  F_drag_n          continuous  (normal quadratic drag force [N])
    #   9  h0                continuous  (env water depth from CSV)
    #  10  Hs                continuous
    #  11  Tp                continuous
    #  12  d1                continuous
    #  13  v1                continuous
    #  14  d2                continuous
    #  15  v2                continuous
    #  16  d3                continuous
    #  17  v3                continuous
    #  18  d4                continuous
    #  19  v4                continuous
    #  20  d5                continuous
    #  21  v5                continuous
    # NOTE: if use_velocity_features/use_drag_features/use_env_features are toggled off, these
    # tuples MUST be overridden in the run cell to match the reduced layout.
    node_continuous_idx: Tuple[int, ...] = (0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21)
    node_binary_idx:     Tuple[int, ...] = (4,)

    edge_continuous_idx: Tuple[int, ...] = (0, 1, 2)
    edge_binary_idx:     Tuple[int, ...] = (3,)

    # Target layout (1): 0 tension [N]
    target_continuous_idx: Tuple[int, ...] = (0,)

    # Static feature standardization (fix of the known thesis-model flaw).
    # graph.x cols:      exclude one-hot cols 1,2,3 (is_anchor/is_fairlead/is_intermediate)
    # graph.edge_attr:   exclude flag cols 7,8,9 (edge_is_anchor/intermediate/fairlead)
    # Zero-variance columns (constants like EA, diameter) are guarded (std -> 1).
    static_node_continuous_idx: Tuple[int, ...] = (0, 4, 5, 6, 7, 8, 9, 10, 11)
    static_edge_continuous_idx: Tuple[int, ...] = (0, 1, 2, 3, 4, 5, 6)

    # Full-scale data pipeline settings
    # num_workers=0 keeps single-process loading (safe on Windows with mmap)
    # Set num_workers=4 when running on Linux or after verifying Windows compat
    num_workers:                 int           = 0
    pin_memory:                  bool          = False
    max_train_windows_per_epoch: Optional[int] = 50_000   # None = use all windows
    n_fit_windows:               int           = 2_000    # windows for normalizer fitting

    # --- AMP & precision -------------------------------------------------
    use_amp:             bool          = False   # fp16 on CUDA, no-op on CPU

    # --- Training efficiency ---------------------------------------------
    validate_every:      int           = 1      # run val every N epochs; always on last epoch
    accumulation_steps:  int           = 1      # gradient accumulation steps
    save_every_n_epochs: int           = 0      # 0 = disabled; periodic checkpoint every N epochs
    save_feasible_checkpoints: bool = False  # save a ckpt ONLY at epochs meeting every selection criterion (Gate A + safety box)
    save_latest_checkpoint: bool = True   # patch sr-13: rolling checkpoint_latest.pt every epoch, so a walltime timeout loses <= 1 epoch (it is NOT checkpoint_epoch*.pt, so the ts-dump loop ignores it)

    # --- Model architecture (passed to build_model_from_config) ----------
    gat_hidden_dim:  int   = 64
    gat_out_dim:     int   = 64
    num_heads:       int   = 4
    gat_dropout:     float = 0.1
    use_global_context: bool = False   # patch 17: line-global readout in GAT encoder (Finding-4 receptive-field fix)
    lstm_hidden_dim: int   = 128
    num_lstm_layers: int   = 1
    lstm_dropout:    float = 0.1
    lstm_bidirectional: bool = False   # nowcast task admits bidirectional; OFF = validated backbone
    head_hidden_dim: int   = 128
    head_dropout:    float = 0.1
    use_query_decoder: bool = False    # Stage 2 super-resolution switch (patch 08)

    # --- Metrics ---------------------------------------------------------
    tension_mape_min: float = 1000.0   # ignore MAPE where |true_tension| < this [N]

    # --- Checkpoint resume -----------------------------------------------
    resume_from_checkpoint: Optional[str] = None

    # --- Batching -------------------------------------------------------
    enable_bucketed_node_count_batching: bool = False
    drop_last_batch:                     bool = False

    # --- Debug ----------------------------------------------------------
    debug_batch_shapes:                  bool = False

    # --- Evaluation gating -------------------------------------------
    run_test_evaluation:                 bool = False  # set True only for final runs

    # --- Training physical metrics (optional, capped subset of train batches) -
    compute_train_physical_metrics: bool = False
    compute_contact_regime_metrics: bool = True   # patch 16: grounded/touchdown/suspended R2
    contact_flag_idx: int = 4                      # dynamic node feature index of contact_flag
    max_train_metric_batches: int = 100

    # --- 8-term Kendall reconstruction loss ------------------------------
    # Terms: [mse, l1, temporal_grad, spectral, constitutive, catenary,
    #         momentum, peak_pinball]
    # Learnable log_vars live on the loss module (excluded from weight decay).
    use_kendall_loss: bool = True
    peak_pinball_tau: float = 0.9
    peak_pinball_thr_n: float = 5750.0  # high-band pinball threshold [N] (~p85 of train tension)
    # --- blended checkpoint-selection rule (patch 09f) ----------------------
    # Gate A: val_global_R2_tension >= running_max - sel_r2_delta.
    # Gate B (safety box, val tension tail): 0 < bias_p90 <= sel_bias90_cap,
    #   0 < bias_p99 <= sel_bias99_cap, underpred_p90/p99 <= sel_up_cap
    #   (tau=0.9 calibration admits ~10-20% tail exceedance; near-0% with a
    #   large positive bias is overshoot, not extra safety).
    # Objective inside A+B: minimize val_temporal_diff_skill_tension.
    sel_r2_delta:   float = 0.0010
    sel_bias90_cap: float = 200.0   # [N]
    sel_bias99_cap: float = 400.0   # [N]
    sel_up_cap:     float = 20.0    # [%]  (legacy, patch 09f; unused by v3 gate)
    # patch 10 (selection v3, 2026-07-20): Gate B is underprediction-RATE only.
    # The bias box of patch 09f was infeasible on the FINAL run (bias_p99 < 0 in
    # ALL 55 epochs) -> zero feasible epochs -> no checkpoints. Rate caps are
    # strict "<" per the user directive.
    sel_up90_cap:   float = 10.0    # [%] underpred rate cap at the p90 tail
    sel_bias90_min: float = float("-inf")  # [N] patch sr-13: Gate B also needs bias_p90 > this. -inf = disabled (pre-sr-13 behaviour); FINAL uses 0.0
    # patch 25 (selection v4, user directive 2026-07-27): the p99 rate cap
    # is REMOVED. On the patch-22 corrected pipeline the p99 tail is 43%
    # (fairlead peak) to 82% (line max) snap-composed, and per-event snap
    # magnitude is aliasing-bounded at the 0.1 s export interval, so
    # up99 < 20% is not reachable by construction. Gating on it gave zero
    # feasible epochs twice (jobs 10466291, 10521852) and both runs then
    # shipped the safety-blind 10-R2 fallback. p90 is only 10-15%
    # snap-composed and carries the tau=0.9 calibration claim, so it stays.
    # up99/bias/undermag remain CSV diagnostics -- measuring where p99
    # lands under a p90-gated selection is the purpose of this run.
    sel_abort_epoch: int = 20       # abort if 0 feasible epochs at this epoch (0 = off)
    # patch 13: per-checkpoint prediction dumps. One small .npz per feasible
    # checkpoint so every candidate epoch can be re-plotted LOCALLY (see
    # plot_test_timeseries.py) without re-running anything heavy.
    ts_dump_all_checkpoints:      bool = True
    ts_dump_max_windows_per_ckpt: int  = 6000   # bounded: ~35-45 s per ckpt
    # True  -> T_phys = relu(T0_seg + EA*(chord_t - chord_0)/chord_0)  (incremental, recommended)
    # False -> T_phys = relu(EA*(chord_t - L0)/L0)                     (literal formulation)
    constitutive_incremental: bool = True

    # --- Checkpoint selection ---------------------------------------------
    # Kendall val_loss contains the 0.5*log_var regularisers (can go negative)
    # and is NOT a quality proxy — select on max val_global_R2_tension instead.
    select_on_r2_tension: bool = True
    # --- patch 19: per-window artifact exclusion ---
    # Share of each dataset's train budget reserved for snap-containing
    # windows. 0.0 = uniform sampling (previous behaviour).
    snap_oversample_frac: float = 0.0

    # --- patch sr-02: paired window draw across node counts ---------------
    # True  -> every N_in sees the SAME windows and the same train/val block
    #          assignment, so the super-resolution N_in sweep is a paired
    #          comparison. Per-node-count budgets are unchanged.
    # False -> exact Stage-1 behaviour (windows drawn independently per N).
    pair_windows_across_node_counts: bool = True

    # --- patch sr-03: Stage-2 query decoder --------------------------------
    # use_query_decoder=False reproduces Stage 1 EXACTLY (no query_decoder
    # parameters exist, forward path unchanged), so a v5 checkpoint loads
    # strict=True and the interpolation baseline runs from this same notebook.
    superres_out_N: int = 21              # target stations; 21 = native FE ceiling
    query_decoder_heads: int = 4
    query_decoder_ffn_dim: int = 256
    query_decoder_fourier_bands: int = 8
    query_decoder_dropout: float = 0.1
    query_decoder_residual_interp: bool = True   # decoder learns a CORRECTION on interpolation
    query_decoder_zero_init: bool = True         # ... starting from exactly zero correction
    # Stage-1 checkpoint to warm-start spatial_encoder / temporal_encoder /
    # prediction_head from (strict=False, so the decoder stays freshly
    # initialised). None = train from scratch.
    warm_start_checkpoint: Optional[str] = None
    # N_out values swept by evaluate_superres_grid() after training.
    superres_grid_out_N: Tuple[int, ...] = (4, 5, 6, 7, 8, 10, 12, 15, 18, 21)

    # --- patch sr-06: Stage-2 mode ----------------------------------------
    # superres_mode drives the DATA (targets at superres_out_N + a query graph
    # per sample); use_query_decoder drives the MODEL. They are separate so the
    # interpolation baseline can run on super-resolved data with a plain
    # Stage-1 model.
    superres_mode: bool = False
    # Stage-1 checkpoint scored as the interpolation baseline after a FINAL run.
    superres_baseline_checkpoint: Optional[str] = None

    # --- patch sr-07: resolution at which the NORMALIZERS are fitted -------
    # True  -> fit at Stage-1 matched resolution (out_N=None), i.e. over the
    #          node-count MIXTURE, exactly as the run that produced the v5
    #          checkpoint did. REQUIRED for a warm start or the interpolation
    #          baseline to denormalise correctly, because the checkpoint stores
    #          no normalizer. Measured cost of getting this wrong: mixture
    #          mean 2779.3 / std 2719.5 N vs N_out=21 mean 2651.5 / std 2547.6 N
    #          = every prediction shrunk ~6.3 %.
    # False -> patch sr-06 ordering (targets already at N_out when fitted).
    superres_fit_norm_at_stage1_resolution: bool = True

    # --- patch sr-17: motion-sensor noise sweep (EVAL ONLY) ----------
    # Declarative guard only: the sweep in the run cell refuses to perturb
    # anything unless this is True, so no training mode can ever silently
    # acquire noise. The levels themselves live in the run cell, where the
    # slurm's HARD GATE 2 can pin them.
    sensor_noise_enabled: bool = False


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [ ]:
# -------------------------------------------------------------
# split utilities
# -------------------------------------------------------------

def get_window_span(dataset: MooringSequenceDatasetPositionTension) -> int:
    return dataset.window_len

def valid_window_starts_inside_raw_interval(
    dataset: MooringSequenceDatasetPositionTension,
    raw_start: int,
    raw_end_exclusive: int,
) -> List[int]:
    span = dataset.window_len
    lo = max(raw_start, 0)
    hi = min(raw_end_exclusive - span + 1, len(dataset))
    starts = list(range(lo, hi))
    # --- patch 19: per-window artifact exclusion ---
    # Drop starts whose [start, start+span) span covers a corrupt sample.
    # This is the single place every candidate list is built, so train,
    # val and test are all covered here.
    if not EXCLUDE_ARTIFACT_WINDOWS:
        return starts
    key = (int(dataset.graph_data.batch_location_id),
           int(dataset.graph_data.case_id))
    bad = ARTIFACT_SAMPLES.get(key)
    if not bad:
        return starts
    keep = []
    for s in starts:
        e = s + span
        if not any(s <= t < e for t in bad):
            keep.append(s)
    return keep

def chunk_raw_time_range(
    raw_start: int,
    raw_end_exclusive: int,
    block_len: int,
) -> List[Tuple[int, int]]:
    blocks = []
    cur = raw_start
    while cur < raw_end_exclusive:
        nxt = min(cur + block_len, raw_end_exclusive)
        blocks.append((cur, nxt))
        cur = nxt
    return blocks

def _sample_windows(starts, max_windows, rng):
    """Randomly subsample starts to at most max_windows entries.

    If max_windows is None or len(starts) <= max_windows, returns starts unchanged.
    Uses random.sample so the result is a new list (no mutation).
    """
    if max_windows is None or len(starts) <= max_windows:
        return starts
    return rng.sample(starts, max_windows)



def _snap_windows_in(dataset, starts):
    """Subset of `starts` whose window covers a catalogued snap event."""
    key = (int(dataset.graph_data.batch_location_id),
           int(dataset.graph_data.case_id))
    ev = SNAP_EVENT_TIMES.get(key)
    if not ev:
        return []
    span = dataset.window_len
    return [s for s in starts if any(s <= t < s + span for t in ev)]


def _snap_enriched_sample(dataset, starts, max_windows, rng, frac, seed,
                          pair_ncs=False):
    """Subsample `starts`, reserving a share of the budget for snaps.

    Snap events are ~2 % of candidate windows and would otherwise be
    thinned in proportion by the per-dataset budget. Reserving
    `frac` of the budget for snap-containing windows raises their share
    without touching the batch sampler, whose per-epoch draw is uniform
    over the pool this builds.

    TRAIN ONLY -- val and test keep calling _sample_windows, so the
    evaluation sets stay representative of operational conditions.

    The uniform draw is taken FIRST and always, so this function consumes
    exactly the same amount of the shared split RNG whatever `frac` is.
    Snap slots are then swapped in using a RNG derived deterministically
    from (seed, loc, case, N). Without that, a frac > 0 run would shift
    the shared stream and silently change which VAL and TEST windows are
    drawn, confounding any with/without-enrichment comparison.

    frac <= 0 reproduces _sample_windows exactly.
    """
    base = _sample_windows(starts, max_windows, rng)
    if frac <= 0.0 or max_windows is None or len(starts) <= max_windows:
        return base
    snap = _snap_windows_in(dataset, starts)
    if not snap:
        return base
    # --- patch sr-02: under paired draws the snap key must NOT depend on
    # target_N either, or a frac > 0 run would break the pairing it enforces.
    key = (int(seed) * 1000003
           + int(dataset.graph_data.batch_location_id) * 10007
           + int(dataset.graph_data.case_id) * 101
           + (0 if pair_ncs else int(dataset.target_N)))
    r_snap = random.Random(key)
    n_snap = min(len(snap), int(round(frac * max_windows)))
    picked = r_snap.sample(snap, n_snap)
    chosen = set(picked)
    picked += [s for s in base if s not in chosen][:max_windows - n_snap]
    picked.sort()
    return picked


def build_leakage_aware_splits(
    load_case_datasets: Dict[Tuple, MooringSequenceDatasetPositionTension],
    cfg: TrainingConfig,
) -> Dict[str, List]:
    """
    Build train / val / test window index lists.

    Dataset keys are (lc_id, case_id, target_N) 3-tuples.

    Split rules:
    - lc_id in train_lc_ids:
        * last 30% raw time per case -> test
        * first 70% raw time per case -> development (block-shuffled train/val)
    - lc_id in test_extra_lc_ids:
        * all windows -> test  (held-out locations)

    Each element of the returned lists is ((lc_id, case_id, target_N), window_idx).
    """
    # --- patch sr-02: per-dataset RNG for a paired N_in draw --------------
    # _shared_rng reproduces Stage 1 exactly (one stream consumed in dict order).
    # When pairing is on, each (lc, case) gets its OWN stream seeded WITHOUT
    # target_N, so all node counts of a case draw identical windows and an
    # identical train/val block shuffle.
    _shared_rng = random.Random(cfg.split_seed)
    _pair_ncs = bool(getattr(cfg, "pair_windows_across_node_counts", False))
    rng = _shared_rng
    split = {"train": [], "val": [], "test": []}

    # --- patch sr-06: cfg.node_counts alone selects the training topologies.
    # The dataset-build cell always constructs ALL node counts because the
    # N_in x N_out grid sweep needs every N_out graph present for the query
    # lookup; this filter keeps the SPLITS restricted to cfg.node_counts.
    # It MUST be applied to the dataset counts below as well: those counts are
    # the divisors of the per-dataset window budget, so leaving unused node
    # counts in them would silently shrink the pool (a 6-topology HPO run over
    # a 10-topology build would get half the windows it asked for).
    _use_ncs = set(int(n) for n in cfg.node_counts)

    # -- Per-dataset window budgets (cap-aware, stratified) ---------------
    # Derive actual dataset counts from the dict keys; no dependency on globals.
    # Per-dataset budgets create stratified sampling: each dataset contributes
    # proportionally. Reproducibility depends on cfg.split_seed.
    _train_keys = [k for k in load_case_datasets
                   if k[0] in cfg.train_lc_ids and k[2] in _use_ncs]
    _test_keys = [k for k in load_case_datasets
                   if k[0] in cfg.test_extra_lc_ids and k[2] in _use_ncs]
    _n_train_ds = max(1, len(_train_keys))
    _n_test_ds  = max(1, len(_test_keys))

    # Each training dataset contributes to both train and val; budget per dataset.
    _per_ds_train = (
        max(1, cfg.max_train_windows // _n_train_ds) if cfg.max_train_windows else None
    )
    _per_ds_val = (
        max(1, cfg.max_val_windows // _n_train_ds) if cfg.max_val_windows else None
    )
    _per_ds_test_held = (
        max(1, cfg.max_test_windows // _n_test_ds) if cfg.max_test_windows else None
    )
    # Time-based test windows from training locations share the same cap pool
    _per_ds_test_time = (
        max(1, cfg.max_test_windows // _n_train_ds) if cfg.max_test_windows else None
    )

    for (lc_id, case_id, target_N), ds in load_case_datasets.items():
        if int(target_N) not in _use_ncs:      # --- patch sr-06
            continue
        rng = (random.Random(cfg.split_seed * 1000003
                             + lc_id * 10007 + case_id * 101)
               if _pair_ncs else _shared_rng)
        num_steps = ds.num_steps

        if lc_id in cfg.train_lc_ids:
            test_raw_start = int(math.floor((1.0 - cfg.test_time_fraction) * num_steps))

            test_window_ids = valid_window_starts_inside_raw_interval(
                ds, raw_start=test_raw_start, raw_end_exclusive=num_steps,
            )
            test_window_ids = _sample_windows(test_window_ids, _per_ds_test_time, rng)
            split["test"].extend(
                ((lc_id, case_id, target_N), w) for w in test_window_ids
            )

            blocks = chunk_raw_time_range(
                raw_start=0, raw_end_exclusive=test_raw_start,
                block_len=cfg.raw_block_len,
            )
            valid_blocks = []
            for b_start, b_end in blocks:
                w_ids = valid_window_starts_inside_raw_interval(
                    ds, raw_start=b_start, raw_end_exclusive=b_end,
                )
                if w_ids:
                    valid_blocks.append((b_start, b_end, w_ids))

            rng.shuffle(valid_blocks)
            n_train = int(math.floor(cfg.train_fraction_within_dev_blocks * len(valid_blocks)))

            # Flatten all block windows per dataset, then sample once (true per-dataset cap)
            train_candidates = [w for _, _, ww in valid_blocks[:n_train] for w in ww]
            val_candidates   = [w for _, _, ww in valid_blocks[n_train:] for w in ww]
            split["train"].extend(
                ((lc_id, case_id, target_N), w)
                for w in _snap_enriched_sample(
                    ds, train_candidates, _per_ds_train, rng,
                    getattr(cfg, "snap_oversample_frac", 0.0),
                    cfg.split_seed, pair_ncs=_pair_ncs)
            )
            split["val"].extend(
                ((lc_id, case_id, target_N), w)
                for w in _sample_windows(val_candidates, _per_ds_val, rng)
            )

        elif lc_id in cfg.test_extra_lc_ids:
            # patch 22: the held-out locations MUST go through the same
            # artifact filter as the dev locations. Before this fix the
            # branch built list(range(len(ds))) and bypassed exclusion
            # entirely, putting 943 MN windows in the test set (job
            # 10521852: p99 thresholds destroyed, 0 feasible epochs/100).
            all_wins = valid_window_starts_inside_raw_interval(
                ds, raw_start=0, raw_end_exclusive=num_steps,
            )
            sampled  = _sample_windows(all_wins, _per_ds_test_held, rng)
            split["test"].extend(
                ((lc_id, case_id, target_N), w) for w in sampled
            )

        else:
            raise ValueError(f"LC {lc_id} is not assigned to any split rule.")

    return split

def print_split_summary(
    load_case_datasets: Dict[Tuple, MooringSequenceDatasetPositionTension],
    split_indices: Dict[str, List],
):
    """Print window counts per split, aggregated by (lc_id, target_N)."""
    print("----- SPLIT SUMMARY -----")
    for split_name, pairs in split_indices.items():
        # Aggregate by (lc_id, target_N) to keep output readable
        counts: Dict[Tuple, int] = {}
        for (lc_id, case_id, target_N), _ in pairs:
            key = (lc_id, target_N)
            counts[key] = counts.get(key, 0) + 1
        total = sum(counts.values())
        print(f"\n{split_name.upper()} total windows: {total}")
        for (lc_id, target_N) in sorted(counts):
            print(f"  LC {lc_id:02d} N={target_N:02d}: {counts[(lc_id, target_N)]} windows")


def _cap_split_windows(split_indices: Dict[str, List], cfg: TrainingConfig) -> Dict[str, List]:
    """Deterministically cap split sizes for small debug runs."""
    rng = random.Random(cfg.split_seed)
    _pair_ncs = bool(getattr(cfg, "pair_windows_across_node_counts", False))
    capped = {}
    caps = {
        "train": cfg.max_train_windows,
        "val": cfg.max_val_windows,
        "test": cfg.max_test_windows,
    }
    for split_name, pairs in split_indices.items():
        pairs = list(pairs)
        cap = caps.get(split_name)
        if cap is not None and len(pairs) > cap:
            if _pair_ncs:
                # Cap by (lc, case, window) GROUP so a retained window survives at
                # EVERY node count. Groups are kept whole and are all the same size
                # (every node count is built for every retained case), so the
                # per-node-count budget is cap // n_node_counts -- identical to the
                # unpaired scheme, only now the same windows at every N_in.
                from collections import defaultdict as _dd_cap
                _groups = _dd_cap(list)
                for _key, _w in pairs:
                    _groups[(_key[0], _key[1], _w)].append((_key, _w))
                _gk = sorted(_groups)
                rng.shuffle(_gk)
                _out, _n = [], 0
                for _g in _gk:
                    _members = _groups[_g]
                    if _n + len(_members) > cap:
                        continue          # never split a group across the cap
                    _out.extend(_members)
                    _n += len(_members)
                pairs = _out
            else:
                rng.shuffle(pairs)
                pairs = pairs[:cap]
            pairs.sort(key=lambda item: (item[0], item[1]))
        capped[split_name] = pairs
    return capped


def assert_windows_are_paired(split_indices: Dict[str, List], node_counts) -> None:
    """--- patch sr-02: verify every drawn window exists at EVERY node count.

    Cheap (a dict build over the pair lists) and worth running once per job:
    a silent pairing failure would confound the whole N_in axis of the
    super-resolution grid with the window draw.
    """
    from collections import defaultdict as _dd_chk
    want = set(int(n) for n in node_counts)
    for split_name, pairs in split_indices.items():
        seen = _dd_chk(set)
        for (lc_id, case_id, target_N), w in pairs:
            seen[(lc_id, case_id, w)].add(int(target_N))
        bad = [g for g, ncs in seen.items() if ncs != want]
        print(f"[paired] {split_name}: {len(seen):,} distinct windows x "
              f"{len(want)} node counts = {len(pairs):,} pairs; "
              f"{len(bad)} unpaired")
        if bad:
            raise AssertionError(
                f"{split_name}: {len(bad)} window(s) not present at every node "
                f"count, e.g. {bad[:3]} -- the N_in sweep would be confounded."
            )


In [10]:
# -------------------------------------------------------------
# subset dataset
# -------------------------------------------------------------

class MultiLoadCaseWindowSubset(Dataset):
    """
    Thin wrapper around multiple MooringSequenceDatasetPositionTension objects.
    Keys in load_case_datasets are (lc_id, case_id, target_N) 3-tuples.
    """
    def __init__(
        self,
        load_case_datasets: Dict[Tuple, MooringSequenceDatasetPositionTension],
        index_pairs: List,
    ):
        self.load_case_datasets = load_case_datasets
        self.index_pairs = index_pairs
# --- patch sr-01: Stage-2 output resolution. None = matched (Stage-1 behaviour).
        # Set it BEFORE creating a loader iterator: persistent_workers is False
        # everywhere, so each new iteration re-forks workers with this value.
        self.out_N = None
        # --- patch sr-17: motion-sensor noise spec (None = clean). Same
        # contract as out_N above: set it BEFORE creating a loader iterator.
        self.sensor_noise = None

    def __len__(self):
        return len(self.index_pairs)

    def __getitem__(self, idx):
        key, window_idx = self.index_pairs[idx]
        lc_id, case_id, node_count = key
        _ds = self.load_case_datasets[key]
        _ds.out_N = self.out_N            # None => matched resolution (Stage 1)
        _ds.sensor_noise = self.sensor_noise   # None => clean (patch sr-17)
        _ds._sn_key = (lc_id, case_id)         # seeds the per-window realisation
        sample = _ds[window_idx]

        out = {
            "graph":      sample["graph"],
            "x_seq":      sample["x_seq"],
            "edge_seq":   sample["edge_seq"],
            "y_seq":      sample["y_seq"],
            "start_idx":  sample["start_idx"],
            "lc_id":      lc_id,
            "case_id":    case_id,
            "node_count": node_count,
            "out_N":      sample["out_N"],
        }
        if self.out_N is not None:
            # The query graph is the static graph ALREADY built for this case at
            # out_N, so it costs no extra memory -- it is a reference into
            # load_case_datasets. Cloned because NormalizedSubset standardises
            # it in place.
            out["query_graph"] = self.load_case_datasets[
                (lc_id, case_id, int(self.out_N))
            ].graph_data.clone()
        return out

def single_item_collate(batch):
    if len(batch) != 1:
        raise ValueError("single_item_collate expects batch_size=1.")
    return batch[0]


def bucketed_node_count_collate(batch):
    """
    Collate function for bucketed batching.
    All samples in the batch must share the same node_count.
    Returns stacked tensors [B, H, N, F] and graphs as List[Data].
    """
    nc = batch[0].get("node_count", batch[0]["x_seq"].shape[1])
    assert all(s.get("node_count", s["x_seq"].shape[1]) == nc for s in batch), (
        f"Batch contains mixed node counts: {[s.get('node_count') for s in batch]}"
    )
    assert all(s["x_seq"].shape    == batch[0]["x_seq"].shape    for s in batch), (
        f"x_seq shape mismatch within bucket batch: {[s['x_seq'].shape for s in batch]}"
    )
    assert all(s["edge_seq"].shape == batch[0]["edge_seq"].shape for s in batch), (
        f"edge_seq shape mismatch: {[s['edge_seq'].shape for s in batch]}"
    )
    assert all(s["y_seq"].shape    == batch[0]["y_seq"].shape    for s in batch), (
        f"y_seq shape mismatch: {[s['y_seq'].shape for s in batch]}"
    )
    out = {
        # Keep each graph as a separate object; static features differ by lc_id/case_id
        # even when node_count matches
        "graphs":     [s["graph"]     for s in batch],              # List[Data], len=B
        "x_seq":      torch.stack([s["x_seq"]    for s in batch]),  # [B, H, N, F_node]
        "edge_seq":   torch.stack([s["edge_seq"] for s in batch]),  # [B, H, E, F_edge]
        "y_seq":      torch.stack([s["y_seq"]    for s in batch]),  # [B, P, N_out, 1]
        "start_idx":  [s["start_idx"]  for s in batch],
        "lc_id":      [s["lc_id"]      for s in batch],
        "case_id":    [s["case_id"]    for s in batch],
        "node_count": nc,
        "out_N":      batch[0].get("out_N"),
    }
    if batch[0].get("query_graph") is not None:
        out["query_graphs"] = [s["query_graph"] for s in batch]
    return out


In [ ]:
# -------------------------------------------------------------
# normalization utilities
# -------------------------------------------------------------

class FeatureStandardizer:
    """
    Per-feature z-score standardization for selected feature columns.

    Works with tensors shaped:
    - node inputs:   [window_len, N, F]
    - edge inputs:   [window_len, E, F]
    - targets:       [window_len, N, F]
    - static feats:  [N, F] / [E, F]
    """
    def __init__(self, feature_idx: Tuple[int, ...], eps: float = 1e-8):
        self.feature_idx = list(feature_idx)
        self.eps = eps
        self.mean = None
        self.std = None

    def fit_from_tensor_list(self, tensor_list: List[torch.Tensor]):
        """
        Aggregate across all dimensions except the last feature dimension.
        """
        selected = []
        for x in tensor_list:
            xs = x[..., self.feature_idx].reshape(-1, len(self.feature_idx))
            selected.append(xs)

        # float64: float32 cancellation gives constant columns like EA=2.3e8
        # a garbage nonzero std that defeats the constant-column guard
        big = torch.cat(selected, dim=0).double()
        self.mean = big.mean(dim=0).float()
        self.std = big.std(dim=0, unbiased=False).clamp_min(self.eps).float()

    def guard_constant_columns(self, rel: float = 1e-6):
        """Set std := 1 for (near-)constant columns.

        Static columns like EA, diameter, area, Cd are bit-identical across
        every graph -> std ~ 0. Without the guard those columns would divide
        by ~eps; with it they standardize to ~0 (centered, unit-scaled).
        Call after fitting.
        """
        const = self.std < (self.mean.abs() + 1.0) * rel
        self.std = torch.where(const, torch.ones_like(self.std), self.std)

    def transform(self, x: torch.Tensor) -> torch.Tensor:
        x = x.clone()
        x_sel = x[..., self.feature_idx]
        x[..., self.feature_idx] = (x_sel - self.mean.to(x.device)) / self.std.to(x.device)
        return x

    def inverse_transform(self, x: torch.Tensor) -> torch.Tensor:
        x = x.clone()
        x_sel = x[..., self.feature_idx]
        x[..., self.feature_idx] = x_sel * self.std.to(x.device) + self.mean.to(x.device)
        return x

def fit_normalizers_from_train_subset(
    train_subset: MultiLoadCaseWindowSubset,
    cfg: TrainingConfig,
):
    """
    Fit normalizers using vectorized batch-Welford online statistics.

    Dynamic tensors (x_seq / edge_seq / y_seq): draws cfg.n_fit_windows random
    windows from train_subset and updates running mean/variance one window at
    a time using pure tensor ops.

    Static tensors (graph.x / graph.edge_attr): fitted over the UNIQUE train
    graphs (each (lc,case,N) dataset contributes its one static graph once,
    not once per window), capped at cfg.n_fit_windows graphs. This fixes the
    thesis-model flaw where static features entered the GAT unnormalized
    (EA = 2.3e8 N, tension0 ~ 1e4-1e5 N, coordinates ~1e1-1e2 m).

    Returns (node_norm, edge_norm, target_norm, static_node_norm, static_edge_norm).
    """
    node_norm        = FeatureStandardizer(cfg.node_continuous_idx,        eps=cfg.eps)
    edge_norm        = FeatureStandardizer(cfg.edge_continuous_idx,        eps=cfg.eps)
    target_norm      = FeatureStandardizer(cfg.target_continuous_idx,      eps=cfg.eps)
    static_node_norm = FeatureStandardizer(cfg.static_node_continuous_idx, eps=cfg.eps)
    static_edge_norm = FeatureStandardizer(cfg.static_edge_continuous_idx, eps=cfg.eps)

    rng     = random.Random(cfg.split_seed)
    indices = rng.sample(range(len(train_subset)), min(cfg.n_fit_windows, len(train_subset)))

    def _batch_welford_merge(count, mean, M2, raw_slice, feat_idx):
        """Merge one window into running statistics using vectorized tensor ops."""
        vals       = raw_slice[..., feat_idx].reshape(-1, len(feat_idx))   # [rows, F]
        n_new      = float(vals.shape[0])
        batch_mean = vals.mean(dim=0)
        batch_var  = vals.var(dim=0, unbiased=False)
        n_combined = count + n_new
        delta      = batch_mean - mean
        new_mean   = (count * mean + n_new * batch_mean) / n_combined
        new_M2     = M2 + batch_var * n_new + delta ** 2 * (count * n_new / n_combined)
        return n_combined, new_mean, new_M2

    def _fit_one(norm, key):
        n_feat          = len(norm.feature_idx)
        count           = 0.0
        mean            = torch.zeros(n_feat)
        M2              = torch.zeros(n_feat)
        for i in indices:
            raw         = train_subset[i][key]
            count, mean, M2 = _batch_welford_merge(count, mean, M2, raw, norm.feature_idx)
        norm.mean = mean
        norm.std  = (M2 / max(count, 1.0)).sqrt().clamp_min(norm.eps)

    _fit_one(node_norm,   "x_seq")
    _fit_one(edge_norm,   "edge_seq")
    _fit_one(target_norm, "y_seq")

    # --- static feature statistics over unique train graphs ---------------
    unique_keys = sorted({key for key, _ in train_subset.index_pairs})
    if len(unique_keys) > cfg.n_fit_windows:
        unique_keys = rng.sample(unique_keys, cfg.n_fit_windows)
    graphs = [train_subset.load_case_datasets[k].graph_data for k in unique_keys]
    static_node_norm.fit_from_tensor_list([g.x for g in graphs])
    static_edge_norm.fit_from_tensor_list([g.edge_attr for g in graphs])
    static_node_norm.guard_constant_columns()
    static_edge_norm.guard_constant_columns()

    return node_norm, edge_norm, target_norm, static_node_norm, static_edge_norm

class NormalizedSubset(Dataset):
    """
    Applies train-fitted normalization on the fly.

    Static graph features are standardized too (graph.x, graph.edge_attr);
    the raw physical copies are kept as graph.x_raw / graph.edge_attr_raw
    because the physics loss terms need tension0 [N] and chord_0 [m] in
    physical units. Mutating the graph is safe: the base dataset returns a
    fresh clone per __getitem__.
    """
    def __init__(
        self,
        base_subset: MultiLoadCaseWindowSubset,
        node_norm: FeatureStandardizer,
        edge_norm: FeatureStandardizer,
        target_norm: FeatureStandardizer,
        static_node_norm: FeatureStandardizer,
        static_edge_norm: FeatureStandardizer,
    ):
        self.base_subset = base_subset
        self.node_norm = node_norm
        self.edge_norm = edge_norm
        self.target_norm = target_norm
        self.static_node_norm = static_node_norm
        self.static_edge_norm = static_edge_norm

    def __len__(self):
        return len(self.base_subset)

    def __getitem__(self, idx):
        sample = self.base_subset[idx]

        graph = sample["graph"]
        graph.x_raw         = graph.x
        graph.edge_attr_raw = graph.edge_attr
        graph.x         = self.static_node_norm.transform(graph.x)
        graph.edge_attr = self.static_edge_norm.transform(graph.edge_attr)

        # --- patch sr-01: Stage-2 query graph -- standardised with the SAME
        # static normalisers as the input graph; x_raw kept for the physics terms.
        qg = sample.get("query_graph")
        if qg is not None:
            qg.x_raw         = qg.x
            qg.edge_attr_raw = qg.edge_attr
            qg.x         = self.static_node_norm.transform(qg.x)
            qg.edge_attr = self.static_edge_norm.transform(qg.edge_attr)

        out = {
            "graph":      graph,
            "x_seq":      self.node_norm.transform(sample["x_seq"]),
            "edge_seq":   self.edge_norm.transform(sample["edge_seq"]),
            "y_seq":      self.target_norm.transform(sample["y_seq"]),
            "start_idx":  sample["start_idx"],
            "lc_id":      sample["lc_id"],
            "case_id":    sample["case_id"],
            "node_count": sample["node_count"],
            "out_N":      sample.get("out_N"),
        }
        if qg is not None:
            out["query_graph"] = qg
        return out


In [ ]:
# -------------------------------------------------------------
# dataloader builder
# -------------------------------------------------------------

class BucketedNodeCountBatchSampler(Sampler):
    """
    Buckets dataset indices by node_count; yields homogeneous batches.

    __init__: builds index->node_count map and full bucket lists (seeded, reproducible).
    __iter__: each epoch, sub-samples max_windows from the full pool (if set),
              shuffles within buckets, builds and shuffles batch list.
              Uses rng = random.Random(seed + epoch) so ordering is epoch-varying
              but reproducible.

    index_pairs: list of ((lc_id, case_id, node_count), window_idx)
    max_windows: per-epoch subsample cap (applied fresh each epoch in __iter__)
    """
    def __init__(self, index_pairs, batch_size, drop_last=False, seed=42, max_windows=None):
        from collections import defaultdict as _dd
        self.batch_size  = batch_size
        self.drop_last   = drop_last
        self.seed        = seed
        self.max_windows = max_windows
        self.epoch       = 0

        # Precompute flat index -> node_count for O(1) lookup in __iter__
        self._index_to_nc = {i: key[2] for i, (key, _) in enumerate(index_pairs)}

        # Build full bucket lists
        buckets = _dd(list)
        for i, nc in self._index_to_nc.items():
            buckets[nc].append(i)
        self._buckets = {nc: list(idxs) for nc, idxs in sorted(buckets.items())}

        # __len__ reflects the full-pool batch count.
        # train_one_epoch must not rely on len(loader) for the final grad-accum
        # flush when max_windows is active (iter may yield fewer batches).
        self._full_len = sum(
            len(idxs) // batch_size
            + (0 if drop_last else int(len(idxs) % batch_size > 0))
            for idxs in self._buckets.values()
        )

    def __iter__(self):
        import random as _random
        from collections import defaultdict as _dd

        rng = _random.Random(self.seed + self.epoch)
        self.epoch += 1

        # Sub-sample max_windows from full pool fresh each epoch
        all_indices = [i for idxs in self._buckets.values() for i in idxs]
        if self.max_windows is not None and len(all_indices) > self.max_windows:
            all_indices = rng.sample(all_indices, self.max_windows)

        # Re-bucket using precomputed O(1) lookup
        buckets = _dd(list)
        for i in all_indices:
            buckets[self._index_to_nc[i]].append(i)

        # Build batches within each bucket, then shuffle the batch list
        all_batches = []
        for nc in sorted(buckets):
            idxs = buckets[nc]
            rng.shuffle(idxs)
            for start in range(0, len(idxs), self.batch_size):
                b = idxs[start : start + self.batch_size]
                if self.drop_last and len(b) < self.batch_size:
                    continue
                all_batches.append(b)

        rng.shuffle(all_batches)
        yield from all_batches

    def __len__(self):
        return self._full_len


def _sr22_filter_windows(index_pairs, drop, window_len):
    """patch sr-22: remove every (key, start) whose window covers a sample of `drop`.

    `drop` maps (loc, case) -> sample indices. A window spans [start, start + window_len)
    and its index IS its start sample (MooringSequenceDatasetPositionTension.__getitem__
    sets start = idx). Pure removal: the order of the surviving pairs is unchanged.
    """
    import bisect as _bisect
    _srt = {(int(k[0]), int(k[1])): sorted(int(t) for t in v) for k, v in drop.items()}
    out = []
    for key, w in index_pairs:
        b = _srt.get((int(key[0]), int(key[1])))
        if b:
            j = _bisect.bisect_left(b, int(w))
            if j < len(b) and b[j] < int(w) + int(window_len):
                continue
        out.append((key, w))
    return out


def build_dataloaders(
    load_case_datasets: Dict[int, MooringSequenceDatasetPositionTension],
    cfg: TrainingConfig,
    test_window_drop=None,   # patch sr-22: {(loc, case): samples}; drops TEST windows only
):

    split_indices = build_leakage_aware_splits(load_case_datasets, cfg)
    split_indices = _cap_split_windows(split_indices, cfg)
    # --- patch sr-22: the evaluation mask removes TEST windows only, AFTER the draw and
    # the cap, so the published draw is reproduced first and then only thinned. The
    # train/val splits and the normalizer fit never see it.
    if test_window_drop:
        _n22 = len(split_indices["test"])
        split_indices["test"] = _sr22_filter_windows(split_indices["test"],
                                                     test_window_drop, cfg.window_len)
        print(f"[sr22] evaluation mask: dropped {_n22 - len(split_indices['test'])} of "
              f"{_n22} test windows")
    print_split_summary(load_case_datasets, split_indices)

    for split_name in ["train", "val", "test"]:
        if not split_indices[split_name]:
            raise ValueError(f"{split_name} split is empty. Adjust debug locs/cases or split settings.")

    train_raw = MultiLoadCaseWindowSubset(load_case_datasets, split_indices["train"])
    val_raw   = MultiLoadCaseWindowSubset(load_case_datasets, split_indices["val"])
    test_raw  = MultiLoadCaseWindowSubset(load_case_datasets, split_indices["test"])

    # --- patch sr-07: FIT THE NORMALIZERS FIRST, at Stage-1 resolution -----
    # Supersedes patch sr-06's ordering, which set out_N before this call and so
    # fitted target_norm on the N_out tension distribution. Stage 1 has no out_N,
    # so the v5 run fitted over the node-count MIXTURE -- and the checkpoint
    # stores NO normalizer, so a warm start or the interpolation baseline would
    # be denormalised with the wrong mean/std. Measured: mixture 2779.3 / 2719.5
    # vs N_out=21 2651.5 / 2547.6 N, i.e. predictions shrunk ~6.3 %, which is the
    # magnitude deficit job 10658637 showed on the DIAGONAL, where it must
    # reproduce v5 and did not.
    #
    # Safe because FeatureStandardizer is PER-CHANNEL (tension is one column, so
    # mean/std are scalars and the node count changes only the number of rows),
    # and because out_N=None is the dataset default and IS Stage-1 behaviour.
    # sr-06's worry (baseline vs decoder under different normalisations) is
    # unaffected: both share this single fit either way.
    _sr_on = bool(getattr(cfg, "superres_mode", False))
    _sr_out_N = int(getattr(cfg, "superres_out_N", 21))
    _sr_fit_stage1 = bool(getattr(
        cfg, "superres_fit_norm_at_stage1_resolution", True))

    if _sr_on and not _sr_fit_stage1:
        for _raw in (train_raw, val_raw, test_raw):
            _raw.out_N = _sr_out_N          # legacy sr-06 ordering

    (node_norm, edge_norm, target_norm,
     static_node_norm, static_edge_norm) = fit_normalizers_from_train_subset(train_raw, cfg)

    _fit_at = ("STAGE-1 matched resolution (node-count mixture)"
               if (_sr_fit_stage1 or not _sr_on) else f"N_out={_sr_out_N}")
    print(f"[norm] target_norm fitted at {_fit_at}")
    print(f"[norm]   tension mean {float(target_norm.mean[0]):.4f} N  "
          f"std {float(target_norm.std[0]):.4f} N")
    print("[norm]   v5 fitted 2791.0876 / 2727.0000 N (recovered from the "
          "checkpoint's criterion t_mean/t_std) -- at FULL budget this refit "
          "should land within ~0.5 % of that.")
    print("[norm]   ~2651 / ~2548 N would mean it ran at N_out=21 (the sr-06 "
          "defect). Small smoke subsets differ legitimately.")

    # Only NOW switch the splits to the Stage-2 output resolution. Workers fork
    # at iterator creation (persistent_workers=False everywhere) and the loaders
    # are constructed below, so this is in time for every pass.
    if _sr_on:
        for _raw in (train_raw, val_raw, test_raw):
            _raw.out_N = _sr_out_N
        print(f"[superres] data mode ON: targets at N_out={_sr_out_N}, "
              f"query graph attached to every sample")

    train_ds = NormalizedSubset(train_raw, node_norm, edge_norm, target_norm,
                                static_node_norm, static_edge_norm)
    val_ds   = NormalizedSubset(val_raw,   node_norm, edge_norm, target_norm,
                                static_node_norm, static_edge_norm)
    test_ds  = NormalizedSubset(test_raw,  node_norm, edge_norm, target_norm,
                                static_node_norm, static_edge_norm)

    _pf = 4 if cfg.num_workers > 0 else None   # prefetch_factor requires num_workers > 0

    if cfg.batch_size == 1:
        # ── single-sample path (unchanged) ───────────────────────────────
        # Per-epoch training sampler: replacement=True avoids allocating a full
        # permutation array, which would require ~1 GB for 270M windows.
        if cfg.max_train_windows_per_epoch is not None:
            n_samples     = min(cfg.max_train_windows_per_epoch, len(train_ds))
            train_sampler = RandomSampler(train_ds, replacement=True, num_samples=n_samples)
            train_shuffle = False
        else:
            train_sampler = None
            train_shuffle = True

        train_loader = DataLoader(
            train_ds,
            batch_size=1,
            sampler=train_sampler,
            shuffle=train_shuffle,
            collate_fn=single_item_collate,
            num_workers=cfg.num_workers,
            timeout=(600 if cfg.num_workers > 0 else 0),
            pin_memory=cfg.pin_memory,
            prefetch_factor=_pf,
            persistent_workers=False,
        )

    elif cfg.enable_bucketed_node_count_batching:
        # ── bucketed batching: one node_count per mini-batch ─────────────
        # max_windows is applied fresh each epoch inside the sampler's __iter__
        bucket_sampler = BucketedNodeCountBatchSampler(
            index_pairs=split_indices["train"],
            batch_size=cfg.batch_size,
            drop_last=cfg.drop_last_batch,
            seed=cfg.split_seed,
            max_windows=cfg.max_train_windows_per_epoch,
        )
        train_loader = DataLoader(
            train_ds,
            batch_sampler=bucket_sampler,
            collate_fn=bucketed_node_count_collate,
            num_workers=cfg.num_workers,
            timeout=(600 if cfg.num_workers > 0 else 0),
            pin_memory=cfg.pin_memory,
            prefetch_factor=_pf,
            persistent_workers=False,
        )

    else:
        raise ValueError(
            f"batch_size={cfg.batch_size} > 1 requires"
            " enable_bucketed_node_count_batching=True"
        )

    # Val / test: bucketed batching when batch_size > 1, else single-item.
    if cfg.batch_size > 1 and cfg.enable_bucketed_node_count_batching:
        val_loader = DataLoader(
            val_ds,
            batch_sampler=BucketedNodeCountBatchSampler(
                split_indices["val"], cfg.batch_size,
                drop_last=False, seed=cfg.split_seed, max_windows=None,
            ),
            collate_fn=bucketed_node_count_collate,
            num_workers=cfg.num_workers,
            timeout=(600 if cfg.num_workers > 0 else 0),
            pin_memory=cfg.pin_memory,
            prefetch_factor=_pf,
            persistent_workers=False,
        )
        test_loader = DataLoader(
            test_ds,
            batch_sampler=BucketedNodeCountBatchSampler(
                split_indices["test"], cfg.batch_size,
                drop_last=False, seed=cfg.split_seed, max_windows=None,
            ),
            collate_fn=bucketed_node_count_collate,
            num_workers=cfg.num_workers,
            timeout=(600 if cfg.num_workers > 0 else 0),
            pin_memory=cfg.pin_memory,
            prefetch_factor=_pf,
            persistent_workers=False,
        )
    else:
        val_loader = DataLoader(
            val_ds,
            batch_size=1,
            shuffle=False,
            collate_fn=single_item_collate,
            num_workers=cfg.num_workers,
            timeout=(600 if cfg.num_workers > 0 else 0),
            pin_memory=cfg.pin_memory,
            prefetch_factor=_pf,
            persistent_workers=False,
        )
        test_loader = DataLoader(
            test_ds,
            batch_size=1,
            shuffle=False,
            collate_fn=single_item_collate,
            num_workers=cfg.num_workers,
            timeout=(600 if cfg.num_workers > 0 else 0),
            pin_memory=cfg.pin_memory,
            prefetch_factor=_pf,
            persistent_workers=False,
        )

    norms = {
        "node_norm":         node_norm,
        "edge_norm":         edge_norm,
        "target_norm":       target_norm,
        "static_node_norm":  static_node_norm,
        "static_edge_norm":  static_edge_norm,
    }

    return train_loader, val_loader, test_loader, norms, split_indices

In [ ]:
# -------------------------------------------------------------
# loss and metrics
# -------------------------------------------------------------
import torch.nn.functional as F


class ReconstructionKendallLoss(nn.Module):
    """
    8-term loss for the co-temporal motion->tension reconstruction task,
    balanced by Kendall homoscedastic-uncertainty weighting.

        L_total = sum_i [ 0.5 * exp(-s_i) * L_i + 0.5 * s_i ],
        s_i = log_vars[i] = log(sigma_i^2), init 0 (sigma = 1, unit weight)

    Learned sigma_i = exp(0.5 * s_i) are themselves a result: they rank the
    terms by fit precision (a large sigma on the constitutive term is the
    quantitative statement that EA-strain is noise-dominated at chain
    stiffness — report it, don't hide it).

    IMPORTANT (C2 lessons, thesis §4.5):
    - all terms are two-sided or floored, so no sigma runaway (the pinball
      term is the C2c fix; never replace it with a one-sided hinge);
    - the Kendall val_loss contains the 0.5*s_i regularisers and can go
      negative — it is NOT a model-quality proxy. Select checkpoints on
      val_global_R2_tension;
    - log_vars must sit in a weight_decay=0 optimizer param group (see
      build_optimizer_and_scheduler).

    All data terms are computed in NORMALIZED tension space; the three physics
    terms are computed in physical units from the raw inputs and re-scaled by
    sigma_T, so every term is O(1) at init and the equal-sigma start is sane.
    """

    TERM_NAMES = (
        "mse", "l1", "temporal_grad", "spectral",
        "constitutive", "catenary", "momentum", "peak_pinball",
    )

    # physical constants — identical to build_mooring_graph()
    RHO_WATER = 1025.0   # [kg/m^3]
    G         = 9.81     # [m/s^2]

    def __init__(self, node_norm, target_norm, cfg):
        super().__init__()
        self.n_terms = len(self.TERM_NAMES)
        self.log_vars = nn.Parameter(torch.zeros(self.n_terms))

        self.tau = float(cfg.peak_pinball_tau)
        self.pin_thr_n = float(getattr(cfg, "peak_pinball_thr_n", 5750.0))
        self.constitutive_incremental = bool(cfg.constitutive_incremental)
        self.ea = float(MOORING_PROPERTIES["EA"])
        # momentum-balance constants (term 6): dt from the cache sampling
        # rate; drag setup identical to the dataset's F_drag_t/F_drag_n
        self.dt   = float(FEATURE_DT)
        self.cd_n = float(MOORING_PROPERTIES["Cd_n"])
        self.cd_t = float(MOORING_PROPERTIES["Cd_t"])
        self.mass_per_length = float(MOORING_PROPERTIES["mass_per_length"])
        # submerged weight per unit length [N/m] — same formula as the graph builder
        self.w_s = float(
            (MOORING_PROPERTIES["mass_per_length"]
             - self.RHO_WATER * MOORING_PROPERTIES["area"]) * self.G
        )

        # inverse-transform stats for x,z (dynamic cols 0,1) and tension target
        px = node_norm.feature_idx.index(0)
        pz = node_norm.feature_idx.index(1)
        self.register_buffer(
            "pos_mean", torch.stack([node_norm.mean[px], node_norm.mean[pz]]).float()
        )
        self.register_buffer(
            "pos_std", torch.stack([node_norm.std[px], node_norm.std[pz]]).float()
        )
        ti = target_norm.feature_idx.index(0)
        self.register_buffer("t_mean", target_norm.mean[ti].clone().float())
        self.register_buffer("t_std",  target_norm.std[ti].clone().float().clamp_min(1e-6))

    # ------------------------------------------------------------------
    # physics helpers (batched only: x_seq [B,W,N,Fdyn], x_static_raw [B,N,12])
    # shared by forward() and run_physics_sanity_check() — one source of truth
    # ------------------------------------------------------------------
    def positions_physical(self, x_seq):
        """Normalized dynamic features -> physical x,z. Returns ([B,W,N], [B,W,N])."""
        x_phys = x_seq[..., 0].float() * self.pos_std[0] + self.pos_mean[0]
        z_phys = x_seq[..., 1].float() * self.pos_std[1] + self.pos_mean[1]
        return x_phys, z_phys

    def constitutive_estimate(self, x_seq, x_static_raw):
        """Physics-based segment tension from input motion [N], shape [B,W,N-1].

        chord_t = instantaneous chord length between adjacent nodes;
        chord_0 = t=0 chord from the raw static graph (cols 4,5 = x0,z0).
        Incremental form: T_phys = relu(T0_seg + EA*(chord_t - chord_0)/chord_0)
        — measuring elongation chord-to-chord cancels the chord-vs-arc bias to
        first order and anchors the estimate at the known pretension.
        Literal form (cfg.constitutive_incremental=False): relu(EA*strain).
        """
        x_phys, z_phys = self.positions_physical(x_seq)
        dx = torch.diff(x_phys, dim=-1)                    # [B,W,N-1] spatial diff
        dz = torch.diff(z_phys, dim=-1)
        chord_t = torch.sqrt(dx * dx + dz * dz + 1e-8)

        x0 = x_static_raw[..., 4].float()                  # [B,N]
        z0 = x_static_raw[..., 5].float()
        chord_0 = torch.sqrt(
            torch.diff(x0, dim=-1) ** 2 + torch.diff(z0, dim=-1) ** 2 + 1e-8
        ).unsqueeze(1)                                     # [B,1,N-1]

        strain = (chord_t - chord_0) / chord_0             # [B,W,N-1]
        if self.constitutive_incremental:
            t0 = x_static_raw[..., 11].float()             # [B,N] tension0
            t0_seg = 0.5 * (t0[..., :-1] + t0[..., 1:]).unsqueeze(1)
            return torch.relu(t0_seg + self.ea * strain)
        return torch.relu(self.ea * strain)

    def catenary_rhs(self, x_seq):
        """Quasi-static equilibrium RHS w_s * dz [N], shape [B,W,N-1].

        Tangential balance of an inextensible cable under submerged weight:
        dT/ds = w_s * dz/ds  =>  T_{i+1} - T_i = w_s * (z_{i+1} - z_i).
        Violated only by dynamics/drag -> natural noise floor (Kendall-safe).
        """
        _, z_phys = self.positions_physical(x_seq)
        return self.w_s * torch.diff(z_phys, dim=-1)

    def momentum_balance(self, t_norm, x_seq, x_static_raw):
        """Discrete nodal momentum balance (Newton's second law) [N].

        For each INTERIOR node i (anchor/fairlead excluded: their motion is
        a prescribed boundary condition, so the reaction there is
        indeterminate from this balance alone):

            m_i * a_i = T_seg[i]*t_hat[i] - T_seg[i-1]*t_hat[i-1]
                        + (0, -w_i) + F_drag_i

        m_i / w_i / diameter come from the RAW static cols 6 / 9 / 7 (end-
        halved by the graph builder); per-node L_eff = m_i / mass_per_length
        recovers the patch-09b half-segment convention exactly. a_i is the
        second central time-difference of the input positions (dt = cache
        step); the elastic force transports the tension along instantaneous
        chord tangents; the drag force re-uses the dataset's quadratic-drag
        formulation (structure velocity, per-node tangent frame). Nodes in
        seabed contact are masked out (bed reaction not modelled here).
        This is the dynamic generalisation of catenary_rhs and the only
        term tying the INPUT kinematics to the OUTPUT tension through
        inertia + drag. Known noise floor (FE added mass, fluid velocity
        missing in drag, continuum-vs-lumped discretisation): Kendall
        down-weights it; run_physics_sanity_check quantifies it.

        t_norm : [B,W,N] normalized tension (prediction or ground truth)
        Returns (lhs, rhs, mask): lhs = m*a, rhs = net force, both
        [B,W-2,N-2,2] physical Newtons (x,z); mask [B,W-2,N-2,1].
        """
        x_phys, z_phys = self.positions_physical(x_seq)
        pos = torch.stack([x_phys, z_phys], dim=-1)              # [B,W,N,2]

        acc = (pos[:, 2:] - 2.0 * pos[:, 1:-1] + pos[:, :-2]) / self.dt ** 2
        vel = (pos[:, 2:] - pos[:, :-2]) / (2.0 * self.dt)       # [B,W-2,N,2]
        pos_c = pos[:, 1:-1]                                     # centred slice

        # elastic transport of tension along instantaneous chord tangents
        t_phys = t_norm.float() * self.t_std + self.t_mean       # [B,W,N]
        t_seg = 0.5 * (t_phys[..., :-1] + t_phys[..., 1:])[:, 1:-1]
        seg = pos_c[:, :, 1:] - pos_c[:, :, :-1]                 # [B,W-2,N-1,2]
        t_hat = seg / seg.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        f_el = (t_seg[..., 1:, None] * t_hat[:, :, 1:]
                - t_seg[..., :-1, None] * t_hat[:, :, :-1])      # [B,W-2,N-2,2]

        # quadratic drag at interior nodes (same convention as the input
        # features: per-node central-difference tangent frame)
        d_node = pos_c[:, :, 2:] - pos_c[:, :, :-2]
        n_hat = d_node / d_node.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        tx, tz = n_hat[..., 0], n_hat[..., 1]
        v_int = vel[:, :, 1:-1]                                  # [B,W-2,N-2,2]
        v_t = v_int[..., 0] * tx + v_int[..., 1] * tz
        v_n = -v_int[..., 0] * tz + v_int[..., 1] * tx
        m_i = x_static_raw[..., 6].float()[:, 1:-1]              # [B,N-2]
        dia = x_static_raw[..., 7].float()[:, 1:-1]
        w_i = x_static_raw[..., 9].float()[:, 1:-1]
        l_eff = (m_i / self.mass_per_length).unsqueeze(1)        # [B,1,N-2]
        a_n = dia.unsqueeze(1) * l_eff                           # projected area
        f_t = -0.5 * self.RHO_WATER * self.cd_t * (math.pi * a_n) * v_t * v_t.abs()
        f_n = -0.5 * self.RHO_WATER * self.cd_n * a_n * v_n * v_n.abs()
        f_dx = f_t * tx - f_n * tz
        f_dz = f_t * tz + f_n * tx

        lhs = m_i[:, None, :, None] * acc[:, :, 1:-1]            # [B,W-2,N-2,2]
        rhs = torch.stack([f_el[..., 0] + f_dx,
                           f_el[..., 1] + f_dz - w_i.unsqueeze(1)], dim=-1)
        # mask seabed-contact nodes (raw 0/1 flag; col 4 is never normalized)
        contact = x_seq[:, 1:-1, 1:-1, 4].float()
        mask = (contact <= 0.5).float().unsqueeze(-1)
        return lhs, rhs, mask

    # ------------------------------------------------------------------
    def sigmas(self):
        """sigma_i = exp(0.5 * log_vars_i) — for per-epoch logging."""
        return torch.exp(0.5 * self.log_vars.detach()).cpu()

    def _compute_terms(self, y_pred, y_true, x_seq, x_static_raw):
        """Unweighted values of the 7 loss terms. Batched inputs only."""
        yp = y_pred[..., 0].float()     # [B,W,N]
        yt = y_true[..., 0].float()

        # 0/1 — extreme-peak MSE + normal-tightness L1 (normalized space)
        l_mse = F.mse_loss(yp, yt)
        l_l1  = F.l1_loss(yp, yt)

        # 2 — temporal trajectory: first-order time differences
        l_temp = F.mse_loss(torch.diff(yp, dim=1), torch.diff(yt, dim=1))

        # 3 — spectral: amplitude spectra along time. norm="forward" divides by W
        #     so magnitudes are W-independent (default "backward" would inflate
        #     the term by ~W^2 at window_len=1000).
        # --- patch sr-09: nvrtc-free complex magnitude -----------------------
        # torch.abs() on a COMPLEX tensor is a *jiterator* op -- torch emits CUDA
        # source and compiles it at run time with nvrtc. On the rebuilt DelftBlue
        # stack nvrtc cannot dlopen libnvrtc-builtins.so.12.5, so job 103651 died
        # on exactly this line, 3 min into the notebook and after a 35-min stage.
        # vector_norm over view_as_real is the same magnitude built from
        # precompiled real-dtype kernels: view_as_real is a zero-copy view, the
        # value matches torch.abs to <= 1 ULP, and -- unlike hypot(re,im) or
        # sqrt(re*re+im*im), which are also nvrtc-free -- its gradient stays
        # FINITE at exactly-zero bins, which a constant-tension window produces.
        # Term 3 is unchanged: still the amplitude spectrum, still norm="forward".
        sp = torch.linalg.vector_norm(
            torch.view_as_real(torch.fft.rfft(yp, dim=1, norm="forward")), dim=-1)
        st = torch.linalg.vector_norm(
            torch.view_as_real(torch.fft.rfft(yt, dim=1, norm="forward")), dim=-1)
        l_spec = F.mse_loss(sp, st)

        # 4 — constitutive EA-strain vs predicted segment tension (sigma_T units)
        t_phys_n = (self.constitutive_estimate(x_seq, x_static_raw) - self.t_mean) / self.t_std
        yp_seg = 0.5 * (yp[..., :-1] + yp[..., 1:])        # prediction at segment midpoints
        l_const = F.mse_loss(yp_seg, t_phys_n)

        # 5 — catenary equilibrium residual (sigma_T units; mean cancels in dT)
        dT_pred_phys = torch.diff(yp, dim=-1) * self.t_std
        l_cat = (((dT_pred_phys - self.catenary_rhs(x_seq)) / self.t_std) ** 2).mean()

        # 6 — nodal momentum balance m*a = F_net (sigma_T units; the dynamic
        #     generalisation of the catenary term; interior non-contact nodes)
        m_lhs, m_rhs, m_mask = self.momentum_balance(yp, x_seq, x_static_raw)
        m_res = (m_lhs - m_rhs) / self.t_std
        l_mom = (m_res.pow(2) * m_mask).sum() / (2.0 * m_mask.sum()).clamp_min(1.0)

        # 7 - high-tension pinball tau=0.9: two-sided asymmetric penalty on EVERY
        #     time-node entry whose TRUE tension exceeds pin_thr_n (physical N,
        #     converted to normalized units via the target normalizer). Spreads the
        #     conservative asymmetry across the whole high-tension regime so the
        #     moderate-high p90-p99 peak band is pushed up too, not only the extreme max.
        thr_norm_pin = (self.pin_thr_n - self.t_mean) / self.t_std
        pin_mask = yt > thr_norm_pin                        # [B,T,N], on TRUE tension
        if pin_mask.any():
            d_pin = (yt - yp)[pin_mask]                     # >0 = underprediction
            l_pin = torch.maximum(self.tau * d_pin, (self.tau - 1.0) * d_pin).mean()
        else:
            l_pin = yp.new_zeros(())

        return torch.stack([l_mse, l_l1, l_temp, l_spec, l_const, l_cat, l_mom, l_pin])

    @staticmethod
    def _ensure_batched(y_pred, y_true, x_seq, x_static_raw):
        if y_pred.dim() == 3:
            return (y_pred.unsqueeze(0), y_true.unsqueeze(0),
                    x_seq.unsqueeze(0), x_static_raw.unsqueeze(0))
        return y_pred, y_true, x_seq, x_static_raw

    @torch.no_grad()
    def calibrate_log_vars(self, model, loader, device, max_batches=5):
        """Data-dependent init: log_vars_i := ln(mean L_i) on the UNTRAINED model.

        Kendall's equilibrium is s_i* = ln(L_i); starting there gives every
        term an equal effective contribution (0.5 each) from step one, so a
        scale-mismatched term (the noise-dominated constitutive term sits at
        O(1e5) vs O(1) for the data terms) cannot swamp early training while
        the log_vars slowly adapt. Call once, after building model+criterion
        and BEFORE train_model().
        """
        was_training = model.training
        model.eval()
        sums = torch.zeros(self.n_terms)
        n = 0
        for bi, batch in enumerate(loader):
            if bi >= max_batches:
                break
            batch = move_sample_to_device(batch, device)
            y_hat = model_forward(model, batch)   # --- patch sr-03:
            x_raw = (torch.stack([g.x_raw for g in batch["graphs"]])
                     if "graphs" in batch else batch["graph"].x_raw)
            # --- patch sr-04: query-station statics when N_out != N_in
            x_raw_q = _batch_static_raw_q(batch)
            args = self._ensure_batched(y_hat.float(), batch["y_seq"].float(),
                                        batch["x_seq"].float(), x_raw.float())
            _yp, _yt, _xs, _xr = args
            _n_out = _yp.shape[-2]
            if _xs.shape[-2] != _n_out:
                _xs = interp_nodes(_xs, _n_out)
                _xr = x_raw_q.float()
                if _xr.dim() == 2:
                    _xr = _xr.unsqueeze(0)
                if _xr.shape[0] == 1 and _yp.shape[0] > 1:
                    _xr = _xr.expand(_yp.shape[0], -1, -1)
            sums += self._compute_terms(_yp, _yt, _xs, _xr).cpu()
            n += 1
        if was_training:
            model.train()
        mean_terms = (sums / max(1, n)).clamp_min(1e-8)
        self.log_vars.data = torch.log(mean_terms).to(self.log_vars.device)
        print("[kendall-calibration] initial term means:",
              {t: float(v) for t, v in zip(self.TERM_NAMES, mean_terms)})
        print("[kendall-calibration] log_vars set to ln(term):",
              {t: round(float(s), 3) for t, s in zip(self.TERM_NAMES, self.log_vars.data.cpu())})

    def forward(self, y_pred, y_true, x_seq, x_static_raw, x_static_raw_q=None):
        """
        y_pred, y_true : [W,N_out,T] or [B,W,N_out,T]  normalized tension
        x_seq          : [W,N_in,F] or [B,W,N_in,F]    normalized dynamic node features
        x_static_raw   : [N_in,12]  or [B,N_in,12]     RAW static node features (input graph)
        x_static_raw_q : [N_out,12] or [B,N_out,12]    RAW static node features at the
                         QUERY stations. Required only when N_out != N_in
                         (patch sr-04, Stage-2 super-resolution).
        """
        y_pred, y_true, x_seq, x_static_raw = self._ensure_batched(
            y_pred, y_true, x_seq, x_static_raw
        )

        # --- patch sr-04: move the physics geometry to the query stations ---
        # Only columns 0 (x), 1 (z) and 4 (contact_flag) of x_seq are read by
        # the physics terms; interpolating the whole tensor is simpler and the
        # cost is negligible. x_raw comes from build_mooring_graph at N_out --
        # NOT interpolated -- because the lumped end-halving in nodal_mass and
        # submerged_weight has no meaningful interpolant.
        n_out = y_pred.shape[-2]
        if x_seq.shape[-2] != n_out:
            if x_static_raw_q is None:
                raise ValueError(
                    "N_out != N_in but no query-station static features were "
                    "passed. Give forward() x_static_raw_q = query_graph.x_raw; "
                    "the physics terms cannot be evaluated without it."
                )
            x_seq = interp_nodes(x_seq, n_out)
            x_static_raw = x_static_raw_q
            if x_static_raw.dim() == 2:
                x_static_raw = x_static_raw.unsqueeze(0)
            if x_static_raw.shape[0] == 1 and y_pred.shape[0] > 1:
                x_static_raw = x_static_raw.expand(y_pred.shape[0], -1, -1)
            x_static_raw = x_static_raw.float()

        terms = self._compute_terms(y_pred, y_true, x_seq, x_static_raw)
        s = self.log_vars
        total = (0.5 * torch.exp(-s) * terms + 0.5 * s).sum()

        term_log = {
            name: float(t.detach().cpu()) for name, t in zip(self.TERM_NAMES, terms)
        }
        return total, term_log


@torch.no_grad()
def compute_physical_metrics_per_target(
    y_hat_norm: torch.Tensor,
    y_true_norm: torch.Tensor,
    target_norm: FeatureStandardizer,
    target_names: List[str],
    mape_eps: float = 1e-8,
    mape_min: float = 1000.0,
    peak_mape_min: float = 1000.0,
):
    """
    Inputs:
        y_hat_norm, y_true_norm: [window_len, N, output_dim] in normalised space.
        mape_min         : per-step MAPE threshold on |true_tension| [N].
        peak_mape_min    : peak MAPE threshold on |true_peak_tension| [N].

    Returns a flat dict. Caller-important keys:
        MAE / RMSE / R2              : {target_name: float}  (mean over time+nodes)
        ss_res / y_sum / y_sq / n_pts: {target_name: float}  (raw sums for global R2)
        MAPE                         : {tension: float|nan, tension_valid_frac: float}
        peak_tension_MAE/_RMSE/_bias/_underpred_rate/_MAPE : float
        peak_tension_abs_errors      : List[float]  (per-node, for global p95)
        temporal_diff_MAE_{name}          : float
        temporal_diff_MAE_flat_{name}     : float  (flat-line predictor baseline)
        temporal_diff_skill_{name}        : float  (model/flat; < 1 = genuine tracking)
    """
    import math as _math
    y_hat_phys  = target_norm.inverse_transform(y_hat_norm.detach().cpu().float())
    y_true_phys = target_norm.inverse_transform(y_true_norm.detach().cpu().float())
    error = y_hat_phys - y_true_phys   # [W, N, D]

    # Per-target MAE / RMSE (mean over time+nodes)
    mae  = error.abs().mean(dim=(0, 1))
    rmse = torch.sqrt((error ** 2).mean(dim=(0, 1)))

    # Per-window R2 (kept for backward compat; averaged by caller)
    y_true_mean_w = y_true_phys.mean(dim=(0, 1), keepdim=True)
    ss_res_w = (error ** 2).sum(dim=(0, 1))
    ss_tot_w = ((y_true_phys - y_true_mean_w) ** 2).sum(dim=(0, 1))
    r2_w     = 1.0 - ss_res_w / ss_tot_w.clamp_min(mape_eps)

    # Raw sums for global R2 accumulation in evaluate()
    ss_res_raw = (error ** 2).sum(dim=(0, 1))
    y_true_sum = y_true_phys.sum(dim=(0, 1))
    y_true_sq  = (y_true_phys ** 2).sum(dim=(0, 1))
    n_per_feat = float(y_true_phys.shape[0] * y_true_phys.shape[1])

    metrics = {
        "MAE":  {name: float(mae[i].item())   for i, name in enumerate(target_names)},
        "RMSE": {name: float(rmse[i].item())  for i, name in enumerate(target_names)},
        "R2":   {name: float(r2_w[i].item())  for i, name in enumerate(target_names)},
        "ss_res": {name: float(ss_res_raw[i].item()) for i, name in enumerate(target_names)},
        "y_sum":  {name: float(y_true_sum[i].item()) for i, name in enumerate(target_names)},
        "y_sq":   {name: float(y_true_sq[i].item())  for i, name in enumerate(target_names)},
        "n_pts":  n_per_feat,
        "MAPE": {},
    }

    # --- MAPE + peak-tension (tension only) ---
    if "tension" in target_names:
        t_idx  = target_names.index("tension")
        true_t = y_true_phys[..., t_idx]   # [W, N]
        pred_t = y_hat_phys[..., t_idx]

        mask = true_t.abs() > mape_min
        if mask.any():
            mape_val   = float(((pred_t - true_t).abs() / true_t.abs())[mask].mean().item() * 100.0)
            valid_frac = float(mask.float().mean().item())
        else:
            mape_val, valid_frac = float("nan"), 0.0
        metrics["MAPE"]["tension"]            = mape_val
        metrics["MAPE"]["tension_valid_frac"] = valid_frac

        true_peak = true_t.amax(dim=0)   # [N]
        pred_peak = pred_t.amax(dim=0)
        peak_err  = pred_peak - true_peak
        peak_abs  = peak_err.abs()
        metrics["peak_tension_MAE"]            = float(peak_abs.mean().item())
        metrics["peak_tension_RMSE"]           = float((peak_err ** 2).mean().sqrt().item())
        metrics["peak_tension_bias"]           = float(peak_err.mean().item())
        metrics["peak_tension_underpred_rate"] = float((peak_err < 0).float().mean().item() * 100.0)
        metrics["peak_tension_abs_errors"]     = peak_abs.tolist()
        metrics["peak_tension_true_peaks"]    = true_peak.tolist()
        metrics["peak_tension_signed_errors"] = peak_err.tolist()

        pmask = true_peak.abs() > peak_mape_min
        if pmask.any():
            metrics["peak_tension_MAPE"] = float(
                (peak_abs[pmask] / true_peak.abs()[pmask]).mean().item() * 100.0
            )
        else:
            metrics["peak_tension_MAPE"] = float("nan")
    else:
        metrics["peak_tension_abs_errors"] = []
        metrics["peak_tension_true_peaks"]    = []
        metrics["peak_tension_signed_errors"] = []

    # --- Temporal accuracy vs the flat-line baseline ---
    # A flat predictor's diff error equals |true_diff|, so skill = model/flat;
    # < 1 means the model genuinely tracks step-to-step changes (the thesis
    # forecaster sat at ~1.7, i.e. WORSE than flat — this metric is the direct
    # test that the reconstruction task fixed the temporal blind spot).
    if error.shape[0] > 1:
        pred_diff  = y_hat_phys[1:] - y_hat_phys[:-1]
        true_diff  = y_true_phys[1:] - y_true_phys[:-1]
        diff_error = (pred_diff - true_diff).abs()
        for i, name in enumerate(target_names):
            model_diff_mae = float(diff_error[..., i].mean().item())
            flat_diff_mae  = float(true_diff[..., i].abs().mean().item())
            metrics[f"temporal_diff_MAE_{name}"]      = model_diff_mae
            metrics[f"temporal_diff_MAE_flat_{name}"] = flat_diff_mae
            metrics[f"temporal_diff_skill_{name}"]    = (
                model_diff_mae / max(flat_diff_mae, 1e-12)
            )
    else:
        for name in target_names:
            metrics[f"temporal_diff_MAE_{name}"]      = float("nan")
            metrics[f"temporal_diff_MAE_flat_{name}"] = float("nan")
            metrics[f"temporal_diff_skill_{name}"]    = float("nan")

    return metrics


In [ ]:
# -------------------------------------------------------------
# physics sanity check — run BEFORE training (paper-ready diagnostic)
# -------------------------------------------------------------

@torch.no_grad()
def run_physics_sanity_check(loader, criterion, target_norm, max_batches=25):
    """
    Score the three physics loss terms against GROUND TRUTH before any training.

    - constitutive: how well relu(T0 + EA*strain) from input motion matches the
      true segment tension. Chain strain is ~T/EA ~ 4e-5, so this term may be
      noise-dominated (chord/interpolation error >> elastic elongation) — this
      check quantifies exactly that, and the number goes in the paper either way.
    - catenary: how well w_s*dz explains the true node-to-node tension
      difference (quasi-static equilibrium; violated only by dynamics/drag).
    - momentum: how well the net force on each interior non-contact node
      (elastic transport of the TRUE tension + submerged weight + quadratic
      drag) explains m*a from the input motion (dynamic equilibrium).

    Uses the SAME criterion methods as the training loss (one source of truth).
    """
    tp_all, tt_all, dt_all, rhs_all = [], [], [], []
    ml_all, mr_all = [], []
    for bi, batch in enumerate(loader):
        if bi >= max_batches:
            break
        # --- patch sr-11: move the WHOLE batch -- including the patch
        # sr-01 query graph(s) -- onto the criterion's device BEFORE
        # anything is read out of it. This used to hand-roll the move for
        # x_seq/y_seq/x_raw only, so the sr-04 block below re-read the
        # query statics from the UN-moved batch and the physics terms
        # mixed cuda+cpu (jobs 117916 / 118966). move_sample_to_device()
        # is the single place that knows every key that has to travel --
        # never re-implement it.
        dev = criterion.pos_std.device
        batch = move_sample_to_device(batch, dev)
        if "graphs" in batch:   # bucketed batch
            x_seq = batch["x_seq"]
            y_seq = batch["y_seq"]
            x_raw = torch.stack([g.x_raw for g in batch["graphs"]])
        else:                    # single sample
            x_seq = batch["x_seq"].unsqueeze(0)
            y_seq = batch["y_seq"].unsqueeze(0)
            x_raw = batch["graph"].x_raw.unsqueeze(0)

        # redundant after the sr-11 move above, kept deliberately: these
        # are no-ops on an already-moved tensor and they document the
        # invariant that every tensor below lives on `dev`.
        x_seq = x_seq.to(dev)
        y_seq = y_seq.to(dev)
        x_raw = x_raw.to(dev)

        # --- patch sr-04: evaluate the physics on the QUERY geometry when the
        # sample is super-resolved, exactly as the loss does.
        x_raw_q = _batch_static_raw_q(batch)
        _n_out = y_seq.shape[-2]
        if x_seq.shape[-2] != _n_out:
            assert x_raw_q is not None, (
                "super-resolved sample carries no query_graph: "
                f"N_in={x_seq.shape[-2]} != N_out={_n_out}. Set out_N on "
                "the dataset subset (patch sr-01)."
            )
            x_seq = interp_nodes(x_seq, _n_out)
            x_raw = x_raw_q.float().to(dev)   # --- patch sr-11: never leave `dev`
            if x_raw.dim() == 2:
                x_raw = x_raw.unsqueeze(0)
            if x_raw.shape[0] == 1 and x_seq.shape[0] > 1:
                x_raw = x_raw.expand(x_seq.shape[0], -1, -1)

        t_phys = criterion.constitutive_estimate(x_seq, x_raw)   # [B,W,N-1] physical
        rhs    = criterion.catenary_rhs(x_seq)                    # [B,W,N-1] physical

        ti = target_norm.feature_idx.index(0)
        y_phys = (
            y_seq[..., 0].float() * float(target_norm.std[ti])
            + float(target_norm.mean[ti])
        )                                                          # [B,W,N]
        tt_seg  = 0.5 * (y_phys[..., :-1] + y_phys[..., 1:])
        dt_true = torch.diff(y_phys, dim=-1)

        tp_all.append(t_phys.flatten());  tt_all.append(tt_seg.flatten())
        dt_all.append(dt_true.flatten()); rhs_all.append(rhs.flatten())

        m_lhs, m_rhs, m_mask = criterion.momentum_balance(
            y_seq[..., 0].float(), x_seq, x_raw
        )
        keep = m_mask.expand_as(m_lhs) > 0.5
        ml_all.append(m_lhs[keep].flatten())
        mr_all.append(m_rhs[keep].flatten())

    tp, tt = torch.cat(tp_all), torch.cat(tt_all)
    dt, rhs = torch.cat(dt_all), torch.cat(rhs_all)
    ml, mr = torch.cat(ml_all), torch.cat(mr_all)

    def _corr(a, b):
        a = a - a.mean()
        b = b - b.mean()
        return float((a * b).mean() / (a.std() * b.std() + 1e-12))

    const_rms = float((tp - tt).pow(2).mean().sqrt())
    cat_res   = float((dt - rhs).pow(2).mean().sqrt())
    dt_rms    = float(dt.pow(2).mean().sqrt())
    mom_res   = float((ml - mr).pow(2).mean().sqrt())
    ma_rms    = float(ml.pow(2).mean().sqrt())

    print("----- PHYSICS SANITY CHECK (ground truth, pre-training) -----")
    print(f"samples: {tp.numel():,} segment-steps over {max_batches} batches")
    print(f"[constitutive] corr(T_phys, T_true_seg)    = {_corr(tp, tt):+.4f}")
    print(f"[constitutive] RMS(T_phys - T_true_seg)    = {const_rms:,.1f} N")
    print(f"[constitutive] RMS / std(T_true_seg)       = {const_rms / (float(tt.std()) + 1e-12):.3f}")
    print(f"[catenary]     corr(dT_true, w_s*dz)       = {_corr(dt, rhs):+.4f}")
    print(f"[catenary]     RMS residual / RMS(dT_true) = {cat_res / (dt_rms + 1e-12):.3f}")
    print(f"[momentum]     corr(m*a, F_net_true)       = {_corr(ml, mr):+.4f}")
    print(f"[momentum]     RMS residual / RMS(m*a)     = {mom_res / (ma_rms + 1e-12):.3f}")
    print(f"[momentum]     RMS(m*a) = {ma_rms:,.1f} N over {ml.numel():,} masked node-steps")
    print("Interpretation: corr near +1 with small relative RMS -> informative term;")
    print("corr ~ 0 or relative RMS >~ 1 -> noise-dominated (Kendall will down-weight;")
    print("report its learned sigma as a diagnostic, not a failure).")

    return {
        "constitutive_corr": _corr(tp, tt),
        "constitutive_rel_rms": const_rms / (float(tt.std()) + 1e-12),
        "catenary_corr": _corr(dt, rhs),
        "catenary_rel_rms": cat_res / (dt_rms + 1e-12),
        "momentum_corr": _corr(ml, mr),
        "momentum_rel_rms": mom_res / (ma_rms + 1e-12),
    }


In [ ]:
# -------------------------------------------------------------
# model/optimizer setup
# -------------------------------------------------------------

def build_model_from_config(
    example_dataset: MooringSequenceDatasetPositionTension,
    cfg: TrainingConfig,
    device: torch.device,
):
    """Build MooringGATLSTM using architecture dims from cfg."""
    model = MooringGATLSTM(
        n_static_node_features=example_dataset.graph_data.x.shape[1],
        n_dynamic_node_features=example_dataset.n_dynamic_node_features,
        n_static_edge_features=example_dataset.graph_data.edge_attr.shape[1],
        n_dynamic_edge_features=example_dataset.n_dynamic_edge_features,
        output_dim=example_dataset.n_targets,

        # architecture dims from TrainingConfig
        gat_hidden_dim=cfg.gat_hidden_dim,
        gat_out_dim=cfg.gat_out_dim,
        num_heads=cfg.num_heads,
        gat_dropout=cfg.gat_dropout,
        gat_use_layernorm=True,
        add_residual_projection=True,
        use_global_context=cfg.use_global_context,

        lstm_hidden_dim=cfg.lstm_hidden_dim,
        num_lstm_layers=cfg.num_lstm_layers,
        lstm_dropout=cfg.lstm_dropout,
        bidirectional=cfg.lstm_bidirectional,
        lstm_use_layernorm=True,

        head_hidden_dim=cfg.head_hidden_dim,
        head_dropout=cfg.head_dropout,

        # patch sr-03: Stage-2 query decoder (False => Stage-1 model exactly)
        use_query_decoder=getattr(cfg, "use_query_decoder", False),
        query_decoder_heads=getattr(cfg, "query_decoder_heads", 4),
        query_decoder_ffn_dim=getattr(cfg, "query_decoder_ffn_dim", 256),
        query_decoder_fourier_bands=getattr(cfg, "query_decoder_fourier_bands", 8),
        query_decoder_dropout=getattr(cfg, "query_decoder_dropout", 0.1),
        query_decoder_residual_interp=getattr(cfg, "query_decoder_residual_interp", True),
        query_decoder_zero_init=getattr(cfg, "query_decoder_zero_init", True),
    ).to(device)

    _ws = getattr(cfg, "warm_start_checkpoint", None)
    if _ws:
        load_stage1_warm_start(model, _ws, device)
    return model


def load_stage1_warm_start(model, ckpt_path, device):
    """patch sr-03: initialise from a Stage-1 (matched-resolution) checkpoint.

    Every Stage-1 weight is node-count agnostic and the decoder was inserted
    WITHOUT changing temporal_encoder / prediction_head shapes, so all three
    prefixes transfer verbatim; strict=False leaves query_decoder.* freshly
    initialised. Loud about what it did -- a silent partial load would be
    indistinguishable from training from scratch.
    """
    try:
        ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    except TypeError:
        ck = torch.load(ckpt_path, map_location=device)
    sd = ck.get("model_state_dict", ck)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    loaded = [k for k in sd if k not in set(unexpected)]
    n_dec = sum(1 for k in missing if k.startswith("query_decoder."))
    print(f"[warm-start] {ckpt_path}")
    print(f"[warm-start]   epoch in checkpoint : {ck.get('epoch')}")
    print(f"[warm-start]   tensors loaded      : {len(loaded)}")
    print(f"[warm-start]   fresh decoder params: {n_dec}")
    if unexpected:
        raise ValueError(
            f"warm start refused: {len(unexpected)} unexpected key(s) in the "
            f"checkpoint, e.g. {unexpected[:3]} -- architecture mismatch."
        )
    stale = [k for k in missing if not k.startswith("query_decoder.")]
    if stale:
        raise ValueError(
            f"warm start refused: {len(stale)} model tensor(s) NOT covered by "
            f"the checkpoint, e.g. {stale[:3]} -- this would silently train "
            f"part of the backbone from scratch."
        )
    del ck


def build_model_from_dataset_example(
    example_dataset: MooringSequenceDatasetPositionTension,
    device: torch.device,
    cfg: TrainingConfig = None,
):
    """Backwards-compatible wrapper. Pass cfg or default TrainingConfig() is used."""
    return build_model_from_config(example_dataset, cfg if cfg is not None else TrainingConfig(), device)

def build_optimizer_and_scheduler(
    model: nn.Module,
    cfg: TrainingConfig,
    criterion: nn.Module = None,
):
    """
    AdamW over the model parameters plus (optionally) the loss module's
    learnable parameters. The Kendall log_vars are homoscedastic-uncertainty
    state, not weights — they get their own param group with weight_decay=0
    so the regulariser cannot bias sigma toward 1.
    """
    param_groups = [
        {"params": list(model.parameters()), "weight_decay": cfg.weight_decay},
    ]
    if criterion is not None:
        loss_params = [p for p in criterion.parameters() if p.requires_grad]
        if loss_params:
            param_groups.append({"params": loss_params, "weight_decay": 0.0})

    optimizer = torch.optim.AdamW(
        param_groups,
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    scheduler = None
    if cfg.use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=cfg.scheduler_factor,
            patience=cfg.scheduler_patience,
            # patch 26: the monitored signal is -R2 (negative), so the
            # relative threshold_mode inverts the improvement bar. "abs"
            # is correct for a metric of any sign. getattr() keeps old
            # checkpoints' pickled cfg objects loadable.
            threshold=getattr(cfg, "scheduler_threshold", 1e-4),
            threshold_mode=getattr(cfg, "scheduler_threshold_mode", "abs"),
            # floor: without it the corrected schedule decays to ~2e-7 and
            # freezes the model (12 decays on v4's trajectory).
            min_lr=getattr(cfg, "scheduler_min_lr", 5e-5),
        )

    return optimizer, scheduler


In [ ]:
# -------------------------------------------------------------
# training and validation loops
# -------------------------------------------------------------


def move_sample_to_device(sample, device):
    result = {
        "x_seq":      sample["x_seq"].to(device, non_blocking=True),
        "edge_seq":   sample["edge_seq"].to(device, non_blocking=True),
        "y_seq":      sample["y_seq"].to(device, non_blocking=True),
        "start_idx":  sample["start_idx"],
        "lc_id":      sample.get("lc_id"),
        "case_id":    sample.get("case_id"),
        "node_count": sample.get("node_count"),
        "out_N":      sample.get("out_N"),
    }
    if "graphs" in sample:
        # Batched: clone each graph individually before moving to device
        # (required to avoid mutating dataset-internal graph objects).
        # Data.to(device) moves ALL tensor attributes, incl. x_raw/edge_attr_raw.
        result["graphs"] = [g.clone().to(device) for g in sample["graphs"]]
    else:
        result["graph"] = sample["graph"].clone().to(device)
# --- patch sr-01: query graph(s) follow the same clone-then-move rule
    if "query_graphs" in sample:
        result["query_graphs"] = [g.clone().to(device) for g in sample["query_graphs"]]
    elif "query_graph" in sample:
        result["query_graph"] = sample["query_graph"].clone().to(device)
    return result


def _batch_static_raw_q(sample):
    """--- patch sr-04: RAW static node features at the QUERY stations.

    Returns [B, N_out, 12] / [N_out, 12], or None for matched resolution
    (Stage-1 samples carry no query graph).
    """
    if "query_graphs" in sample:
        return torch.stack([g.x_raw for g in sample["query_graphs"]])
    if "query_graph" in sample:
        return sample["query_graph"].x_raw
    return None


def _batch_static_raw(sample):
    """Raw physical static node features for the physics loss terms.

    Returns [B, N, 14] (bucketed batch) or [N, 14] (single sample)."""
    if "graphs" in sample:
        return torch.stack([g.x_raw for g in sample["graphs"]])
    return sample["graph"].x_raw


def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    cfg: TrainingConfig,
    scaler,
    target_norm=None,
    target_names=None,
):
    """
    Returns (avg_total_loss, avg_grad_norm, train_phys_metrics, avg_terms).

    avg_total_loss : mean Kendall-weighted total (contains the 0.5*log_var
                     regularisers — interpretable as an objective, NOT as a
                     quality metric).
    avg_terms      : {term_name: mean unweighted value} for the 7 loss terms.
    train_phys_metrics is {} when cfg.compute_train_physical_metrics is False.
    """
    import math as _math
    model.train()
    criterion.train()

    amp_enabled       = cfg.use_amp and device.type == "cuda"
    running_loss      = 0.0
    term_sums         = {}
    running_grad_norm = 0.0
    num_batches       = 0
    optimizer_steps   = 0
    optimizer.zero_grad(set_to_none=True)
    pending_grad = False
    n_skipped = 0

    # clip the loss module's log_vars together with the model weights
    _clip_params = list(model.parameters()) + list(criterion.parameters())

    do_train_phys = (
        cfg.compute_train_physical_metrics
        and target_norm is not None
        and target_names is not None
    )
    _tnames = target_names or []
    phys_struct = {k: {n: 0.0 for n in _tnames}
                   for k in ("MAE", "RMSE", "R2", "ss_res", "y_sum", "y_sq")}
    phys_n_pts  = {n: 0.0 for n in _tnames}
    phys_flat   = {}
    phys_flat_c = {}
    phys_peak_abs = []
    phys_nb     = 0

    for step_idx, sample in enumerate(loader):
        sample = move_sample_to_device(sample, device)

        if cfg.debug_batch_shapes and step_idx < 3:
            nc = sample.get("node_count", "?")
            gs = len(sample["graphs"]) if "graphs" in sample else 1
            print(f"  [batch debug] step={step_idx}  x_seq={tuple(sample['x_seq'].shape)}"
                  f"  node_count={nc}  n_graphs={gs}")

        with torch.amp.autocast("cuda", enabled=amp_enabled):
            y_hat = model_forward(model, sample)   # --- patch sr-03:

        # Kendall loss in float32 OUTSIDE autocast (rfft + physics need fp32)
        with torch.amp.autocast("cuda", enabled=False):
            x_raw = _batch_static_raw(sample)
            loss, term_log = criterion(
                y_hat.float(), sample["y_seq"].float(),
                sample["x_seq"].float(), x_raw.float(),
                x_static_raw_q=_batch_static_raw_q(sample),   # --- patch sr-04
            )

        loss_scaled = loss / cfg.accumulation_steps if cfg.accumulation_steps > 1 else loss

        # per-batch NaN/Inf loss guard: skip a corrupt batch instead of crashing the run
        if not torch.isfinite(loss_scaled):
            n_skipped += 1
            if n_skipped <= 10:
                print(f"  [nan-guard] non-finite loss at step {step_idx}; skipping batch (corrupt data?)")
            continue
        scaler.scale(loss_scaled).backward()
        pending_grad = True

        if (step_idx + 1) % cfg.accumulation_steps == 0:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                _clip_params,
                cfg.grad_clip_max_norm if cfg.grad_clip_max_norm is not None else float("inf"),
            )
            running_grad_norm += float(grad_norm)
            optimizer_steps   += 1
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            pending_grad = False

        # Log the unscaled (pre-division) loss for interpretable CSV
        running_loss += loss.item()
        for k, v in term_log.items():
            term_sums[k] = term_sums.get(k, 0.0) + v
        num_batches += 1

        if do_train_phys and phys_nb < cfg.max_train_metric_batches:
            y_hat_det = y_hat.detach()
            y_seq     = sample["y_seq"]
            items_h = [y_hat_det[b] for b in range(y_hat_det.shape[0])] if y_hat_det.dim() == 4 else [y_hat_det]
            items_t = [y_seq[b]     for b in range(y_hat_det.shape[0])] if y_hat_det.dim() == 4 else [y_seq]
            for yh, yt in zip(items_h, items_t):
                m = compute_physical_metrics_per_target(
                    y_hat_norm=yh, y_true_norm=yt,
                    target_norm=target_norm, target_names=target_names,
                )
                phys_nb += 1
                for metric in ("MAE", "RMSE", "R2", "ss_res", "y_sum", "y_sq"):
                    for n in target_names:
                        phys_struct[metric][n] += m[metric][n]
                for n in target_names:
                    phys_n_pts[n] += m["n_pts"]
                phys_peak_abs.extend(m.get("peak_tension_abs_errors", []))
                _skip = {"MAE","RMSE","R2","MAPE","ss_res","y_sum","y_sq",
                         "n_pts","peak_tension_abs_errors","peak_tension_true_peaks","peak_tension_signed_errors"}
                for k in list(m["MAPE"].keys()) + [k for k in m if k not in _skip]:
                    val = m["MAPE"].get(k) if k in m["MAPE"] else m[k]
                    import math as _m2
                    if k not in phys_flat:
                        phys_flat[k]   = 0.0
                        phys_flat_c[k] = 0
                    if not _m2.isnan(val):
                        phys_flat[k]   += val
                        phys_flat_c[k] += 1

    if pending_grad:
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(
            _clip_params,
            cfg.grad_clip_max_norm if cfg.grad_clip_max_norm is not None else float("inf"),
        )
        running_grad_norm += float(grad_norm)
        optimizer_steps   += 1
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    avg_total_loss = running_loss / max(1, num_batches)
    avg_grad_norm  = running_grad_norm / max(1, optimizer_steps)
    avg_terms      = {k: v / max(1, num_batches) for k, v in term_sums.items()}

    train_phys_metrics = {}
    if do_train_phys and phys_nb > 0:
        _meps = 1e-8
        def _gr2(ss, ys, yq, np_, tnames):
            out = {}
            for n in tnames:
                n_t = np_[n]
                if n_t < 2:
                    out[n] = float("nan"); continue
                gm = ys[n] / n_t
                ss_tot = yq[n] - n_t * gm ** 2
                out[n] = float(1.0 - ss[n] / max(ss_tot, _meps))
            return out
        train_phys_metrics = {
            "MAE":       {n: phys_struct["MAE"][n]  / phys_nb for n in target_names},
            "RMSE":      {n: phys_struct["RMSE"][n] / phys_nb for n in target_names},
            "R2":        {n: phys_struct["R2"][n]   / phys_nb for n in target_names},
            "global_R2": _gr2(phys_struct["ss_res"], phys_struct["y_sum"],
                               phys_struct["y_sq"], phys_n_pts, target_names),
            "MAPE": {},
        }
        for k, s in phys_flat.items():
            cnt = phys_flat_c[k]
            val = s / cnt if cnt > 0 else float("nan")
            if k in ("tension", "tension_valid_frac"):
                train_phys_metrics["MAPE"][k] = val
            else:
                train_phys_metrics[k] = val
        if phys_peak_abs:
            sp = sorted(phys_peak_abs)
            p95_idx = max(0, int(0.95 * len(sp)) - 1)
            train_phys_metrics["peak_tension_p95_abs_error"] = float(sp[p95_idx])
            train_phys_metrics["peak_tension_max_abs_error"] = float(sp[-1])

    return avg_total_loss, avg_grad_norm, train_phys_metrics, avg_terms


def _pw_item(v, i):
    """patch sr-18: read element i of a collated field.

    bucketed_node_count_collate returns list[int] per batch; single_item_collate
    returns the bare scalar. Both reach evaluate(), so both are handled.
    """
    if isinstance(v, (list, tuple)):
        return v[i] if i < len(v) else None
    return v


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
    mape_min: float = 1000.0,
    per_nc_metrics: Optional[dict] = None,
    per_window_rows: Optional[list] = None,   # patch sr-18
    cfg=None,
):
    """
    Evaluate model over loader. Returns (avg_loss, avg_metrics).

    avg_loss is the mean Kendall total — logged for objective continuity but it
    contains the 0.5*log_var regularisers (can go negative) and MUST NOT be
    used for model selection. Select on avg_metrics["global_R2"]["tension"].

    avg_metrics adds: global_R2, temporal_diff_MAE/_flat/_skill, aggregated
    peak_tension_p95/max_abs_error, peak_tension_MAPE, and the per-term means
    as loss_term_{name}.

    per_nc_metrics (optional out-param): if a dict is passed, it is populated
    as {node_count: avg_metrics_dict} with per-node-count breakdown.
    """
    import math as _math
    from collections import defaultdict as _dd
    model.eval()
    criterion.eval()

    running_loss   = 0.0
    term_sums      = {}
    n_loss_batches = 0
    num_windows    = 0

    struct_sums = {
        "MAE":    {n: 0.0 for n in target_names},
        "RMSE":   {n: 0.0 for n in target_names},
        "R2":     {n: 0.0 for n in target_names},
        "ss_res": {n: 0.0 for n in target_names},
        "y_sum":  {n: 0.0 for n in target_names},
        "y_sq":   {n: 0.0 for n in target_names},
        "n_pts":  {n: 0.0 for n in target_names},
    }
    flat_sums   = {}
    flat_counts = {}
    all_peak_abs_errors = []
    all_peak_true_peaks    = []
    all_peak_signed_errors = []

    # patch 16: per-contact-regime per-node R2 (grounded/touchdown/suspended).
    # Uniform-sample yardstick for the receptive-field fix (Finding 4). Per-node
    # R2 is affine-invariant, so it is computed in NORMALISED tension space.
    _regime_r2 = {"grounded": [], "touchdown": [], "suspended": []}
    _cidx = getattr(cfg, "contact_flag_idx", 4) if cfg is not None else 4
    _do_regime = ("tension" in target_names) and (
        getattr(cfg, "compute_contact_regime_metrics", True) if cfg is not None else True
    )
    _t_idx = target_names.index("tension") if "tension" in target_names else 0

    do_nc = per_nc_metrics is not None
    nc_struct   = _dd(lambda: {k: {n: 0.0 for n in target_names}
                                for k in ("MAE","RMSE","R2","ss_res","y_sum","y_sq","n_pts")})
    nc_flat     = _dd(dict)
    nc_flat_c   = _dd(dict)
    nc_peak_abs = _dd(list)
    # patch 14: per-topology tail safety needs the SIGNED errors and the TRUE
    # peaks per node count (abs errors alone only support p95/max).
    nc_peak_true   = _dd(list)
    nc_peak_signed = _dd(list)
    nc_nb       = _dd(int)

    for sample in loader:
        sample = move_sample_to_device(sample, device)
        nc_val = sample.get("node_count")

        y_hat_out = model_forward(model, sample)   # --- patch sr-03:

        # batch-level Kendall loss (same objective as training)
        x_raw = _batch_static_raw(sample)
        loss, term_log = criterion(
            y_hat_out.float(), sample["y_seq"].float(),
            sample["x_seq"].float(), x_raw.float(),
            x_static_raw_q=_batch_static_raw_q(sample),   # --- patch sr-04
        )
        running_loss += loss.item()
        for k, v in term_log.items():
            term_sums[k] = term_sums.get(k, 0.0) + v
        n_loss_batches += 1

        if y_hat_out.dim() == 4:
            y_hat_items  = [y_hat_out[b]        for b in range(y_hat_out.shape[0])]
            y_true_items = [sample["y_seq"][b]  for b in range(y_hat_out.shape[0])]
        else:
            y_hat_items  = [y_hat_out]
            y_true_items = [sample["y_seq"]]

        _xseq_all = sample["x_seq"]
        if _xseq_all.dim() == 4:
            _xseq_items = [_xseq_all[b] for b in range(_xseq_all.shape[0])]
        else:
            _xseq_items = [_xseq_all]

        for _wi, (y_hat, y_true) in enumerate(zip(y_hat_items, y_true_items)):
            num_windows += 1

            # patch 16: per-node tension R2 binned by seabed-contact regime
            if _do_regime and y_true.dim() == 3:
                _xw = _xseq_items[_wi]
                if _xw.dim() == 3 and _xw.shape[-1] > _cidx:
                    _cf   = _xw[:, :, _cidx].float()
                    _frac = _cf.mean(dim=0)
                    # --- patch sr-04: the contact fraction lives at the INPUT
                    # stations, the per-node R2 at the OUTPUT stations. Align
                    # them along arc length, or N_in > N_out indexes past the
                    # end of the R2 list (and N_in < N_out silently drops
                    # nodes). The per-node TIME MEAN is interpolated rather
                    # than the binary flag per step -- "what fraction of the
                    # window is this arc position grounded" interpolates
                    # cleanly, a 0/1 flag does not.
                    _n_out_r = y_true.shape[1]
                    if _frac.shape[0] != _n_out_r:
                        _frac = interp_nodes(_frac.view(-1, 1), _n_out_r).view(-1)
                    _yt   = y_true[:, :, _t_idx].float()
                    _yh   = y_hat[:, :, _t_idx].float()
                    _ssr  = ((_yh - _yt) ** 2).sum(dim=0)
                    _sst  = ((_yt - _yt.mean(dim=0, keepdim=True)) ** 2).sum(dim=0)
                    _r2n  = 1.0 - _ssr / _sst.clamp_min(1e-8)
                    _fr   = _frac.detach().cpu().tolist()
                    _r2l  = _r2n.detach().cpu().tolist()
                    _vl   = (_sst > 1e-8).detach().cpu().tolist()
                    for _ni in range(len(_fr)):
                        if not _vl[_ni]:
                            continue
                        _fv = _fr[_ni]
                        if _fv >= 0.98:
                            _regime_r2["grounded"].append(_r2l[_ni])
                        elif _fv > 0.02:
                            _regime_r2["touchdown"].append(_r2l[_ni])
                        else:
                            _regime_r2["suspended"].append(_r2l[_ni])

            m = compute_physical_metrics_per_target(
                y_hat_norm=y_hat,
                y_true_norm=y_true,
                target_norm=target_norm,
                target_names=target_names,
                mape_min=mape_min,
            )

            for metric in ("MAE", "RMSE", "R2"):
                for n in target_names:
                    struct_sums[metric][n] += m[metric][n]
            for rk in ("ss_res", "y_sum", "y_sq"):
                for n in target_names:
                    struct_sums[rk][n] += m[rk][n]
            for n in target_names:
                struct_sums["n_pts"][n] += m["n_pts"]
            all_peak_abs_errors.extend(m.get("peak_tension_abs_errors", []))
            all_peak_true_peaks.extend(m.get("peak_tension_true_peaks", []))
            all_peak_signed_errors.extend(m.get("peak_tension_signed_errors", []))

            # --- patch sr-18: per-window sufficient statistics  ------------
            # Emitted only when a caller passes a list, so the per-epoch
            # validation inside train_model is untouched and costs nothing.
            # The RAW SUMS are the point: a mean of per-window R2 is not the
            # pooled global_R2, so only ss_res/y_sum/y_sq/n_pts let an offline
            # subset (the wave-train repeat split, a location, an N_in, a noise
            # level) reproduce the published headline EXACTLY.
            if per_window_rows is not None:
                _pwn = target_names[0]
                per_window_rows.append({
                    "lc_id":     _pw_item(sample.get("lc_id"), _wi),
                    "case_id":   _pw_item(sample.get("case_id"), _wi),
                    "n_in":      nc_val,
                    "out_N":     sample.get("out_N"),
                    "start_idx": _pw_item(sample.get("start_idx"), _wi),
                    "n_pts":  m["n_pts"],
                    "ss_res": m["ss_res"][_pwn],
                    "y_sum":  m["y_sum"][_pwn],
                    "y_sq":   m["y_sq"][_pwn],
                    "mae":    m["MAE"][_pwn],
                    "rmse":   m["RMSE"][_pwn],
                    "r2":     m["R2"][_pwn],
                    "mape":            m["MAPE"].get(_pwn, float("nan")),
                    "mape_valid_frac": m["MAPE"].get(_pwn + "_valid_frac", float("nan")),
                    "td_mae":      m.get("temporal_diff_MAE_" + _pwn, float("nan")),
                    "td_mae_flat": m.get("temporal_diff_MAE_flat_" + _pwn, float("nan")),
                    "td_skill":    m.get("temporal_diff_skill_" + _pwn, float("nan")),
                    "peak_MAE":    m.get("peak_tension_MAE", float("nan")),
                    "peak_bias":   m.get("peak_tension_bias", float("nan")),
                })

            _skip = {"MAE","RMSE","R2","MAPE","ss_res","y_sum","y_sq",
                     "n_pts","peak_tension_abs_errors","peak_tension_true_peaks","peak_tension_signed_errors"}
            flat_keys = list(m["MAPE"].keys()) + [k for k in m if k not in _skip]
            for k in flat_keys:
                val = m["MAPE"].get(k) if k in m["MAPE"] else m[k]
                if k not in flat_sums:
                    flat_sums[k]   = 0.0
                    flat_counts[k] = 0
                if not _math.isnan(val):
                    flat_sums[k]   += val
                    flat_counts[k] += 1

            if do_nc and nc_val is not None:
                nc_nb[nc_val] += 1
                for metric in ("MAE", "RMSE", "R2"):
                    for n in target_names:
                        nc_struct[nc_val][metric][n] += m[metric][n]
                for rk in ("ss_res", "y_sum", "y_sq"):
                    for n in target_names:
                        nc_struct[nc_val][rk][n] += m[rk][n]
                for n in target_names:
                    nc_struct[nc_val]["n_pts"][n] += m["n_pts"]
                nc_peak_abs[nc_val].extend(m.get("peak_tension_abs_errors", []))
                nc_peak_true[nc_val].extend(m.get("peak_tension_true_peaks", []))
                nc_peak_signed[nc_val].extend(m.get("peak_tension_signed_errors", []))
                for k in flat_keys:
                    val = m["MAPE"].get(k) if k in m["MAPE"] else m[k]
                    if k not in nc_flat[nc_val]:
                        nc_flat[nc_val][k]   = 0.0
                        nc_flat_c[nc_val][k] = 0
                    if not _math.isnan(val):
                        nc_flat[nc_val][k]   += val
                        nc_flat_c[nc_val][k] += 1

    avg_loss = running_loss / max(1, n_loss_batches)
    _meps = 1e-8

    def _global_r2(ss_d, ys_d, yq_d, np_d, tnames):
        out = {}
        for n in tnames:
            n_t = np_d[n]
            if n_t < 2:
                out[n] = float("nan"); continue
            gm = ys_d[n] / n_t
            ss_tot = yq_d[n] - n_t * gm ** 2
            out[n] = float(1.0 - ss_d[n] / max(ss_tot, _meps))
        return out

    def _agg_p95_max(errs):
        if not errs:
            return float("nan"), float("nan")
        se = sorted(errs)
        return float(se[max(0, int(0.95 * len(se)) - 1)]), float(se[-1])

    def _peak_tail_stats(true_pk, signed_err, q):
        # returns (thr, underpred_rate, bias, MAE, conditional_undermag)
        # Tail-conditioned peak safety (2026-07-17 selection amendment):
        # stats over node-window peaks whose TRUE peak lies above the
        # pooled q-quantile — the design-critical extreme states. The val
        # set is fixed across ladder runs, so thresholds are comparable.
        if not true_pk:
            return (float("nan"),) * 5
        pairs = sorted(zip(true_pk, signed_err), key=lambda t: t[0])
        k = max(1, int(round((1.0 - q) * len(pairs))))
        tail = pairs[len(pairs) - k:]
        thr  = float(tail[0][0])
        errs = [e for _t, e in tail]
        under = 100.0 * sum(1 for e in errs if e < 0) / len(errs)
        bias  = float(sum(errs) / len(errs))
        mae   = float(sum(abs(e) for e in errs) / len(errs))
        # patch 10: CONDITIONAL underprediction magnitude -- mean |error| over
        # the tail entries that actually MISS low. Diagnostic only (never gated):
        # rate and magnitude are anti-correlated, so a rate-only gate is blind to
        # a few catastrophic misses dragging bias negative.
        _neg = [abs(e) for e in errs if e < 0]
        undermag = float(sum(_neg) / len(_neg)) if _neg else 0.0
        return thr, under, bias, mae, undermag

    def _build_avg(struct_s, flat_s, flat_c, nb, pk_list):
        avg = {
            "MAE":       {n: struct_s["MAE"][n]  / max(1, nb) for n in target_names},
            "RMSE":      {n: struct_s["RMSE"][n] / max(1, nb) for n in target_names},
            "R2":        {n: struct_s["R2"][n]   / max(1, nb) for n in target_names},
            "global_R2": _global_r2(
                struct_s["ss_res"], struct_s["y_sum"], struct_s["y_sq"],
                struct_s["n_pts"], target_names
            ),
            "MAPE": {},
        }
        for k, s in flat_s.items():
            cnt = flat_c[k]
            val = s / cnt if cnt > 0 else float("nan")
            if k in ("tension", "tension_valid_frac"):
                avg["MAPE"][k] = val
            else:
                avg[k] = val
        p95, pmax = _agg_p95_max(pk_list)
        avg["peak_tension_p95_abs_error"] = p95
        avg["peak_tension_max_abs_error"] = pmax
        return avg

    avg_metrics = _build_avg(
        struct_sums, flat_sums, flat_counts, num_windows, all_peak_abs_errors
    )
    for _q, _tag in ((0.90, "p90"), (0.99, "p99")):
        _thr, _und, _bias, _mae, _umag = _peak_tail_stats(
            all_peak_true_peaks, all_peak_signed_errors, _q
        )
        avg_metrics[f"peak_tension_thr_{_tag}"]            = _thr
        avg_metrics[f"peak_tension_underpred_rate_{_tag}"] = _und
        avg_metrics[f"peak_tension_bias_{_tag}"]           = _bias
        avg_metrics[f"peak_tension_MAE_{_tag}"]            = _mae
        avg_metrics[f"peak_tension_undermag_{_tag}"]       = _umag

    # patch 16: aggregate per-contact-regime R2 (median + %<0 + count)
    import statistics as _stats16
    for _rg in ("grounded", "touchdown", "suspended"):
        _vals = _regime_r2[_rg]
        if _vals:
            avg_metrics[f"contact_{_rg}_R2_median"]  = float(_stats16.median(_vals))
            avg_metrics[f"contact_{_rg}_R2_fracneg"] = float(
                100.0 * sum(1 for v in _vals if v < 0) / len(_vals))
            avg_metrics[f"contact_{_rg}_n"]          = len(_vals)
        else:
            avg_metrics[f"contact_{_rg}_R2_median"]  = float("nan")
            avg_metrics[f"contact_{_rg}_R2_fracneg"] = float("nan")
            avg_metrics[f"contact_{_rg}_n"]          = 0

    if do_nc:
        per_nc_metrics.clear()
        for nc_key, nb_val in nc_nb.items():
            _m_nc = _build_avg(
                nc_struct[nc_key], nc_flat[nc_key], nc_flat_c[nc_key],
                nb_val, nc_peak_abs[nc_key]
            )
            # patch 14: tail-conditioned peak safety PER TOPOLOGY. The quantile is
            # taken WITHIN this node count, so each topology is judged on its own
            # extreme states; thr_{p90,p99} is emitted too, because the absolute
            # tension level of "the top 1%" differs between topologies and must be
            # quoted alongside any cross-topology comparison.
            for _q, _tag in ((0.90, "p90"), (0.99, "p99")):
                _t, _u, _b, _ma, _um = _peak_tail_stats(
                    nc_peak_true[nc_key], nc_peak_signed[nc_key], _q
                )
                _m_nc[f"peak_tension_thr_{_tag}"]            = _t
                _m_nc[f"peak_tension_underpred_rate_{_tag}"] = _u
                _m_nc[f"peak_tension_bias_{_tag}"]           = _b
                _m_nc[f"peak_tension_MAE_{_tag}"]            = _ma
                _m_nc[f"peak_tension_undermag_{_tag}"]       = _um
            per_nc_metrics[nc_key] = _m_nc

    for k, v in term_sums.items():
        avg_metrics[f"loss_term_{k}"] = v / max(1, n_loss_batches)
    return avg_loss, avg_metrics


In [ ]:

def _sr14_resume_csv(path, fieldnames, start_epoch, _csv):
    """Open a metrics CSV so that a RESUMED run extends it instead of wiping it.

    Before patch sr-14 both metrics CSVs were opened "w", so submission 2 of a
    multi-submission FINAL run destroyed submission 1's history.  Rows at or
    after `start_epoch` are dropped, so re-running an epoch cannot duplicate it.
    Returns (file, writer, kept_rows).
    """
    kept = []
    if start_epoch > 1 and path.exists():
        try:
            with open(path, newline="", encoding="utf-8") as _f:
                for _r in _csv.DictReader(_f):
                    try:
                        if int(float(_r["epoch"])) < start_epoch:
                            kept.append(_r)
                    except (KeyError, TypeError, ValueError):
                        continue
        except Exception as _e:
            print("[resume-csv] could not read %s (%s) -- starting a fresh file"
                  % (path.name, _e))
            kept = []
    _fh = open(path, "w", newline="", encoding="utf-8")
    _w = _csv.DictWriter(_fh, fieldnames=fieldnames, extrasaction="ignore")
    _w.writeheader()
    for _r in kept:
        _w.writerow(_r)
    _fh.flush()
    return _fh, _w, kept


def _sr14_recover_counters(kept_rows):
    """Rebuild the loop counters a resume would otherwise reset.

    Returns (running_max_r2, n_feasible, epochs_without_improvement).
    Feasibility is read off the TIERED selection_score rather than by re-deriving
    Gate A/B, so it cannot drift from the gate: tier 0 (both gates passed) writes
    temporal_diff_skill (~0.26-0.5), tier 1 writes 10-R2 (~9.0) and tier 2 writes
    100-R2 (~99).  A cut at 1.0 separates them with three orders of margin.
    """
    _r2s, _scores = [], []
    for _r in kept_rows:
        try:
            _r2s.append(float(_r["val_global_R2_tension"]))
        except (KeyError, TypeError, ValueError):
            pass
        try:
            _scores.append(float(_r["selection_score"]))
        except (KeyError, TypeError, ValueError):
            pass
    _rmax = max(_r2s) if _r2s else -float("inf")
    _nfeas = sum(1 for _s in _scores if _s < 1.0)
    _ewi = 0
    if _scores:
        _best_i = min(range(len(_scores)), key=lambda _i: _scores[_i])
        _ewi = len(_scores) - 1 - _best_i
    return _rmax, _nfeas, _ewi


def _sr13_bias_ok(_b90, _cfg):
    """patch sr-13 Gate-B term: bias_p90 must exceed cfg.sel_bias90_min.

    Returns True unconditionally when sel_bias90_min is -inf (the default),
    so the pre-sr-13 gate is reproduced bit-for-bit.  With a finite floor a
    NaN bias FAILS the gate rather than silently passing.
    """
    import math as _m
    _floor = float(getattr(_cfg, "sel_bias90_min", float("-inf")))
    if _floor == float("-inf"):
        return True
    return (not _m.isnan(_b90)) and _b90 > _floor

# -------------------------------------------------------------
# full training runner with checkpointing and early stopping
# -------------------------------------------------------------

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion,
    device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
    cfg: TrainingConfig,
    run_seed: int,
):
    import csv as _csv, dataclasses as _dc, json as _json, math as _math

    checkpoint_dir = Path(cfg.checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = checkpoint_dir / f"best_mooring_gat_lstm_seed_{run_seed}.pt"

    _term_names = list(getattr(criterion, "TERM_NAMES", ()))

    # -- config.json: written once at training start ---------------------------
    config_path = checkpoint_dir / "config.json"
    with open(config_path, "w", encoding="utf-8") as _f:
        _json.dump(_dc.asdict(cfg), _f, indent=2, default=str)

    # -- metrics_history.csv: stable fieldnames determined before loop ---------
    _tgt = target_names
    _csv_base_fields = (
        ["epoch", "lr",
         "train_loss",            # alias for train_total_loss — backward compat
         "train_total_loss", "grad_norm", "amp_scale"]
        + [f"train_term_{t}" for t in _term_names]
        + [f"sigma_{t}"      for t in _term_names]
    )
    _train_phys_fields = (
        [f"train_MAE_{n}"       for n in _tgt] +
        [f"train_RMSE_{n}"      for n in _tgt] +
        [f"train_global_R2_{n}" for n in _tgt] +
        ["train_MAPE_tension", "train_MAPE_tension_valid_frac"] +
        ["train_peak_tension_MAE", "train_peak_tension_bias",
         "train_peak_tension_underpred_rate",
         "train_peak_tension_p95_abs_error"]
    )
    _csv_val_fields = (
        [f"val_MAE_{n}"       for n in _tgt] +
        [f"val_RMSE_{n}"      for n in _tgt] +
        [f"val_R2_{n}"        for n in _tgt] +
        [f"val_global_R2_{n}" for n in _tgt] +
        ["val_MAPE_tension", "val_MAPE_tension_valid_frac",
         "val_peak_tension_MAE", "val_peak_tension_RMSE",
         "val_peak_tension_bias", "val_peak_tension_underpred_rate",
         "val_peak_tension_MAPE",
         "val_peak_tension_p95_abs_error", "val_peak_tension_max_abs_error",
         "val_peak_tension_thr_p90", "val_peak_tension_underpred_rate_p90",
         "val_peak_tension_bias_p90", "val_peak_tension_MAE_p90",
         "val_peak_tension_thr_p99", "val_peak_tension_underpred_rate_p99",
         "val_peak_tension_bias_p99", "val_peak_tension_MAE_p99",
         "val_peak_tension_undermag_p90", "val_peak_tension_undermag_p99"] +
        ["val_contact_grounded_R2_median", "val_contact_grounded_R2_fracneg", "val_contact_grounded_n",
         "val_contact_touchdown_R2_median", "val_contact_touchdown_R2_fracneg", "val_contact_touchdown_n",
         "val_contact_suspended_R2_median", "val_contact_suspended_R2_fracneg", "val_contact_suspended_n"] +
        [f"val_temporal_diff_MAE_{n}"      for n in _tgt] +
        [f"val_temporal_diff_MAE_flat_{n}" for n in _tgt] +
        [f"val_temporal_diff_skill_{n}"    for n in _tgt] +
        [f"val_term_{t}" for t in _term_names] +
        ["val_loss", "selection_score"]
    )
    _csv_fields  = _csv_base_fields + _train_phys_fields + _csv_val_fields
    history_path = checkpoint_dir / "metrics_history.csv"
    # (opened after the resume block -- patch sr-14)
    # -- metrics_by_node_count.csv -------------------------------------------
    _nc_csv_fields = [
        "epoch", "split", "node_count",
        "MAE_tension", "RMSE_tension", "R2_tension", "global_R2_tension",
        "MAPE_tension",
        "peak_tension_MAE", "peak_tension_bias",
        "peak_tension_underpred_rate", "peak_tension_MAPE",
        "peak_tension_p95_abs_error", "peak_tension_max_abs_error",
        # patch 14: per-topology tail safety (quantile taken within each node count)
        "peak_tension_thr_p90", "peak_tension_underpred_rate_p90",
        "peak_tension_bias_p90", "peak_tension_MAE_p90", "peak_tension_undermag_p90",
        "peak_tension_thr_p99", "peak_tension_underpred_rate_p99",
        "peak_tension_bias_p99", "peak_tension_MAE_p99", "peak_tension_undermag_p99",
        "temporal_diff_MAE_tension", "temporal_diff_MAE_flat_tension",
        "temporal_diff_skill_tension",
    ]
    _nc_path   = checkpoint_dir / "metrics_by_node_count.csv"
    # (opened after the resume block -- patch sr-14)
    history = {
        "train_loss": [],
        "grad_norm":  [],
        "val_loss":   [],
        "val_metrics": [],
    }

    # AMP GradScaler — no-op on CPU (enabled=False passes through cleanly)
    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.use_amp and device.type == "cuda"))
    best_state                 = None
    best_criterion_state       = None
    epochs_without_improvement = 0
    start_epoch                = 1
    best_selection_score       = float("inf")
    best_val_loss_at_selection = float("inf")
    _sel_running_max_r2        = -float("inf")  # patch 09f: Gate-A running max
    _n_feasible_epochs         = 0              # patch 10: feasible-checkpoint count

    # -- checkpoint resume -----------------------------------------------
    if cfg.resume_from_checkpoint is not None:
        _ckpt = torch.load(cfg.resume_from_checkpoint, map_location=device, weights_only=False)
        model.load_state_dict(_ckpt["model_state_dict"])
        optimizer.load_state_dict(_ckpt["optimizer_state_dict"])
        if _ckpt.get("criterion_state_dict"):
            criterion.load_state_dict(_ckpt["criterion_state_dict"])
        if scheduler is not None and _ckpt.get("scheduler_state_dict"):
            scheduler.load_state_dict(_ckpt["scheduler_state_dict"])
        if _ckpt.get("scaler_state_dict"):
            scaler.load_state_dict(_ckpt["scaler_state_dict"])
        start_epoch = _ckpt.get("epoch", 0) + 1
        best_selection_score       = _ckpt.get("best_selection_score",
                                                _ckpt.get("best_val_loss", float("inf")))
        best_val_loss_at_selection = _ckpt.get("best_val_loss_at_selection",
                                                _ckpt.get("best_val_loss", float("inf")))
        print(f"Resumed from {cfg.resume_from_checkpoint}, "
              f"starting at epoch {start_epoch}, "
              f"best_selection_score={best_selection_score:.6f}")
        # -- patch sr-15: restore the best-so-far STATE, not just its score --
        # best_state is otherwise set only by an improvement INSIDE this
        # submission.  A resumed run whose best epoch lies in an earlier
        # submission therefore returned its LAST-epoch model, and the test
        # eval, the ts dump and the whole N_in x N_out grid silently scored the
        # wrong epoch while the log still named the best checkpoint (found on
        # FINAL_WARM job 129246: best = ep33 in submission 1, ep34-53 all
        # infeasible).  best_model_path is written together with best_state at
        # every improvement, so the file IS the best-so-far of the whole run.
        if best_model_path.exists():
            _bk = torch.load(best_model_path, map_location="cpu", weights_only=False)
            _bk_ep = int(_bk.get("epoch", -1))
            _bk_score = float(_bk.get("best_selection_score", float("nan")))
            if not (0 < _bk_ep < start_epoch):
                raise RuntimeError(
                    "[resume-best] %s holds epoch %d, which is not before the "
                    "resume point (epoch %d) -- it is not this run's best-so-far "
                    "checkpoint" % (best_model_path.name, _bk_ep, start_epoch))
            if not abs(_bk_score - float(best_selection_score)) <= 1e-9:
                raise RuntimeError(
                    "[resume-best] %s records selection_score %.9f but the resumed "
                    "checkpoint records best_selection_score %.9f -- the best file "
                    "does not belong to this run"
                    % (best_model_path.name, _bk_score, float(best_selection_score)))
            best_state           = _bk["model_state_dict"]
            best_criterion_state = _bk.get("criterion_state_dict")
            print("[resume-best] restored best-so-far state from %s (epoch %d, "
                  "selection_score=%.6f); it is what gets evaluated unless an "
                  "epoch of this submission improves on it"
                  % (best_model_path.name, _bk_ep, _bk_score))
            del _bk
        else:
            print("[resume-best] WARNING: %s not found -- unless an epoch of this "
                  "submission improves, the LAST epoch will be evaluated"
                  % best_model_path.name)


    # -- patch sr-14: open both metrics CSVs only AFTER the resume block, so a
    # resumed run EXTENDS them instead of truncating away submission 1.
    _csv_file, _csv_writer, _sr14_hist = _sr14_resume_csv(
        history_path, _csv_fields, start_epoch, _csv)
    _nc_file, _nc_writer, _ = _sr14_resume_csv(
        _nc_path, _nc_csv_fields, start_epoch, _csv)
    if start_epoch > 1:
        _sel_running_max_r2, _n_feasible_epochs, epochs_without_improvement = \
            _sr14_recover_counters(_sr14_hist)
        print("[resume-csv] preserved %d earlier epoch row(s) in %s"
              % (len(_sr14_hist), history_path.name))
        print("[resume-csv] recovered running_max_R2=%.6f | n_feasible=%d | "
              "epochs_without_improvement=%d"
              % (_sel_running_max_r2, _n_feasible_epochs,
                 epochs_without_improvement))
        if _n_feasible_epochs == 0:
            print("[resume-csv] WARNING: submission 1 had NO feasible epoch -- "
                  "the sel_abort_epoch guard is still armed and will fire.")
    def _ckpt_payload(epoch):
        return {
            "model_state_dict":           model.state_dict(),
            "criterion_state_dict":       criterion.state_dict(),
            "optimizer_state_dict":       optimizer.state_dict(),
            "scheduler_state_dict":       scheduler.state_dict() if scheduler else None,
            "scaler_state_dict":          scaler.state_dict(),
            "epoch":                      epoch,
            "best_selection_score":       best_selection_score,
            "best_val_loss_at_selection": best_val_loss_at_selection,
            "best_val_loss":              best_val_loss_at_selection,
            "cfg":                        _dc.asdict(cfg),
        }

    for epoch in range(start_epoch, cfg.num_epochs + 1):
        train_total_loss, grad_norm, train_phys, train_terms = \
            train_one_epoch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                criterion=criterion,
                device=device,
                cfg=cfg,
                scaler=scaler,
                target_norm=target_norm,
                target_names=target_names,
            )

        history["train_loss"].append(train_total_loss)
        history["grad_norm"].append(grad_norm)

        amp_scale  = scaler.get_scale()
        current_lr = optimizer.param_groups[0]["lr"]

        if _math.isnan(train_total_loss) or _math.isinf(train_total_loss):
            print(f"CRITICAL: train_total_loss={train_total_loss} at epoch {epoch} -- "
                  f"AMP overflow or corrupt batch. Stopping training.")
            break
        if amp_scale < 1.0:
            print(f"WARNING: AMP loss scale={amp_scale:.4f} at epoch {epoch} -- "
                  f"gradients likely underflowing. Set use_amp=False if loss stalls.")

        # Kendall sigma_i = exp(0.5 * log_vars_i) — logged every epoch; the
        # trajectory (and final ranking) is itself a result for the paper.
        _sig_vals = {}
        if hasattr(criterion, "sigmas") and _term_names:
            _sig = criterion.sigmas().tolist()
            _sig_vals = {t: float(s) for t, s in zip(_term_names, _sig)}

        _row = {
            "epoch":            epoch,
            "lr":               current_lr,
            "train_loss":       train_total_loss,
            "train_total_loss": train_total_loss,
            "grad_norm":        grad_norm,
            "amp_scale":        amp_scale,
        }
        _row.update({f"train_term_{t}": train_terms.get(t, "") for t in _term_names})
        _row.update({f"sigma_{t}": _sig_vals.get(t, "") for t in _term_names})
        if train_phys:
            _row.update({f"train_MAE_{n}":  train_phys["MAE"].get(n, "")  for n in target_names})
            _row.update({f"train_RMSE_{n}": train_phys["RMSE"].get(n, "") for n in target_names})
            _row.update({f"train_global_R2_{n}": train_phys["global_R2"].get(n, "") for n in target_names})
            _row["train_MAPE_tension"]              = train_phys["MAPE"].get("tension", "")
            _row["train_MAPE_tension_valid_frac"]   = train_phys["MAPE"].get("tension_valid_frac", "")
            _row["train_peak_tension_MAE"]           = train_phys.get("peak_tension_MAE", "")
            _row["train_peak_tension_bias"]          = train_phys.get("peak_tension_bias", "")
            _row["train_peak_tension_underpred_rate"]= train_phys.get("peak_tension_underpred_rate", "")
            _row["train_peak_tension_p95_abs_error"] = train_phys.get("peak_tension_p95_abs_error", "")

        is_last_epoch = (epoch == cfg.num_epochs)
        # validate_every: run validation every N epochs and always on the last epoch.
        # Early-stopping patience counts validation events, not training epochs.
        do_validate = (epoch % cfg.validate_every == 0) or is_last_epoch

        if do_validate:
            _nc_out = {}
            val_loss, val_metrics = evaluate(
                model=model,
                loader=val_loader,
                criterion=criterion,
                device=device,
                target_norm=target_norm,
                target_names=target_names,
                mape_min=cfg.tension_mape_min,
                per_nc_metrics=_nc_out,
                cfg=cfg,
            )

            history["val_loss"].append(val_loss)
            history["val_metrics"].append(val_metrics)

            # -- checkpoint selection -------------------------------------
            # Kendall val_loss contains the 0.5*log_var regularisers (can go
            # negative) and is NOT a quality proxy — select on global R2.
            _sel_in_gate = _sel_in_box = False
            if getattr(cfg, "select_on_r2_tension", False):
                _r2t = val_metrics["global_R2"].get("tension", float("nan"))
                _sched_score = -float(_r2t) if not _math.isnan(_r2t) else float("inf")
                # ---- blended checkpoint selection (patch 09f) ----------------
                # Tiered score, LOWER = better:
                #   tier 0  Gate A AND safety box : score = temporal_diff_skill
                #   tier 1  Gate A only           : score = 10.0  - R2  (~9.0)
                #   tier 2  outside Gate A        : score = 100.0 - R2  (~99.0)
                # Gate A uses the RUNNING max R2 (causal approximation; the
                # binding post-run pick is pub_select_epoch.py on the CSV).
                if not _math.isnan(_r2t):
                    _sel_running_max_r2 = max(_sel_running_max_r2, float(_r2t))
                _sb90 = float(val_metrics.get("peak_tension_bias_p90", float("nan")))
                _sb99 = float(val_metrics.get("peak_tension_bias_p99", float("nan")))
                _su90 = float(val_metrics.get("peak_tension_underpred_rate_p90", float("nan")))
                _su99 = float(val_metrics.get("peak_tension_underpred_rate_p99", float("nan")))
                _sskl = float(val_metrics.get("temporal_diff_skill_tension", float("nan")))
                _sel_in_gate = (not _math.isnan(_r2t)) and (
                    float(_r2t) >= _sel_running_max_r2 - cfg.sel_r2_delta
                )
                # patch 25 (selection v4, 2026-07-27): Gate B = the p90
                # underprediction RATE only. The p99 cap is GONE -- on the
                # corrected pipeline the p99 tail is 43-82% snap-composed and
                # snap magnitude is aliasing-bounded at the 0.1 s export
                # interval, so it is not a cap the model can meet. p90 is only
                # 10-15% snap-composed and carries the tau=0.9 calibration
                # claim, so it stays. up99/bias/undermag stay CSV diagnostics.
                _sel_in_box = (
                    not any(_math.isnan(_v) for _v in (_su90, _sskl))
                    and _su90 < cfg.sel_up90_cap
                    and _sr13_bias_ok(_sb90, cfg)
                )
                if _sel_in_gate and _sel_in_box:
                    selection_score = _sskl
                elif _sel_in_gate:
                    selection_score = 10.0 - float(_r2t)
                else:
                    selection_score = 100.0 - (0.0 if _math.isnan(_r2t) else float(_r2t))
            else:
                _sched_score = val_loss
                selection_score = val_loss

            if scheduler is not None:
                # patch 09f: LR schedule steps on the SAME signal as rounds 1-2
                # (-R2, or val_loss in the non-R2 branch) so training dynamics
                # stay ladder-comparable; only checkpoint selection is blended.
                scheduler.step(_sched_score)

            _r2_str  = " | ".join([f"val_gR2_{k}={v:.6f}" for k, v in val_metrics["global_R2"].items()])
            _sig_str = " ".join([f"sig_{t}={_sig_vals.get(t, float('nan')):.3f}" for t in _term_names])
            print(
                f"Epoch {epoch:03d} | "
                f"train_loss={train_total_loss:.6f} | grad_norm={grad_norm:.4f} | "
                f"val_loss={val_loss:.6f} | {_r2_str} | "
                f"val_MAE_tension={val_metrics['MAE'].get('tension', float('nan')):.2f} N | "
                f"underpred={val_metrics.get('peak_tension_underpred_rate', float('nan')):.1f}% | "
                f"{_sig_str}"
            )

            # -- append a completed row with val metrics to CSV ---------------
            _row.update({
                "val_loss":                        val_loss,
                "val_MAPE_tension":                val_metrics["MAPE"].get("tension", ""),
                "val_MAPE_tension_valid_frac":     val_metrics["MAPE"].get("tension_valid_frac", ""),
                "val_peak_tension_MAE":            val_metrics.get("peak_tension_MAE", ""),
                "val_peak_tension_RMSE":           val_metrics.get("peak_tension_RMSE", ""),
                "val_peak_tension_bias":           val_metrics.get("peak_tension_bias", ""),
                "val_peak_tension_underpred_rate": val_metrics.get("peak_tension_underpred_rate", ""),
                "val_peak_tension_MAPE":           val_metrics.get("peak_tension_MAPE", ""),
                "val_peak_tension_p95_abs_error":  val_metrics.get("peak_tension_p95_abs_error", ""),
                "val_peak_tension_max_abs_error":  val_metrics.get("peak_tension_max_abs_error", ""),
                "val_peak_tension_thr_p90": val_metrics.get("peak_tension_thr_p90", ""),
                "val_peak_tension_underpred_rate_p90": val_metrics.get("peak_tension_underpred_rate_p90", ""),
                "val_peak_tension_bias_p90": val_metrics.get("peak_tension_bias_p90", ""),
                "val_peak_tension_MAE_p90": val_metrics.get("peak_tension_MAE_p90", ""),
                "val_peak_tension_undermag_p90": val_metrics.get("peak_tension_undermag_p90", ""),
                "val_peak_tension_undermag_p99": val_metrics.get("peak_tension_undermag_p99", ""),
                "val_peak_tension_thr_p99": val_metrics.get("peak_tension_thr_p99", ""),
                "val_peak_tension_underpred_rate_p99": val_metrics.get("peak_tension_underpred_rate_p99", ""),
                "val_peak_tension_bias_p99": val_metrics.get("peak_tension_bias_p99", ""),
                "val_peak_tension_MAE_p99": val_metrics.get("peak_tension_MAE_p99", ""),
                "val_contact_grounded_R2_median": val_metrics.get("contact_grounded_R2_median", ""),
                "val_contact_grounded_R2_fracneg": val_metrics.get("contact_grounded_R2_fracneg", ""),
                "val_contact_grounded_n": val_metrics.get("contact_grounded_n", ""),
                "val_contact_touchdown_R2_median": val_metrics.get("contact_touchdown_R2_median", ""),
                "val_contact_touchdown_R2_fracneg": val_metrics.get("contact_touchdown_R2_fracneg", ""),
                "val_contact_touchdown_n": val_metrics.get("contact_touchdown_n", ""),
                "val_contact_suspended_R2_median": val_metrics.get("contact_suspended_R2_median", ""),
                "val_contact_suspended_R2_fracneg": val_metrics.get("contact_suspended_R2_fracneg", ""),
                "val_contact_suspended_n": val_metrics.get("contact_suspended_n", ""),
                **{f"val_MAE_{n}":       val_metrics["MAE"][n]       for n in target_names},
                **{f"val_RMSE_{n}":      val_metrics["RMSE"][n]      for n in target_names},
                **{f"val_R2_{n}":        val_metrics["R2"][n]        for n in target_names},
                **{f"val_global_R2_{n}": val_metrics["global_R2"][n] for n in target_names},
            })
            for n in target_names:
                _row[f"val_temporal_diff_MAE_{n}"]      = val_metrics.get(f"temporal_diff_MAE_{n}", "")
                _row[f"val_temporal_diff_MAE_flat_{n}"] = val_metrics.get(f"temporal_diff_MAE_flat_{n}", "")
                _row[f"val_temporal_diff_skill_{n}"]    = val_metrics.get(f"temporal_diff_skill_{n}", "")
            _row.update({f"val_term_{t}": val_metrics.get(f"loss_term_{t}", "") for t in _term_names})
            for _nc_key, _nc_m in _nc_out.items():
                _nc_row = {
                    "epoch": epoch, "split": "val", "node_count": _nc_key,
                    "MAE_tension":       _nc_m["MAE"].get("tension", ""),
                    "RMSE_tension":      _nc_m["RMSE"].get("tension", ""),
                    "R2_tension":        _nc_m["R2"].get("tension", ""),
                    "global_R2_tension": _nc_m["global_R2"].get("tension", ""),
                    "MAPE_tension":               _nc_m["MAPE"].get("tension", ""),
                    "peak_tension_MAE":           _nc_m.get("peak_tension_MAE", ""),
                    "peak_tension_bias":          _nc_m.get("peak_tension_bias", ""),
                    "peak_tension_underpred_rate":_nc_m.get("peak_tension_underpred_rate", ""),
                    "peak_tension_MAPE":          _nc_m.get("peak_tension_MAPE", ""),
                    "peak_tension_p95_abs_error": _nc_m.get("peak_tension_p95_abs_error", ""),
                    "peak_tension_max_abs_error": _nc_m.get("peak_tension_max_abs_error", ""),
                    "peak_tension_thr_p90": _nc_m.get("peak_tension_thr_p90", ""),
                    "peak_tension_underpred_rate_p90": _nc_m.get("peak_tension_underpred_rate_p90", ""),
                    "peak_tension_bias_p90": _nc_m.get("peak_tension_bias_p90", ""),
                    "peak_tension_MAE_p90": _nc_m.get("peak_tension_MAE_p90", ""),
                    "peak_tension_undermag_p90": _nc_m.get("peak_tension_undermag_p90", ""),
                    "peak_tension_thr_p99": _nc_m.get("peak_tension_thr_p99", ""),
                    "peak_tension_underpred_rate_p99": _nc_m.get("peak_tension_underpred_rate_p99", ""),
                    "peak_tension_bias_p99": _nc_m.get("peak_tension_bias_p99", ""),
                    "peak_tension_MAE_p99": _nc_m.get("peak_tension_MAE_p99", ""),
                    "peak_tension_undermag_p99": _nc_m.get("peak_tension_undermag_p99", ""),
                    "temporal_diff_MAE_tension":      _nc_m.get("temporal_diff_MAE_tension", ""),
                    "temporal_diff_MAE_flat_tension": _nc_m.get("temporal_diff_MAE_flat_tension", ""),
                    "temporal_diff_skill_tension":    _nc_m.get("temporal_diff_skill_tension", ""),
                }
                _nc_writer.writerow(_nc_row)
            _nc_file.flush()

            _row["selection_score"] = selection_score
            _csv_writer.writerow(_row)
            _csv_file.flush()

            improved = (best_selection_score - selection_score) > cfg.min_delta
            if improved:
                best_selection_score       = selection_score
                best_val_loss_at_selection = val_loss
                best_state           = copy.deepcopy(model.state_dict())
                best_criterion_state = copy.deepcopy(criterion.state_dict())
                torch.save(_ckpt_payload(epoch), best_model_path)
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            # -- periodic checkpoint (save_every_n_epochs) ---------------
            if cfg.save_every_n_epochs > 0 and epoch % cfg.save_every_n_epochs == 0:
                _periodic_path = checkpoint_dir / f"checkpoint_epoch{epoch:04d}.pt"
                torch.save(_ckpt_payload(epoch), _periodic_path)
                print(f"  Periodic checkpoint saved: {_periodic_path}")

            # -- feasible-only checkpoint (user 2026-07-19) --------------
            # Write a checkpoint ONLY when every selection criterion is
            # met (Gate-A running-max R2 + the full safety box). Every
            # file on disk is then a valid pub_select_epoch.py pick
            # candidate; infeasible epochs are skipped (disk-lean vs
            # save_every_n_epochs, which writes ~2-3x more files).
            if getattr(cfg, "save_feasible_checkpoints", False) and _sel_in_gate and _sel_in_box:
                _feas_path = checkpoint_dir / f"checkpoint_epoch{epoch:04d}.pt"
                torch.save(_ckpt_payload(epoch), _feas_path)
                _um90 = float(val_metrics.get("peak_tension_undermag_p90", float("nan")))
                _um99 = float(val_metrics.get("peak_tension_undermag_p99", float("nan")))
                print(f"  Feasible checkpoint saved (ep{epoch:03d}): "
                      f"skill={_sskl:.4f} up90={_su90:.1f}% up99={_su99:.1f}% "
                      f"| diag b90={_sb90:+.0f} b99={_sb99:+.0f} "
                      f"undermag90={_um90:.0f} undermag99={_um99:.0f}")
                _n_feasible_epochs += 1

            # -- ROLLING checkpoint (patch sr-13) ------------------------
            # The feasible-only policy above can leave long stretches with
            # NO checkpoint (HPO_WARM job 124311 saved nothing after ep37
            # of 60), so a walltime timeout could resume tens of epochs
            # earlier and the run might never advance.  This bounds the
            # loss to one epoch.  Atomic tmp+replace so a SIGKILL mid-save
            # cannot truncate the resume anchor.  The name deliberately does
            # NOT match checkpoint_epoch*.pt, so the per-checkpoint ts-dump
            # loop is unaffected.
            if getattr(cfg, "save_latest_checkpoint", False):
                _lt = checkpoint_dir / "checkpoint_latest.pt.tmp"
                torch.save(_ckpt_payload(epoch), _lt)
                _lt.replace(checkpoint_dir / "checkpoint_latest.pt")

            if epochs_without_improvement >= cfg.early_stopping_patience:
                print(f"Early stopping triggered at epoch {epoch}.")
                break

            # patch 10: loud warning if the safety gate is still infeasible.
            # Silence here is what let the FINAL run ship its worst-safety epoch.
            if epoch == 15 and _n_feasible_epochs == 0:
                print("  [WARN] no feasible epoch after 15 epochs -- the safety "
                      "gate (up90 < %.1f%%) may be unreachable; only the fallback "
                      "best-R2 checkpoint would exist." % (cfg.sel_up90_cap,))

            # patch 25: HARD ABORT. A warning nobody acts on is exactly why job
            # 10521852 ran all 100 epochs with 0 feasible and then shipped the
            # safety-blind 10-R2 fallback. If the gate is still empty at
            # cfg.sel_abort_epoch, kill the run (~2 h) instead of at 16 h. The
            # CSVs are flushed every epoch, so metrics_history.csv survives and
            # shows exactly where up90/up99 sat. Read from val_metrics (not the
            # _s* locals) so this is safe when select_on_r2_tension is False.
            _abort_ep = int(getattr(cfg, "sel_abort_epoch", 0) or 0)
            if _abort_ep > 0 and epoch >= _abort_ep and _n_feasible_epochs == 0:
                _a90 = float(val_metrics.get("peak_tension_underpred_rate_p90", float("nan")))
                _a99 = float(val_metrics.get("peak_tension_underpred_rate_p99", float("nan")))
                _ar2 = float(val_metrics.get("global_R2", {}).get("tension", float("nan")))
                _csv_file.flush()
                _nc_file.flush()
                raise RuntimeError(
                    "[ABORT] no feasible epoch by epoch %d (Gate A + up90 < %.1f%%). "
                    "Latest epoch: up90=%.1f%% up99=%.1f%% R2=%.5f. Training stopped "
                    "so the run cannot ship a safety-blind best-R2 checkpoint. "
                    "Inspect metrics_history.csv in %s."
                    % (epoch, cfg.sel_up90_cap, _a90, _a99, _ar2, str(checkpoint_dir))
                )

        else:
            history["val_loss"].append(None)
            history["val_metrics"].append(None)
            # Write base row with empty val fields for skipped epochs
            _csv_writer.writerow(_row)
            _csv_file.flush()
            print(
                f"Epoch {epoch:03d} | "
                f"train_loss={train_total_loss:.6f} | "
                f"grad_norm={grad_norm:.4f} | amp_scale={amp_scale:.0f} | val: skipped"
            )

    _csv_file.close()
    _nc_file.close()

    # export CSV -> XLSX for direct cluster inspection (requires openpyxl)
    try:
        import pandas as _pd
        _pd.read_csv(history_path).to_excel(history_path.with_suffix('.xlsx'), index=False)
        _pd.read_csv(_nc_path).to_excel(_nc_path.with_suffix('.xlsx'), index=False)
        print(f"[xlsx] exported {history_path.with_suffix('.xlsx')} and {_nc_path.with_suffix('.xlsx')}")
    except Exception as _xlsx_err:
        print(f"[warn] XLSX export failed: {_xlsx_err}")

    if best_state is not None:
        model.load_state_dict(best_state)
    if best_criterion_state is not None:
        criterion.load_state_dict(best_criterion_state)

    return model, history, str(best_model_path)


In [ ]:
# -------------------------------------------------------------
# test evaluation
# -------------------------------------------------------------


@torch.no_grad()
def test_model(
    model,
    test_loader,
    criterion,
    device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
    cfg=None,
    tag: str = "",          # patch sr-18: suffixes every output file
):
    _nc_out_test = {}
    _pw_rows = []           # patch sr-18: per-window sufficient stats
    test_loss, test_metrics = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
        target_norm=target_norm,
        target_names=target_names,
        per_nc_metrics=_nc_out_test,
        per_window_rows=_pw_rows,
        cfg=cfg,
    )

    print("\n----- TEST RESULTS -----")
    print(f"test_loss = {test_loss:.6f}  (Kendall objective — NOT a quality metric)")

    for metric_name in ["MAE", "RMSE", "R2"]:
        for k, v in test_metrics[metric_name].items():
            print(f"test_{metric_name}_{k} = {v:.6f}")

    if "global_R2" in test_metrics:
        for _k, _v in test_metrics["global_R2"].items():
            print(f"test_global_R2_{_k} = {_v:.6f}")

    if "tension" in test_metrics["MAPE"]:
        print(f"test_MAPE_tension = {test_metrics['MAPE']['tension']:.6f}")

    for _k in ("peak_tension_MAE", "peak_tension_RMSE",
               "peak_tension_bias", "peak_tension_underpred_rate",
               "peak_tension_MAPE", "peak_tension_p95_abs_error",
               "peak_tension_max_abs_error",
               "peak_tension_thr_p90", "peak_tension_underpred_rate_p90",
               "peak_tension_bias_p90", "peak_tension_MAE_p90",
               "peak_tension_thr_p99", "peak_tension_underpred_rate_p99",
               "peak_tension_bias_p99", "peak_tension_MAE_p99",
               "peak_tension_undermag_p90", "peak_tension_undermag_p99"):
        if _k in test_metrics:
            print(f"test_{_k} = {test_metrics[_k]:.6f}")

    # Temporal tracking vs the flat-line baseline (skill < 1 = genuine tracking)
    for _tname in target_names:
        for _kk in (f"temporal_diff_MAE_{_tname}",
                    f"temporal_diff_MAE_flat_{_tname}",
                    f"temporal_diff_skill_{_tname}"):
            if _kk in test_metrics:
                print(f"test_{_kk} = {test_metrics[_kk]:.6f}")

    # Unweighted loss-term means (diagnostic)
    for _kk in sorted(k for k in test_metrics if k.startswith("loss_term_")):
        print(f"test_{_kk} = {test_metrics[_kk]:.6f}")

    # -- persist test metrics to CSV (+ XLSX), mirroring the train/val files --
    if cfg is not None and getattr(cfg, "checkpoint_dir", None):
        from pathlib import Path as _Path
        import csv as _csv_t
        _out_dir = _Path(cfg.checkpoint_dir)
        _out_dir.mkdir(parents=True, exist_ok=True)
        _tn = list(target_names)
        _g = test_metrics.get("global_R2", {})
        # aggregate: tidy (metric, value), full val-style set with a test_ prefix
        _rows = [("test_loss", test_loss)]
        for _n in _tn: _rows.append((f"test_MAE_{_n}", test_metrics["MAE"].get(_n, "")))
        for _n in _tn: _rows.append((f"test_RMSE_{_n}", test_metrics["RMSE"].get(_n, "")))
        for _n in _tn: _rows.append((f"test_R2_{_n}", test_metrics["R2"].get(_n, "")))
        for _n in _tn: _rows.append((f"test_global_R2_{_n}", _g.get(_n, "")))
        _rows.append(("test_MAPE_tension", test_metrics["MAPE"].get("tension", "")))
        _rows.append(("test_MAPE_tension_valid_frac", test_metrics["MAPE"].get("tension_valid_frac", "")))
        for _k in ("peak_tension_MAE", "peak_tension_RMSE", "peak_tension_bias",
                   "peak_tension_underpred_rate", "peak_tension_MAPE",
                   "peak_tension_p95_abs_error", "peak_tension_max_abs_error",
                   "peak_tension_thr_p90", "peak_tension_underpred_rate_p90",
                   "peak_tension_bias_p90", "peak_tension_MAE_p90",
                   "peak_tension_thr_p99", "peak_tension_underpred_rate_p99",
                   "peak_tension_bias_p99", "peak_tension_MAE_p99",
                   "peak_tension_undermag_p90", "peak_tension_undermag_p99"):
            _rows.append((f"test_{_k}", test_metrics.get(_k, "")))
        for _rg in ("grounded", "touchdown", "suspended"):
            for _sfx in ("R2_median", "R2_fracneg", "n"):
                _rows.append((f"test_contact_{_rg}_{_sfx}",
                              test_metrics.get(f"contact_{_rg}_{_sfx}", "")))
        for _n in _tn:
            _rows.append((f"test_temporal_diff_MAE_{_n}",
                          test_metrics.get(f"temporal_diff_MAE_{_n}", "")))
            _rows.append((f"test_temporal_diff_MAE_flat_{_n}",
                          test_metrics.get(f"temporal_diff_MAE_flat_{_n}", "")))
            _rows.append((f"test_temporal_diff_skill_{_n}",
                          test_metrics.get(f"temporal_diff_skill_{_n}", "")))
        for _kk in sorted(k for k in test_metrics if k.startswith("loss_term_")):
            _rows.append((f"test_{_kk}", test_metrics.get(_kk, "")))

        # --- patch sr-18: per-window sufficient statistics  ----------------------
        # One row per scored window, keyed by (lc, case, N_in, out_N, start).
        # Offline this reproduces the pooled global_R2 for ANY subset from the
        # raw sums, and the window-averaged MAE / MAPE / skill the same way
        # _build_avg does -- which is what makes the wave-train repeat split
        # (overlap = clip(start + W - 12064, 0, W) > 0 on the development
        # sites) comparable with the published headline.
        _pw_fields = ['lc_id', 'case_id', 'n_in', 'out_N', 'start_idx', 'n_pts', 'ss_res', 'y_sum', 'y_sq', 'mae', 'rmse', 'r2', 'mape', 'mape_valid_frac', 'td_mae', 'td_mae_flat', 'td_skill', 'peak_MAE', 'peak_bias']
        _pw_csv = _out_dir / ("test_per_window_stats%s.csv" % tag)
        with open(_pw_csv, "w", newline="", encoding="utf-8") as _f:
            _w = _csv_t.DictWriter(_f, fieldnames=_pw_fields, extrasaction="ignore")
            _w.writeheader()
            _w.writerows(_pw_rows)
        print("[per-window] wrote %s (%d rows) to %s"
              % (_pw_csv.name, len(_pw_rows), _out_dir))
        _agg_csv = _out_dir / ("test_metrics%s.csv" % tag)
        with open(_agg_csv, "w", newline="", encoding="utf-8") as _f:
            _w = _csv_t.writer(_f); _w.writerow(["metric", "value"]); _w.writerows(_rows)
        # per-node-count: same columns as metrics_by_node_count.csv, split=test
        _nc_fields = [
            "split", "node_count",
            "MAE_tension", "RMSE_tension", "R2_tension", "global_R2_tension",
            "MAPE_tension",
            "peak_tension_MAE", "peak_tension_bias",
            "peak_tension_underpred_rate", "peak_tension_MAPE",
            "peak_tension_p95_abs_error", "peak_tension_max_abs_error",
            # patch 14: per-topology tail safety (quantile within each node count)
            "peak_tension_thr_p90", "peak_tension_underpred_rate_p90",
            "peak_tension_bias_p90", "peak_tension_MAE_p90", "peak_tension_undermag_p90",
            "peak_tension_thr_p99", "peak_tension_underpred_rate_p99",
            "peak_tension_bias_p99", "peak_tension_MAE_p99", "peak_tension_undermag_p99",
            "temporal_diff_MAE_tension", "temporal_diff_MAE_flat_tension",
            "temporal_diff_skill_tension",
        ]
        _nc_csv = _out_dir / ("test_metrics_by_node_count%s.csv" % tag)
        with open(_nc_csv, "w", newline="", encoding="utf-8") as _f:
            _w = _csv_t.DictWriter(_f, fieldnames=_nc_fields, extrasaction="ignore")
            _w.writeheader()
            for _nc_key in sorted(_nc_out_test.keys()):
                _m = _nc_out_test[_nc_key]
                _w.writerow({
                    "split": "test", "node_count": _nc_key,
                    "MAE_tension":       _m["MAE"].get("tension", ""),
                    "RMSE_tension":      _m["RMSE"].get("tension", ""),
                    "R2_tension":        _m["R2"].get("tension", ""),
                    "global_R2_tension": _m["global_R2"].get("tension", ""),
                    "MAPE_tension": _m["MAPE"].get("tension", ""),
                    "peak_tension_MAE":  _m.get("peak_tension_MAE", ""),
                    "peak_tension_bias": _m.get("peak_tension_bias", ""),
                    "peak_tension_underpred_rate": _m.get("peak_tension_underpred_rate", ""),
                    "peak_tension_MAPE":          _m.get("peak_tension_MAPE", ""),
                    "peak_tension_p95_abs_error": _m.get("peak_tension_p95_abs_error", ""),
                    "peak_tension_max_abs_error": _m.get("peak_tension_max_abs_error", ""),
                    "peak_tension_thr_p90": _m.get("peak_tension_thr_p90", ""),
                    "peak_tension_underpred_rate_p90": _m.get("peak_tension_underpred_rate_p90", ""),
                    "peak_tension_bias_p90": _m.get("peak_tension_bias_p90", ""),
                    "peak_tension_MAE_p90": _m.get("peak_tension_MAE_p90", ""),
                    "peak_tension_undermag_p90": _m.get("peak_tension_undermag_p90", ""),
                    "peak_tension_thr_p99": _m.get("peak_tension_thr_p99", ""),
                    "peak_tension_underpred_rate_p99": _m.get("peak_tension_underpred_rate_p99", ""),
                    "peak_tension_bias_p99": _m.get("peak_tension_bias_p99", ""),
                    "peak_tension_MAE_p99": _m.get("peak_tension_MAE_p99", ""),
                    "peak_tension_undermag_p99": _m.get("peak_tension_undermag_p99", ""),
                    "temporal_diff_MAE_tension":      _m.get("temporal_diff_MAE_tension", ""),
                    "temporal_diff_MAE_flat_tension": _m.get("temporal_diff_MAE_flat_tension", ""),
                    "temporal_diff_skill_tension":    _m.get("temporal_diff_skill_tension", ""),
                })
        try:
            import pandas as _pd_t
            _pd_t.read_csv(_agg_csv).to_excel(_agg_csv.with_suffix(".xlsx"), index=False)
            _pd_t.read_csv(_nc_csv).to_excel(_nc_csv.with_suffix(".xlsx"), index=False)
            print(f"[test-metrics] wrote {_agg_csv.name}, {_nc_csv.name} (+ xlsx) to {_out_dir}")
        except Exception as _xe:
            print(f"[test-metrics] CSV written; XLSX export failed: {_xe}")

    return {
        "test_loss": test_loss,
        "test_metrics": test_metrics,
    }


# -------------------------------------------------------------
# patch sr-04: Stage-2 N_in x N_out super-resolution grid
# -------------------------------------------------------------

# Per-CELL fields: exactly the columns metrics_by_node_count.csv carries, so a
# grid cell is directly comparable to a Stage-1 per-topology row.
SUPERRES_GRID_FIELDS = [
    "MAE_tension", "RMSE_tension", "R2_tension", "global_R2_tension",
    "MAPE_tension",
    "peak_tension_MAE", "peak_tension_bias", "peak_tension_underpred_rate",
    "peak_tension_MAPE", "peak_tension_p95_abs_error",
    "peak_tension_max_abs_error",
    "peak_tension_thr_p90", "peak_tension_underpred_rate_p90",
    "peak_tension_bias_p90", "peak_tension_MAE_p90", "peak_tension_undermag_p90",
    "peak_tension_thr_p99", "peak_tension_underpred_rate_p99",
    "peak_tension_bias_p99", "peak_tension_MAE_p99", "peak_tension_undermag_p99",
    "temporal_diff_MAE_tension", "temporal_diff_MAE_flat_tension",
    "temporal_diff_skill_tension",
]

# Contact-regime R2 is accumulated ONLY on the aggregate in evaluate() -- the
# per-node-count path never receives it -- so it appears in the pooled file and
# would be blank in every grid cell. Keeping the two lists separate avoids
# shipping a column that is empty by construction.
SUPERRES_POOLED_FIELDS = SUPERRES_GRID_FIELDS + [
    "contact_grounded_R2_median", "contact_grounded_R2_fracneg", "contact_grounded_n",
    "contact_touchdown_R2_median", "contact_touchdown_R2_fracneg", "contact_touchdown_n",
    "contact_suspended_R2_median", "contact_suspended_R2_fracneg", "contact_suspended_n",
]


def _grid_value(m, field, tname="tension"):
    """Read one metric out of an evaluate() metrics dict.

    evaluate() mixes two conventions: flat keys (peak_tension_*,
    temporal_diff_*, contact_*) and per-target nested dicts
    (m["global_R2"]["tension"]). Handle both, and return "" when absent rather
    than a None that would later fail float().
    """
    v = m.get(field)
    if isinstance(v, dict):
        v = v.get(tname)
    if v is None and field.endswith("_" + tname):
        parent = m.get(field[: -(len(tname) + 1)])
        if isinstance(parent, dict):
            v = parent.get(tname)
    return "" if v is None else v


def _base_window_subset(loader):
    """Unwrap NormalizedSubset -> MultiLoadCaseWindowSubset (owns out_N)."""
    ds = loader.dataset
    while hasattr(ds, "base_subset"):
        ds = ds.base_subset
    if not hasattr(ds, "out_N"):
        raise TypeError(
            "loader.dataset does not expose out_N -- patch sr-01 is not applied."
        )
    return ds


# --- patch sr-17: motion-sensor noise -- sweep-side helpers -------------------
def set_sensor_noise(loader, spec):
    """Install a noise spec on the loader's window subset (None = clean).

    Mirrors how evaluate_superres_grid mutates out_N. persistent_workers is
    False everywhere, so the next iterator re-forks its workers and they pick
    the spec up. Returns the previous spec so a caller can restore it.
    """
    base = _base_window_subset(loader)
    prev = getattr(base, "sensor_noise", None)
    base.sensor_noise = spec
    return prev


def sensor_noise_spec_absolute(level_id, name, sigma_p=0.0, sigma_v=0.0,
                               sigma_b=0.0):
    """A level in PHYSICAL units -- the instrument-grounded axis.

    sigma_p [m] white noise on the measured x and z; sigma_v [m/s] white noise
    on the reported velocities; sigma_b [m] a constant per-station bias.
    sigma_v = 0 gives the derivative-coupled sensor (the pessimistic bound).

    grafe2024virtual Table 6 quotes an rms heave accuracy of 0.05 m and an rms
    velocity accuracy of 0.05 m/s for a commercial GNSS-IMU, which is the
    anchor for this axis.
    """
    return {"id": int(level_id), "name": str(name), "axis": "absolute",
            "sigma_x": float(sigma_p),  "sigma_z": float(sigma_p),
            "sigma_vx": float(sigma_v), "sigma_vz": float(sigma_v),
            "sigma_bx": float(sigma_b), "sigma_bz": float(sigma_b)}


def sensor_noise_spec_snr(level_id, snr_db, node_norm, cfg):
    """A level in per-channel SNR [dB] -- the xie2026sparse-comparable axis.

    sigma_c = std_c * 10 ** (-SNR / 20), with std_c the channel's TRAINING-set
    standard deviation taken from the fitted normalizer. Two deliberate
    choices, both worth stating in the paper:

      * std, not root-mean-square, so "signal power" is the AC power about the
        mean. x_abs carries a large DC offset (tens of metres) that no sensor
        error scales with, and using it would inflate the noise absurdly.
      * a TRAIN statistic, not a per-window one, so the level is deterministic,
        reproducible and leaks nothing from the test set.

    Note this axis scales the noise by each channel's own variance, so it
    makes the sensors on a barely-moving grounded station almost perfect --
    exactly where the model is weakest. The absolute axis does not. The two
    arms therefore disagree by construction, and that asymmetry is a result.
    """
    idx  = list(node_norm.feature_idx)
    cols = {"sigma_x": 0, "sigma_z": 1, "sigma_vx": 2, "sigma_vz": 3}
    scale = 10.0 ** (-float(snr_db) / 20.0)
    spec = {"id": int(level_id), "name": "snr%gdB" % float(snr_db),
            "axis": "snr", "snr_db": float(snr_db),
            "sigma_bx": 0.0, "sigma_bz": 0.0}
    for key, col in cols.items():
        if col not in idx:
            raise ValueError("node column %d is not normalised -- cannot "
                             "derive an SNR level for it" % col)
        spec[key] = float(node_norm.std[idx.index(col)]) * scale
    return spec


def sensor_noise_report(spec, node_norm):
    """One line per channel: sigma, the equivalent SNR, and normalised sigma.

    Emitted for every level so the robustness table can be read against
    xie2026sparse (dB), yan2026dacla (normalised sigma) or grafe2024virtual
    (absolute rms) without re-deriving anything offline.
    """
    idx  = list(node_norm.feature_idx)
    cols = [("x_abs", "sigma_x", 0), ("z_abs", "sigma_z", 1),
            ("x_vel", "sigma_vx", 2), ("z_vel", "sigma_vz", 3)]
    out = []
    for cname, key, col in cols:
        sd = float(spec.get(key, 0.0) or 0.0)
        std = float(node_norm.std[idx.index(col)]) if col in idx else float("nan")
        snr = (20.0 * math.log10(std / sd)) if sd > 0.0 else float("inf")
        out.append((cname, sd, snr, (sd / std) if std else float("nan"), std))
    return out


@torch.no_grad()
def evaluate_superres_grid(
    model, test_loader, criterion, device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
    cfg=None,
    out_N_values=None,
    tag: str = "",
):
    """Score the model over the full N_in x N_out grid.

    N_in is whatever each bucket carries (the loader is bucketed by node
    count); N_out is swept by setting out_N on the underlying window subset and
    re-running the SAME evaluate() used for every Stage-1 number, so the grid
    and the headline metrics are computed by identical code.

    The DIAGONAL recovers the Stage-1 topology-generalisation result and is the
    built-in control. The UPPER TRIANGLE (N_in < N_out) is the new capability.

    WARNING when reading the output: peak quantiles are taken WITHIN each cell,
    so absolute levels differ between cells. Always quote peak_tension_thr_p90 /
    _thr_p99 alongside any underprediction rate.

    Writes superres_grid{tag}.csv into cfg.checkpoint_dir and returns the rows.
    """
    import csv as _csv
    import time as _time

    if out_N_values is None:
        out_N_values = getattr(
            cfg, "superres_grid_out_N", (4, 5, 6, 7, 8, 10, 12, 15, 18, 21)
        )
    out_N_values = [int(v) for v in out_N_values]

    base = _base_window_subset(test_loader)
    prev_out_N = base.out_N

    rows = []
    pooled = []
    try:
        for n_out in out_N_values:
            t0 = _time.time()
            base.out_N = n_out
            per_nc = {}
            _loss, _agg = evaluate(
                model=model, loader=test_loader, criterion=criterion,
                device=device, target_norm=target_norm,
                target_names=target_names,
                mape_min=(cfg.tension_mape_min if cfg is not None else 1000.0),
                per_nc_metrics=per_nc, cfg=cfg,
            )
            # pooled-over-N_in aggregate for this output resolution: the
            # headline "reconstruct N_out stations from any N_in" number
            prow = {"N_out": int(n_out)}
            for f in SUPERRES_POOLED_FIELDS:
                prow[f] = _grid_value(_agg, f)
            pooled.append(prow)
            for n_in in sorted(per_nc):
                m = per_nc[n_in]
                row = {"N_in": int(n_in), "N_out": int(n_out)}
                for f in SUPERRES_GRID_FIELDS:
                    row[f] = _grid_value(m, f)
                rows.append(row)
            print(f"[grid] N_out={n_out:2d}: {len(per_nc)} N_in cells "
                  f"in {_time.time() - t0:.0f} s")
    finally:
        base.out_N = prev_out_N

    if cfg is not None and getattr(cfg, "checkpoint_dir", None):
        from pathlib import Path as _P
        out_dir = _P(cfg.checkpoint_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        path = out_dir / f"superres_grid{tag}.csv"
        with open(path, "w", newline="", encoding="utf-8") as fh:
            w = _csv.DictWriter(fh, fieldnames=["N_in", "N_out"] + SUPERRES_GRID_FIELDS,
                                extrasaction="ignore")
            w.writeheader()
            w.writerows(rows)
        print(f"[grid] wrote {path}  ({len(rows)} cells)")

        ppath = out_dir / f"superres_grid_pooled{tag}.csv"
        with open(ppath, "w", newline="", encoding="utf-8") as fh:
            w = _csv.DictWriter(fh, fieldnames=["N_out"] + SUPERRES_POOLED_FIELDS,
                                extrasaction="ignore")
            w.writeheader()
            w.writerows(pooled)
        print(f"[grid] wrote {ppath}  ({len(pooled)} rows, pooled over N_in)")

        # compact console view of the headline surface
        print()
        print("[grid] global_R2_tension   rows = N_in, cols = N_out")
        n_ins = sorted({r["N_in"] for r in rows})
        n_outs = sorted({r["N_out"] for r in rows})
        look = {(r["N_in"], r["N_out"]): r for r in rows}
        print("  N_in\\N_out " + "".join(f"{c:>9d}" for c in n_outs))
        for a in n_ins:
            cells = []
            for b in n_outs:
                r = look.get((a, b))
                try:
                    cells.append(f"{float(r['global_R2_tension']):9.5f}")
                except (TypeError, ValueError):
                    cells.append("        -")
            print(f"  {a:>10d} " + "".join(cells))

    return rows


In [ ]:
# -------------------------------------------------------------
# patch sr-05: interpolation baseline (Stage-1 model + resample)
# -------------------------------------------------------------
#
# THE control for Stage 2. Run the shipped matched-resolution model at N_in and
# push its predicted tension to the N_out stations by arc-length interpolation.
# The query decoder must beat this surface, cell by cell, or the capability
# reduces to interpolation.


class InterpolatedStage1Model(nn.Module):
    """Matched-resolution model whose OUTPUT is interpolated to the queries.

    Wrapping instead of special-casing the evaluator means the baseline goes
    through exactly the same evaluate() / evaluate_superres_grid() /
    dump_test_timeseries() code as the Stage-2 model, on the same paired
    windows -- so the two surfaces subtract cell by cell.

    Note this interpolates the PREDICTED TENSION, whereas the query decoder's
    residual skip interpolates the node EMBEDDINGS. The two coincide only if
    the LSTM and head were linear, which they are not; the difference between
    the surfaces is exactly what the decoder buys.
    """

    def __init__(self, stage1_model: nn.Module):
        super().__init__()
        if getattr(stage1_model, "query_decoder", None) is not None:
            raise ValueError(
                "InterpolatedStage1Model expects a MATCHED-resolution model "
                "(use_query_decoder=False); it was given one that already has "
                "a query decoder."
            )
        self.stage1_model = stage1_model

    def forward(self, graph_or_graphs, x_seq, edge_seq, query_graph_or_graphs=None):
        y = self.stage1_model(graph_or_graphs, x_seq, edge_seq)   # [..., W, N_in, T]
        if query_graph_or_graphs is None:
            return y
        q = query_graph_or_graphs
        n_out = (q[0] if isinstance(q, (list, tuple)) else q).x.size(0)
        return interp_nodes(y, n_out)                             # [..., W, N_out, T]


def build_interpolation_baseline(example_dataset, cfg, device, checkpoint_path):
    """Build the Stage-1 model, load `checkpoint_path` STRICTLY, and wrap it.

    strict=True is deliberate and load-bearing: it fails loudly if this
    notebook's Stage-1 path has drifted from the one that produced the
    checkpoint, which is the integrity check the whole baseline rests on.
    """
    import copy as _copy

    base_cfg = _copy.deepcopy(cfg)
    base_cfg.use_query_decoder = False
    base_cfg.warm_start_checkpoint = None      # loaded strictly below instead

    model = build_model_from_config(
        example_dataset=example_dataset, cfg=base_cfg, device=device
    )
    assert getattr(model, "query_decoder", None) is None, (
        "baseline model must have no query decoder"
    )

    try:
        ck = torch.load(checkpoint_path, map_location=device, weights_only=False)
    except TypeError:
        ck = torch.load(checkpoint_path, map_location=device)
    sd = ck.get("model_state_dict", ck)

    # A Stage-2 checkpoint here is an easy mistake to make and produces an
    # opaque 20-key state_dict error at the very END of a long run. Name it.
    dec_keys = [k for k in sd if k.startswith("query_decoder.")]
    if dec_keys:
        raise ValueError(
            f"{checkpoint_path} is a STAGE-2 checkpoint ({len(dec_keys)} "
            f"query_decoder.* tensors). The interpolation baseline needs a "
            f"MATCHED-resolution Stage-1 checkpoint, e.g. "
            f"checkpoints_P1_matched_v5/best_mooring_gat_lstm_seed_42.pt -- "
            f"check cfg.superres_baseline_checkpoint."
        )

    model.load_state_dict(sd, strict=True)     # raises on ANY other mismatch
    model.eval()

    n_par = sum(p.numel() for p in model.parameters())
    print(f"[baseline] {checkpoint_path}")
    print(f"[baseline]   checkpoint epoch : {ck.get('epoch')}")
    print(f"[baseline]   tensors loaded   : {len(sd)} (strict=True)")
    print(f"[baseline]   parameters       : {n_par:,}")
    print("[baseline]   strict load OK -> this notebook's Stage-1 forward path "
          "is unchanged from the run that produced the checkpoint")
    del ck
    return InterpolatedStage1Model(model).to(device)


@torch.no_grad()
def run_interpolation_baseline(
    example_dataset, cfg, device, checkpoint_path,
    test_loader, criterion, target_norm, target_names,
    out_N_values=None,
    tag="_baseline",   # patch sr-19: per-level tag so a sweep does not overwrite
):
    """Score the interpolation baseline over the full N_in x N_out grid.

    Writes superres_grid_baseline.csv next to the decoder's superres_grid.csv.
    Costs one evaluation pass per N_out value and needs no training.
    """
    print()
    print("=" * 70)
    print("INTERPOLATION BASELINE — Stage-1 model at N_in, prediction resampled "
          "to N_out")
    print("=" * 70)
    model = build_interpolation_baseline(
        example_dataset=example_dataset, cfg=cfg, device=device,
        checkpoint_path=checkpoint_path,
    )
    rows = evaluate_superres_grid(
        model=model, test_loader=test_loader, criterion=criterion,
        device=device, target_norm=target_norm, target_names=target_names,
        cfg=cfg, out_N_values=out_N_values, tag=tag,
    )
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return rows


def compare_superres_grids(decoder_rows, baseline_rows, cfg=None,
                           metric="global_R2_tension", higher_is_better=True):
    """patch sr-05: decoder-minus-baseline delta surface, printed and saved.

    A POSITIVE delta means the decoder beats interpolation on that cell (the
    sign is flipped automatically for lower-is-better metrics, so positive
    always means better).
    """
    import csv as _csv

    def _key(rows):
        return {(int(r["N_in"]), int(r["N_out"])): r for r in rows}

    dec, bas = _key(decoder_rows), _key(baseline_rows)
    cells = sorted(set(dec) & set(bas))
    sgn = 1.0 if higher_is_better else -1.0

    out = []
    for (a, b) in cells:
        try:
            d = float(dec[(a, b)][metric])
            v = float(bas[(a, b)][metric])
        except (TypeError, ValueError, KeyError):
            continue      # metric blank in one of the two grids -- skip the cell
        out.append({"N_in": a, "N_out": b, "decoder": d, "baseline": v,
                    "delta_better": sgn * (d - v)})

    n_ins = sorted({r["N_in"] for r in out})
    n_outs = sorted({r["N_out"] for r in out})
    look = {(r["N_in"], r["N_out"]): r for r in out}
    print()
    print(f"[grid] DECODER minus BASELINE on {metric} "
          f"(positive = decoder better)")
    print("  N_in\\N_out " + "".join(f"{c:>11d}" for c in n_outs))
    for a in n_ins:
        line = []
        for b in n_outs:
            r = look.get((a, b))
            line.append("          -" if r is None
                        else f"{r['delta_better']:>+11.5f}")
        print(f"  {a:>10d} " + "".join(line))

    n_win = sum(1 for r in out if r["delta_better"] > 0)
    n_up = sum(1 for r in out if r["N_in"] < r["N_out"])
    n_up_win = sum(1 for r in out if r["N_in"] < r["N_out"] and r["delta_better"] > 0)
    print(f"[grid] decoder wins {n_win}/{len(out)} cells overall, "
          f"{n_up_win}/{n_up} in the SUPER-RESOLUTION upper triangle")

    if cfg is not None and getattr(cfg, "checkpoint_dir", None):
        from pathlib import Path as _P
        p = _P(cfg.checkpoint_dir) / f"superres_grid_delta_{metric}.csv"
        with open(p, "w", newline="", encoding="utf-8") as fh:
            w = _csv.DictWriter(
                fh, fieldnames=["N_in", "N_out", "decoder", "baseline", "delta_better"]
            )
            w.writeheader()
            w.writerows(out)
        print(f"[grid] wrote {p}")
    return out


In [ ]:
# -------------------------------------------------------------
# test-set prediction time-series dump (patches 11 / 13 / 15)
# -------------------------------------------------------------
# Runs on the cluster right after test_model(). Writes a SMALL .npz so the
# figures can be re-plotted locally without rebuilding the dataset or the model.
#   patch 13: one dump per feasible checkpoint (tag=), sampler epoch pinned so
#             every checkpoint is scored on the IDENTICAL windows.
#   patch 15: per-TOPOLOGY window selection, so every node count is represented.


@torch.no_grad()
def dump_test_timeseries(
    model,
    test_loader,
    device,
    target_norm,
    target_names,
    cfg,
    n_extreme: int = 8,
    n_reservoir: int = 12,
    n_nc_extreme: int = 3,
    n_nc_reservoir: int = 2,
    n_snap: int = 8,
    n_nc_snap: int = 2,
    site_windows=((10, 11), (4,)),   # --- patch sr-21: (sites, node counts) to target
    n_site_clean: int = 24,          # snap-free windows per (site, node count)
    n_site_snap: int = 12,           # snap-bearing windows per (site, node count)
    max_windows: int = None,
    tag: str = "",
    make_plots: bool = True,
    pin_sampler_epoch: int = 0,
):
    """Collect predicted vs true tension time series on the held-out test set.

    Returns the output directory, or None if anything failed (never raises --
    this runs after a multi-hour training job and must not destroy its results).
    """
    from pathlib import Path as _P

    import numpy as _np

    try:
        t_idx = list(target_names).index("tension")
    except ValueError:
        print("[ts-dump] no 'tension' target -- skipped")
        return None

    out_dir = _P(cfg.checkpoint_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    model.eval()
    rng = _np.random.default_rng(42)

    peak_true_all, peak_pred_all, meta_all = [], [], []
    # patch 23: all-node line peak (a snap often peaks at an interior
    # node, so a fairlead-only peak can miss it entirely)
    lpeak_true_all, lpeak_pred_all = [], []
    # patch 23: per catalogued snap event inside a scored window --
    # (lc, case, node_count, window_start, event_t) + line max at event_t
    snap_ev, snap_true, snap_pred = [], [], []
    # --- patch sr-20: the whole station row around each snap event, so
    # capture can be read at the snap's OWN station (and with a +-2-sample
    # tolerance), not only as the line maximum. NaN-padded to 21 stations.
    _SR20_OFF = (-2, -1, 0, 1, 2)
    _SR20_MAXN = 21
    snap_nbhd_true, snap_nbhd_pred, snap_row_nout = [], [], []
    hi_sel_d, un_sel_d, reservoir = {}, {}, []    # pooled across topologies
    nc_hi, nc_un, nc_res, nc_seen = {}, {}, {}, {}  # per node count
    all_events = set()        # patch 29: distinct events seen, for the log
    # patch 27: reservoir over SNAP-CONTAINING windows. hi/un are pure tail
    # selectors (measured on the v4 dump: 76/76 of their windows contain a
    # snap), so without this the only snap traces available are the most
    # extreme ones -- no representative example exists.
    snap_res, snap_seen = [], 0
    nc_snap, nc_snap_seen = {}, {}
    # --- patch sr-21: reservoirs per (site, node count, class). The pooled and
    # per-node-count selectors above are blind to SITE, so a site that is a small
    # share of the draw is represented only by whatever the tail selectors pick --
    # which for loc10 is four windows from two simulations, all carrying the
    # out-pulling residue, and for loc11 a single snap-bearing window. Both classes
    # reject any window holding a masked artifact sample or an out-pulling sample,
    # so every window stored here is clean by construction.
    _SITE_LOCS, _SITE_NCS = (set(site_windows[0]), set(site_windows[1])) if site_windows else (set(), set())
    site_res, site_seen = {}, {}
    tiebreak = 0
    n_seen = 0

    def _payload(true_w, pred_w, lc, case, nc, start, evrows=()):
        # float32 [W, N] -- ~84 KB per array at N=21
        return {
            "true": true_w.astype(_np.float32),
            "pred": pred_w.astype(_np.float32),
            "lc_id": int(lc), "case_id": int(case),
            "node_count": int(nc), "start_idx": int(start),
            # patch 27: snap-event row indices WITHIN this window, so the
            # spike can be annotated without re-deriving the snap_ev join
            "evrows": _np.asarray(sorted(evrows), dtype=_np.int32),
        }

    _EVQ = 3          # decimals kept in the float part of the event key

    # patch 29: DE-DUPLICATE BY PHYSICAL EVENT.
    # Any window start in [t-999, t] covers a peak at absolute sample t, and the
    # test set draws ~5k windows per node count from the same case pool, so ONE
    # FE event is drawn at many offsets AND many topologies. The old bounded
    # heap ranked WINDOWS, so a single event swept an entire 8-panel grid
    # (measured on the v5 dump: ts_highest_peaks = 8 panels, 2 distinct events;
    # 50,000 test windows carry only 20,588 distinct events).
    #
    # Event key = (lc, case, true fairlead peak). The fairlead is 21-node index
    # 20 at EVERY topology, so arc-length resampling leaves its peak bit-
    # identical across duplicates; this key was verified to give exactly the
    # same 66 groups as the absolute-sample key on the v5 dump.
    #
    # The kept representative is the window whose peak sits FURTHEST FROM EITHER
    # EDGE. The model is bidirectional, so an event near a window boundary loses
    # one side of its context -- measured at 8.5 kN vs 11.3 kN for the same
    # event at sample 992/1000 versus mid-window.
    def _push(sel, key, pl, cap, evkey, cent):
        """Bounded top-`cap` selection over DISTINCT physical events."""
        nonlocal tiebreak
        slot = sel.get(evkey)
        if slot is not None:                 # same event, a different window
            slot["n"] += 1
            if key > slot["key"]:
                slot["key"] = key            # only `miss` varies within an event
            if cent > slot["cent"]:
                slot["cent"], slot["pl"] = cent, pl
            return
        tiebreak += 1
        sel[evkey] = {"key": key, "cent": cent, "pl": pl, "n": 1, "o": tiebreak}
        # Hold an over-cap pool so an event evicted early cannot be lost when a
        # later window of it raises its score. Trimmed to `cap` in _ranked().
        if len(sel) > cap * 3 + 4:
            worst = min(sel, key=lambda k: (sel[k]["key"], -sel[k]["o"]))
            del sel[worst]

    # Pin the batch sampler's shuffle epoch for the duration of this pass.
    # BucketedNodeCountBatchSampler.__iter__ seeds rng with (seed + epoch) and
    # then increments epoch, so consecutive passes reshuffle. Under a bounded
    # max_windows that would give every checkpoint a DIFFERENT subset and make
    # the per-checkpoint dumps incomparable. Restored right after the loop.
    _bsamp = getattr(test_loader, "batch_sampler", None)
    _prev_ep = getattr(_bsamp, "epoch", None)
    if pin_sampler_epoch is not None and _prev_ep is not None:
        _bsamp.epoch = int(pin_sampler_epoch)

    for sample in test_loader:
        sample = move_sample_to_device(sample, device)
        y_hat = model_forward(model, sample)   # --- patch sr-03:
        y_true = sample["y_seq"]

        if y_hat.dim() == 4:
            hats  = [y_hat[b]  for b in range(y_hat.shape[0])]
            trues = [y_true[b] for b in range(y_hat.shape[0])]
            lcs    = sample["lc_id"]
            cases  = sample["case_id"]
            starts = sample["start_idx"]
        else:
            hats, trues = [y_hat], [y_true]
            lcs    = [sample["lc_id"]]
            cases  = [sample["case_id"]]
            starts = [sample["start_idx"]]
        nc = int(sample.get("node_count", hats[0].shape[1]))

        for k, (yh, yt) in enumerate(zip(hats, trues)):
            # physical units via the SAME inverse transform the metrics use
            ph = target_norm.inverse_transform(yh.detach().cpu().float())
            pt = target_norm.inverse_transform(yt.detach().cpu().float())
            pred_w = ph[..., t_idx].numpy()     # [W, N]
            true_w = pt[..., t_idx].numpy()

            fl_true = float(true_w[:, -1].max())   # fairlead = last node
            fl_pred = float(pred_w[:, -1].max())
            miss    = fl_true - fl_pred            # > 0 = underprediction
            peak_true_all.append(fl_true)
            peak_pred_all.append(fl_pred)
            meta_all.append((int(lcs[k]), int(cases[k]), nc,
                             int(starts[k])))
            lpeak_true_all.append(float(true_w.max()))
            lpeak_pred_all.append(float(pred_w.max()))
            # snap-event capture: line max at the exact event timestep
            _rows = []                                   # patch 27
            _evs = SNAP_EVENT_TIMES.get((int(lcs[k]), int(cases[k])))
            if _evs:
                _s = int(starts[k])
                _e = _s + int(true_w.shape[0])
                for _t in _evs:
                    if _s <= _t < _e:
                        _r = int(_t) - _s
                        _rows.append(_r)                 # patch 27
                        snap_ev.append((int(lcs[k]), int(cases[k]), nc,
                                        _s, int(_t)))
                        snap_true.append(float(true_w[_r].max()))
                        snap_pred.append(float(pred_w[_r].max()))
                        # --- patch sr-20: every station, event sample +-2
                        _nst20 = int(true_w.shape[1])
                        assert _nst20 <= _SR20_MAXN, _nst20
                        _tt20 = _np.full((len(_SR20_OFF), _SR20_MAXN), _np.nan, _np.float32)
                        _pp20 = _np.full((len(_SR20_OFF), _SR20_MAXN), _np.nan, _np.float32)
                        for _j20, _o20 in enumerate(_SR20_OFF):
                            if 0 <= _r + _o20 < int(true_w.shape[0]):
                                _tt20[_j20, :_nst20] = true_w[_r + _o20]
                                _pp20[_j20, :_nst20] = pred_w[_r + _o20]
                        snap_nbhd_true.append(_tt20)
                        snap_nbhd_pred.append(_pp20)
                        snap_row_nout.append(_nst20)

            pl = _payload(true_w, pred_w, lcs[k], cases[k], nc, starts[k],
                          evrows=_rows)

            # --- pooled selections -------------------------------------------
            # patch 29: rank DISTINCT events, keeping the best-centred window
            _evkey = (int(lcs[k]), int(cases[k]), round(fl_true, _EVQ))
            _prow  = int(_np.argmax(true_w[:, -1]))
            _cent  = min(_prow, int(true_w.shape[0]) - 1 - _prow)
            all_events.add(_evkey)
            _push(hi_sel_d, fl_true, pl, n_extreme, _evkey, _cent)
            _push(un_sel_d, miss,    pl, n_extreme, _evkey, _cent)
            n_seen += 1
            if len(reservoir) < n_reservoir:
                reservoir.append(pl)
            else:
                j = int(rng.integers(0, n_seen))
                if j < n_reservoir:
                    reservoir[j] = pl

            # --- per-topology selections (patch 15) --------------------------
            # per topology the key de-duplicates OFFSETS only, which is what
            # keeps node count the legitimate comparison axis it is
            _push(nc_hi.setdefault(nc, {}), fl_true, pl, n_nc_extreme,
                  _evkey, _cent)
            _push(nc_un.setdefault(nc, {}), miss,    pl, n_nc_extreme,
                  _evkey, _cent)
            _r = nc_res.setdefault(nc, [])
            nc_seen[nc] = nc_seen.get(nc, 0) + 1
            if len(_r) < n_nc_reservoir:
                _r.append(pl)
            else:
                j = int(rng.integers(0, nc_seen[nc]))
                if j < n_nc_reservoir:
                    _r[j] = pl

            # --- patch 27: representative SNAP windows (pooled + per topology)
            if _rows:
                snap_seen += 1
                if len(snap_res) < n_snap:
                    snap_res.append(pl)
                else:
                    j = int(rng.integers(0, snap_seen))
                    if j < n_snap:
                        snap_res[j] = pl
                _sr = nc_snap.setdefault(nc, [])
                nc_snap_seen[nc] = nc_snap_seen.get(nc, 0) + 1
                if len(_sr) < n_nc_snap:
                    _sr.append(pl)
                else:
                    j = int(rng.integers(0, nc_snap_seen[nc]))
                    if j < n_nc_snap:
                        _sr[j] = pl

            # --- patch sr-21: per-(site, node count, class) reservoirs
            _lc21, _cs21 = int(lcs[k]), int(cases[k])
            if _lc21 in _SITE_LOCS and nc in _SITE_NCS:
                _s21 = int(starts[k])
                _e21 = _s21 + int(true_w.shape[0])
                _dirty21 = any(
                    _s21 <= _t21 < _e21
                    for _t21 in ARTIFACT_SAMPLES.get((_lc21, _cs21), ()))
                if not _dirty21:
                    _dirty21 = any(
                        _s21 <= _t21 < _e21
                        for _t21 in OUTPULL_SAMPLES.get((_lc21, _cs21), ()))
                if not _dirty21:
                    _cls21 = "sn" if _rows else "cl"
                    _key21 = (_lc21, nc, _cls21)
                    _cap21 = n_site_snap if _rows else n_site_clean
                    _b21 = site_res.setdefault(_key21, [])
                    site_seen[_key21] = site_seen.get(_key21, 0) + 1
                    if len(_b21) < _cap21:
                        _b21.append(pl)
                    else:
                        j = int(rng.integers(0, site_seen[_key21]))
                        if j < _cap21:
                            _b21[j] = pl

        if max_windows is not None and len(peak_true_all) >= max_windows:
            break

    if pin_sampler_epoch is not None and _prev_ep is not None:
        _bsamp.epoch = _prev_ep   # restore; next call re-pins to the same order

    if not peak_true_all:
        print("[ts-dump] no test windows collected -- skipped")
        return None

    peak_true_all = _np.asarray(peak_true_all, dtype=_np.float64)
    peak_pred_all = _np.asarray(peak_pred_all, dtype=_np.float64)
    meta_all      = _np.asarray(meta_all, dtype=_np.int32)
    lpeak_true_all = _np.asarray(lpeak_true_all, dtype=_np.float64)
    lpeak_pred_all = _np.asarray(lpeak_pred_all, dtype=_np.float64)
    snap_ev   = _np.asarray(snap_ev, dtype=_np.int32).reshape(-1, 5)
    snap_true = _np.asarray(snap_true, dtype=_np.float64)
    snap_pred = _np.asarray(snap_pred, dtype=_np.float64)
    # patch sr-20
    _nb_shape20 = (0, len(_SR20_OFF), _SR20_MAXN)
    snap_nbhd_true = (_np.stack(snap_nbhd_true) if snap_nbhd_true
                      else _np.zeros(_nb_shape20, _np.float32))
    snap_nbhd_pred = (_np.stack(snap_nbhd_pred) if snap_nbhd_pred
                      else _np.zeros(_nb_shape20, _np.float32))
    snap_row_nout = _np.asarray(snap_row_nout, dtype=_np.int32)
    if snap_true.size:
        _cap = snap_pred / _np.maximum(snap_true, 1.0)
        print(f'[ts-dump]{tag} snap events in scored windows: {snap_true.size} | median capture pred/true {float(_np.median(_cap)):.3f} | underpredicted {100.0 * float((_cap < 1.0).mean()):.1f}%')
    else:
        print(f'[ts-dump]{tag} no catalogued snap event fell in a scored window')

    def _ranked(sel, cap):
        out = []
        for s in sorted(sel.values(), key=lambda s: -s["key"])[:cap]:
            p = dict(s["pl"])
            p["nwin"] = s["n"]               # windows collapsed into this event
            out.append(p)
        return out

    hi_sel = _ranked(hi_sel_d, n_extreme)
    un_sel = _ranked(un_sel_d, n_extreme)
    med    = float(_np.median(peak_true_all))
    ty_sel = sorted(reservoir, key=lambda p: abs(float(p["true"][:, -1].max()) - med))

    under_rate = 100.0 * float((peak_pred_all < peak_true_all).mean())
    node_counts = sorted(nc_hi.keys())
    print(f"[ts-dump]{tag} {len(peak_true_all)} test windows | fairlead-peak "
          f"underprediction {under_rate:.1f}% | median true peak {med/1000:.2f} kN "
          f"| topologies {node_counts}")
    # patch 27: how many scored windows carried a snap, and how many traces kept
    print(f"[ts-dump]{tag} distinct physical events: {len(all_events)} "
          f"in {len(peak_true_all)} windows "
          f"(mean multiplicity {len(peak_true_all) / max(len(all_events), 1):.2f}) "
          f"| hi/un grids now hold {len(hi_sel)}/{len(un_sel)} DISTINCT events")
    print(f"[ts-dump]{tag} snap-containing windows: {snap_seen} "
          f"({100.0 * snap_seen / max(len(peak_true_all), 1):.2f}% of scored) "
          f"| full traces kept: pooled {len(snap_res)}, per-topology "
          f"{ {k: len(v) for k, v in sorted(nc_snap.items())} }")

    # ---------------- save the small .npz FIRST (never lose the data) --------
    payload = {
        "peak_true_all": peak_true_all,
        "peak_pred_all": peak_pred_all,
        "meta_all":      meta_all,
        "lpeak_true_all": lpeak_true_all,
        "lpeak_pred_all": lpeak_pred_all,
        "snap_ev":        snap_ev,
        "snap_true":      snap_true,
        "snap_pred":      snap_pred,
        # patch sr-20: [n_rows, 5, 21] around each event, NaN-padded
        "snap_nbhd_true":    snap_nbhd_true,
        "snap_nbhd_pred":    snap_nbhd_pred,
        "snap_nbhd_offsets": _np.asarray(_SR20_OFF, dtype=_np.int32),
        "snap_row_nout":     snap_row_nout,
        "dt":            _np.float64(FEATURE_DT),
        "node_counts":   _np.asarray(node_counts, dtype=_np.int32),
    }

    # --- patch sr-21: report what the site buckets caught
    if site_res or _SITE_LOCS:
        _rep21 = {f"loc{a}/N{b}/{c}": f"{len(site_res.get((a, b, c), []))}/{site_seen.get((a, b, c), 0)}"
                  for a in sorted(_SITE_LOCS) for b in sorted(_SITE_NCS) for c in ("cl", "sn")}
        print(f"[ts-dump]{tag} site windows kept/clean-seen: {_rep21}")

    def _store(prefix, sel):
        for i, p in enumerate(sel):
            payload[f"{prefix}{i}_true"] = p["true"]
            payload[f"{prefix}{i}_pred"] = p["pred"]
            payload[f"{prefix}{i}_meta"] = _np.array(
                [p["lc_id"], p["case_id"], p["node_count"], p["start_idx"]],
                dtype=_np.int64)
            # patch 27: snap-event rows inside this window (may be empty)
            payload[f"{prefix}{i}_evrows"] = p.get(
                "evrows", _np.zeros(0, dtype=_np.int32))
            # patch 29: how many test windows collapsed into this event
            payload[f"{prefix}{i}_nwin"] = _np.int64(p.get("nwin", 1))

    _store("hi", hi_sel)
    _store("un", un_sel)
    _store("ty", ty_sel)
    # patch 27: representative snap windows, ordered by true line peak so the
    # first is the most energetic of an UNBIASED draw (not of the whole tail)
    _store("sn", sorted(snap_res, key=lambda p: -float(p["true"].max())))
    for _nc in node_counts:
        _store(f"nc{_nc}_hi", _ranked(nc_hi[_nc], n_nc_extreme))
        _store(f"nc{_nc}_un", _ranked(nc_un[_nc], n_nc_extreme))
        _store(f"nc{_nc}_sn", sorted(nc_snap.get(_nc, []),
                                     key=lambda p: -float(p["true"].max())))
        _nc_med = float(_np.median(peak_true_all[meta_all[:, 2] == _nc]))
        _store(f"nc{_nc}_ty", sorted(
            nc_res.get(_nc, []),
            key=lambda p: abs(float(p["true"][:, -1].max()) - _nc_med)))

    # --- patch sr-21: site-targeted buckets, ordered by (case, start) so the local
    # scripts see a stable index. Each bucket is an unbiased reservoir over the
    # CLEAN windows of that site, class 'cl' = snap-free, class 'sn' = snap-bearing.
    for _a21 in sorted(_SITE_LOCS):
        for _b21 in sorted(_SITE_NCS):
            for _c21 in ("cl", "sn"):
                _sel21 = site_res.get((_a21, _b21, _c21), [])
                if _sel21:
                    _store(f"site{_a21:02d}_n{_b21:02d}_{_c21}",
                           sorted(_sel21, key=lambda p: (p["case_id"], p["start_idx"])))

    npz_path = out_dir / f"test_timeseries_dump{tag}.npz"
    _np.savez_compressed(npz_path, **payload)
    print(f"[ts-dump] wrote {npz_path.name} "
          f"({npz_path.stat().st_size/1e6:.1f} MB) -- re-plot locally from this")

    # ---------------- figures (failure here must not lose the .npz) ---------
    # Per-checkpoint dumps pass make_plots=False: the .npz is the deliverable and
    # plot_test_timeseries.py renders it locally in ~2 s.
    if not make_plots:
        return out_dir
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as _plt

        DT = float(FEATURE_DT)
        C_TRUE, C_PRED, C_GRAY = "#1F2A37", "#4884D6", "#6B7280"

        def _grid(sel, title, fname):
            if not sel:
                return
            n = min(len(sel), 8)
            rows = (n + 1) // 2
            fig, axes = _plt.subplots(rows, 2, figsize=(13, 2.6 * rows),
                                      squeeze=False)
            for i in range(rows * 2):
                ax = axes[i // 2][i % 2]
                if i >= n:
                    ax.axis("off")
                    continue
                p = sel[i]
                tr, pr = p["true"][:, -1], p["pred"][:, -1]
                t = _np.arange(len(tr)) * DT
                ax.plot(t, tr / 1000.0, color=C_TRUE, lw=1.2, label="FE truth")
                ax.plot(t, pr / 1000.0, color=C_PRED, lw=1.1, ls="--",
                        label="prediction")
                ax.axhline(tr.max() / 1000.0, color=C_GRAY, lw=0.7, ls=":")
                ax.set_title(
                    f"loc{p['lc_id']:02d} case{p['case_id']:04d} N={p['node_count']}"
                    f" | true peak {tr.max()/1000:.1f} kN,"
                    f" pred {pr.max()/1000:.1f} kN",
                    fontsize=8)
                ax.tick_params(labelsize=7)
                ax.grid(alpha=0.2)
                if i == 0:
                    ax.legend(fontsize=7, frameon=False, loc="upper right")
            fig.supxlabel("time within window [s]", fontsize=9)
            fig.supylabel("fairlead tension [kN]", fontsize=9)
            fig.suptitle(title, fontsize=11)
            fig.tight_layout()
            fig.savefig(out_dir / fname, dpi=140)
            _plt.close(fig)
            print(f"[ts-dump] wrote {fname}")

        _grid(hi_sel, "Highest-load test windows -- fairlead tension",
              f"ts_highest_peaks{tag}.png")
        _grid(un_sel, "Worst fairlead-peak UNDERPREDICTIONS -- test windows",
              f"ts_worst_underpred{tag}.png")
        _grid(ty_sel, "Typical (median-peak) test windows -- fairlead tension",
              f"ts_typical{tag}.png")

        fig, ax = _plt.subplots(figsize=(6.2, 6.2))
        lim = [0, max(peak_true_all.max(), peak_pred_all.max()) / 1000 * 1.05]
        ax.plot(lim, lim, color=C_GRAY, ls="--", lw=1, label="perfect (y = x)")
        ax.scatter(peak_true_all / 1000, peak_pred_all / 1000, s=8,
                   color=C_PRED, alpha=0.35, edgecolor="none",
                   label=f"test windows (n={len(peak_true_all)})")
        ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
        ax.set_xlabel("true fairlead peak tension [kN]")
        ax.set_ylabel("predicted fairlead peak tension [kN]")
        ax.set_title(f"Fairlead peak parity -- held-out test\n"
                     f"{under_rate:.1f}% of windows underpredicted", fontsize=10)
        ax.legend(fontsize=8, frameon=False, loc="upper left")
        ax.grid(alpha=0.2)
        fig.tight_layout()
        fig.savefig(out_dir / f"peak_parity_fairlead{tag}.png", dpi=140)
        _plt.close(fig)
        print(f"[ts-dump] wrote peak_parity_fairlead{tag}.png")
    except Exception as _pe:
        print(f"[ts-dump] .npz saved; plotting failed ({_pe}) -- re-plot locally")

    return out_dir


In [ ]:
def run_multi_seed_test_experiment(
    load_case_datasets: Dict[int, MooringSequenceDatasetPositionTension],
    cfg: TrainingConfig,
    device: torch.device,
):
    first_key = sorted(load_case_datasets.keys())[0]
    example_dataset = load_case_datasets[first_key]

    # Fixed split for all seeds
    train_loader, val_loader, test_loader, norms, split_indices = build_dataloaders(
        load_case_datasets=load_case_datasets,
        cfg=cfg,
    )

    all_results = []

    for run_seed in cfg.seed_list:
        print("\n" + "=" * 70)
        print(f"STARTING RUN FOR SEED {run_seed}")
        print("=" * 70)

        set_seed(run_seed)

        model = build_model_from_config(
            example_dataset=example_dataset,
            cfg=cfg,
            device=device,
        )

        criterion = ReconstructionKendallLoss(
            node_norm=norms["node_norm"],
            target_norm=norms["target_norm"],
            cfg=cfg,
        ).to(device)
        optimizer, scheduler = build_optimizer_and_scheduler(model, cfg, criterion=criterion)

        model, history, model_path = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            criterion=criterion,
            device=device,
            target_norm=norms["target_norm"],
            target_names=example_dataset.target_feature_names,
            cfg=cfg,
            run_seed=run_seed,
        )

        test_results = test_model(
            model=model,
            test_loader=test_loader,
            criterion=criterion,
            device=device,
            target_norm=norms["target_norm"],
            target_names=example_dataset.target_feature_names,
            cfg=cfg,
        )

        run_result = {
            "seed": run_seed,
            "model_path": model_path,
            "history": history,
            "test_loss": test_results["test_loss"],
            "test_metrics": test_results["test_metrics"],
        }
        all_results.append(run_result)

    # Aggregate test loss across seeds
    test_losses = [r["test_loss"] for r in all_results]
    mean_test_loss = float(np.mean(test_losses))
    std_test_loss = float(np.std(test_losses))

    # Aggregate test across seeds
    target_names = example_dataset.target_feature_names
    metric_names_all_targets = ["MAE", "RMSE", "R2"]
    mean_test_metrics = {metric: {} for metric in metric_names_all_targets}
    std_test_metrics = {metric: {} for metric in metric_names_all_targets}
    mean_test_metrics["MAPE"] = {}
    std_test_metrics["MAPE"] = {}

    for metric in metric_names_all_targets:
        for name in target_names:
            vals = [r["test_metrics"][metric][name] for r in all_results]
            mean_test_metrics[metric][name] = float(np.mean(vals))
            std_test_metrics[metric][name] = float(np.std(vals))

    if "tension" in target_names:
        vals = [r["test_metrics"]["MAPE"].get("tension", float("nan")) for r in all_results]
        mean_test_metrics["MAPE"]["tension"] = float(np.nanmean(vals))
        std_test_metrics["MAPE"]["tension"] = float(np.nanstd(vals))


    print("\n" + "=" * 70)
    print("MULTI-SEED TEST SUMMARY")
    print("=" * 70)
    print(f"Seeds used: {list(cfg.seed_list)}")
    print(f"Mean test loss across seeds = {mean_test_loss:.6f}")
    print(f"Std  test loss across seeds = {std_test_loss:.6f}")

    for metric in ["MAE", "RMSE", "R2"]:
        print(f"\n{metric}:")
        for name in target_names:
            print(
                f"  {name}: mean = {mean_test_metrics[metric][name]:.6f}, "
                f"std = {std_test_metrics[metric][name]:.6f}"
            )

    if "tension" in mean_test_metrics["MAPE"]:
        print("\nMAPE:")
        print(
            f"  tension: mean = {mean_test_metrics['MAPE']['tension']:.6f}, "
            f"std = {std_test_metrics['MAPE']['tension']:.6f}"
        )

    summary = {
        "all_results": all_results,
        "mean_test_loss": mean_test_loss,
        "std_test_loss": std_test_loss,
        "mean_test_metrics": mean_test_metrics,
        "std_test_metrics": std_test_metrics,
    }
    return summary

In [19]:
# -------------------------------------------------------------
# real data loader
# -------------------------------------------------------------

import os as _os

DATA_ROOT = Path(r"C:\Users\thano\Desktop\data")  # local path; not needed in cache-only runs
CACHE_DIR = Path(
    _os.environ.get(
        "MOORING_CACHE_DIR",
        "/scratch/tkonstantaras/mooring_data/cache_npy",
    )
)
AVAILABLE_LOC_IDS = [1, 3, 4, 5, 6, 7, 8, 9, 10, 11]
N_CASES = 300

# Corrupt FE simulation cases -- UNION of two independent criteria that COEXIST
# (both are excluded from dataset construction so neither can contaminate the
# FeatureStandardizer or any training/val/test window):
#   RULE A (legacy, thesis Ch.2 §2.5): non-finite (NaN/Inf) tension OR peak
#          |T| > 1.5 MN.  -> _CORRUPT_CASES_LEGACY (53 cases, kept verbatim).
#   RULE B (2026-07-22 rescan, scan_corrupt_cases.py): per-sample spatial-
#          coherence blowup (iso>10 AND (coh>3 OR T>3x fairlead_max)) OR
#          non-finite; the 56 ADDITIONAL cases beyond RULE A, taken from
#          corrupt_scan.csv (n_blowup>0 OR n_nonfinite>0).
#          -> _CORRUPT_CASES_RESCAN_NEW (56 cases). RULE B reclassifies
#          (8,83)=1.42 MN (previously kept as a snap load) as an FE blowup.
_CORRUPT_CASES_LEGACY = frozenset({
    (1, 63), (1, 202), (1, 215),
    (4, 45), (4, 133), (4, 164), (4, 270),
    (5, 31), (5, 35), (5, 46), (5, 108), (5, 194), (5, 197), (5, 202), (5, 262), (5, 264),
    (6, 22), (6, 131), (6, 134),
    (7, 16), (7, 32), (7, 83), (7, 132), (7, 137), (7, 174), (7, 212), (7, 214), (7, 220), (7, 221), (7, 225), (7, 229),
    (8, 37), (8, 94), (8, 136), (8, 143), (8, 247),
    (9, 43), (9, 167), (9, 193),
    (10, 22), (10, 56), (10, 77), (10, 187), (10, 188), (10, 224), (10, 234), (10, 239), (10, 275), (10, 278),
    (11, 3), (11, 22), (11, 61), (11, 89),
})
_CORRUPT_CASES_RESCAN_NEW = frozenset({
    (1, 11), (1, 116), (1, 131), (1, 142), (1, 185), (1, 243), (1, 253), (1, 298),
    (3, 129), (3, 164),
    (4, 2), (4, 29), (4, 72), (4, 79), (4, 99), (4, 114), (4, 186), (4, 239), (4, 296),
    (5, 19), (5, 22), (5, 133), (5, 224), (5, 232),
    (6, 65), (6, 120), (6, 126), (6, 222),
    (7, 2), (7, 31), (7, 62), (7, 80), (7, 85), (7, 92), (7, 117), (7, 169), (7, 203), (7, 244),
    (8, 77), (8, 83), (8, 133),
    (9, 34), (9, 65), (9, 150), (9, 156), (9, 176), (9, 218), (9, 238),
    (10, 81), (10, 100), (10, 173), (10, 205), (10, 286),
    (11, 11), (11, 43), (11, 240),
})
# Both rules coexist: the effective blacklist is their union (53 + 56 = 109).
CORRUPT_CASES = _CORRUPT_CASES_LEGACY | _CORRUPT_CASES_RESCAN_NEW
assert len(CORRUPT_CASES) == 109, (
    f"blacklist size {len(CORRUPT_CASES)} != 109 "
    f"(legacy {len(_CORRUPT_CASES_LEGACY)} + rescan {len(_CORRUPT_CASES_RESCAN_NEW)})"
)
# --- patch 19: per-window artifact exclusion ---
# Corrupt samples are ~0.0024 % of the record but the 109-case blacklist
# discards the whole simulation, taking 86 % of the dataset's snap-load
# events with it. Exclude the affected WINDOWS instead; keep whole-case
# exclusion only where the corruption is non-finite (NaN/Inf), which would
# poison the normalizer fit if a single sample were missed.
EXCLUDE_ARTIFACT_WINDOWS = True

# --- patch 22: sustained-divergence ceiling ---
# The shape rule ANDs iso > 10, so it only ever sees a spike that stands
# out from its own neighbours. Once a solve has DIVERGED the neighbouring
# samples are diverged too (iso ~ 1) and the rule is structurally blind:
# 638 corrupt samples escape it, the largest at 1.3e18 N inside a window
# that per-window shape masking alone would have kept.
#
# Justification is PROVENANCE, not magnitude. Measured over the full cache:
#   * the ceiling fires in 36 cases, ALL already flagged by the shape or
#     non-finite rules -- it implicates ZERO new simulations, so it is a
#     within-case refinement of WHICH samples are dropped;
#   * 558/558 ceiling-only samples lie in cases whose record elsewhere
#     exceeds 1 MN, i.e. a diverged solve;
#   * 50 kN is 1.53x the maximum tension anywhere in the 2891 cases the
#     shape rule never fires on (32,751 N) and 2.45x the largest verified
#     physical snap (20,411 N), so it is set from cases it does not filter.
# Sensitivity 35/50/75 kN -> 1030/1030/1053 reachable snap events.
ARTIFACT_TENSION_CEILING_N = 50_000.0

ARTIFACT_SAMPLES = {    # (loc, case) -> corrupt sample indices  [scan_artifact_samples.py]
    (1, 11): (9195,),
    (1, 63): (3678, 5776, 5985, 7508, 7509, 7510, 7511, 7512, 7513, 7514, 7515, 7516, 7517, 7518, 7519, 7520, 7521, 7522, 7523, 7596, 8083, 8096, 8098, 8273, 10455,),
    (1, 116): (12794,),
    (1, 131): (9191,),
    (1, 142): (9202, 10454,),
    (1, 185): (4084,),
    (1, 202): (361, 731, 1705, 1933, 2132, 3659, 3662, 3681, 3829, 3830, 3831, 3832, 3833, 3834, 3835, 3836, 3837, 3838, 3839, 3840, 3842, 3843, 3851, 3878, 3882, 3884, 6456, 6596, 6625, 6952, 6954, 7535, 8892, 8903, 8904, 8905, 8906, 8907,),
    (1, 215): (682, 683, 697, 737, 2063, 2081, 2092, 2093, 2094, 2095, 2096, 2097, 2098, 2099, 2100, 2101, 2102, 2103, 2104, 2210, 3651, 3700, 3711, 3712, 3713, 3714, 3715,),
    (1, 243): (4748, 9744,),
    (1, 253): (9753,),
    (1, 298): (12730, 12734,),
    (3, 129): (5625,),
    (3, 164): (673, 7533, 7537,),
    (4, 2): (3876,),
    (4, 29): (9764,),
    (4, 45): (702, 735, 2043, 9755, 10229, 10230, 10233, 10234, 10238, 10256, 10259, 12799,),
    (4, 72): (5356,),
    (4, 79): (2070,),
    (4, 99): (9767,),
    (4, 114): (1845, 10221, 10237, 12801,),
    (4, 133): (7569, 9757, 9758, 9759, 9760, 9761, 12632,),
    (4, 164): (7575, 8760, 8770, 8775,),
    (4, 186): (1950,),
    (4, 239): (6954, 8114,),
    (4, 270): (701, 728, 2154,),
    (4, 296): (9759,),
    (5, 19): (7565,),
    (5, 22): (744, 10224,),
    (5, 31): (8956, 8958, 8966, 8969, 8975, 8976, 8978, 8987, 11045,),
    (5, 35): (8922, 8923, 8924, 8925, 8926, 8927, 8928, 8929, 8930, 8934, 8940, 8946, 8948, 8951, 8960, 8966, 8968, 8971, 8986, 8987, 8988, 8992, 9143,),
    (5, 46): (1954, 1955, 1956, 1958, 1959, 1960, 1961, 1963, 1968, 1969, 1970, 1971, 1974, 1975, 1978, 1979, 1981, 1988, 1996, 2000, 6584, 8958,),
    (5, 108): (13411, 13412, 13414, 13415,),
    (5, 133): (7565,),
    (5, 194): (1957, 1962, 1965, 1972, 10478, 10482,),
    (5, 197): (8951, 8953, 8955, 8956, 8959, 8963, 8965, 8968, 8970, 9009, 10599,),
    (5, 202): (616, 617, 618, 619, 620, 621, 622, 623, 624, 625,),
    (5, 224): (2576, 8111,),
    (5, 232): (8957, 8960, 8961,),
    (5, 262): (603, 612, 613, 614, 615, 616, 617, 618,),
    (5, 264): (7553, 7555, 7557, 7559, 7561, 7564, 7566, 7569, 7572, 7586, 8965,),
    (6, 22): (7488, 8109, 8300, 8305, 8309, 8874, 8898, 8985, 9674, 9680, 9684,),
    (6, 65): (743,),
    (6, 120): (703,),
    (6, 126): (739, 12803,),
    (6, 131): (710, 714, 722, 724, 743, 750, 754, 755, 12802,),
    (6, 134): (711, 714, 716, 719, 725, 728, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1035, 1044, 1066, 1888, 1917, 1927, 1930, 1957, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2056, 2057, 2059, 2060, 2067, 2074, 2099, 2112, 2118, 6027, 10297, 10544, 10549, 10566, 10567, 10568, 10570, 10571, 10575, 10577, 10578, 11892, 11897, 11898, 11914, 11915, 12104, 12130, 12131, 12145, 12199, 12200, 12201, 12202, 12203, 12204, 12205, 12206, 12207, 12208, 12209, 12210, 12211, 12212, 12213, 12214, 12215, 12241, 12244, 12262, 12273, 12286, 12295, 12765, 12788, 12813, 12991, 13014, 13073, 13074, 13075, 13076, 13077, 13078, 13079, 13080, 13081, 13082, 13083, 13084, 13085, 13086, 13087, 13088, 13089, 13090, 13091, 13092, 13093, 13094, 13095, 13096, 13097, 13099, 13116, 13136, 13140, 13163,),
    (6, 222): (12703, 12719,),
    (7, 2): (9760,),
    (7, 16): (13792,),
    (7, 31): (8097,),
    (7, 32): (12172,),
    (7, 62): (9762,),
    (7, 80): (7566,),
    (7, 83): (11992,),
    (7, 85): (9758,),
    (7, 92): (9769,),
    (7, 117): (6880, 9754,),
    (7, 132): (6876, 7568, 9673, 9767, 13297,),
    (7, 137): (13567,),
    (7, 169): (4100,),
    (7, 174): (743, 8114, 8117, 8121, 8133, 8135, 10233, 10249,),
    (7, 203): (7555,),
    (7, 212): (1737, 1739, 1787, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2023, 2024, 2026, 2045, 2047, 2118, 2120, 2145, 2148, 4090, 4092, 4093, 4098, 4107, 4134, 4136, 4151, 4152, 4153, 4154, 4157, 4158, 4159, 4160, 4169, 4170, 4171, 4172, 4183, 4191, 4202, 4205, 4215, 4663, 4664, 4669, 4670, 4671, 4672, 4679, 4691, 4693, 4711, 4725, 6591, 6654, 6655, 6658, 6659, 6660, 6663, 6669, 6671, 6696, 6704, 7572, 8372, 8917, 8918, 8919, 8920, 8921, 8922,),
    (7, 214): (4792,),
    (7, 220): (12622,),
    (7, 221): (2032,),
    (7, 225): (4792,),
    (7, 229): (10192,),
    (7, 244): (7563,),
    (8, 37): (4648, 4650,),
    (8, 77): (682,),
    (8, 83): (2067, 2069, 6583, 12803,),
    (8, 94): (2081, 8084, 8085, 10218,),
    (8, 133): (11607,),
    (8, 136): (601, 602, 603, 604, 605, 606, 608, 609, 678, 743, 747, 748, 749, 750, 751, 752, 754, 755, 756, 771, 2094, 2095, 2099, 2100, 3818, 4099, 6636, 6637, 6638, 6639, 6640, 6641, 6643, 6648, 7531, 8897, 9775, 10219, 10594, 10595, 10596, 10597, 10598, 10599, 10600, 10601, 10602, 10605, 10619, 10620, 10623, 10631, 10651, 10707, 11864, 12652, 12664, 12665, 12666, 12667, 12668, 12669, 12670, 12671, 12674, 12675, 12678, 12748, 12774, 12806, 12810, 12812, 12813, 12814, 12815, 12816, 12817, 12818, 12825, 12826, 12834, 13087, 13764,),
    (8, 143): (2076, 3691, 4641, 4642, 4670, 4698, 5655, 5674, 9776, 12652,),
    (8, 247): (3691, 4074, 4085, 4102, 4116, 4642, 5622, 5625, 8092, 8875, 8909, 8910, 8911, 9749, 9755, 12791,),
    (9, 34): (9186, 10959, 12618,),
    (9, 43): (544, 2543, 4647, 5418, 6769, 7538, 7994, 9706, 9707, 9708, 9709, 9710, 9711, 9712, 9713, 9714, 9715, 9716, 9717, 9720, 9723, 9725, 9726, 9732, 9733, 9743, 9778, 9855, 11017,),
    (9, 65): (8094,),
    (9, 150): (7534,),
    (9, 156): (723, 2015,),
    (9, 167): (1930, 6562, 6566, 6597, 6599,),
    (9, 176): (9035,),
    (9, 193): (580, 596, 597, 598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 615, 616, 622, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 767, 1705, 2037, 6834, 8081, 8864, 10233, 10588, 10589, 10590, 10591, 10592, 10593, 10594, 10595, 10596, 10597, 10598, 10599, 10600, 10601, 10602, 10603, 10605, 10607, 10612, 10614, 10620, 10725, 12644, 12660, 12661, 12662, 12663, 12664, 12665, 12666, 12667, 12668, 12669, 12670, 12671, 12672, 12673, 12677, 12796, 12807, 12808, 12809, 12810, 12811, 12812, 12813, 12814, 12815, 12816, 12817, 12818, 12819, 12820, 12821, 12822, 12823, 12824, 12825, 12826, 12828, 12833, 12837, 12838, 12847,),
    (9, 218): (8854,),
    (9, 238): (4072, 9742, 9745,),
    (10, 22): (705, 715, 737, 4104, 4107, 4112, 5649, 5653, 5658, 5660, 5675, 5678, 5705, 8868, 8870, 8882, 8884, 8885, 8888, 8922, 8927, 8928, 8929, 8940, 8941, 8979, 8981, 8988, 9000, 9763, 10212, 10219, 10222, 10223, 10225, 10226, 10227, 10228, 10231, 10235, 10238, 10240, 10259, 10262, 10271, 10274, 10319, 12764, 12765, 12774,),
    (10, 56): (729, 1899, 3773, 6775, 8129, 8131, 8134, 8135, 8141, 10553, 10577, 10578, 10585, 10587, 12765, 12796, 12799, 12812, 13177,),
    (10, 77): (562, 564, 631, 656, 657, 658, 659, 660, 661, 662, 663, 664, 665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675, 676, 677, 678, 679, 686, 690, 691, 696, 706, 708, 712, 726, 742, 1770, 1886, 2040, 2043, 2046, 2049, 2051, 2056, 2057, 2112, 2114, 2121, 2132, 2964, 2966, 2967, 2969, 2976, 2977, 2981, 2994, 2998, 3000, 3015, 3665, 3671, 3676, 3795, 3801, 3813, 3818, 3823, 3830, 3834, 4117, 4133, 4722, 6591, 6592, 6594, 6599, 6615, 6616, 6621, 6632, 6633, 6898, 7548, 7550, 7554, 7593, 7596, 7601, 8117, 8979, 8989, 8992, 8999, 9765, 10244, 10246, 10252, 10253, 10258, 10259, 10268, 10281, 10299, 10539, 10543, 10574, 10986, 11206, 11904, 12109, 12110, 12115, 12118, 12248, 12249, 12251, 12254, 12256, 12257, 12262, 12265, 12489, 12631, 12634, 12636, 12638, 12653, 12660, 12663, 12665, 12668, 12720, 12721, 12722, 12723, 12724, 12725, 12726, 12727, 12728, 12729, 12730, 12731, 12732, 12733, 12734, 12735, 12736, 12737, 12738, 12739, 12740, 12741, 12742, 12743, 12748, 12749, 12750, 12751, 12752, 12753, 12785, 12799,),
    (10, 81): (8304,),
    (10, 100): (3663, 3666, 6977,),
    (10, 173): (744,),
    (10, 187): (698, 706, 708, 1959, 2066, 2069, 2445, 6603, 8122, 8136, 8144, 8146, 8304, 8596, 8680, 8866, 8867, 8871, 8875, 8876, 8877, 8881, 8889, 8912, 8971, 8991, 9004, 9669, 9672, 9674, 9675, 9677, 9678, 9683, 9684, 9687, 9700, 9711, 9726, 9728, 10244, 10251,),
    (10, 188): (702, 704, 705, 708, 710, 711, 713, 740, 757, 767, 768, 769, 2044, 2958, 2961, 2966, 2998, 3002, 6630, 10256, 10547, 12767, 12770, 12835,),
    (10, 205): (2082, 10245,),
    (10, 224): (401, 555, 558, 560, 566, 571, 574, 584, 608, 661, 662, 663, 664, 665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675, 676, 677, 678, 679, 680, 682, 703, 706, 709, 711, 715, 729, 741, 1352, 1877, 1879, 1899, 2039, 2052, 2098, 2450, 2964, 2965, 2966, 2967, 2968, 2975, 2978, 2979, 2980, 2981, 2989, 2990, 2994, 3001, 3008, 3011, 3027, 3041, 3662, 3680, 3799, 3811, 3820, 4134, 4678, 4685, 4687, 4693, 4753, 5632, 5654, 5989, 6036, 6716, 6718, 6720, 6727, 6732, 6757, 6845, 6906, 6915, 6926, 6946, 8873, 8880, 8886, 8895, 8983, 9776, 9789, 9831, 9833, 9834, 9835, 9836, 9837, 9839, 10242, 10250, 10260, 10265, 10269, 10270, 10283, 10284, 10290, 10317, 10321, 10328, 10403, 10432, 10541, 10558, 10686, 10724, 12267, 12272, 12293, 12300, 12322, 12467, 12468, 12469, 12470, 12471, 12472, 12475, 12476, 12484, 12618, 12627, 12630, 12631, 12632, 12639, 12641, 12650, 12655, 12657, 12663, 12680, 12725, 12726, 12727, 12728, 12729, 12730, 12731, 12732, 12733, 12734, 12735, 12736, 12737, 12738, 12739, 12740, 12741, 12742, 12743, 12744, 12778, 12792, 13174, 13176, 13319,),
    (10, 234): (400, 599, 657, 658, 659, 660, 661, 662, 663, 664, 665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675, 676, 683, 685, 687, 692, 693, 694, 695, 702, 709, 710, 715, 728, 756, 771, 1074, 1264, 1284, 1784, 1892, 1901, 1953, 2039, 2047, 2048, 2093, 3667, 3669, 3672, 3676, 3688, 3713, 4105, 4115, 4683, 5784, 6592, 6603, 6720, 6721, 6723, 6748, 6888, 6889, 6890, 6891, 6909, 7558, 7843, 7844, 7846, 7857, 7886, 8862, 9780, 9785, 9786, 9790, 9793, 9794, 9797, 9799, 9817, 10271, 10285, 10540, 10541, 10543, 10545, 10689, 11897, 11907, 11916, 11934, 11942, 11951, 11962, 12260, 12274, 12340, 12464, 12466, 12470, 12659, 12721, 12722, 12723, 12724, 12725, 12726, 12727, 12728, 12729, 12730, 12731, 12732, 12733, 12734, 12735, 12736, 12737, 12738, 12739, 12740, 12745, 12746, 12747, 12748, 12749, 12750, 12752, 12755, 12763, 12771, 12790, 12827, 12828, 13120,),
    (10, 239): (709, 710, 711, 716, 717, 718, 2025, 2029, 2031,),
    (10, 275): (708, 1868, 1869, 1870, 1871, 1877, 1883, 1905, 1906, 1956, 1958, 1960, 1962, 1967, 2046, 3687, 3688, 3689, 3798, 3799, 3803, 3893, 3921, 4106, 4768, 5650, 6611, 6622, 6725, 8031, 8865, 8867, 8869, 8874, 9062, 9063, 9073, 9082, 9674, 9675, 9677, 9686, 10273,),
    (10, 278): (697, 698, 714, 892, 10290, 10597, 12253, 12768, 12791, 12822, 13143, 13197, 13231,),
    (10, 286): (12802,),
    (11, 3): (1346, 6582, 6963, 6997, 7021, 9764,),
    (11, 11): (1955, 5011,),
    (11, 22): (2025, 2049, 2147, 2148, 2149, 2150, 2159, 2161, 2163, 2167, 2178, 2181, 2184, 2193, 3878, 6577, 6579, 6580, 6581, 6583, 6584, 6587, 6589, 6594, 6612, 6615, 8923, 8924, 8925, 8926, 8927, 8928, 8932, 8933, 8934, 8935, 8943, 8949, 8951, 8955, 8960, 8962, 8979, 12441,),
    (11, 43): (7557,),
    (11, 61): (278, 2037, 2043, 3718, 3719, 3720, 3721, 3722, 3723, 3724, 3725, 3726, 3727, 3728, 3729, 3730, 3731, 3732, 3733, 3734, 3735, 3736, 3737, 3742, 3745, 3746, 3752, 3755, 3782, 3783, 3784, 3786, 3823, 3827, 3846, 3851, 3854, 3858, 3861, 3864, 3866, 3867, 3869, 3872, 3907, 3910, 3914, 3919, 3920, 3923, 3926, 3933, 3935, 3938, 4200, 6951, 7555, 8101, 8103, 8104, 8918, 8919, 8920, 8921, 8922, 8923, 8924, 8925, 8926, 8927, 8928,),
    (11, 89): (6583, 6584, 6585, 6586, 6587, 6588, 6589, 6607, 6616, 6619, 6625, 6628, 6653, 6673, 8956,),
    (11, 240): (6597, 7577,),
}

SNAP_EVENT_TIMES = {    # (loc, case) -> snap-load sample indices [scan_snap_events.py]
    # patch 29: regenerated from the CORRECTED catalogue.
    #   guard A  anchor may not out-pull the fairlead by > 3x (removes whole-line
    #            inverted-gradient artifacts that the coh/fairlead_max branches
    #            of the artifact rule are both structurally blind to)
    #   guard B  artifact rule references p99.9 of the fairlead series, not its
    #            max, which is itself contaminated in 26 cases
    #   guard C  the pipeline's own 50 kN ceiling + non-finite veto, so the
    #            catalogue can never name a sample the MASK excludes
    #   refractory grouping now keeps the STRONGEST sample of each 2.0 s group
    #            instead of the first (274/1611 groups pointed at a precursor
    #            more than 5 % weaker; p50 1.56x, max 939x)
    # 1611 -> 1388 events in 200 cases; peak p50 5,946 N,
    # p99 28,978 N, max 44,085 N (was 1.08 GN).
    # UNUSED while snap_oversample_frac == 0.0 -- it feeds only
    # _snap_windows_in()/_snap_enriched_sample(), never the exclusion mask.
    (1, 11): (10472,),
    (1, 63): (569, 643, 1881, 2018, 2298, 2445, 2511, 2951, 3681, 3812, 4099, 5218, 5302, 5368, 5439, 5583, 5646, 5795, 7497, 7524, 7586, 7656, 8025, 8112, 8604, 9063, 9138, 10473, 11049, 11624, 12633, 12706, 12766, 13538,),
    (1, 116): (730, 2063, 12800,),
    (1, 130): (10471,),
    (1, 142): (564, 639, 698, 2293, 2442, 3808, 5292, 5361, 5434, 5572, 5799, 7488, 7572, 8303, 11620, 12703, 12762,),
    (1, 177): (1957, 2549, 4099, 5008, 7482, 8025, 8104, 8880, 9136, 9770, 10471, 11050,),
    (1, 185): (3672, 12763,),
    (1, 202): (288, 420, 722, 1361, 1763, 1965, 2074, 2179, 2441, 2967, 3814, 3858, 3886, 3918, 3941, 4110, 4130, 4696, 4701, 4778, 5869, 6507, 6730, 7001, 7538, 7600, 8114, 8154, 8896,),
    (1, 215): (590, 723, 1257, 1375, 1752, 1780, 1881, 2054, 2076, 2110, 2181, 2229, 2250, 2438, 2962, 3648,),
    (1, 221): (7489, 7557,),
    (1, 231): (9054, 10470,),
    (1, 243): (7490, 7560, 8296, 9057, 9132, 10235, 10476,),
    (1, 253): (4098, 7553, 8879,),
    (1, 298): (12768,),
    (3, 5): (1951, 8104, 8293, 8876, 9054,),
    (3, 9): (720,),
    (3, 32): (4888,),
    (3, 82): (7477, 7547, 10466,),
    (3, 100): (8868,),
    (3, 110): (4093, 8867,),
    (3, 114): (8867,),
    (3, 129): (4099, 7551, 8869,),
    (3, 152): (2433, 2494, 3608, 4881, 7482,),
    (3, 164): (702, 4096, 7554, 8104, 8869, 9763, 12742, 12762,),
    (4, 1): (2517,),
    (4, 2): (1961, 2054, 2156, 4116, 6604, 7586, 8123, 9773, 9794,),
    (4, 21): (7569, 9774,),
    (4, 29): (7573, 9775,),
    (4, 34): (9775,),
    (4, 37): (9773,),
    (4, 45): (722, 2073, 4683, 6605, 7575, 8124, 8147, 8984, 9787, 10266, 10293, 12784, 12805,),
    (4, 56): (7571,),
    (4, 59): (9776,),
    (4, 64): (713, 2055, 2157, 4114, 5658, 6606, 7587, 8129, 8984, 12777,),
    (4, 65): (1963, 2049, 8119, 8972, 9789,),
    (4, 72): (7565,),
    (4, 79): (719, 2049, 2087, 7573, 8149, 9780, 12798,),
    (4, 89): (2052, 7571, 8126,),
    (4, 99): (7571, 9775,),
    (4, 114): (742, 1907, 2059, 8132, 10265, 12262,),
    (4, 117): (7574, 8117,),
    (4, 127): (7571, 9776,),
    (4, 133): (7576, 9791, 9805, 9825, 10488,),
    (4, 135): (2049, 4112, 4855, 5372, 7574, 8117, 9226, 9783, 10487, 11063,),
    (4, 137): (7571,),
    (4, 164): (1967, 2054, 2153, 4114, 5372, 5651, 6889, 8124, 8778, 8980, 9776,),
    (4, 186): (1970, 2053, 4114, 5375, 7585, 8979, 9788,),
    (4, 187): (7571, 9775,),
    (4, 190): (9771,),
    (4, 195): (5373, 7502,),
    (4, 204): (9771,),
    (4, 207): (2516, 2575, 7567,),
    (4, 214): (2050, 8124, 9780,),
    (4, 222): (7572, 9778,),
    (4, 239): (2056, 2157, 4114, 5659, 7576, 8144, 8984, 9786, 12778,),
    (4, 243): (2052, 2154, 5655, 8124, 8979,),
    (4, 270): (2073, 2181, 3799, 5678, 6609, 6988, 7578, 8125, 8148, 9009, 9786, 12792,),
    (4, 296): (1969, 2050, 4112, 4926, 5372, 6881, 7575, 8118, 9148, 9782, 10487, 11064,),
    (4, 300): (7566, 9771,),
    (5, 22): (750, 1926, 2093, 8145, 10276, 12279, 12808,),
    (5, 31): (1985, 2079, 2164, 4134, 5664, 6615, 7586, 8145, 8979, 9005, 9024, 9078, 9793,),
    (5, 35): (1983, 2083, 2161, 5661, 6614, 7573, 8121, 8144, 8781, 8961, 8983, 9012, 9038, 9083, 9785, 11382, 13420,),
    (5, 46): (1362, 1964, 2008, 2076, 2100, 2167, 3909, 4127, 6634, 7585, 8145, 9019, 9080, 9794, 12783, 13426,),
    (5, 48): (9777,),
    (5, 57): (9783,),
    (5, 77): (7573,),
    (5, 97): (7576, 9782,),
    (5, 105): (9775,),
    (5, 108): (1974, 4118, 4859, 5376, 7587, 9787, 10331, 10495, 13442, 13459,),
    (5, 128): (1967, 7578, 9785,),
    (5, 133): (7576,),
    (5, 143): (9776,),
    (5, 171): (7578, 8974,),
    (5, 178): (7572,),
    (5, 183): (7575, 9780,),
    (5, 194): (1990, 9783, 10489, 10506,),
    (5, 197): (2059, 4120, 6609, 7582, 8126, 8964, 9008, 9790, 10614,),
    (5, 201): (7573, 9778,),
    (5, 202): (404, 580,),
    (5, 207): (7573,),
    (5, 211): (7577,),
    (5, 213): (9775,),
    (5, 224): (2525, 2585, 5811, 6810, 7510, 7582, 12785,),
    (5, 231): (7575, 9776,),
    (5, 232): (5662, 6611, 7582, 8133, 8984, 9799,),
    (5, 234): (9781,),
    (5, 235): (1970, 2057, 4120, 7582, 8126, 8981, 9797,),
    (5, 259): (9782,),
    (5, 262): (431, 570,),
    (5, 264): (2058, 4115, 5661, 6609, 7584, 7587, 7607, 8985, 9799,),
    (5, 270): (7573,),
    (5, 291): (7574, 9778,),
    (5, 296): (7575,),
    (6, 22): (4144, 8047, 8301, 8917, 8993, 9094, 9681,),
    (6, 102): (11632,),
    (6, 120): (12801,),
    (6, 126): (12809,),
    (6, 131): (737, 767, 794, 8169, 10298, 12819,),
    (6, 134): (227, 429, 721, 941, 1148, 1333, 1949, 2084, 2108, 2940, 3696, 4733, 6624, 8174, 9563, 9853, 9948, 10146, 10304, 10338, 10581, 10616, 11911, 12141, 12167, 12316, 12338, 12718, 12775, 12810, 13100, 13164, 13185, 13213, 13396,),
    (6, 182): (2469, 2528, 3641, 7448, 7519, 7579,),
    (6, 222): (2470, 2527, 3640, 5459, 5749, 5817, 7446, 7520, 7589,),
    (7, 5): (2579, 7571,),
    (7, 16): (7575, 9782,),
    (7, 31): (7577, 9783,),
    (7, 42): (7571, 9776,),
    (7, 62): (7572, 9776,),
    (7, 68): (738, 2073, 12801,),
    (7, 85): (9777,),
    (7, 92): (7575, 9782,),
    (7, 111): (5374,),
    (7, 117): (1971, 7587, 8104, 8127, 8981, 9789,),
    (7, 125): (2059, 7580, 12782,),
    (7, 132): (1973, 2055, 4116, 5378, 7582, 8125, 8979, 9230, 9778,),
    (7, 133): (2058, 5659, 7578, 8131, 9795,),
    (7, 169): (9782,),
    (7, 174): (1917, 2067, 8116, 8161, 8169, 10236, 10286, 12279,),
    (7, 191): (9777,),
    (7, 203): (8123, 9782,),
    (7, 212): (324, 612, 709, 1377, 1753, 1784, 1879, 2081, 2095, 2179, 2192, 2218, 2260, 2468, 2976, 3718, 3816, 3926, 4100, 4126, 4212, 4266, 4281, 4675, 4699, 4729, 4787, 5679, 5882, 6631, 6731, 6760, 6793, 6904, 7000, 7588, 8121, 8414, 8803, 8899,),
    (7, 220): (8126, 9783,),
    (7, 221): (719, 2070, 5661, 6608, 8149, 8984, 9796, 12782,),
    (7, 244): (7575, 9778,),
    (8, 16): (7551, 10470,),
    (8, 37): (1959, 4101, 4679, 4764, 4844, 5006, 7485, 8026, 8102, 8294, 8877, 9060, 9206, 9770, 10476, 10968, 11053,),
    (8, 46): (2497, 7484,),
    (8, 77): (719, 12783,),
    (8, 83): (745, 2063, 3685, 4694, 6610, 6729, 8146, 10283, 10568,),
    (8, 94): (733, 1899, 2087, 3688, 4699, 6618, 6737, 8086, 8872, 9004, 10267, 10290, 12796, 12811,),
    (8, 109): (723, 12787,),
    (8, 136): (418, 613, 737, 790, 1094, 1753, 1802, 1905, 3823, 4114, 4231, 4718, 5677, 6621, 6661, 6749, 6867, 6919, 7135, 7577, 8157, 8407, 8870, 9005, 10229, 10277, 10621, 10729, 11623, 11917, 12482, 12695, 12782, 12842, 13151, 13333,),
    (8, 143): (588, 741, 1894, 4638, 4669, 4689, 4718, 5660, 5680, 5706, 6620, 6747, 6882, 7007, 7121, 7565, 8980, 9721, 9822, 10580, 12636, 12656, 12793,),
    (8, 162): (8874,),
    (8, 220): (7485,),
    (8, 229): (744, 2074, 12801,),
    (8, 247): (728, 2072, 2958, 3696, 3791, 3902, 4105, 5619, 5667, 6726, 7003, 7570, 8122, 8999, 9780, 9804, 10258,),
    (9, 34): (1958, 4099, 4841, 5007, 5353, 6788, 7480, 7555, 8025, 8106, 8293, 9041, 9061, 9137, 9715, 9749, 9776, 9850, 10469, 11050,),
    (9, 43): (562, 632, 1966, 2545, 2571, 3129, 3394, 4117, 4582, 4665, 4777, 4847, 5019, 5359, 5436, 5778, 6792, 6864, 7486, 7561, 8031, 8125, 8296, 8602, 8968, 9064, 9143, 9215, 9685, 9747, 9782, 9791, 9870, 9875, 10481, 10631, 10782, 10871, 10969, 11045, 11120, 11294, 12625,),
    (9, 47): (7552,),
    (9, 65): (1951, 4100, 7555, 8104, 8291, 8877, 9766,),
    (9, 94): (697, 4101, 7555, 8103, 8872, 9763, 12762,),
    (9, 98): (2494, 7484, 7548,),
    (9, 150): (8101,),
    (9, 156): (728, 2033,),
    (9, 167): (697, 4111, 6616, 7559, 8110, 8292, 8886, 9773, 10237, 12760,),
    (9, 172): (5779,),
    (9, 176): (10467,),
    (9, 193): (414, 614, 640, 739, 781, 837, 1085, 1275, 1726, 1757, 1923, 2077, 3691, 3806, 4096, 4707, 5667, 6006, 6614, 6746, 6877, 6996, 7557, 8108, 8150, 8869, 8993, 9763, 9920, 10259, 10287, 10699, 10737, 11913, 12287, 12478, 12687, 12801, 12941, 13016, 13165,),
    (9, 227): (7486, 7554,),
    (9, 229): (10466,),
    (9, 238): (4101, 6591, 7555, 8103, 9765,),
    (10, 3): (8922,),
    (10, 22): (723, 3724, 4130, 5654, 5682, 5707, 6767, 7031, 8179, 8871, 8933, 8937, 9033, 10277, 10329, 12670, 12784, 12833,),
    (10, 56): (761, 1274, 1916, 2009, 3691, 3828, 3850, 6041, 6807, 7021, 8151, 8189, 10288, 10574, 10610, 10620, 10746, 12736, 12818, 13336, 13369,),
    (10, 57): (738,),
    (10, 77): (435, 615, 697, 764, 781, 1928, 1984, 2037, 2087, 2137, 2194, 2997, 3055, 3735, 3983, 4134, 4256, 4723, 5656, 5688, 5839, 6114, 6608, 6635, 6649, 6670, 6754, 6782, 7172, 7613, 7638, 8898, 8918, 9013, 9031, 9040, 9577, 9798, 10249, 10273, 10315, 10599, 11523, 11556, 11926, 12298, 12329, 12640, 12662, 12805, 12848, 13000, 13202,),
    (10, 100): (739, 3667, 7020, 7047, 8164, 9007,),
    (10, 147): (4800, 8920,),
    (10, 173): (751, 12813,),
    (10, 187): (712, 756, 1394, 1991, 2067, 2199, 2468, 3939, 4155, 4251, 4726, 5692, 6645, 6822, 7026, 7520, 7616, 8064, 8128, 8149, 8172, 8424, 8805, 8902, 8925, 8996, 9008, 9108, 9705, 9734, 9818, 10245, 10279, 11018,),
    (10, 188): (439, 594, 724, 764, 771, 801, 823, 1068, 1156, 1288, 1939, 2975, 3012, 3024, 3060, 3704, 3733, 4703, 5699, 6767, 6804, 6895, 7027, 8153, 10253, 10288, 10550, 11923, 12295, 12326, 12502, 12654, 12822, 13173, 13217, 13353,),
    (10, 205): (733, 762, 10290, 12794, 12823,),
    (10, 216): (7518,),
    (10, 224): (561, 602, 758, 1148, 1414, 1829, 1933, 1990, 2060, 2089, 2515, 3055, 3708, 3840, 4152, 4686, 4734, 4757, 5696, 6071, 6719, 6842, 6927, 6950, 7052, 7167, 8155, 8188, 8457, 8881, 8923, 9003, 9796, 9838, 9847, 9947, 9979, 10267, 10293, 10314, 10339, 10600, 11536, 11668, 12140, 12311, 12508, 12647, 12696, 12802, 12810, 13184, 13225, 13334, 13477,),
    (10, 234): (610, 701, 754, 775, 933, 1158, 1899, 1986, 2096, 2127, 2978, 3691, 3729, 3739, 3965, 4136, 5789, 6025, 6626, 6724, 6766, 6795, 6902, 6942, 7058, 7133, 7596, 7876, 7908, 8916, 9944, 9972, 10296, 10328, 10560, 10727, 11935, 11976, 12126, 12268, 12371, 12407, 12483, 12532, 12654, 12764, 12823, 12997, 13174, 13329,),
    (10, 239): (746, 2023, 12786,),
    (10, 250): (7513,),
    (10, 275): (622, 1406, 1920, 1924, 1961, 2000, 2009, 2057, 2090, 2202, 3755, 3911, 3934, 4156, 4255, 6613, 6638, 7520, 7623, 8136, 8174, 8333, 8894, 8923, 9111, 9119, 9727, 9809, 10155, 10293, 11018, 12391, 12685,),
    (10, 278): (254, 441, 743, 1166, 1307, 1909, 2078, 2103, 2132, 3727, 6621, 10579, 10747, 12122, 12319, 12351, 12504, 12812, 12830, 13181, 13228, 13254,),
    (10, 281): (7511,),
    (10, 286): (12808,),
    (11, 3): (722, 2078, 2165, 3806, 3907, 4121, 6624, 6964, 7049, 7594, 8139, 9003, 9795, 12786, 13426,),
    (11, 11): (1977, 4116, 4860, 7579, 8123, 8975, 9153, 9788, 10491, 11067, 11385,),
    (11, 22): (295, 588, 724, 1366, 1753, 1972, 2153, 2205, 2224, 2247, 3919, 4130, 5685, 6591, 6618, 6649, 7586, 8148, 8952, 8992, 9019, 9082, 9793, 10566, 12360, 12442, 12652, 12789, 13430,),
    (11, 29): (7575,),
    (11, 36): (2522, 2582, 5377, 7506, 7576,),
    (11, 50): (9779,),
    (11, 61): (325, 402, 591, 729, 829, 1375, 1759, 1871, 1968, 2071, 2162, 2200, 3775, 3819, 3853, 3928, 3955, 4133, 4697, 5686, 6615, 6635, 6747, 6883, 6956, 6990, 7593, 8143, 8173, 8402, 8893,),
    (11, 80): (7576, 9782,),
    (11, 84): (7570,),
    (11, 85): (7578, 9781,),
    (11, 87): (727,),
    (11, 89): (1969, 2060, 2162, 4123, 5663, 6618, 6651, 6676, 8129, 8984, 9796, 13422,),
    (11, 112): (7573, 9778,),
    (11, 127): (727, 12795,),
    (11, 132): (7569,),
    (11, 147): (9776,),
    (11, 148): (9781,),
    (11, 150): (7576,),
    (11, 164): (7573,),
    (11, 169): (7572, 9777,),
    (11, 186): (7575,),
    (11, 187): (7578,),
    (11, 205): (9781,),
    (11, 240): (2058, 5659, 8132, 9786,),
    (11, 252): (7573,),
    (11, 256): (7578, 9781,),
    (11, 264): (7575,),
    (11, 278): (7575,),
    (11, 291): (4118, 5657, 7580, 8124, 8978, 9786,),
    (11, 296): (3697, 5809, 7583,),
}
assert sum(len(v) for v in SNAP_EVENT_TIMES.values()) == 1388, 'snap catalogue is not the patch-29 corrected one'

# --- patch sr-21: samples where the anchor out-pulls the fairlead by more than a
# factor of three (scan_anchor_outpull.py -> anchor_outpull_samples.csv, the
# residue of Appendix D). Read ONLY by the site-window filter in the dump cell.
# It is deliberately NOT part of the exclusion mask -- adding it there would
# change the split, the training pool and every published metric.
OUTPULL_SAMPLES = {    # (loc, case) -> out-pulling sample indices
    (1, 63): [8098],
    (1, 116): [12794],
    (1, 202): [731, 3659, 3681, 3695, 3835, 3841, 3843, 3882, 6625, 6954, 8146, 8892],
    (1, 215): [682, 683, 737, 1776, 1901, 2063, 2104, 3687, 3700, 3711],
    (1, 298): [728],
    (4, 45): [702, 10229, 10238],
    (4, 114): [12801],
    (4, 133): [9758],
    (4, 164): [8775],
    (5, 22): [744, 2087],
    (5, 31): [8958, 8975, 8976],
    (5, 35): [8948, 8960, 8968, 8986],
    (5, 46): [1963, 1969, 1971, 1974, 1978, 1979, 1980, 1988],
    (5, 108): [13411],
    (5, 197): [8963, 8970],
    (5, 262): [603, 615, 618],
    (5, 264): [7553, 7559, 7566],
    (6, 22): [1993, 4139, 8306, 8307, 8905, 8998, 9674, 9717, 9794],
    (6, 65): [743, 12807],
    (6, 120): [703, 739, 2092, 8801, 12817, 12818],
    (6, 126): [739, 12803],
    (6, 131): [710, 717, 745, 746, 786, 2081, 2086, 3700, 3820, 4702, 6627, 6632, 6755, 7016, 8163, 8906, 10291, 10581, 12800, 12801, 12802, 12821, 12822],
    (6, 134): [457, 588, 589, 711, 717, 726, 730, 740, 745, 746, 756, 762, 767, 951, 952, 956, 1159, 1160, 1165, 1917, 1918, 1920, 1922, 1946, 1960, 1965, 2056, 2103, 2124, 4727, 6027, 6641, 6642, 8168, 8169, 8991, 9968, 9969, 10141, 10297, 10331, 10544, 10566, 10570, 10571, 10577, 10589, 10619, 10624, 10742, 11892, 11897, 11898, 11899, 11903, 11944, 12104, 12130, 12164, 12165, 12176, 12276, 12298, 12330, 12496, 12497, 12793, 12794, 12800, 12818, 12831, 12832, 13014, 13018, 13225],
    (6, 222): [7504],
    (7, 174): [743, 2085, 8115],
    (7, 212): [2007, 2014, 2023, 4154, 4171, 4669, 4672, 6654, 6660, 6661, 6671, 6672, 8919],
    (8, 83): [2067, 6583, 10278, 12803],
    (8, 94): [741, 6613, 8084, 8139],
    (8, 136): [602, 612, 616, 617, 618, 619, 678, 743, 746, 758, 760, 761, 762, 764, 766, 772, 1770, 2095, 2102, 2103, 2104, 3818, 4711, 6638, 6645, 6646, 7009, 7129, 10219, 10288, 10591, 10603, 10604, 10620, 10651, 12278, 12652, 12669, 12676, 12681, 12683, 12684, 12685, 12686, 12687, 12689, 12691, 12692, 12774, 12806, 12824, 12828, 12829, 12830, 12832, 12833, 12835, 12836],
    (8, 143): [1766, 2076, 3691, 3815, 5674, 8152, 10271, 10576, 12652],
    (8, 229): [738],
    (8, 247): [2067, 3691, 6622, 8909, 8913, 8917, 9758, 9799, 12791],
    (9, 34): [9714],
    (9, 43): [7511, 8908, 9710, 9726, 9727, 9732, 9733],
    (9, 156): [2015],
    (9, 193): [597, 598, 605, 610, 615, 616, 622, 765, 768, 769, 770, 1917, 2037, 10233, 10573, 10595, 10601, 10620, 12282, 12659, 12663, 12668, 12812, 12814, 12819, 12824, 12828, 12831, 12837, 12847],
    (10, 22): [601, 739, 759, 2082, 2083, 3704, 3826, 3832, 4107, 4112, 4707, 6632, 6638, 6760, 7024, 8172, 8870, 8884, 8886, 8888, 8889, 8890, 8891, 8903, 8928, 8929, 8940, 8979, 8988, 8999, 9000, 9780, 10223, 10224, 10234, 10236, 10238, 10304, 10586, 10587, 12665, 12765, 12828],
    (10, 56): [429, 587, 590, 608, 609, 725, 746, 770, 1943, 1950, 2092, 3773, 3844, 4739, 5685, 6034, 6629, 6634, 6772, 6775, 6798, 7016, 8129, 8135, 8174, 9004, 10312, 10553, 10577, 10584, 10587, 10606, 10739, 11931, 12288, 12657, 12761, 12786, 12796, 12814, 12834, 12835, 13175, 13364],
    (10, 57): [733, 12796],
    (10, 77): [450, 564, 631, 686, 691, 708, 712, 726, 728, 731, 735, 736, 744, 759, 952, 1136, 1160, 1273, 1783, 1784, 1788, 1953, 1954, 1958, 2043, 2046, 2051, 2057, 2074, 2097, 2970, 2976, 3665, 3667, 3674, 3700, 3727, 3795, 3801, 3813, 3820, 3831, 3851, 4129, 4250, 4251, 4590, 4748, 5683, 5703, 6022, 6592, 6594, 6596, 6599, 6615, 6621, 6632, 6772, 6773, 6898, 6902, 7029, 7164, 7593, 7596, 7884, 7891, 8117, 8165, 8166, 8992, 8993, 9028, 9572, 9792, 9811, 9948, 10152, 10156, 10244, 10258, 10308, 10442, 10539, 10543, 10567, 10582, 10595, 10611, 10728, 11206, 11547, 11665, 11892, 11943, 11949, 12112, 12118, 12119, 12123, 12124, 12146, 12155, 12156, 12248, 12270, 12489, 12631, 12649, 12665, 12666, 12682, 12683, 12775, 12777, 12788, 12797, 12815, 12823, 13016, 13020, 13218, 13219, 13357],
    (10, 100): [734, 3712, 4139, 6641, 6642, 7040, 8157, 8896, 9002, 9800, 12797, 12798],
    (10, 173): [744, 12807, 12808],
    (10, 187): [320, 1388, 1873, 1899, 1959, 2000, 2066, 2070, 2079, 2192, 3714, 3715, 3934, 4114, 4146, 4244, 4696, 4813, 5685, 6515, 6620, 6725, 6898, 7021, 8043, 8051, 8122, 8718, 8796, 8797, 8866, 8875, 8885, 9014, 9672, 9674, 9699, 9728, 9783, 9811, 10161, 12384, 12790],
    (10, 188): [433, 591, 610, 611, 705, 708, 709, 710, 713, 757, 767, 1148, 1149, 1283, 1772, 1773, 1954, 2073, 3720, 3827, 5694, 5695, 6630, 6785, 6798, 6891, 7022, 8175, 8910, 9009, 9010, 10256, 10310, 10311, 10547, 10570, 10600, 10601, 10713, 11950, 12289, 12497, 12668, 12669, 12770, 12791, 12830, 13209, 13210, 13347],
    (10, 205): [756, 2082, 10245, 12817, 12818],
    (10, 224): [222, 555, 560, 562, 563, 576, 626, 710, 728, 747, 771, 1137, 1276, 1277, 1301, 1395, 1765, 1766, 1877, 1948, 1954, 2072, 2098, 2507, 2964, 2966, 2968, 2969, 2978, 2985, 3018, 3667, 3680, 3697, 3716, 3726, 3814, 3828, 3849, 4126, 4147, 4262, 4692, 5694, 6035, 6036, 6064, 6591, 6613, 6614, 6617, 6647, 6718, 6722, 6723, 6765, 6878, 6915, 7029, 7030, 7159, 7603, 7733, 7881, 8020, 8030, 8031, 8117, 8169, 8448, 8880, 8891, 8895, 8896, 8899, 8900, 8998, 8999, 9030, 9586, 9789, 9791, 9819, 9833, 9835, 9837, 9972, 10250, 10283, 10284, 10297, 10317, 10328, 10432, 10557, 10570, 10585, 10607, 10686, 10723, 10747, 11212, 11213, 11531, 11662, 11918, 12137, 12156, 12272, 12300, 12374, 12467, 12468, 12469, 12471, 12472, 12475, 12484, 12519, 12618, 12631, 12644, 12660, 12680, 12793, 12833, 13174, 13216, 13319, 13340, 13359, 13360, 13469],
    (10, 234): [237, 400, 423, 448, 449, 586, 590, 617, 683, 687, 692, 694, 708, 709, 710, 711, 712, 713, 716, 717, 720, 730, 731, 755, 768, 950, 955, 1150, 1284, 1784, 1789, 1953, 2039, 2078, 2093, 3042, 3667, 3670, 3672, 3690, 3708, 3726, 3818, 3819, 3844, 4132, 4240, 4586, 4590, 4593, 4683, 4726, 5682, 5784, 5875, 6046, 6622, 6623, 6642, 6643, 6721, 6751, 6764, 6790, 6890, 6892, 7844, 7863, 8173, 8174, 8179, 8911, 9001, 9016, 9602, 9610, 9780, 9790, 9792, 9818, 9966, 9967, 10288, 10322, 10580, 10610, 10744, 10750, 11214, 11897, 11906, 11907, 11927, 11932, 11962, 11970, 12137, 12146, 12147, 12150, 12284, 12464, 12466, 12468, 12469, 12526, 12652, 12676, 12677, 12747, 12748, 12757, 12758, 12763, 12784, 12785, 12800, 12808, 12821, 12827, 13013, 13014, 13120, 13200, 13207],
    (10, 239): [709, 711, 758, 2093],
    (10, 275): [322, 616, 722, 1398, 1872, 1881, 1883, 1899, 1906, 1956, 1957, 1958, 1959, 1960, 1963, 1993, 2085, 2086, 3688, 3702, 3722, 3798, 3816, 3821, 3841, 3950, 4124, 4146, 4683, 4713, 5660, 5699, 6513, 6532, 6609, 6649, 6720, 6892, 7024, 8041, 8054, 8169, 8874, 8885, 8918, 8919, 8984, 9019, 9063, 9696, 9801, 9808, 10279, 10592, 12386, 12679, 12796, 13462],
    (10, 278): [247, 435, 588, 729, 730, 762, 892, 936, 956, 961, 1116, 1160, 1161, 1979, 2122, 3719, 4737, 6642, 6643, 6753, 8168, 9962, 10290, 10594, 10595, 10600, 10740, 11931, 11937, 12139, 12143, 12341, 12499, 12652, 12657, 12768, 12789, 12791, 12798, 12806, 12822, 12827, 13014, 13015, 13143, 13147, 13197, 13221, 13222, 13249],
    (10, 286): [738, 12802],
    (11, 3): [7037, 8950],
    (11, 22): [2149, 6577, 6579, 6580, 8962],
    (11, 61): [3721, 3729, 3731, 3734, 3735, 3736, 3742, 3745, 3746, 3868, 3869, 3870, 3871, 3872, 3914, 3920, 3923, 3935, 8921],
    (11, 89): [6584, 6586, 6587, 6588, 6589, 6590, 6607],
}
assert len(OUTPULL_SAMPLES) == 55, len(OUTPULL_SAMPLES)
assert sum(len(v) for v in OUTPULL_SAMPLES.values()) == 1047, 'out-pulling scan is not the Appendix-D one'

# --- patch sr-22: the EVALUATION mask's catenary rule (RULE C, CLAUDE.md 12.14) ---
# Samples in which a station carrying more than 1 kN exceeds the fairlead tension at
# the same instant (scan_artifact_samples.py RULE C, k = 1.0), i.e.
# artifact_samples_cat.csv minus artifact_samples.csv. The solver's line is
# frictionless on the seabed and axially quasi-static at dt = 0.1 s, so no station can
# out-pull the fairlead; these samples also miss the axial momentum balance against
# the recorded motion by 7-25 kN. The two shipped models were TRAINED without this
# rule, so it is NOT part of the split: ARTIFACT_SAMPLES builds every draw exactly as
# published, and only the EVAL_CLEAN mode reads this table, to drop the TEST windows
# that cover one (build_dataloaders(test_window_drop=...)).
CATENARY_SAMPLES = {
    (1, 63): (643, 5317, 5318, 5322, 7446, 7447, 7507, 7524, 7525, 7526, 7532, 7536, 7586, 7624, 7678, 8112, 12706, 12766,),
    (1, 116): (730, 2062, 2063, 12800,),
    (1, 142): (7514,),
    (1, 202): (420, 736, 1779, 1780, 1965, 2074, 2095, 2179, 3695, 3813, 3814, 3818, 3841, 3844, 3845, 3846, 3847, 3848, 3849, 3853, 3854, 3855, 3871, 3886, 3940, 3941, 3950, 4129, 4130, 4135, 4696, 4701, 5673, 5674, 5675, 6507, 6637, 6638, 6730, 7001, 7514, 7536, 7595, 8146, 8154, 8896, 8902,),
    (1, 215): (589, 590, 594, 736, 741, 742, 743, 748, 749, 1257, 1375, 1776, 1780, 1901, 1902, 2054, 2084, 2090, 2091, 2105, 2106, 2107, 2108, 2110, 2113, 2114, 2118, 2119, 2120, 2121, 2123, 2250, 3648, 3687, 3706, 3707, 3708, 3709, 3710,),
    (1, 298): (728, 729,),
    (3, 9): (720, 726, 12783, 12784,),
    (3, 32): (2485,),
    (3, 164): (12742,),
    (4, 21): (7557,),
    (4, 45): (2073, 10293, 10294, 10302, 12805,),
    (4, 79): (719, 2073, 2087, 8149, 12798,),
    (4, 114): (737, 738, 1907, 2080, 2081, 10265, 12262,),
    (4, 133): (9806,),
    (4, 164): (6863, 8776, 8805, 8806,),
    (4, 195): (7488,),
    (4, 270): (3722, 3723, 3799, 12791, 12792,),
    (5, 22): (750, 1919, 1920, 2087, 2093, 8145, 10276, 12279, 12807, 12808, 12813,),
    (5, 31): (8957, 8977, 8979, 8982, 9005, 9024,),
    (5, 35): (2083, 8936, 8944, 8947, 8949, 8950, 8961, 9038, 9083,),
    (5, 46): (1957, 1962, 1964, 1966, 1967, 1977, 1980, 1982, 1983, 1985, 1986, 2008, 2076, 2100, 7585, 8924,),
    (5, 108): (13459,),
    (5, 194): (1990, 10506,),
    (5, 197): (8957, 8958, 8962, 8964, 8967, 9007, 9008, 10614,),
    (5, 202): (580,),
    (5, 262): (430, 431, 436, 611,),
    (5, 264): (7554, 7570, 7583, 7584, 7607,),
    (6, 22): (1993, 1997, 1998, 4139, 4144, 7510, 8024, 8025, 8026, 8027, 8028, 8029, 8030, 8031, 8032, 8033, 8035, 8036, 8038, 8039, 8040, 8044, 8047, 8049, 8050, 8051, 8052, 8053, 8054, 8055, 8056, 8057, 8058, 8059, 8301, 8302, 8303, 8306, 8307, 8343, 8344, 8794, 8795, 8796, 8875, 8878, 8879, 8880, 8905, 8906, 8909, 8917, 8918, 8992, 8993, 8998, 9001, 9002, 9003, 9094, 9681, 9685, 9687, 9694, 9695, 9716, 9717, 9721, 9722, 9794, 9798, 9799, 10244,),
    (6, 37): (8907,),
    (6, 65): (749, 750, 12806, 12807, 12813,),
    (6, 120): (595, 596, 597, 598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 613, 614, 739, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766, 767, 768, 769, 770, 2092, 2093, 2097, 8801, 8810, 8811, 12135, 12136, 12137, 12138, 12139, 12140, 12141, 12142, 12143, 12144, 12145, 12146, 12147, 12148, 12149, 12150, 12151, 12152, 12153, 12154, 12155, 12156, 12157, 12158, 12159, 12160, 12161, 12295, 12296, 12659, 12660, 12661, 12662, 12663, 12664, 12665, 12666, 12667, 12668, 12669, 12670, 12671, 12672, 12673, 12674, 12675, 12676, 12677, 12801, 12817, 12818, 12822, 12823, 12824, 12825, 12826, 12827, 12828, 12829, 12830, 12831, 12832, 12833, 12834, 12835,),
    (6, 126): (12809,),
    (6, 131): (603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619, 716, 717, 723, 737, 740, 744, 745, 746, 756, 757, 767, 772, 773, 786, 794, 2080, 2081, 2086, 2089, 2090, 2091, 2092, 2093, 2094, 2095, 2096, 2097, 3700, 3704, 3705, 3707, 3708, 3709, 3710, 3711, 3712, 3713, 3714, 3715, 3716, 3717, 3718, 3719, 3720, 3820, 3821, 3824, 4702, 5682, 5683, 6626, 6627, 6632, 6635, 6636, 6637, 6638, 6639, 6640, 6641, 6642, 6643, 6644, 6645, 6646, 6647, 6755, 6889, 6890, 6891, 6892, 6893, 6894, 6895, 6896, 6897, 6898, 6899, 6900, 6901, 6902, 6903, 6904, 6905, 7016, 8023, 8024, 8025, 8026, 8027, 8028, 8029, 8030, 8031, 8032, 8033, 8034, 8035, 8036, 8037, 8038, 8039, 8040, 8041, 8042, 8043, 8044, 8045, 8046, 8047, 8048, 8049, 8050, 8051, 8052, 8053, 8054, 8055, 8056, 8057, 8058, 8059, 8162, 8163, 8169, 8905, 8906, 8910, 8913, 8914, 10291, 10298, 10581, 10582, 10585, 10586, 10587, 10588, 10590, 10591, 10592, 10593, 10594, 10595, 10596, 10597, 12666, 12667, 12668, 12669, 12670, 12671, 12672, 12673, 12674, 12675, 12676, 12677, 12678, 12679, 12680, 12681, 12682, 12683, 12800, 12801, 12803, 12819, 12821, 12822, 12826,),
    (6, 134): (227, 228, 429, 430, 457, 465, 466, 588, 589, 592, 593, 594, 595, 596, 597, 598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 715, 717, 721, 723, 726, 727, 729, 730, 731, 732, 733, 734, 735, 736, 737, 738, 739, 740, 741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766, 767, 768, 769, 770, 771, 772, 773, 774, 775, 776, 777, 778, 779, 780, 781, 782, 783, 784, 785, 786, 787, 788, 927, 941, 951, 952, 955, 956, 957, 958, 959, 960, 961, 962, 963, 964, 965, 966, 967, 968, 969, 970, 971, 972, 973, 974, 975, 976, 977, 978, 979, 980, 981, 982, 983, 984, 985, 986, 987, 988, 989, 990, 1036, 1037, 1038, 1039, 1040, 1045, 1135, 1148, 1159, 1160, 1165, 1166, 1168, 1169, 1170, 1171, 1172, 1173, 1174, 1175, 1333, 1344, 1760, 1761, 1762, 1763, 1764, 1765, 1766, 1767, 1768, 1769, 1770, 1771, 1772, 1773, 1774, 1775, 1776, 1777, 1778, 1779, 1780, 1781, 1782, 1783, 1784, 1785, 1786, 1787, 1788, 1789, 1790, 1791, 1792, 1793, 1794, 1795, 1796, 1797, 1798, 1799, 1800, 1801, 1802, 1803, 1804, 1805, 1806, 1807, 1918, 1919, 1920, 1922, 1923, 1928, 1929, 1931, 1933, 1934, 1946, 1949, 1950, 1959, 1960, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2061, 2062, 2064, 2084, 2088, 2103, 2104, 2108, 2109, 2124, 2131, 2939, 2940, 3558, 3559, 3560, 3561, 3562, 3563, 3564, 3565, 3566, 3567, 3568, 3569, 3570, 3571, 3572, 3573, 3574, 3575, 3576, 3577, 3578, 3579, 3580, 3581, 3582, 3583, 3584, 3585, 3586, 3587, 3588, 3589, 3590, 3591, 3592, 3593, 3594, 3595, 3696, 3807, 3808, 3809, 3810, 3811, 3812, 3813, 3814, 3815, 3816, 3817, 3818, 3819, 3820, 3821, 3822, 3823, 3824, 3825, 3826, 3827, 3828, 3829, 3830, 3831, 3832, 3833, 3834, 3835, 3836, 3837, 3838, 3839, 3840, 3841, 4576, 4577, 4578, 4579, 4580, 4581, 4582, 4583, 4584, 4585, 4586, 4587, 4588, 4589, 4590, 4591, 4592, 4593, 4594, 4595, 4596, 4597, 4598, 4599, 4600, 4601, 4602, 4603, 4604, 4605, 4606, 4607, 4608, 4609, 4610, 4611, 4612, 4727, 4728, 4733, 5669, 5670, 5671, 5672, 5673, 5674, 5675, 5676, 5677, 5678, 5679, 5680, 5681, 5682, 5683, 5684, 5685, 5686, 5687, 5688, 5689, 5690, 5691, 5692, 5693, 5694, 5695, 5696, 5697, 5698, 5699, 5700, 5701, 5702, 5703, 5704, 6030, 6031, 6032, 6033, 6034, 6035, 6036, 6037, 6038, 6039, 6040, 6041, 6042, 6043, 6044, 6045, 6046, 6047, 6048, 6049, 6050, 6624, 6641, 6642, 6647, 6648, 8168, 8169, 8174, 8785, 8786, 8787, 8788, 8789, 8790, 8791, 8792, 8793, 8794, 8795, 8796, 8797, 8798, 8799, 8800, 8991, 8994, 8995, 8996, 8997, 8998, 8999, 9000, 9001, 9002, 9003, 9004, 9005, 9006, 9007, 9008, 9009, 9010, 9011, 9012, 9013, 9014, 9015, 9016, 9017, 9018, 9019, 9020, 9021, 9022, 9023, 9024, 9025, 9026, 9027, 9028, 9029, 9404, 9405, 9406, 9407, 9408, 9409, 9410, 9411, 9412, 9413, 9414, 9415, 9416, 9417, 9418, 9419, 9420, 9421, 9422, 9423, 9424, 9425, 9426, 9427, 9428, 9429, 9430, 9431, 9432, 9433, 9434, 9435, 9436, 9437, 9563, 9779, 9780, 9781, 9782, 9783, 9784, 9785, 9786, 9787, 9788, 9789, 9790, 9791, 9792, 9793, 9794, 9795, 9796, 9797, 9798, 9799, 9800, 9801, 9802, 9803, 9804, 9805, 9806, 9807, 9808, 9809, 9810, 9811, 9812, 9813, 9814, 9815, 9816, 9817, 9818, 9819, 9820, 9948, 9968, 9969, 9973, 9974, 9976, 9977, 9978, 9979, 9980, 9981, 10141, 10142, 10144, 10145, 10146, 10147, 10148, 10149, 10150, 10151, 10152, 10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162, 10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172, 10173, 10174, 10175, 10176, 10177, 10178, 10179, 10180, 10181, 10182, 10183, 10184, 10185, 10304, 10331, 10338, 10559, 10569, 10572, 10573, 10574, 10576, 10579, 10582, 10585, 10589, 10592, 10619, 10622, 10623, 10624, 10625, 10742, 11899, 11900, 11901, 11903, 11911, 11922, 11936, 11937, 11944, 11945, 11947, 11948, 11949, 11950, 11951, 11952, 11953, 11954, 11955, 11956, 11957, 11958, 11959, 11960, 11961, 11962, 11963, 11964, 12132, 12141, 12146, 12151, 12152, 12153, 12162, 12163, 12164, 12165, 12176, 12177, 12179, 12180, 12181, 12182, 12183, 12216, 12217, 12218, 12219, 12220, 12221, 12264, 12275, 12276, 12277, 12298, 12316, 12329, 12330, 12338, 12339, 12496, 12497, 12501, 12637, 12638, 12639, 12640, 12641, 12642, 12643, 12644, 12645, 12646, 12647, 12648, 12649, 12650, 12651, 12652, 12653, 12654, 12655, 12656, 12657, 12658, 12659, 12660, 12661, 12662, 12663, 12664, 12665, 12666, 12667, 12668, 12669, 12670, 12671, 12672, 12673, 12674, 12675, 12676, 12677, 12678, 12679, 12680, 12681, 12682, 12683, 12684, 12685, 12686, 12687, 12688, 12689, 12690, 12691, 12692, 12718, 12775, 12793, 12794, 12796, 12797, 12798, 12799, 12800, 12801, 12802, 12803, 12804, 12805, 12806, 12807, 12808, 12809, 12810, 12811, 12812, 12814, 12815, 12816, 12817, 12818, 12820, 12821, 12822, 12823, 12824, 12825, 12830, 12831, 12832, 12834, 12835, 12836, 12837, 12838, 12839, 12840, 12841, 12842, 12843, 12844, 12845, 12846, 12847, 12848, 12849, 12850, 12851, 13018, 13019, 13020, 13021, 13022, 13023, 13024, 13025, 13026, 13027, 13028, 13029, 13030, 13031, 13032, 13033, 13034, 13035, 13036, 13037, 13038, 13039, 13040, 13041, 13042, 13043, 13044, 13045, 13046, 13047, 13048, 13049, 13050, 13051, 13052, 13053, 13054, 13098, 13100, 13102, 13104, 13164, 13185, 13201, 13213, 13225, 13229, 13230, 13231, 13232, 13233, 13234, 13235, 13236, 13237, 13238, 13239, 13305, 13396,),
    (6, 182): (2428, 2429, 2469, 2528, 3641, 7448, 7519, 7579,),
    (6, 222): (666, 2470, 2527, 3640, 5321, 5388, 5749, 5817, 7446, 7479, 7480, 7481, 7482, 7483, 7484, 7485, 7486, 7487, 7504, 7508, 7520, 7521, 7589, 12730, 12731,),
    (7, 68): (737, 738, 743, 2073, 12801,),
    (7, 117): (8104,),
    (7, 174): (749, 1917, 2085, 8115, 8116, 8161, 8169, 10236, 10286, 12279, 12806, 12807,),
    (7, 212): (1738, 1740, 1741, 1784, 2000, 2001, 2020, 2022, 2028, 2029, 2038, 2064, 2144, 2179, 2192, 2218, 2260, 3717, 3718, 3724, 3725, 3726, 3811, 3812, 3871, 3926, 4091, 4100, 4101, 4145, 4146, 4147, 4148, 4155, 4156, 4162, 4173, 4174, 4175, 4176, 4177, 4178, 4182, 4184, 4185, 4186, 4187, 4206, 4212, 4281, 4673, 4675, 4729, 6647, 6649, 6650, 6651, 6652, 6653, 6657, 6661, 6662, 6670, 6672, 6673, 6674, 6675, 6676, 7557, 8121, 8414,),
    (8, 37): (4679,),
    (8, 77): (719, 12783, 12788,),
    (8, 83): (739, 740, 745, 2079, 2080, 3685, 3690, 4694, 6610, 6729, 8146, 8153, 10278, 10568, 12807,),
    (8, 94): (741, 742, 1899, 2079, 2087, 3688, 3692, 4699, 6613, 6618, 6737, 8139, 8145, 9003, 9004, 9009, 10268, 10282, 10283, 10571, 10572, 12805, 12806, 12811,),
    (8, 109): (723, 12787,),
    (8, 136): (418, 588, 589, 592, 597, 611, 612, 613, 614, 615, 616, 617, 618, 619, 622, 745, 746, 753, 758, 759, 760, 761, 762, 763, 764, 765, 766, 767, 768, 772, 773, 774, 1087, 1088, 1268, 1269, 1770, 1797, 1798, 1802, 1803, 1804, 1919, 1920, 2084, 2085, 2088, 2089, 2090, 2091, 2098, 2102, 2103, 2104, 2105, 2106, 3694, 3695, 3699, 3709, 3823, 4114, 4231, 4711, 4718, 5676, 5677, 5682, 6620, 6621, 6625, 6645, 6646, 6647, 6650, 6651, 6652, 6653, 6654, 6655, 6656, 6662, 6748, 6749, 6755, 6867, 6913, 6914, 7009, 7129, 7135, 7577, 8053, 8054, 8055, 8058, 8157, 8407, 8896, 9000, 9001, 9005, 10222, 10288, 10556, 10580, 10581, 10590, 10591, 10592, 10593, 10603, 10604, 10606, 10607, 10610, 10624, 10626, 10728, 10729, 10735, 11917, 12278, 12482, 12487, 12660, 12672, 12676, 12681, 12682, 12683, 12684, 12685, 12686, 12687, 12689, 12690, 12691, 12692, 12693, 12694, 12696, 12809, 12811, 12821, 12822, 12824, 12828, 12829, 12830, 12832, 12833, 12835, 12836, 12837, 12838, 13151, 13333,),
    (8, 143): (588, 589, 736, 737, 741, 1766, 1894, 2081, 3696, 3815, 3816, 4638, 4643, 4644, 4718, 5680, 5706, 6619, 6620, 6624, 6638, 6639, 6742, 6743, 6882, 7006, 7007, 7012, 7121, 8056, 8152, 8886, 8887, 8905, 8906, 8912, 8980, 9721, 9822, 9832, 10270, 10271, 10278, 10279, 10576, 10580, 10597, 12656, 12800, 12801, 12804,),
    (8, 229): (737, 738, 744, 2069, 2070, 12801, 12802,),
    (8, 247): (727, 728, 732, 2067, 2072, 3696, 4105, 4128, 4129, 4691, 4692, 5619, 5623, 5677, 5678, 6571, 6622, 6726, 6998, 6999, 8906, 8907, 8908, 8913, 8914, 8915, 8916, 8917, 8918, 8920, 8998, 8999, 9733, 9758, 9799, 10258,),
    (9, 34): (1938, 9137, 9713, 9714,),
    (9, 43): (1986, 1987, 3117, 4791, 4792, 7510, 7511, 7512, 8031, 8296, 8907, 8908, 8909, 9719, 9727, 9728, 9729, 9736, 9737, 9791, 9854, 9870,),
    (9, 98): (5335,),
    (9, 156): (727, 728, 733, 2059, 2060, 12791, 12792,),
    (9, 167): (6616, 8877, 8878,),
    (9, 173): (7543,),
    (9, 193): (414, 594, 595, 613, 614, 617, 618, 619, 620, 621, 640, 647, 739, 740, 741, 742, 764, 765, 766, 768, 769, 770, 771, 772, 773, 774, 777, 778, 1085, 1275, 1757, 1795, 1796, 1801, 1802, 1917, 1918, 1995, 2083, 2084, 3690, 3691, 3695, 3806, 4707, 5666, 5667, 6006, 6614, 6634, 6635, 6636, 6641, 6740, 6741, 6877, 6996, 7001, 8150, 8992, 8993, 9920, 10280, 10281, 10573, 10574, 10586, 10587, 10604, 10608, 10609, 10617, 10618, 10619, 10699, 10726, 10737, 11913, 11918, 12282, 12478, 12482, 12648, 12658, 12659, 12678, 12679, 12680, 12683, 12684, 12685, 12804, 12805, 12806, 12827, 12829, 12830, 12831, 12832, 12834, 13008, 13016, 13165, 13331, 13332,),
    (10, 22): (601, 723, 739, 740, 759, 764, 2082, 2083, 2087, 2088, 2090, 2091, 2092, 2093, 2094, 2095, 3704, 3705, 3708, 3709, 3710, 3711, 3712, 3713, 3714, 3715, 3716, 3717, 3724, 3826, 3832, 3835, 4113, 4122, 4127, 4130, 4707, 4708, 5650, 5651, 5652, 5654, 5664, 5682, 5706, 5707, 6632, 6637, 6638, 6641, 6642, 6643, 6644, 6645, 6759, 6760, 6767, 6887, 6888, 6889, 6890, 6891, 6892, 6893, 6894, 6895, 6896, 6897, 6898, 6899, 6900, 6901, 6902, 6903, 6904, 6905, 7024, 7031, 8024, 8025, 8026, 8027, 8028, 8029, 8030, 8031, 8032, 8033, 8034, 8035, 8036, 8037, 8038, 8039, 8040, 8041, 8042, 8043, 8044, 8045, 8046, 8047, 8048, 8049, 8050, 8051, 8052, 8053, 8054, 8055, 8056, 8057, 8058, 8172, 8173, 8179, 8869, 8871, 8876, 8883, 8886, 8887, 8889, 8890, 8891, 8892, 8893, 8894, 8896, 8897, 8898, 8899, 8900, 8903, 8918, 8926, 8932, 8937, 8996, 8999, 9028, 9029, 9033, 9780, 9798, 10224, 10229, 10230, 10232, 10233, 10234, 10236, 10239, 10277, 10293, 10303, 10304, 10329, 10337, 10586, 10587, 10591, 12665, 12670, 12775, 12784, 12787, 12788, 12791, 12805, 12806, 12828, 12833,),
    (10, 56): (428, 429, 433, 587, 588, 590, 591, 608, 609, 613, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 713, 725, 726, 728, 730, 746, 747, 761, 770, 774, 951, 952, 953, 954, 955, 956, 957, 958, 959, 960, 961, 1111, 1112, 1273, 1274, 1762, 1763, 1764, 1765, 1766, 1767, 1768, 1769, 1770, 1771, 1772, 1773, 1774, 1775, 1776, 1777, 1778, 1779, 1780, 1781, 1782, 1783, 1784, 1785, 1786, 1787, 1788, 1789, 1790, 1791, 1792, 1793, 1794, 1795, 1796, 1797, 1798, 1799, 1800, 1801, 1802, 1803, 1804, 1805, 1898, 1900, 1902, 1916, 1943, 1950, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 2092, 2093, 2096, 2097, 3690, 3691, 3694, 3695, 3696, 3697, 3698, 3699, 3700, 3701, 3702, 3703, 3704, 3705, 3706, 3707, 3708, 3709, 3710, 3711, 3712, 3713, 3828, 3843, 3844, 3850, 4580, 4581, 4582, 4583, 4584, 4585, 4586, 4587, 4588, 4589, 4590, 4591, 4592, 4593, 4594, 4595, 4596, 4597, 4598, 4599, 4600, 4601, 4602, 4603, 4604, 4605, 4606, 4607, 4608, 4609, 4738, 4739, 4748, 5685, 5691, 5692, 6033, 6034, 6041, 6628, 6629, 6634, 6636, 6637, 6638, 6639, 6640, 6641, 6642, 6643, 6644, 6645, 6646, 6647, 6648, 6649, 6772, 6798, 6799, 6807, 7015, 7016, 7021, 8021, 8022, 8023, 8024, 8025, 8026, 8027, 8028, 8029, 8030, 8031, 8032, 8033, 8034, 8035, 8036, 8037, 8038, 8039, 8040, 8041, 8042, 8043, 8044, 8045, 8046, 8047, 8048, 8049, 8050, 8051, 8052, 8137, 8147, 8151, 8174, 8175, 8179, 8189, 9004, 9005, 10144, 10145, 10146, 10147, 10148, 10149, 10150, 10151, 10152, 10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162, 10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172, 10173, 10174, 10175, 10288, 10312, 10313, 10319, 10574, 10576, 10579, 10580, 10583, 10584, 10588, 10589, 10595, 10606, 10607, 10610, 10620, 10621, 10738, 10739, 10746, 11931, 11935, 11936, 12125, 12126, 12127, 12128, 12129, 12130, 12131, 12132, 12133, 12134, 12135, 12136, 12137, 12138, 12139, 12140, 12141, 12142, 12143, 12144, 12145, 12146, 12147, 12148, 12149, 12150, 12151, 12152, 12153, 12154, 12155, 12156, 12157, 12158, 12159, 12160, 12161, 12162, 12163, 12164, 12165, 12166, 12167, 12168, 12169, 12170, 12171, 12288, 12289, 12293, 12657, 12660, 12661, 12662, 12663, 12664, 12665, 12666, 12667, 12668, 12669, 12670, 12671, 12672, 12673, 12674, 12675, 12676, 12677, 12678, 12679, 12680, 12681, 12682, 12683, 12684, 12685, 12686, 12687, 12688, 12689, 12690, 12691, 12692, 12736, 12761, 12768, 12780, 12786, 12789, 12790, 12791, 12792, 12793, 12794, 12795, 12797, 12813, 12814, 12818, 12828, 12834, 12835, 12840, 12841, 12842, 13015, 13016, 13017, 13018, 13019, 13020, 13021, 13022, 13023, 13024, 13175, 13176, 13336, 13364, 13369,),
    (10, 57): (733, 738, 12796, 12797,),
    (10, 77): (435, 450, 451, 455, 565, 580, 581, 583, 584, 585, 586, 587, 588, 589, 590, 591, 592, 593, 594, 595, 596, 597, 598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619, 630, 635, 636, 680, 681, 682, 683, 684, 685, 687, 688, 689, 692, 693, 694, 695, 697, 698, 702, 704, 707, 709, 710, 711, 713, 714, 718, 720, 721, 722, 723, 724, 725, 727, 728, 729, 730, 731, 732, 733, 734, 735, 736, 737, 738, 739, 740, 741, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766, 767, 768, 769, 770, 772, 773, 780, 781, 952, 953, 956, 957, 958, 959, 960, 961, 962, 963, 964, 965, 966, 967, 968, 969, 970, 971, 972, 973, 974, 975, 976, 977, 978, 979, 980, 981, 982, 983, 1071, 1136, 1137, 1159, 1160, 1167, 1168, 1253, 1273, 1274, 1277, 1278, 1279, 1280, 1281, 1282, 1283, 1284, 1285, 1286, 1287, 1288, 1772, 1783, 1784, 1787, 1788, 1789, 1790, 1791, 1792, 1793, 1794, 1795, 1796, 1797, 1798, 1799, 1800, 1801, 1802, 1803, 1804, 1805, 1806, 1807, 1808, 1809, 1810, 1811, 1812, 1813, 1814, 1928, 1941, 1953, 1954, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2037, 2058, 2066, 2067, 2073, 2074, 2076, 2077, 2087, 2097, 2110, 2111, 2122, 2123, 2137, 2194, 2968, 2970, 2971, 2972, 2973, 2974, 2978, 2979, 2980, 2983, 2984, 2997, 3055, 3564, 3565, 3566, 3567, 3568, 3569, 3570, 3571, 3572, 3573, 3574, 3575, 3576, 3577, 3578, 3579, 3580, 3581, 3582, 3583, 3584, 3585, 3586, 3587, 3588, 3589, 3590, 3591, 3592, 3593, 3594, 3666, 3667, 3668, 3669, 3672, 3673, 3674, 3675, 3677, 3678, 3680, 3700, 3703, 3719, 3726, 3727, 3734, 3735, 3820, 3821, 3831, 3832, 3850, 3851, 3983, 4128, 4129, 4134, 4250, 4251, 4256, 4568, 4569, 4590, 4593, 4594, 4595, 4596, 4597, 4598, 4599, 4600, 4601, 4602, 4603, 4604, 4605, 4606, 4607, 4608, 4609, 4610, 4611, 4612, 4613, 4614, 4615, 4616, 4617, 4618, 4619, 4620, 4621, 4622, 4719, 4723, 4748, 5656, 5682, 5683, 5688, 5703, 5839, 5870, 5871, 5872, 5873, 5874, 5875, 5876, 5877, 5878, 5879, 5880, 5881, 5882, 5883, 5884, 5885, 5886, 5887, 5888, 5889, 5890, 5891, 5892, 6022, 6114, 6593, 6595, 6596, 6597, 6600, 6608, 6618, 6622, 6623, 6624, 6649, 6670, 6754, 6772, 6773, 6781, 6782, 6864, 6902, 6906, 6907, 6908, 6909, 6910, 6911, 6912, 6913, 6914, 6915, 6916, 6917, 6918, 6919, 6920, 6921, 6922, 6923, 7006, 7007, 7029, 7035, 7036, 7163, 7164, 7172, 7551, 7597, 7599, 7600, 7602, 7613, 7630, 7631, 7638, 7884, 7891, 8012, 8013, 8014, 8015, 8016, 8017, 8018, 8019, 8020, 8021, 8022, 8023, 8024, 8025, 8026, 8027, 8028, 8029, 8030, 8031, 8032, 8033, 8034, 8035, 8036, 8037, 8038, 8039, 8040, 8041, 8042, 8043, 8044, 8045, 8046, 8047, 8048, 8049, 8050, 8051, 8052, 8053, 8054, 8055, 8056, 8057, 8058, 8059, 8060, 8061, 8062, 8063, 8064, 8124, 8138, 8145, 8165, 8166, 8169, 8170, 8171, 8172, 8173, 8174, 8175, 8176, 8177, 8178, 8179, 8180, 8296, 8297, 8298, 8299, 8300, 8301, 8302, 8303, 8304, 8305, 8306, 8307, 8308, 8309, 8310, 8311, 8312, 8313, 8314, 8315, 8316, 8317, 8318, 8319, 8320, 8321, 8322, 8323, 8324, 8325, 8326, 8327, 8328, 8329, 8330, 8331, 8332, 8777, 8778, 8779, 8780, 8781, 8782, 8783, 8784, 8785, 8786, 8787, 8788, 8789, 8790, 8791, 8792, 8793, 8794, 8795, 8796, 8797, 8798, 8799, 8800, 8801, 8802, 8803, 8894, 8895, 8898, 8918, 8922, 8987, 8993, 8994, 8997, 9004, 9005, 9006, 9010, 9013, 9017, 9028, 9040, 9414, 9415, 9416, 9417, 9418, 9419, 9420, 9421, 9422, 9423, 9424, 9425, 9426, 9427, 9428, 9429, 9430, 9431, 9571, 9572, 9577, 9776, 9777, 9792, 9798, 9810, 9811, 9815, 9816, 9817, 9818, 9819, 9820, 9821, 9822, 9823, 9824, 9825, 9948, 9952, 9960, 9961, 9962, 9963, 9964, 9965, 9966, 9967, 9968, 10132, 10133, 10152, 10156, 10157, 10158, 10159, 10160, 10161, 10162, 10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172, 10173, 10174, 10175, 10176, 10177, 10178, 10179, 10180, 10181, 10182, 10183, 10184, 10185, 10186, 10187, 10247, 10249, 10260, 10261, 10262, 10280, 10300, 10308, 10315, 10323, 10324, 10442, 10443, 10567, 10570, 10571, 10572, 10573, 10582, 10583, 10595, 10596, 10598, 10599, 10610, 10611, 10614, 10615, 10616, 10617, 10728, 10729, 10731, 10732, 10733, 10734, 10735, 10736, 10737, 10738, 10739, 10740, 10741, 10742, 10743, 10744, 10745, 10746, 10747, 11209, 11210, 11211, 11212, 11213, 11214, 11215, 11216, 11217, 11218, 11219, 11220, 11221, 11222, 11223, 11224, 11225, 11226, 11227, 11228, 11229, 11230, 11231, 11232, 11233, 11481, 11523, 11547, 11556, 11665, 11892, 11898, 11926, 11942, 11943, 11949, 11950, 12107, 12112, 12113, 12119, 12120, 12121, 12123, 12124, 12125, 12126, 12127, 12128, 12129, 12130, 12131, 12132, 12133, 12134, 12135, 12136, 12137, 12138, 12139, 12140, 12141, 12142, 12143, 12144, 12145, 12146, 12147, 12148, 12149, 12150, 12151, 12152, 12153, 12155, 12156, 12158, 12159, 12160, 12161, 12162, 12163, 12164, 12165, 12166, 12167, 12168, 12169, 12170, 12171, 12172, 12173, 12174, 12175, 12176, 12177, 12178, 12179, 12247, 12253, 12255, 12258, 12270, 12274, 12298, 12307, 12328, 12329, 12338, 12492, 12493, 12494, 12495, 12496, 12497, 12498, 12499, 12500, 12501, 12502, 12503, 12504, 12505, 12506, 12507, 12508, 12509, 12510, 12511, 12512, 12513, 12623, 12627, 12628, 12630, 12640, 12642, 12646, 12649, 12650, 12655, 12656, 12657, 12658, 12661, 12662, 12666, 12673, 12676, 12682, 12683, 12686, 12687, 12688, 12689, 12690, 12691, 12692, 12693, 12694, 12695, 12696, 12697, 12698, 12699, 12700, 12701, 12744, 12745, 12746, 12747, 12754, 12755, 12774, 12775, 12777, 12778, 12779, 12780, 12781, 12782, 12783, 12784, 12786, 12787, 12788, 12789, 12790, 12791, 12792, 12793, 12794, 12795, 12796, 12797, 12798, 12800, 12801, 12802, 12803, 12804, 12805, 12806, 12807, 12808, 12809, 12810, 12811, 12812, 12813, 12814, 12815, 12816, 12817, 12818, 12819, 12820, 12821, 12822, 12823, 12824, 12825, 12834, 12839, 12840, 12848, 13000, 13016, 13020, 13021, 13022, 13023, 13024, 13025, 13026, 13027, 13028, 13029, 13030, 13031, 13032, 13033, 13034, 13035, 13036, 13037, 13038, 13039, 13040, 13041, 13042, 13043, 13044, 13045, 13046, 13047, 13202, 13218, 13219, 13226, 13227, 13320, 13331, 13332, 13333, 13334, 13357, 13358,),
    (10, 100): (733, 734, 739, 3667, 3668, 3669, 3711, 3712, 3716, 4139, 4144, 6641, 6642, 6646, 7020, 7040, 7047, 8157, 8164, 8895, 8896, 8900, 8902, 8903, 8905, 8906, 8907, 8908, 8909, 8910, 8911, 8921, 9002, 9007, 9800, 9801, 12797, 12798, 12802,),
    (10, 147): (8920,),
    (10, 173): (751, 12807, 12808, 12813,),
    (10, 187): (320, 321, 712, 735, 736, 1387, 1388, 1394, 1872, 1873, 1875, 1876, 1877, 1878, 1879, 1880, 1881, 1882, 1883, 1884, 1885, 1899, 1902, 1903, 1904, 1905, 1906, 1907, 1908, 1953, 1954, 1955, 1956, 1957, 1958, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1969, 1975, 1989, 1990, 1991, 2000, 2003, 2067, 2070, 2073, 2074, 2075, 2076, 2077, 2078, 2079, 2082, 2083, 2084, 2085, 2086, 2087, 2088, 2089, 2104, 2105, 2191, 2192, 2199, 3714, 3715, 3720, 3810, 3811, 3813, 3814, 3815, 3816, 3817, 3820, 3821, 3822, 3823, 3824, 3828, 3933, 3934, 3939, 4016, 4017, 4018, 4019, 4020, 4021, 4022, 4023, 4024, 4025, 4026, 4027, 4028, 4029, 4030, 4031, 4032, 4033, 4034, 4035, 4036, 4037, 4038, 4039, 4040, 4041, 4042, 4043, 4044, 4045, 4046, 4047, 4048, 4049, 4050, 4051, 4052, 4053, 4054, 4055, 4056, 4113, 4114, 4115, 4131, 4132, 4146, 4151, 4244, 4251, 4696, 4699, 4700, 4701, 4702, 4703, 4704, 4705, 4706, 4707, 4708, 4709, 4710, 4711, 4712, 4713, 4813, 5563, 5564, 5565, 5566, 5567, 5568, 5569, 5570, 5571, 5572, 5573, 5574, 5575, 5576, 5577, 5578, 5580, 5581, 5582, 5583, 5584, 5587, 5685, 5692, 6510, 6515, 6517, 6518, 6519, 6520, 6521, 6522, 6523, 6524, 6525, 6526, 6527, 6528, 6529, 6530, 6531, 6532, 6533, 6534, 6535, 6536, 6537, 6538, 6539, 6540, 6541, 6542, 6543, 6544, 6545, 6546, 6547, 6548, 6549, 6620, 6624, 6625, 6627, 6628, 6629, 6630, 6631, 6645, 6652, 6653, 6724, 6725, 6728, 6729, 6730, 6731, 6732, 6733, 6734, 6735, 6822, 6898, 6899, 6902, 6903, 6904, 6905, 6906, 6907, 7021, 7022, 7026, 7482, 7488, 7489, 7490, 7491, 7492, 7493, 7494, 7495, 7496, 7497, 7498, 7501, 7502, 7503, 7504, 7505, 7609, 7610, 7616, 8015, 8016, 8017, 8018, 8019, 8020, 8021, 8022, 8023, 8024, 8025, 8026, 8027, 8028, 8032, 8043, 8044, 8051, 8054, 8055, 8056, 8057, 8058, 8059, 8060, 8061, 8062, 8063, 8064, 8067, 8123, 8128, 8149, 8172, 8173, 8298, 8299, 8300, 8301, 8302, 8303, 8305, 8313, 8319, 8325, 8424, 8592, 8593, 8600, 8601, 8602, 8631, 8632, 8645, 8646, 8717, 8718, 8796, 8797, 8799, 8800, 8801, 8802, 8803, 8804, 8805, 8806, 8807, 8808, 8809, 8810, 8811, 8812, 8813, 8814, 8815, 8816, 8817, 8818, 8819, 8820, 8821, 8822, 8823, 8824, 8825, 8869, 8870, 8872, 8878, 8879, 8880, 8882, 8885, 8888, 8890, 8891, 8892, 8893, 8895, 8896, 8897, 8898, 8899, 8900, 8901, 8902, 8914, 8921, 8924, 8925, 8975, 8976, 8977, 8986, 8987, 8992, 8996, 8997, 9008, 9013, 9014, 9017, 9018, 9102, 9103, 9670, 9681, 9685, 9686, 9688, 9689, 9698, 9699, 9705, 9716, 9722, 9731, 9733, 9734, 9783, 9788, 9789, 9811, 9812, 9816, 9818, 9819, 10161, 10164, 10165, 10245, 10252, 10289, 10290, 10877, 12384, 12385, 12389, 12790, 12794, 13443, 13444,),
    (10, 188): (433, 439, 591, 592, 594, 610, 611, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 706, 707, 709, 712, 714, 716, 723, 724, 734, 735, 747, 748, 749, 750, 764, 771, 810, 823, 1068, 1148, 1149, 1156, 1283, 1288, 1772, 1773, 1775, 1776, 1777, 1778, 1779, 1780, 1781, 1782, 1783, 1784, 1785, 1786, 1787, 1788, 1789, 1790, 1791, 1792, 1793, 1794, 1795, 1796, 1797, 1798, 1799, 1800, 1801, 1802, 1803, 1804, 1805, 1806, 1807, 1808, 1809, 1939, 1954, 1955, 1958, 1959, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 2064, 2073, 2074, 2076, 2077, 2078, 2079, 2080, 2081, 2082, 2083, 2084, 2085, 2086, 2087, 2088, 2089, 2090, 2091, 2092, 2093, 2094, 2095, 2096, 2097, 2098, 2099, 2100, 2101, 2969, 2975, 2999, 3003, 3004, 3012, 3059, 3060, 3072, 3704, 3720, 3721, 3724, 3733, 3827, 3831, 3832, 3833, 3834, 3835, 3836, 3837, 3838, 3839, 3840, 4577, 4578, 4579, 4580, 4581, 4582, 4583, 4584, 4585, 4586, 4587, 4588, 4589, 4590, 4591, 4592, 4593, 4594, 4595, 4596, 4597, 4598, 4599, 4600, 4601, 4602, 4603, 4604, 4605, 4606, 4607, 4608, 4609, 4610, 4611, 4612, 4613, 4614, 4703, 5694, 5695, 5699, 6633, 6634, 6635, 6636, 6637, 6638, 6639, 6640, 6641, 6642, 6643, 6644, 6645, 6646, 6647, 6648, 6649, 6650, 6767, 6785, 6798, 6804, 6890, 6891, 6895, 6897, 6898, 6899, 6900, 6901, 6902, 6903, 6904, 6906, 6907, 7022, 7027, 7145, 7146, 8018, 8019, 8020, 8021, 8022, 8023, 8024, 8025, 8026, 8027, 8028, 8029, 8030, 8031, 8032, 8033, 8034, 8035, 8036, 8037, 8038, 8039, 8040, 8041, 8042, 8043, 8044, 8045, 8046, 8047, 8048, 8049, 8050, 8051, 8052, 8053, 8054, 8055, 8056, 8057, 8058, 8153, 8174, 8175, 8181, 8182, 8909, 8910, 8914, 9008, 9009, 9010, 9014, 9015, 9791, 9792, 9793, 9794, 9797, 9798, 9799, 9800, 9801, 9802, 9803, 9804, 9805, 9806, 9807, 9808, 9809, 9810, 9811, 9812, 9813, 9814, 10142, 10143, 10144, 10145, 10146, 10147, 10148, 10149, 10150, 10151, 10152, 10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162, 10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172, 10173, 10174, 10175, 10176, 10177, 10178, 10250, 10251, 10253, 10288, 10310, 10311, 10317, 10318, 10550, 10553, 10570, 10571, 10573, 10574, 10600, 10601, 10604, 10605, 10606, 10607, 10608, 10609, 10610, 10611, 10713, 11923, 11924, 11950, 11957, 11958, 12125, 12126, 12127, 12128, 12129, 12130, 12131, 12132, 12133, 12134, 12135, 12136, 12137, 12138, 12139, 12140, 12141, 12142, 12143, 12144, 12145, 12146, 12147, 12148, 12149, 12150, 12151, 12152, 12153, 12154, 12155, 12156, 12157, 12158, 12159, 12160, 12161, 12162, 12163, 12164, 12165, 12166, 12167, 12168, 12169, 12170, 12171, 12172, 12289, 12295, 12326, 12332, 12496, 12497, 12502, 12654, 12668, 12669, 12671, 12672, 12673, 12674, 12675, 12676, 12677, 12678, 12679, 12680, 12681, 12682, 12683, 12684, 12685, 12686, 12687, 12688, 12689, 12690, 12691, 12692, 12693, 12694, 12695, 12696, 12697, 12762, 12778, 12791, 12794, 12795, 12796, 12797, 12798, 12799, 12800, 12801, 12802, 12803, 12804, 12805, 12806, 12813, 12822, 12830, 12836, 12837, 12838, 13173, 13209, 13210, 13217, 13346, 13347, 13353,),
    (10, 205): (733, 755, 756, 762, 10290, 10291, 12772, 12794, 12817, 12818, 12823,),
    (10, 216): (7518,),
    (10, 224): (222, 227, 228, 407, 418, 419, 421, 422, 423, 424, 425, 426, 427, 428, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 476, 477, 559, 561, 562, 563, 565, 572, 573, 575, 576, 578, 579, 580, 581, 582, 583, 585, 586, 587, 588, 589, 590, 591, 592, 593, 594, 595, 596, 597, 598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 621, 625, 626, 627, 629, 630, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 681, 683, 684, 685, 686, 687, 688, 689, 690, 691, 692, 701, 704, 707, 708, 710, 712, 713, 714, 716, 717, 718, 719, 720, 721, 722, 723, 724, 726, 727, 728, 730, 731, 732, 733, 734, 735, 736, 737, 738, 739, 740, 747, 748, 750, 751, 752, 753, 754, 755, 758, 762, 763, 770, 771, 774, 775, 776, 777, 950, 951, 952, 953, 954, 955, 956, 957, 958, 959, 960, 961, 962, 963, 964, 965, 966, 967, 968, 969, 970, 1137, 1147, 1148, 1257, 1258, 1276, 1277, 1280, 1281, 1282, 1283, 1284, 1291, 1301, 1302, 1305, 1394, 1395, 1400, 1414, 1415, 1765, 1766, 1768, 1769, 1770, 1771, 1772, 1773, 1774, 1775, 1776, 1777, 1778, 1779, 1780, 1781, 1782, 1783, 1784, 1785, 1786, 1787, 1788, 1789, 1790, 1791, 1792, 1793, 1794, 1795, 1796, 1797, 1798, 1799, 1800, 1801, 1802, 1803, 1804, 1805, 1806, 1807, 1808, 1809, 1810, 1811, 1812, 1829, 1933, 1948, 1954, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1989, 1990, 2019, 2020, 2021, 2022, 2023, 2053, 2056, 2058, 2059, 2060, 2072, 2073, 2075, 2076, 2077, 2078, 2097, 2102, 2103, 2104, 2105, 2106, 2107, 2108, 2109, 2110, 2507, 2515, 2969, 2970, 2971, 2974, 2982, 2983, 2984, 2985, 2988, 2992, 2995, 2997, 3005, 3009, 3010, 3012, 3018, 3055, 3567, 3568, 3569, 3570, 3571, 3572, 3573, 3574, 3575, 3576, 3577, 3578, 3579, 3580, 3581, 3582, 3583, 3584, 3585, 3586, 3587, 3588, 3589, 3590, 3591, 3592, 3593, 3594, 3595, 3596, 3667, 3668, 3670, 3671, 3672, 3673, 3674, 3681, 3683, 3685, 3686, 3689, 3690, 3692, 3693, 3694, 3697, 3700, 3708, 3709, 3716, 3717, 3726, 3727, 3731, 3732, 3733, 3796, 3798, 3814, 3816, 3817, 3818, 3819, 3828, 3831, 3832, 3840, 3848, 3849, 3852, 3853, 4125, 4126, 4130, 4135, 4137, 4138, 4147, 4152, 4159, 4160, 4261, 4262, 4268, 4269, 4569, 4570, 4571, 4572, 4573, 4574, 4575, 4576, 4577, 4578, 4579, 4580, 4581, 4582, 4583, 4584, 4585, 4586, 4587, 4588, 4589, 4590, 4591, 4592, 4593, 4594, 4595, 4596, 4597, 4598, 4599, 4600, 4601, 4602, 4603, 4604, 4605, 4606, 4607, 4608, 4609, 4610, 4611, 4612, 4613, 4614, 4615, 4616, 4617, 4618, 4619, 4620, 4621, 4622, 4682, 4686, 4691, 4692, 4700, 4723, 4724, 4734, 4751, 5552, 5553, 5554, 5555, 5556, 5557, 5558, 5559, 5560, 5561, 5562, 5563, 5564, 5565, 5566, 5567, 5568, 5569, 5570, 5571, 5572, 5573, 5574, 5575, 5576, 5577, 5578, 5579, 5580, 5581, 5582, 5583, 5584, 5585, 5586, 5587, 5588, 5589, 5590, 5656, 5694, 5696, 6035, 6064, 6065, 6071, 6509, 6510, 6511, 6512, 6591, 6592, 6594, 6595, 6613, 6614, 6616, 6617, 6618, 6619, 6620, 6621, 6622, 6623, 6624, 6625, 6626, 6627, 6628, 6629, 6630, 6631, 6646, 6647, 6650, 6651, 6652, 6653, 6654, 6655, 6656, 6657, 6658, 6659, 6660, 6717, 6719, 6722, 6723, 6725, 6728, 6729, 6758, 6765, 6878, 6879, 6881, 6882, 6883, 6884, 6885, 6886, 6887, 6888, 6889, 6890, 6891, 6892, 6893, 6894, 6895, 6896, 6897, 6898, 6899, 6900, 6905, 6916, 6917, 6922, 6923, 6984, 7029, 7030, 7034, 7037, 7038, 7052, 7128, 7159, 7167, 7603, 7607, 7733, 7737, 7880, 7881, 7887, 7888, 8019, 8020, 8022, 8023, 8024, 8025, 8026, 8027, 8028, 8029, 8030, 8031, 8032, 8033, 8034, 8035, 8036, 8037, 8038, 8039, 8040, 8041, 8042, 8043, 8044, 8045, 8046, 8047, 8048, 8049, 8050, 8051, 8052, 8053, 8054, 8055, 8056, 8057, 8058, 8059, 8060, 8061, 8062, 8063, 8064, 8065, 8066, 8067, 8117, 8126, 8155, 8168, 8169, 8175, 8176, 8177, 8187, 8188, 8293, 8294, 8295, 8296, 8297, 8298, 8299, 8300, 8301, 8302, 8303, 8304, 8305, 8306, 8307, 8308, 8309, 8310, 8311, 8312, 8313, 8314, 8315, 8316, 8317, 8318, 8319, 8320, 8321, 8322, 8323, 8324, 8325, 8326, 8327, 8328, 8329, 8330, 8331, 8332, 8333, 8334, 8335, 8336, 8448, 8457, 8779, 8780, 8781, 8782, 8783, 8784, 8785, 8786, 8787, 8788, 8789, 8790, 8791, 8792, 8793, 8794, 8795, 8796, 8797, 8798, 8799, 8800, 8801, 8802, 8803, 8874, 8881, 8882, 8883, 8887, 8888, 8889, 8890, 8891, 8893, 8896, 8897, 8898, 8899, 8900, 8901, 8902, 8904, 8905, 8907, 8908, 8909, 8911, 8912, 8913, 8917, 8922, 8923, 8998, 8999, 9003, 9020, 9030, 9031, 9586, 9594, 9595, 9790, 9791, 9794, 9796, 9797, 9801, 9802, 9804, 9805, 9806, 9807, 9808, 9809, 9810, 9819, 9824, 9847, 9947, 9971, 9972, 9979, 10136, 10137, 10138, 10139, 10140, 10141, 10142, 10143, 10144, 10145, 10146, 10147, 10148, 10149, 10150, 10151, 10152, 10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162, 10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172, 10173, 10174, 10175, 10176, 10177, 10178, 10179, 10180, 10181, 10182, 10183, 10184, 10185, 10186, 10187, 10253, 10255, 10256, 10263, 10267, 10285, 10286, 10289, 10292, 10293, 10297, 10303, 10314, 10315, 10318, 10352, 10353, 10556, 10557, 10559, 10570, 10582, 10585, 10586, 10587, 10588, 10589, 10590, 10591, 10596, 10597, 10600, 10606, 10607, 10610, 10611, 10612, 10613, 10614, 10615, 10616, 10617, 10723, 10747, 10753, 10754, 11212, 11213, 11216, 11217, 11218, 11219, 11220, 11221, 11222, 11223, 11224, 11225, 11226, 11227, 11228, 11229, 11230, 11231, 11232, 11233, 11531, 11532, 11536, 11662, 11668, 11918, 11919, 12137, 12140, 12156, 12157, 12159, 12160, 12161, 12162, 12163, 12164, 12165, 12166, 12167, 12168, 12169, 12170, 12171, 12172, 12173, 12174, 12175, 12176, 12177, 12256, 12275, 12276, 12311, 12374, 12473, 12474, 12491, 12507, 12508, 12519, 12633, 12635, 12637, 12644, 12645, 12647, 12648, 12651, 12652, 12653, 12654, 12656, 12658, 12659, 12660, 12662, 12664, 12665, 12666, 12667, 12668, 12669, 12670, 12671, 12672, 12673, 12674, 12676, 12677, 12678, 12679, 12683, 12684, 12685, 12686, 12687, 12688, 12689, 12690, 12691, 12692, 12695, 12696, 12699, 12700, 12701, 12702, 12703, 12704, 12745, 12746, 12747, 12748, 12749, 12750, 12751, 12752, 12753, 12754, 12755, 12756, 12774, 12775, 12780, 12781, 12782, 12783, 12784, 12785, 12786, 12787, 12788, 12789, 12790, 12791, 12793, 12795, 12796, 12797, 12798, 12799, 12800, 12801, 12802, 12803, 12804, 12805, 12806, 12807, 12808, 12810, 12811, 12812, 12813, 12816, 12817, 12822, 12823, 12832, 12833, 12836, 12837, 12838, 12839, 12840, 12841, 13014, 13015, 13016, 13017, 13018, 13019, 13020, 13021, 13022, 13023, 13024, 13025, 13026, 13027, 13028, 13029, 13030, 13031, 13032, 13033, 13034, 13184, 13216, 13225, 13226, 13333, 13334, 13340, 13341, 13343, 13359, 13360, 13363, 13364, 13365, 13366, 13367, 13368, 13469, 13477,),
    (10, 234): (237, 241, 401, 413, 423, 448, 449, 452, 453, 586, 587, 589, 590, 591, 592, 593, 594, 595, 596, 597, 598, 600, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636, 637, 638, 677, 678, 679, 680, 681, 682, 684, 686, 688, 689, 690, 696, 697, 699, 701, 703, 705, 706, 708, 711, 712, 713, 714, 716, 717, 718, 719, 720, 721, 722, 723, 724, 725, 726, 727, 729, 730, 731, 732, 733, 734, 735, 736, 737, 738, 739, 740, 741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 754, 755, 758, 759, 760, 761, 762, 767, 768, 774, 775, 778, 779, 780, 781, 782, 933, 950, 955, 956, 957, 958, 959, 960, 961, 962, 963, 964, 965, 966, 967, 968, 969, 970, 971, 972, 973, 974, 975, 976, 977, 978, 979, 980, 981, 982, 983, 984, 1150, 1157, 1158, 1260, 1287, 1783, 1788, 1789, 1790, 1791, 1792, 1793, 1794, 1795, 1796, 1797, 1798, 1799, 1800, 1801, 1802, 1803, 1804, 1805, 1806, 1807, 1808, 1809, 1810, 1811, 1812, 1813, 1814, 1899, 1914, 1915, 1919, 1920, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1986, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2065, 2078, 2084, 2122, 2123, 2127, 3042, 3564, 3565, 3566, 3567, 3568, 3569, 3570, 3571, 3572, 3573, 3574, 3575, 3576, 3577, 3578, 3579, 3580, 3581, 3582, 3583, 3584, 3585, 3586, 3587, 3588, 3589, 3590, 3591, 3592, 3593, 3594, 3670, 3671, 3673, 3674, 3681, 3690, 3691, 3708, 3719, 3726, 3739, 3818, 3819, 3822, 3823, 3824, 3825, 3826, 3827, 3828, 3829, 3834, 3843, 3844, 3847, 3848, 3965, 3973, 4110, 4131, 4132, 4136, 4239, 4240, 4244, 4245, 4586, 4590, 4591, 4592, 4593, 4594, 4595, 4596, 4597, 4598, 4599, 4600, 4601, 4602, 4603, 4604, 4605, 4606, 4607, 4608, 4609, 4610, 4611, 4612, 4613, 4614, 4615, 4616, 4617, 4618, 4619, 4620, 4621, 4725, 4726, 4731, 5682, 5685, 5686, 5687, 5688, 5689, 5690, 5691, 5692, 5693, 5694, 5695, 5696, 5705, 5706, 5789, 5875, 5879, 5880, 5881, 5882, 5883, 5884, 5885, 5886, 5887, 5888, 5889, 5890, 5891, 5892, 6025, 6046, 6050, 6622, 6623, 6626, 6627, 6642, 6643, 6647, 6648, 6649, 6650, 6651, 6652, 6653, 6654, 6655, 6656, 6657, 6658, 6722, 6724, 6725, 6727, 6730, 6751, 6764, 6790, 6892, 6893, 6894, 6895, 6899, 6902, 6922, 6923, 6942, 7000, 7001, 7009, 7010, 7011, 7012, 7013, 7014, 7015, 7016, 7017, 7018, 7019, 7020, 7021, 7022, 7023, 7024, 7025, 7026, 7027, 7028, 7029, 7057, 7058, 7133, 7596, 7600, 7845, 7847, 7848, 7849, 7850, 7851, 7863, 7876, 7895, 7899, 7908, 8013, 8014, 8015, 8016, 8017, 8018, 8019, 8020, 8021, 8022, 8023, 8024, 8025, 8026, 8027, 8028, 8029, 8030, 8031, 8032, 8033, 8034, 8035, 8036, 8037, 8038, 8039, 8040, 8041, 8042, 8043, 8044, 8045, 8046, 8047, 8048, 8049, 8050, 8051, 8052, 8053, 8054, 8055, 8056, 8057, 8058, 8059, 8060, 8061, 8062, 8063, 8064, 8173, 8174, 8179, 8182, 8300, 8301, 8302, 8303, 8304, 8305, 8306, 8307, 8308, 8309, 8310, 8311, 8312, 8313, 8314, 8315, 8316, 8317, 8318, 8319, 8320, 8321, 8322, 8323, 8324, 8325, 8326, 8327, 8328, 8329, 8330, 8331, 8332, 8780, 8781, 8782, 8783, 8784, 8785, 8786, 8787, 8788, 8789, 8790, 8791, 8792, 8793, 8794, 8795, 8796, 8797, 8798, 8799, 8800, 8801, 8802, 8803, 8911, 8916, 9001, 9016, 9019, 9020, 9021, 9022, 9023, 9024, 9025, 9026, 9027, 9414, 9415, 9416, 9417, 9418, 9419, 9420, 9421, 9422, 9423, 9424, 9425, 9426, 9427, 9428, 9429, 9430, 9431, 9563, 9564, 9602, 9610, 9789, 9791, 9792, 9806, 9807, 9808, 9818, 9821, 9822, 9825, 9944, 9966, 9967, 9972, 9975, 10133, 10134, 10135, 10136, 10137, 10138, 10139, 10140, 10141, 10142, 10143, 10144, 10145, 10146, 10147, 10148, 10149, 10150, 10151, 10152, 10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162, 10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172, 10173, 10174, 10175, 10176, 10177, 10178, 10179, 10180, 10181, 10182, 10183, 10184, 10185, 10186, 10288, 10292, 10294, 10296, 10308, 10322, 10323, 10328, 10555, 10556, 10558, 10560, 10562, 10563, 10564, 10565, 10566, 10567, 10568, 10569, 10570, 10571, 10572, 10573, 10574, 10575, 10576, 10577, 10578, 10579, 10580, 10581, 10582, 10583, 10584, 10585, 10586, 10587, 10588, 10589, 10590, 10591, 10592, 10593, 10594, 10595, 10596, 10604, 10605, 10610, 10611, 10613, 10614, 10615, 10616, 10617, 10727, 10743, 10744, 10750, 11214, 11217, 11218, 11219, 11220, 11221, 11222, 11223, 11224, 11225, 11226, 11227, 11228, 11229, 11230, 11231, 11232, 11233, 11906, 11911, 11927, 11932, 11933, 11935, 11940, 11941, 11943, 11958, 11963, 11964, 11970, 11976, 12126, 12137, 12138, 12140, 12141, 12142, 12143, 12144, 12145, 12146, 12147, 12148, 12149, 12150, 12151, 12152, 12153, 12154, 12155, 12156, 12157, 12158, 12159, 12160, 12161, 12162, 12163, 12164, 12165, 12166, 12167, 12168, 12169, 12170, 12171, 12172, 12173, 12174, 12175, 12176, 12177, 12178, 12179, 12268, 12284, 12285, 12288, 12365, 12366, 12371, 12402, 12403, 12465, 12468, 12469, 12476, 12483, 12484, 12516, 12517, 12525, 12526, 12532, 12652, 12654, 12655, 12656, 12657, 12658, 12660, 12661, 12662, 12663, 12664, 12665, 12666, 12667, 12668, 12669, 12670, 12671, 12672, 12673, 12674, 12675, 12676, 12677, 12679, 12680, 12681, 12682, 12683, 12684, 12685, 12686, 12687, 12688, 12689, 12690, 12691, 12692, 12693, 12694, 12695, 12696, 12697, 12698, 12699, 12700, 12701, 12702, 12741, 12742, 12743, 12744, 12751, 12753, 12756, 12757, 12758, 12761, 12765, 12769, 12770, 12773, 12774, 12784, 12785, 12787, 12788, 12789, 12791, 12792, 12793, 12794, 12795, 12796, 12797, 12798, 12799, 12800, 12801, 12802, 12803, 12804, 12805, 12806, 12807, 12808, 12809, 12810, 12811, 12812, 12813, 12814, 12815, 12816, 12817, 12820, 12821, 12823, 12826, 12831, 12834, 12842, 12843, 12846, 12997, 13013, 13014, 13017, 13018, 13019, 13020, 13021, 13022, 13023, 13024, 13025, 13026, 13027, 13028, 13029, 13030, 13031, 13032, 13033, 13034, 13035, 13036, 13037, 13038, 13039, 13040, 13041, 13042, 13043, 13044, 13045, 13046, 13047, 13048, 13174, 13200, 13207, 13329, 13337, 13338, 13339, 13340, 13341, 13342, 13343, 13344,),
    (10, 239): (715, 746, 757, 758, 764, 765, 2023, 2057, 2093, 2094, 12786,),
    (10, 275): (321, 322, 327, 616, 617, 622, 722, 725, 750, 751, 754, 1271, 1272, 1273, 1274, 1275, 1276, 1277, 1278, 1279, 1280, 1281, 1282, 1283, 1284, 1285, 1398, 1406, 1872, 1873, 1874, 1875, 1880, 1881, 1887, 1891, 1895, 1896, 1899, 1900, 1901, 1907, 1908, 1909, 1910, 1913, 1915, 1919, 1920, 1924, 1957, 1959, 1963, 1964, 1970, 1972, 1977, 1985, 1992, 1993, 1999, 2000, 2009, 2057, 2085, 2086, 2090, 2091, 2092, 2093, 2094, 2095, 2096, 2097, 2098, 2099, 2100, 2101, 2102, 2195, 2196, 2202, 3676, 3690, 3691, 3692, 3700, 3701, 3702, 3703, 3704, 3705, 3721, 3722, 3741, 3742, 3755, 3815, 3816, 3821, 3841, 3842, 3845, 3911, 3934, 3950, 3951, 4015, 4016, 4017, 4018, 4019, 4020, 4021, 4022, 4023, 4024, 4025, 4026, 4027, 4028, 4029, 4030, 4031, 4032, 4033, 4034, 4035, 4036, 4037, 4038, 4039, 4040, 4041, 4042, 4043, 4044, 4045, 4046, 4047, 4048, 4049, 4050, 4051, 4052, 4053, 4054, 4055, 4056, 4107, 4124, 4125, 4145, 4146, 4151, 4156, 4248, 4249, 4250, 4683, 4686, 4713, 4714, 4717, 4718, 4764, 4769, 4812, 4813, 5561, 5562, 5563, 5564, 5565, 5566, 5567, 5568, 5569, 5570, 5571, 5572, 5573, 5574, 5575, 5576, 5577, 5578, 5579, 5580, 5581, 5582, 5583, 5584, 5585, 5586, 5587, 5588, 5589, 5590, 5591, 5592, 5593, 5594, 5595, 5596, 5597, 5660, 5698, 5699, 5704, 6512, 6513, 6515, 6516, 6517, 6518, 6519, 6520, 6521, 6522, 6523, 6524, 6525, 6526, 6527, 6531, 6532, 6534, 6535, 6536, 6537, 6538, 6539, 6540, 6541, 6542, 6543, 6544, 6545, 6546, 6547, 6548, 6609, 6612, 6613, 6614, 6618, 6619, 6620, 6627, 6649, 6720, 6724, 6731, 6753, 6754, 6892, 6895, 6896, 6897, 6898, 6899, 6900, 6901, 6902, 6903, 6904, 6905, 6906, 7024, 7025, 7030, 7488, 7489, 7490, 7491, 7492, 7493, 7494, 7495, 7496, 7497, 7498, 7499, 7500, 7501, 7502, 7503, 7504, 7505, 7509, 7613, 7614, 7623, 8016, 8017, 8018, 8019, 8020, 8021, 8022, 8023, 8024, 8025, 8026, 8027, 8028, 8029, 8040, 8041, 8043, 8044, 8045, 8046, 8047, 8054, 8057, 8058, 8059, 8060, 8061, 8062, 8063, 8064, 8065, 8136, 8169, 8333, 8784, 8785, 8786, 8787, 8788, 8789, 8790, 8791, 8792, 8793, 8794, 8795, 8796, 8797, 8798, 8799, 8800, 8801, 8802, 8803, 8804, 8805, 8806, 8807, 8808, 8809, 8810, 8811, 8812, 8813, 8814, 8815, 8816, 8817, 8818, 8819, 8863, 8866, 8868, 8872, 8873, 8876, 8877, 8878, 8879, 8880, 8881, 8882, 8884, 8885, 8886, 8887, 8888, 8889, 8890, 8891, 8892, 8893, 8894, 8895, 8896, 8897, 8898, 8899, 8902, 8918, 8919, 8923, 8924, 8984, 8985, 8987, 8988, 8989, 8990, 8991, 8992, 8993, 9019, 9020, 9023, 9064, 9066, 9067, 9074, 9078, 9111, 9112, 9113, 9678, 9679, 9680, 9681, 9682, 9683, 9684, 9685, 9687, 9688, 9689, 9690, 9691, 9692, 9693, 9694, 9695, 9696, 9697, 9698, 9699, 9700, 9701, 9702, 9703, 9704, 9705, 9706, 9707, 9708, 9718, 9727, 9728, 9801, 9808, 9809, 10154, 10155, 10158, 10159, 10160, 10161, 10162, 10163, 10164, 10165, 10166, 10233, 10279, 10280, 10285, 10592, 10593, 10597, 12386, 12391, 12678, 12679, 12685, 12795, 12796, 12800, 12801, 12804, 13334, 13335, 13336, 13337, 13338, 13339, 13340, 13341, 13342, 13343, 13344, 13345, 13346, 13347, 13348, 13349, 13462, 13471,),
    (10, 278): (247, 248, 254, 435, 441, 588, 589, 591, 592, 593, 594, 595, 596, 597, 598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623, 624, 715, 729, 730, 732, 733, 734, 735, 736, 737, 738, 739, 740, 741, 742, 743, 744, 746, 747, 748, 749, 750, 753, 762, 765, 766, 767, 768, 769, 770, 771, 772, 773, 774, 775, 776, 777, 778, 779, 809, 936, 937, 955, 956, 961, 962, 963, 964, 965, 966, 967, 968, 969, 970, 971, 972, 973, 974, 975, 976, 977, 978, 979, 980, 981, 1116, 1160, 1161, 1166, 1306, 1307, 1314, 1765, 1766, 1767, 1768, 1769, 1770, 1771, 1772, 1773, 1774, 1775, 1776, 1777, 1778, 1779, 1780, 1781, 1782, 1783, 1784, 1785, 1786, 1787, 1788, 1789, 1790, 1791, 1792, 1793, 1794, 1795, 1796, 1797, 1798, 1799, 1800, 1801, 1909, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947, 1948, 1949, 1978, 1979, 1985, 2059, 2078, 2103, 2121, 2122, 2132, 3719, 3727, 4587, 4588, 4589, 4590, 4591, 4592, 4593, 4594, 4595, 4596, 4597, 4598, 4737, 6621, 6642, 6643, 6649, 6753, 8167, 8168, 8174, 8175, 9795, 9796, 9797, 9798, 9801, 9802, 9803, 9804, 9805, 9806, 9807, 9808, 9809, 9962, 9966, 10140, 10141, 10142, 10143, 10144, 10145, 10146, 10147, 10148, 10149, 10150, 10151, 10152, 10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162, 10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172, 10173, 10174, 10175, 10176, 10579, 10594, 10595, 10600, 10601, 10603, 10604, 10605, 10606, 10607, 10608, 10609, 10610, 10739, 10740, 10747, 11931, 11937, 11940, 11941, 11942, 11943, 11944, 11945, 11946, 11947, 11948, 12122, 12139, 12143, 12144, 12145, 12146, 12147, 12148, 12149, 12150, 12151, 12152, 12153, 12154, 12155, 12156, 12157, 12158, 12159, 12160, 12161, 12162, 12163, 12164, 12165, 12166, 12167, 12168, 12169, 12170, 12171, 12172, 12173, 12174, 12175, 12176, 12255, 12319, 12341, 12342, 12351, 12498, 12499, 12504, 12652, 12656, 12657, 12658, 12659, 12660, 12661, 12662, 12663, 12664, 12665, 12666, 12667, 12668, 12669, 12670, 12671, 12672, 12673, 12674, 12675, 12676, 12677, 12678, 12679, 12680, 12681, 12682, 12683, 12684, 12685, 12686, 12687, 12688, 12788, 12789, 12797, 12798, 12803, 12805, 12806, 12812, 12824, 12826, 12827, 12828, 12829, 12830, 12831, 12832, 12833, 12834, 12835, 12836, 12837, 12838, 12839, 12840, 12841, 12842, 12843, 12844, 13014, 13015, 13018, 13019, 13020, 13021, 13022, 13023, 13024, 13025, 13026, 13027, 13028, 13029, 13030, 13031, 13032, 13033, 13034, 13035, 13036, 13037, 13038, 13039, 13040, 13041, 13042, 13043, 13044, 13045, 13144, 13145, 13146, 13147, 13181, 13199, 13200, 13221, 13222, 13228, 13249, 13254, 13370, 13371,),
    (10, 281): (7511,),
    (10, 286): (738, 739, 12808,),
    (11, 3): (3806, 6624, 7037, 7049, 8927, 8928, 8950, 9003, 9010,),
    (11, 22): (2151, 2152, 2153, 2247, 6618, 6649, 6656, 8922, 8936, 8940, 8941, 8953, 8956, 8958, 8963, 8992, 9019, 12442,),
    (11, 61): (729, 761, 2071, 2077, 3695, 3696, 3738, 3739, 3740, 3741, 3743, 3747, 3748, 3749, 3750, 3753, 3754, 3757, 3868, 3870, 3871, 3873, 3874, 3909, 3928, 3955, 4126, 4133, 4697, 5676, 5686, 6615, 6635, 6636, 6747, 8099, 8162, 8163, 8173, 8893, 8917,),
    (11, 87): (727, 12790,),
    (11, 89): (6590, 6591, 6592, 6598, 6608, 6614, 6651,),
    (11, 127): (727, 12790, 12791,),
}
assert len(CATENARY_SAMPLES) == 80, len(CATENARY_SAMPLES)
assert sum(len(v) for v in CATENARY_SAMPLES.values()) == 7772, 'catenary table is not the 2026-09-23 RULE C scan'

# --- patch sr-22: the snap catalogue under the evaluation mask (scan_snap_events.py
# guard D, k = 1.0 -> snap_catalogue_cat.csv). Read only by the EVAL_CLEAN dumps,
# which swap it in for SNAP_EVENT_TIMES and restore the published one afterwards.
SNAP_EVENT_TIMES_CAT = {
    (1, 11): (10472,),
    (1, 63): (569, 1881, 2018, 2298, 2445, 2511, 2951, 3681, 3812, 4099, 5218, 5302, 5368, 5439, 5583, 5646, 5795, 7497, 7579, 7656, 8025, 8604, 9063, 9138, 10473, 11049, 11624, 12633, 13538,),
    (1, 130): (10471,),
    (1, 142): (564, 639, 698, 2293, 2442, 3808, 5292, 5361, 5434, 5572, 5799, 7488, 7572, 8303, 11620, 12703, 12762,),
    (1, 177): (1957, 2549, 4099, 5008, 7482, 8025, 8104, 8880, 9136, 9770, 10471, 11050,),
    (1, 185): (3672, 12763,),
    (1, 202): (288, 405, 722, 1361, 1763, 1957, 2184, 2441, 2967, 3858, 3918, 4110, 4679, 4778, 5869, 7538, 7600, 8114,),
    (1, 215): (723, 1752, 1881, 2076, 2181, 2229, 2257, 2438, 2962,),
    (1, 221): (7489, 7557,),
    (1, 231): (9054, 10470,),
    (1, 243): (7490, 7560, 8296, 9057, 9132, 10235, 10476,),
    (1, 253): (4098, 7553, 8879,),
    (1, 298): (12768,),
    (3, 5): (1951, 8104, 8293, 8876, 9054,),
    (3, 32): (4888,),
    (3, 82): (7477, 7547, 10466,),
    (3, 100): (8868,),
    (3, 110): (4093, 8867,),
    (3, 114): (8867,),
    (3, 129): (4099, 7551, 8869,),
    (3, 152): (2433, 2494, 3608, 4881, 7482,),
    (3, 164): (702, 4096, 7554, 8104, 8869, 9763, 12762,),
    (4, 1): (2517,),
    (4, 2): (1961, 2054, 2156, 4116, 6604, 7586, 8123, 9773, 9794,),
    (4, 21): (7569, 9774,),
    (4, 29): (7573, 9775,),
    (4, 34): (9775,),
    (4, 37): (9773,),
    (4, 45): (722, 2056, 4683, 6605, 7575, 8124, 8147, 8984, 9787, 10266, 12784,),
    (4, 56): (7571,),
    (4, 59): (9776,),
    (4, 64): (713, 2055, 2157, 4114, 5658, 6606, 7587, 8129, 8984, 12777,),
    (4, 65): (1963, 2049, 8119, 8972, 9789,),
    (4, 72): (7565,),
    (4, 79): (2049, 2079, 7573, 9780, 12782,),
    (4, 89): (2052, 7571, 8126,),
    (4, 99): (7571, 9775,),
    (4, 114): (742, 2059, 8132,),
    (4, 117): (7574, 8117,),
    (4, 127): (7571, 9776,),
    (4, 133): (7576, 9791, 9805, 9825, 10488,),
    (4, 135): (2049, 4112, 4855, 5372, 7574, 8117, 9226, 9783, 10487, 11063,),
    (4, 137): (7571,),
    (4, 164): (1967, 2054, 2153, 4114, 5372, 5651, 6889, 8124, 8778, 8980, 9776,),
    (4, 186): (1970, 2053, 4114, 5375, 7585, 8979, 9788,),
    (4, 187): (7571, 9775,),
    (4, 190): (9771,),
    (4, 195): (5373, 7502,),
    (4, 204): (9771,),
    (4, 207): (2516, 2575, 7567,),
    (4, 214): (2050, 8124, 9780,),
    (4, 222): (7572, 9778,),
    (4, 239): (2056, 2157, 4114, 5659, 7576, 8144, 8984, 9786, 12778,),
    (4, 243): (2052, 2154, 5655, 8124, 8979,),
    (4, 270): (2073, 2181, 3805, 5678, 6609, 6988, 7578, 8125, 8148, 9009, 9786, 12796,),
    (4, 296): (1969, 2050, 4112, 4926, 5372, 6881, 7575, 8118, 9148, 9782, 10487, 11064,),
    (4, 300): (7566, 9771,),
    (5, 22): (1926, 8127,),
    (5, 31): (1985, 2079, 2164, 4134, 5664, 6615, 7586, 8145, 8981, 9015, 9078, 9793,),
    (5, 35): (1983, 2161, 5661, 6614, 7573, 8121, 8144, 8781, 8983, 9012, 9087, 9785, 11382, 13420,),
    (5, 46): (1362, 2002, 2061, 2081, 2167, 3909, 4127, 6634, 8145, 9019, 9080, 9794, 12783, 13426,),
    (5, 48): (9777,),
    (5, 57): (9783,),
    (5, 77): (7573,),
    (5, 97): (7576, 9782,),
    (5, 105): (9775,),
    (5, 108): (1974, 4118, 4859, 5376, 7587, 9787, 10331, 10495, 13442, 13465,),
    (5, 128): (1967, 7578, 9785,),
    (5, 133): (7576,),
    (5, 143): (9776,),
    (5, 171): (7578, 8974,),
    (5, 178): (7572,),
    (5, 183): (7575, 9780,),
    (5, 194): (1995, 9783, 10489, 10512,),
    (5, 197): (2059, 4120, 6609, 7582, 8126, 8973, 9013, 9790,),
    (5, 201): (7573, 9778,),
    (5, 202): (404, 585,),
    (5, 207): (7573,),
    (5, 211): (7577,),
    (5, 213): (9775,),
    (5, 224): (2525, 2585, 5811, 6810, 7510, 7582, 12785,),
    (5, 231): (7575, 9776,),
    (5, 232): (5662, 6611, 7582, 8133, 8984, 9799,),
    (5, 234): (9781,),
    (5, 235): (1970, 2057, 4120, 7582, 8126, 8981, 9797,),
    (5, 259): (9782,),
    (5, 262): (570,),
    (5, 264): (2058, 4115, 5661, 6609, 7567, 7587, 7616, 8985, 9799,),
    (5, 270): (7573,),
    (5, 291): (7574, 9778,),
    (5, 296): (7575,),
    (6, 102): (11632,),
    (6, 134): (9853, 10581, 10616, 12167, 12727,),
    (6, 222): (5459,),
    (7, 5): (2579, 7571,),
    (7, 16): (7575, 9782,),
    (7, 31): (7577, 9783,),
    (7, 42): (7571, 9776,),
    (7, 62): (7572, 9776,),
    (7, 85): (9777,),
    (7, 92): (7575, 9782,),
    (7, 111): (5374,),
    (7, 117): (1971, 7587, 8127, 8981, 9789,),
    (7, 125): (2059, 7580, 12782,),
    (7, 132): (1973, 2055, 4116, 5378, 7582, 8125, 8979, 9230, 9778,),
    (7, 133): (2058, 5659, 7578, 8131, 9795,),
    (7, 169): (9782,),
    (7, 174): (1923, 2067, 8118, 8146, 10273,),
    (7, 191): (9777,),
    (7, 203): (8123, 9782,),
    (7, 212): (324, 612, 709, 1377, 1753, 1789, 1879, 2081, 2108, 2169, 2268, 2468, 2976, 3703, 3816, 3930, 4126, 4228, 4266, 4290, 4688, 4706, 4787, 5679, 5882, 6631, 6731, 6760, 6793, 6904, 7000, 7588, 8133, 8803, 8899,),
    (7, 220): (8126, 9783,),
    (7, 221): (719, 2070, 5661, 6608, 8149, 8984, 9796, 12782,),
    (7, 244): (7575, 9778,),
    (8, 16): (7551, 10470,),
    (8, 37): (1959, 4101, 4764, 4844, 5006, 7485, 8026, 8102, 8294, 8877, 9060, 9206, 9770, 10476, 10968, 11053,),
    (8, 46): (2497, 7484,),
    (8, 83): (2063, 4700, 6614, 10283, 10572,),
    (8, 94): (733, 1906, 4706, 6743, 8086, 8872, 10267, 10290, 12796,),
    (8, 109): (728,),
    (8, 136): (737, 790, 1094, 1753, 1905, 4098, 4236, 6661, 6919, 8412, 8870, 10229, 10277, 10621, 10719, 11623, 12695, 12782, 12842, 13156,),
    (8, 143): (571, 4669, 4689, 5660, 5712, 6747, 7565, 12636, 12793,),
    (8, 162): (8874,),
    (8, 220): (7485,),
    (8, 229): (2074, 12806,),
    (8, 247): (2958, 3678, 3791, 3902, 4108, 5667, 7003, 7570, 8122, 8981, 9780, 9804,),
    (9, 34): (1958, 4099, 4841, 5007, 5353, 6788, 7480, 7555, 8025, 8106, 8293, 9041, 9061, 9715, 9749, 9776, 9850, 10469, 11050,),
    (9, 43): (562, 632, 1966, 2545, 2571, 3129, 3394, 4117, 4582, 4665, 4777, 4847, 5019, 5359, 5436, 5778, 6792, 6864, 7486, 7561, 8039, 8125, 8602, 8968, 9064, 9143, 9215, 9685, 9747, 9782, 9875, 10481, 10631, 10782, 10871, 10969, 11045, 11120, 11294, 12625,),
    (9, 47): (7552,),
    (9, 65): (1951, 4100, 7555, 8104, 8291, 8877, 9766,),
    (9, 94): (697, 4101, 7555, 8103, 8872, 9763, 12762,),
    (9, 98): (2494, 7484, 7548,),
    (9, 150): (8101,),
    (9, 156): (2033,),
    (9, 167): (697, 4111, 7559, 8110, 8292, 8886, 9773, 10237, 12760,),
    (9, 172): (5779,),
    (9, 176): (10467,),
    (9, 193): (419, 648, 734, 781, 837, 1092, 1260, 1726, 1761, 1923, 2077, 4096, 4693, 6746, 7557, 8108, 8869, 9763, 9925, 10259, 10287, 10717, 12287, 12687, 12801, 12941, 13148,),
    (9, 227): (7486, 7554,),
    (9, 229): (10466,),
    (9, 238): (4101, 6591, 7555, 8103, 9765,),
    (10, 3): (8922,),
    (10, 22): (8933, 10313, 12818,),
    (10, 56): (2009, 12820,),
    (10, 77): (4736, 6635, 9031, 10273, 12677,),
    (10, 147): (4800,),
    (10, 187): (756, 2468, 4155, 4726, 7520, 8180, 8916, 9108, 10279, 11018,),
    (10, 188): (782, 801, 2964, 3024,),
    (10, 224): (1914, 2089, 4757, 6842, 6927, 6950, 9838, 9851, 10339, 12510,),
    (10, 234): (698, 2096, 2978, 3729, 6766, 6795, 6915, 7880, 7912, 12407, 12764, 12836,),
    (10, 250): (7513,),
    (10, 275): (1914, 1961, 4255, 6638, 7520, 8174, 8912, 9097, 9119, 10293, 11018,),
    (10, 278): (12801,),
    (11, 3): (722, 2078, 2165, 3907, 4121, 6964, 7054, 7594, 8139, 9795, 12786, 13426,),
    (11, 11): (1977, 4116, 4860, 7579, 8123, 8975, 9153, 9788, 10491, 11067, 11385,),
    (11, 22): (295, 588, 724, 1366, 1753, 1972, 2164, 2205, 2224, 3919, 4130, 5685, 6591, 6637, 6653, 7586, 8148, 8952, 9006, 9082, 9793, 10566, 12360, 12652, 12789, 13430,),
    (11, 29): (7575,),
    (11, 36): (2522, 2582, 5377, 7506, 7576,),
    (11, 50): (9779,),
    (11, 61): (325, 402, 591, 731, 829, 1375, 1759, 1871, 1968, 2087, 2162, 2200, 3775, 3819, 3853, 3917, 3964, 6619, 6883, 6956, 6990, 7593, 8143, 8402, 8898,),
    (11, 80): (7576, 9782,),
    (11, 84): (7570,),
    (11, 85): (7578, 9781,),
    (11, 89): (1969, 2060, 2162, 4123, 5663, 6618, 6641, 6676, 8129, 8984, 9796, 13422,),
    (11, 112): (7573, 9778,),
    (11, 127): (731, 12795,),
    (11, 132): (7569,),
    (11, 147): (9776,),
    (11, 148): (9781,),
    (11, 150): (7576,),
    (11, 164): (7573,),
    (11, 169): (7572, 9777,),
    (11, 186): (7575,),
    (11, 187): (7578,),
    (11, 205): (9781,),
    (11, 240): (2058, 5659, 8132, 9786,),
    (11, 252): (7573,),
    (11, 256): (7578, 9781,),
    (11, 264): (7575,),
    (11, 278): (7575,),
    (11, 291): (4118, 5657, 7580, 8124, 8978, 9786,),
    (11, 296): (3697, 5809, 7583,),
}
assert sum(len(v) for v in SNAP_EVENT_TIMES_CAT.values()) == 902, 'snap catalogue is not the guard-D one'

CORRUPT_CASES_NONFINITE = frozenset({
    (7, 16),
    (7, 32),
    (7, 83),
    (7, 132),
    (7, 137),
    (7, 212),
    (7, 214),
    (7, 220),
    (7, 225),
    (7, 229),
    (11, 61),
})

# Cases whose FE run DIVERGED AND TERMINATED EARLY: the cached record is
# shorter than one window (W=1000 @ 10 Hz), so they yield ZERO windows under
# ANY exclusion policy -- per-window masking cannot salvage a record that is
# too short to contain a single window. They were covered by RULE A (peak
# > 1.5 MN) and dropped whole while CORRUPT_CASES was the 109-case union;
# with EXCLUDE_ARTIFACT_WINDOWS=True (patch 19) they must be named explicitly,
# or dataset construction raises 'Not enough time steps' (job 10521328).
#   (5, 202): 626 cached steps, blowup at sample 620
#   (5, 262): 619 cached steps, blowup at sample 614
# Measured from the .npy headers of all 3000 cached cases -- these are the ONLY
# two below 1000 steps (next shortest: loc01/case0215 at 3716 steps).
CORRUPT_CASES_TOO_SHORT = frozenset({
    (5, 202),
    (5, 262),
})

CORRUPT_CASES_ALL = CORRUPT_CASES
CORRUPT_CASES = ((CORRUPT_CASES_NONFINITE | CORRUPT_CASES_TOO_SHORT)
                 if EXCLUDE_ARTIFACT_WINDOWS else CORRUPT_CASES_ALL)
assert len(CORRUPT_CASES_ALL) == 109, len(CORRUPT_CASES_ALL)
assert len(CORRUPT_CASES_NONFINITE) == 11, len(CORRUPT_CASES_NONFINITE)
assert len(CORRUPT_CASES_TOO_SHORT) == 2, len(CORRUPT_CASES_TOO_SHORT)
assert CORRUPT_CASES_TOO_SHORT <= CORRUPT_CASES_ALL, \
    'too-short cases must be a SUBSET of the 109-case union (they were RULE-A drops)'
print(f'[patch19] artifact windows excluded: {EXCLUDE_ARTIFACT_WINDOWS} | '
      f'cases dropped whole: {len(CORRUPT_CASES)} | '
      f'cases with per-window masking: {len(ARTIFACT_SAMPLES)} | '
      f'snap events catalogued: {sum(len(v) for v in SNAP_EVENT_TIMES.values())}')

N_NODES_FULL = 21
CSV_SEP = ";"

# Actual input CSV columns:
# SN is the 1-indexed case number. h0 is the water depth for that case.
ENV_COL_NAMES = [
    "h0", "Hs", "Tp",
    "d1", "v1",
    "d2", "v2",
    "d3", "v3",
    "d4", "v4",
    "d5", "v5",
]
DEPTH_COL_NAMES = ["d1", "d2", "d3", "d4", "d5"]
REQUIRED_ENV_COLUMNS = ["SN", *ENV_COL_NAMES]

def _loc_folder(loc_id: int) -> Path:
    return DATA_ROOT / f"batchRieke_loc{loc_id:02d}"

def _case_dat_path(loc_id: int, case_id: int) -> Path:
    return _loc_folder(loc_id) / f"case_{case_id:04d}" / "gnl_data1.dat"

def inspect_env_csv(loc_id: int) -> None:
    """Print column names and first row of the env CSV for loc_id."""
    input_dir = _loc_folder(loc_id) / "input"
    csvs = sorted(input_dir.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV found in {input_dir}")
    df = pd.read_csv(csvs[0], sep=CSV_SEP)
    print(f"Loc {loc_id:02d} env CSV: {csvs[0].name}")
    print(f"  Columns ({len(df.columns)}): {list(df.columns)}")
    print(f"  First row: {df.iloc[0].to_dict()}")
    print(f"  Shape: {df.shape}")

def load_env_csv(loc_id: int) -> pd.DataFrame:
    """Return the env DataFrame for a location, using the actual semicolon CSV schema."""
    input_dir = _loc_folder(loc_id) / "input"
    csvs = sorted(input_dir.glob("*.csv"))
    if len(csvs) != 1:
        raise FileNotFoundError(f"Expected 1 CSV in {input_dir}, found: {[p.name for p in csvs]}")

    df = pd.read_csv(csvs[0], sep=CSV_SEP)
    missing = [c for c in REQUIRED_ENV_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"CSV {csvs[0].name} is missing columns {missing}. "
            f"Found columns: {list(df.columns)}"
        )
    if len(df) < N_CASES:
        raise ValueError(f"Expected at least {N_CASES} cases in {csvs[0].name}, found {len(df)}.")
    return df

def case_env_row(env_df: pd.DataFrame, case_id: int) -> pd.Series:
    """Return the CSV row whose SN equals the 1-indexed case number."""
    matches = env_df.loc[env_df["SN"] == case_id]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one row with SN={case_id}, found {len(matches)}.")
    return matches.iloc[0]

def case_water_depth(env_df: pd.DataFrame, case_id: int) -> float:
    """Return h0 [m] from the CSV for one case."""
    return float(case_env_row(env_df, case_id)["h0"])

def case_current_depths(env_df: pd.DataFrame, case_id: int) -> list:
    """Return d1..d5 [m] from the CSV for one case."""
    row = case_env_row(env_df, case_id)
    return [float(row[col]) for col in DEPTH_COL_NAMES]

def case_env_tensor(env_df: pd.DataFrame, case_id: int, T: int) -> torch.Tensor:
    """
    Return a [T, 13] float32 tensor with constant env features for one case.

    Parameters
    ----------
    env_df  : DataFrame returned by load_env_csv().
    case_id : 1-indexed case number; matched to the SN column.
    T       : number of time steps (tensor rows will all be identical).
    """
    row_values = case_env_row(env_df, case_id)[ENV_COL_NAMES].to_numpy(dtype=np.float32)
    row_t = torch.tensor(row_values, dtype=torch.float32)   # [13]
    return row_t.unsqueeze(0).expand(T, -1).contiguous()    # [T, 13]

def load_gnl_dat(dat_path: Path, n_nodes: int = 21):
    """
    Load one gnl_data1.dat file.

    Exact file layout:
        - No header
        - Comma-separated
        - 87 columns total for 21 nodes
        - Column 0: time / timestep
        - Columns 1 and 2: unused, drop them
        - Then repeated per node:
            reference, x_abs, z_abs, tension

    Raw expected number of columns:
        3 + 4*n_nodes

    After dropping columns 1 and 2:
        1 + 4*n_nodes

    Returns:
        time:    [T]
        x_abs:   [T, n_nodes]
        z_abs:   [T, n_nodes]
        tension: [T, n_nodes]
    """
    dat_path = Path(dat_path)
    if not dat_path.exists():
        raise FileNotFoundError(f"Missing .dat file: {dat_path}")

    raw_df = pd.read_csv(dat_path, sep=",", header=None)

    raw_expected_cols = 3 + 4 * n_nodes
    if raw_df.shape[1] != raw_expected_cols:
        raise ValueError(
            f"{dat_path} has {raw_df.shape[1]} raw columns, but expected "
            f"{raw_expected_cols} for n_nodes={n_nodes}. "
            "Expected layout: time, unused1, unused2, then "
            "[reference, x_abs, z_abs, tension] repeated for each node."
        )

    df = raw_df.drop(columns=[1, 2])

    expected_cols_after_drop = 1 + 4 * n_nodes
    if df.shape[1] != expected_cols_after_drop:
        raise ValueError(
            f"After dropping columns 1 and 2, {dat_path} has {df.shape[1]} columns, "
            f"but expected {expected_cols_after_drop}."
        )

    time = df.iloc[:, 0].to_numpy(dtype=np.float32)
    node_values = df.iloc[:, 1:].to_numpy(dtype=np.float32)

    if node_values.shape[1] % 4 != 0:
        raise ValueError(
            f"After dropping columns 1 and 2, node data has "
            f"{node_values.shape[1]} columns, which is not divisible by 4."
        )

    inferred_nodes = node_values.shape[1] // 4
    if inferred_nodes != n_nodes:
        raise ValueError(
            f"Inferred {inferred_nodes} nodes from file, but n_nodes={n_nodes}. "
            "Check N_FULL or the file format."
        )

    node_data = node_values.reshape(node_values.shape[0], inferred_nodes, 4)

    reference = node_data[:, :, 0]  # arc-length s [m] per node
    x_abs     = node_data[:, :, 1]
    z_abs     = node_data[:, :, 2]
    tension   = node_data[:, :, 3]

    return time, x_abs, z_abs, tension, reference

def preprocess_all_data(cache_dir: Path, loc_ids, case_ids):
    """
    One-time preprocessing: convert each .dat file to four uncompressed .npy files
    (x_abs, z_abs, tension, reference) plus one env.npy per case.

    Uses uncompressed .npy so np.load(..., mmap_mode='r') works.
    Skips cases whose output directory already exists.
    """
    for loc_id in loc_ids:
        env_df = load_env_csv(loc_id)
        for case_id in case_ids:
            if (loc_id, case_id) in corrupt_cases:
                continue
            case_dir = cache_dir / f"loc{loc_id:02d}" / f"case_{case_id:04d}"
            if (case_dir / "x_abs.npy").exists():
                continue
            case_dir.mkdir(parents=True, exist_ok=True)
            _time, x_abs, z_abs, tension, reference = load_gnl_dat(
                _case_dat_path(loc_id, case_id), n_nodes=N_NODES_FULL
            )
            env_row = case_env_row(env_df, case_id)[ENV_COL_NAMES].to_numpy(np.float32)
            np.save(case_dir / "x_abs.npy",     x_abs.astype(np.float32))     # [T, 21]
            np.save(case_dir / "z_abs.npy",     z_abs.astype(np.float32))
            np.save(case_dir / "tension.npy",   tension.astype(np.float32))
            np.save(case_dir / "reference.npy", reference.astype(np.float32))
            np.save(case_dir / "env.npy",       env_row)                        # [13] env row
        print(f"loc {loc_id:02d}: done")
    print("Preprocessing complete.")


In [ ]:
# preprocess_all_data(
#     cache_dir=CACHE_DIR,
#     loc_ids=AVAILABLE_LOC_IDS,
#     case_ids=list(range(1, N_CASES + 1)),
# )

loc 01: done
loc 03: done
loc 04: done
loc 05: done
loc 06: done


C:\Users\thano\AppData\Local\Temp\ipykernel_24952\4084240722.py:120: DtypeWarning: Columns (0: 86) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(dat_path, sep=",", header=None)


loc 07: done
loc 08: done
loc 09: done
loc 10: done


C:\Users\thano\AppData\Local\Temp\ipykernel_24952\4084240722.py:120: DtypeWarning: Columns (0: 82) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(dat_path, sep=",", header=None)


loc 11: done
Preprocessing complete.


In [ ]:
# -------------------------------------------------------------
# build load_case_datasets from .npy cache (lazy mmap)
# -------------------------------------------------------------
#
# This debug cell intentionally loads a small real-data subset so the full
# pipeline can be verified quickly. Increase DEBUG_* values later for the
# full experiment.

DEBUG_LOC_IDS = [4]
DEBUG_CASE_IDS = [1]
DEBUG_NODE_COUNTS = [4, 8, 12, 21]
DEBUG_MAX_TIMESTEPS = 3000       # W=1000 -> 2001 windows; use None to load all ~13,800 steps

WINDOW_LEN = 1000                # co-temporal input/output window (100 s @ 10 Hz)
USE_ENV_FEATURES = True
USE_VELOCITY_FEATURES = True
USE_DRAG_FEATURES = True


# ------------------------------------------------------------------
# Full-scale lazy dataset builder (uses .npy cache via mmap)
# ------------------------------------------------------------------

def build_lazy_load_case_datasets(
    loc_ids=AVAILABLE_LOC_IDS,
    case_ids=range(1, N_CASES + 1),
    node_counts=(4, 5, 6, 7, 8, 10, 12, 15, 18, 21),
    cache_dir=CACHE_DIR,
    window_len=WINDOW_LEN,
    contact_tol=1e-6,
    max_time_steps=None,
    corrupt_cases=frozenset(),
    use_env_features=USE_ENV_FEATURES,
    use_velocity_features=USE_VELOCITY_FEATURES,
    use_drag_features=USE_DRAG_FEATURES,
):
    """
    Build one lazy MooringSequenceDatasetPositionTension per (loc_id, case_id, target_N).

    Data is read from the pre-processed .npy cache (see preprocess_all_data).
    Each dataset object stores only a directory path -- no time-series data in RAM.
    File handles are opened per DataLoader worker on first access.

    Parameters
    ----------
    max_time_steps : int, optional
        Limit exposed time steps for debug runs. None = use all steps.
    """
    bad_locs = sorted(set(loc_ids) - set(AVAILABLE_LOC_IDS))
    if bad_locs:
        raise ValueError(f"Unavailable locations requested: {bad_locs}.")

    datasets = {}
    _short_skipped = []          # (loc, case, n_steps) records shorter than one window

    for loc_id in loc_ids:
        print(f"\n--- Location {loc_id:02d} ---")

        for case_id in case_ids:
            if (loc_id, case_id) in corrupt_cases:
                continue
            case_dir = cache_dir / f"loc{loc_id:02d}" / f"case_{case_id:04d}"
            if not (case_dir / "x_abs.npy").exists():
                raise FileNotFoundError(
                    f"Cache missing for loc {loc_id}, case {case_id}. "
                    "Run preprocess_all_data() first."
                )

            # An FE run that diverged and terminated early can be SHORTER than
            # one window, which yields zero windows under any exclusion policy.
            # Skip it here rather than raising deep inside the dataset
            # constructor -- an unattended cluster job must not die on this.
            _n_avail = np.load(case_dir / 'x_abs.npy', mmap_mode='r').shape[0]
            if max_time_steps is not None:
                _n_avail = min(_n_avail, max_time_steps)
            if _n_avail < window_len:
                _short_skipped.append((loc_id, case_id, int(_n_avail)))
                print(f'  [skip] loc {loc_id:02d} case {case_id:04d}: '
                       f'{_n_avail} steps < window_len={window_len}')
                continue

            # Read only t=0 row for graph construction -- 21 floats, negligible
            x0 = torch.tensor(np.load(case_dir / "x_abs.npy",     mmap_mode="r")[0], dtype=torch.float32)
            z0 = torch.tensor(np.load(case_dir / "z_abs.npy",     mmap_mode="r")[0], dtype=torch.float32)
            t0 = torch.tensor(np.load(case_dir / "tension.npy",   mmap_mode="r")[0], dtype=torch.float32)
            s0 = torch.tensor(np.load(case_dir / "reference.npy", mmap_mode="r")[0], dtype=torch.float32)

            # env.npy layout: [h0, Hs, Tp, d1, v1, d2, v2, d3, v3, d4, v4, d5, v5]
            env_row        = np.load(case_dir / "env.npy")
            water_depth    = float(env_row[0])                               # h0
            current_depths = [float(env_row[i]) for i in (3, 5, 7, 9, 11)]  # d1..d5

            for target_N in node_counts:
                graph = build_mooring_graph(
                    x0_nodes=resample_fe_output(x0.unsqueeze(0), target_N).squeeze(0),
                    z0_nodes=resample_fe_output(z0.unsqueeze(0), target_N).squeeze(0),
                    tension0_nodes=resample_fe_output(t0.unsqueeze(0), target_N).squeeze(0),
                    s0_nodes=resample_fe_output(s0.unsqueeze(0), target_N).squeeze(0),
                    water_depth=water_depth,
                    depths_current=current_depths,
                )
                graph.batch_location_id = loc_id
                graph.case_id           = case_id

                datasets[(loc_id, case_id, target_N)] = MooringSequenceDatasetPositionTension(
                    graph_data=graph,
                    case_dir=case_dir,
                    target_N=target_N,
                    window_len=window_len,
                    contact_tol=contact_tol,
                    max_time_steps=max_time_steps,
                    use_env_features=use_env_features,
                    use_velocity_features=use_velocity_features,
                    use_drag_features=use_drag_features,
                )

        print(f"  loc {loc_id:02d}: {len(list(case_ids))} cases x {len(list(node_counts))} node counts built")

    print(f"\nTotal lazy datasets: {len(datasets):,}")
    if _short_skipped:
        print(f'[builder] skipped {len(_short_skipped)} case(s) shorter than window_len={window_len}: '
              f'{_short_skipped}')
    return datasets


# Stage 2 (patch sr-06): ALWAYS build every node count -- the N_in x N_out
# grid sweep looks the query graph up as load_case_datasets[(lc, case, N_out)],
# so all ten must exist even for a 6-topology HPO run. cfg.node_counts alone
# selects which topologies enter the SPLITS (build_leakage_aware_splits).
# Publication Stage 1: all locs (train + held-out), cases 1-300, all node_counts, full time series
load_case_datasets = build_lazy_load_case_datasets(
    loc_ids=[1, 3, 4, 5, 6, 7, 8, 9, 10, 11],
    case_ids=range(1, 301),
    node_counts=(4, 5, 6, 7, 8, 10, 12, 15, 18, 21),
    cache_dir=CACHE_DIR,
    window_len=WINDOW_LEN,
    max_time_steps=None,
    corrupt_cases=CORRUPT_CASES,
    use_env_features=USE_ENV_FEATURES,
    use_velocity_features=USE_VELOCITY_FEATURES,
)

# ------------------------------------------------------------------
# Consistency checks
# ------------------------------------------------------------------
assert len(load_case_datasets) > 0, "load_case_datasets is empty."

first_key = sorted(load_case_datasets.keys())[0]
example_ds = load_case_datasets[first_key]

ref_dyn_node  = example_ds.n_dynamic_node_features
ref_dyn_edge  = example_ds.n_dynamic_edge_features
ref_target    = example_ds.n_targets
ref_window    = example_ds.window_len
ref_node_feat = example_ds.graph_data.x.shape[1]
assert ref_node_feat == 12, f"Expected 12 static node features, got {ref_node_feat}"
ref_edge_feat = example_ds.graph_data.edge_attr.shape[1]
assert ref_edge_feat == 10, f"Expected 10 static edge features, got {ref_edge_feat}"
assert ref_target == 1, f"Expected 1 target (tension), got {ref_target}"

# LEAK CHECK: tension must never appear among the model inputs
assert "tension" not in example_ds.dynamic_feature_names, \
    "LEAK: tension found in dynamic input features!"

for (lc_id, case_id, target_N), ds in load_case_datasets.items():
    tag = f"({lc_id},{case_id},{target_N})"
    assert ds.n_dynamic_node_features == ref_dyn_node, f"{tag}: dyn node feat mismatch"
    assert ds.n_dynamic_edge_features == ref_dyn_edge, f"{tag}: dyn edge feat mismatch"
    assert ds.n_targets               == ref_target,   f"{tag}: target dim mismatch"
    assert ds.window_len == ref_window,                f"{tag}: window_len mismatch"
    assert ds.graph_data.x.shape[1]        == ref_node_feat, f"{tag}: static node feat mismatch"
    assert ds.graph_data.edge_attr.shape[1] == ref_edge_feat, f"{tag}: static edge feat mismatch"

print_dataset_summary(example_ds)


In [ ]:
# =============================================================================
# Publication STAGE 2 — SPATIAL SUPER-RESOLUTION  (N_in -> N_out)
# built by _nb_patch_sr_06_runcell.py  (patch sr-06)
# =============================================================================
# Stage 1 is closed: v5 (job 10532625, ep32) reconstructs the tension field at
# MATCHED resolution, N_out == N_in, because the per-node head is sized to the
# input node count. Stage 2 breaks that coupling with a query-based
# cross-attention decoder between the GAT encoder and the LSTM, so the dense
# 21-node field can be reconstructed from as few as 4 motion sensors.
# N_out = 21 is the ceiling: 21 is the native FE resolution and therefore the
# finest ground truth that exists.
#
# What must be true for the result to mean anything:
#   * the decoder must BEAT interpolation of the Stage-1 prediction, cell by
#     cell, or the capability is just interpolation (mode BASELINE measures it);
#   * the N_in axis must be a PAIRED comparison -- patch sr-02 makes every node
#     count draw identical windows, so N_in is not confounded with the draw.
#
# ============== RUN SELECTOR — exactly ONE _MODE line may be active ==========
# Uncomment the line for the job being submitted; each slurm script's GATE 2
# refuses to start unless the active line is the one it expects.
# _MODE = ("BASELINE",      "./checkpoints_P1_matched_v5/best_mooring_gat_lstm_seed_42.pt")
# _MODE = ("HPO_WARM",      "./checkpoints_P1_matched_v5/best_mooring_gat_lstm_seed_42.pt")
# _MODE = ("HPO_SCRATCH",   None)
# _MODE = ("FINAL_WARM",    "./checkpoints_P1_matched_v5/best_mooring_gat_lstm_seed_42.pt")
# _MODE = ("FINAL_SCRATCH", None)
# _MODE = ("EVAL_NOISE",    "./checkpoints_P2_final_warm/best_mooring_gat_lstm_seed_42.pt")
# _MODE = ("EVAL_SNAPROWS", "./checkpoints_P2_final_warm/best_mooring_gat_lstm_seed_42.pt")   # patch sr-20
_MODE = ("EVAL_CLEAN",    "./checkpoints_P2_final_warm/best_mooring_gat_lstm_seed_42.pt")   # patch sr-22
# =============================================================================
import time

_TAG, _CKPT_IN = _MODE
_IS_BASELINE = (_TAG == "BASELINE")
_IS_HPO      = _TAG.startswith("HPO")
_IS_FINAL    = _TAG.startswith("FINAL")
_IS_WARM     = _TAG.endswith("_WARM")
# --- patch sr-19: EVAL_NOISE: eval-only, on the SHIPPED Stage-2 checkpoint.
_IS_NOISE    = (_TAG == "EVAL_NOISE")
# --- patch sr-20: EVAL_SNAPROWS: eval-only, both SHIPPED models.
_IS_SNAPROWS = (_TAG == "EVAL_SNAPROWS")
# --- patch sr-22: EVAL_CLEAN: eval-only, both SHIPPED models, evaluation mask.
_IS_CLEAN    = (_TAG == "EVAL_CLEAN")
_IS_EVAL     = _IS_BASELINE or _IS_NOISE or _IS_SNAPROWS or _IS_CLEAN   # none of these trains
assert _IS_EVAL or _IS_HPO or _IS_FINAL, f"unknown _MODE tag {_TAG}"
assert (_CKPT_IN is not None) == (_IS_EVAL or _IS_WARM), (
    "a checkpoint path is required for BASELINE and *_WARM modes, and must be "
    "None otherwise"
)

_CKPT_DIR = "./checkpoints_P2_" + _TAG.lower()
_RESUME_SCAN_DIR = _CKPT_DIR          # MUST match cfg.checkpoint_dir

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"=== STAGE 2 MODE: {_TAG} -> {_CKPT_DIR} ===")

# A100 throughput package: TF32 matmuls, identical to every ladder run
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# --- patch sr-08: cuDNN RNN escape hatch -------------------------------------
# Job 103567 died in build_model_from_config -> LSTM.to(device) with
#   RuntimeError: cuDNN error: CUDNN_STATUS_BAD_PARAM
# from torch._cudnn_rnn_flatten_weight. The LSTM is the ordinary v5 one (input
# 128, hidden 384, 2 layers, bidirectional) that trained for 7.5 h on this
# cluster under the OLD module stack, so the parameters are not bad -- the cuDNN
# behind py-torch/2.5.1 on the rebuilt cluster rejects a valid RNN descriptor.
# flatten_parameters() short-circuits on torch.backends.cudnn.is_acceptable(),
# which returns False as soon as cuDNN is disabled, so the crashing call becomes
# unreachable and the LSTM falls back to the ATen kernels. This model has NO
# convolutions, so cuDNN only ever served the LSTM: numerics are identical and
# the only cost is RNN speed.
# The slurm's HARD GATE 1-rnn probes the real LSTM on the real GPU before
# staging and exports SR_DISABLE_CUDNN=1 only when it has proven the fallback is
# both NEEDED and WORKING -- so full speed returns by itself once cuDNN is fixed.
if os.environ.get("SR_DISABLE_CUDNN", "") == "1":
    torch.backends.cudnn.enabled = False
try:
    _sr_cudnn_ver = torch.backends.cudnn.version()
except Exception:
    _sr_cudnn_ver = None
print("[cudnn] enabled=" + str(torch.backends.cudnn.enabled)
      + " version=" + str(_sr_cudnn_ver)
      + " SR_DISABLE_CUDNN=" + repr(os.environ.get("SR_DISABLE_CUDNN", "")))

# --- budgets -----------------------------------------------------------------
# BASELINE deliberately shares the FINAL budgets, node counts, split_seed and
# n_fit_windows so its normalizers and its test draw are bit-identical to the
# FINAL run's. Only the training is skipped.
if _IS_HPO:
    _NODE_COUNTS = (4, 6, 8, 12, 15, 21)
    _POOL, _PER_EP, _VAL, _TEST, _FIT = 60_000, 8_000, 2_500, 6_000, 2_000
    _EPOCHS, _PATIENCE, _TEST_EVAL = 60, 60, False
else:
    _NODE_COUNTS = (4, 5, 6, 7, 8, 10, 12, 15, 18, 21)
    _POOL, _PER_EP, _VAL, _TEST, _FIT = 1_000_000, 30_000, 10_000, 50_000, 5_000
    _EPOCHS, _PATIENCE, _TEST_EVAL = 78, 30, True   # patch sr-16: 60 -> 78 (FINAL_WARM run 2 trains ep54-78)

# --- patch sr-12: VRAM. Stage 2 pins N_out=21, so the LSTM and the head run
# at B*21 sequences on EVERY bucket (Stage 1 ran them at B*N_in, so only the
# N=21 bucket ever reached this size). GATE 1-rnn also has to disable cuDNN on
# the rebuilt stack, and the ATen LSTM keeps its per-timestep activations for
# backward where cuDNN kept a compact reserve. Jobs 121890/121892 OOM'd inside
# epoch 1 on an 80 GB A100 at batch 32 (76.82 GiB allocated, asking 1.92 GiB
# more -- exactly one [B*21, 1000, 768] LSTM readout). Measured activation
# model: ~28.6 kB*B*W*N_in + ~143.6 kB*B*W*N_out => ~108 GiB at batch 32,
# ~54 GiB at batch 16. Memory is linear in B, so halving it is the whole fix.
# Effective batch and optimizer-step count are UNCHANGED (16 x 2 = 32).
# BASELINE stays at 32/1: it is no_grad-only and job 105091's interpolation
# bar was produced at that setting.
_BATCH, _ACCUM = (32, 1) if _IS_BASELINE else (16, 2)

# --- AUTO-RESUME (survive a walltime timeout) -------------------------------
# Training modes only: BASELINE trains nothing, so a stale checkpoint in its
# directory must never be picked up.


def _pick_resume_checkpoint(ckpt_dir):
    """Newest resumable checkpoint in ckpt_dir, or None. Verify-loads newest
    first so a checkpoint truncated by a mid-save SIGKILL is skipped."""
    from pathlib import Path as _P
    _p = _P(ckpt_dir)
    if not _p.exists():
        return None
    _cands = []
    for _f in _p.glob("checkpoint_epoch*.pt"):
        _stem = _f.name[len("checkpoint_epoch"):-3]   # strip prefix + ".pt"
        if _stem.isdigit():
            _cands.append((int(_stem), _f))
    # patch sr-13: the rolling per-epoch checkpoint is the primary resume
    # anchor -- feasible-only saving can leave 15-23 epoch gaps.
    _latest = _p / "checkpoint_latest.pt"
    if _latest.exists():
        try:
            _le = torch.load(_latest, map_location="cpu", weights_only=False)
            _cands.append((int(_le.get("epoch", -1)), _latest))
            del _le
        except Exception:
            print("[auto-resume] checkpoint_latest.pt unreadable -- skipped")
    for _f in _p.glob("best_*.pt"):
        try:
            _be = torch.load(_f, map_location="cpu", weights_only=False)
            _cands.append((int(_be.get("epoch", -1)), _f))
            del _be
        except Exception:
            print("[auto-resume] best ckpt " + _f.name + " unreadable -- skipped")
    if not _cands:
        return None
    _cands.sort(key=lambda c: -c[0])
    for _ep, _f in _cands:
        try:
            _ck = torch.load(_f, map_location="cpu", weights_only=False)
            if isinstance(_ck, dict) and "model_state_dict" in _ck and "epoch" in _ck:
                del _ck
                return (str(_f), _ep)
            del _ck
        except Exception:
            print("[auto-resume] candidate " + _f.name + " corrupt -- trying older")
    return None


_resume_hit = None if _IS_EVAL else _pick_resume_checkpoint(_RESUME_SCAN_DIR)
if _resume_hit is not None:
    _resume_path, _resume_ep = _resume_hit
    print("[auto-resume] *** RESUMING from " + _resume_path + " (saved epoch "
          + str(_resume_ep) + ") -> training continues at epoch "
          + str(_resume_ep + 1) + " ***")
else:
    _resume_path = None
    print("[auto-resume] no valid checkpoint to resume -> fresh start")

cfg = TrainingConfig(
    train_lc_ids=(1, 3, 4, 5, 6, 7, 8, 9),
    test_extra_lc_ids=(10, 11),
    node_counts=_NODE_COUNTS,
    # --- reconstruction task ----------------------------------------------
    window_len=WINDOW_LEN,
    use_env_features=USE_ENV_FEATURES,
    use_velocity_features=USE_VELOCITY_FEATURES,
    raw_block_len=2000,
    # --- STAGE 2 -----------------------------------------------------------
    # superres_mode: the DATA carries 21-node targets + a query graph.
    # use_query_decoder: the MODEL gains the decoder. BASELINE turns the model
    # side OFF so a v5 checkpoint loads strict=True and is scored with its
    # prediction interpolated to N_out.
    superres_mode=True,
    superres_out_N=21,
    use_query_decoder=(not _IS_BASELINE),
    query_decoder_heads=4,
    query_decoder_ffn_dim=256,
    query_decoder_fourier_bands=8,
    query_decoder_dropout=0.1,
    query_decoder_residual_interp=True,   # decoder learns a CORRECTION on interpolation
    query_decoder_zero_init=True,         # ... starting from exactly zero correction
    warm_start_checkpoint=(_CKPT_IN if _IS_WARM else None),
    superres_baseline_checkpoint="./checkpoints_P1_matched_v5/best_mooring_gat_lstm_seed_42.pt",
    superres_grid_out_N=(4, 5, 6, 7, 8, 10, 12, 15, 18, 21),
    pair_windows_across_node_counts=True,
    sensor_noise_enabled=(_IS_NOISE or _IS_CLEAN),   # patch sr-19/22: provably off in every training mode
    # --- window budgets -----------------------------------------------------
    max_train_windows=_POOL,
    max_val_windows=_VAL,
    max_test_windows=_TEST,
    max_train_windows_per_epoch=_PER_EP,
    n_fit_windows=_FIT,
    # --- optimization -------------------------------------------------------
    num_epochs=_EPOCHS,
    learning_rate=1e-3,
    weight_decay=1e-4,
    batch_size=_BATCH,
    accumulation_steps=_ACCUM,
    grad_clip_max_norm=1.0,
    # --- architecture: the locked Stage-1 winner (PC1, 2-layer bidirectional)
    gat_hidden_dim=128,
    gat_out_dim=128,
    num_heads=4,
    gat_dropout=0.1,
    lstm_hidden_dim=384,
    num_lstm_layers=2,
    lstm_dropout=0.1,
    lstm_bidirectional=True,
    head_hidden_dim=128,
    head_dropout=0.1,
    use_global_context=True,
    # --- schedule / stopping ------------------------------------------------
    # patch 26 LR fix, stated explicitly so config.json records it and GATE 2
    # can assert on it: the scheduler is fed -R2 (NEGATIVE), for which PyTorch's
    # default threshold_mode="rel" inverts the improvement bar and decay never
    # fires. "abs" is correct for a metric of either sign; min_lr is part of the
    # fix, not a tweak.
    use_scheduler=True,
    scheduler_factor=0.5,
    scheduler_patience=5,
    scheduler_threshold=1e-4,
    scheduler_threshold_mode="abs",
    scheduler_min_lr=5e-5,
    early_stopping_patience=_PATIENCE,
    min_delta=1e-5,
    # --- loaders / runtime --------------------------------------------------
    enable_bucketed_node_count_batching=True,
    drop_last_batch=False,
    num_workers=8,
    pin_memory=(device.type == "cuda"),
    use_amp=False,
    validate_every=1,
    # --- checkpointing: FEASIBLE-ONLY (disk-lean) ---------------------------
    save_every_n_epochs=(0 if _IS_HPO else 5),   # patch sr-13: periodic ckpts at FINAL
    save_latest_checkpoint=True,
    save_feasible_checkpoints=True,
    debug_batch_shapes=False,
    seed_list=(42,),
    checkpoint_dir=_CKPT_DIR,
    resume_from_checkpoint=_resume_path,
    run_test_evaluation=_TEST_EVAL,
    compute_train_physical_metrics=True,
    max_train_metric_batches=25,
    # --- SELECTION v4 (unchanged from Stage 1) ------------------------------
    # Gate A: val_global_R2_tension >= running_max - sel_r2_delta.
    # Gate B: val_peak_tension_underpred_rate_p90 < sel_up90_cap. p90 ONLY --
    # the p99 tail is 43-82 % snap-composed, so a p99 rate cap gave zero
    # feasible epochs twice and both runs shipped the safety-blind fallback.
    # Objective: min val_temporal_diff_skill_tension among feasible epochs.
    snap_oversample_frac=0.0,
    sel_r2_delta=0.0010,
    # patch sr-16 (user, 2026-09-10): Gate B is OFF at FINAL. Run 1 of FINAL_WARM
    # (job 129246) showed the rate cap and the bias floor are one sign statistic
    # (corr -0.84): they rejected every epoch after ep33, although those epochs are
    # ~25 % more accurate and carry LOWER tail under-mass. FINAL now selects on
    # Gate A + temporal skill; the HPO modes keep their 10 % cap.
    sel_up90_cap=(10.0 if _IS_HPO else float("inf")),   # patch sr-16: no rate cap at FINAL; HPO keeps 10 %
    sel_bias90_min=float("-inf"),   # patch sr-16: no bias floor (HPO never had one)
    # patch sr-10: the epoch-20 hard abort is OFF for the HPO arms.  Stage 2
    # is judged on accuracy + temporal fidelity, and the interpolation bar's
    # pooled up90 is already 10.94 % (> the 10 % cap), so the abort could kill
    # an accuracy-first run on a deprioritised metric.  Gate B itself is
    # unchanged, so n_feasible stays a diagnostic.  FINAL keeps the abort.
    sel_abort_epoch=(0 if _IS_HPO else 20),
)

assert _RESUME_SCAN_DIR == cfg.checkpoint_dir, (
    "auto-resume scan dir != cfg.checkpoint_dir -- keep them in sync")
assert cfg.superres_mode, "Stage 2 requires superres_mode=True"

set_seed(cfg.seed_list[0])

t0 = time.time()
train_loader, val_loader, test_loader, norms, split_indices = build_dataloaders(
    load_case_datasets=load_case_datasets, cfg=cfg,
)
print(f"build_dataloaders took {time.time() - t0:.1f} s")

# The N_in sweep is only interpretable if every node count scored the SAME
# windows. Cheap to verify, catastrophic to get wrong silently.
assert_windows_are_paired(split_indices, cfg.node_counts)

# Normalizer fingerprint. BASELINE and FINAL share budgets/seed, so these two
# numbers MUST agree between the two jobs -- if they do not, the baseline and
# the decoder were scored under different normalisations and the delta surface
# is meaningless.
_tn = norms["target_norm"]
print(f"[norm] tension mean {float(_tn.mean[0]):.6f} N  "
      f"std {float(_tn.std[0]):.6f} N   (compare across Stage-2 jobs)")

first_key = sorted(load_case_datasets.keys())[0]
example_dataset = load_case_datasets[first_key]
assert cfg.window_len == example_dataset.window_len, (
    f"cfg.window_len={cfg.window_len} does not match the dataset build "
    f"constant WINDOW_LEN={example_dataset.window_len}"
)
_built_ncs = sorted({key[2] for key in load_case_datasets.keys()})
assert set(cfg.node_counts) <= set(_built_ncs), (
    f"cfg.node_counts={sorted(cfg.node_counts)} is not covered by the built "
    f"datasets {_built_ncs}")
assert set(cfg.superres_grid_out_N) <= set(_built_ncs), (
    f"grid N_out values {sorted(cfg.superres_grid_out_N)} are not all built "
    f"({_built_ncs}) -- the query-graph lookup would KeyError mid-sweep")

criterion = ReconstructionKendallLoss(
    node_norm=norms["node_norm"],
    target_norm=norms["target_norm"],
    cfg=cfg,
).to(device)

# =============================================================================
# MODE: BASELINE — no training. Score v5 + interpolation over the whole grid.
# =============================================================================
if _IS_BASELINE:
    from pathlib import Path as _P
    _P(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)

    # Reuse v5's own Kendall log_vars so the loss-term diagnostics in the grid
    # are on the same footing as the run that produced the checkpoint.
    try:
        _bck = torch.load(_CKPT_IN, map_location=device, weights_only=False)
    except TypeError:
        _bck = torch.load(_CKPT_IN, map_location=device)
    if _bck.get("criterion_state_dict"):
        criterion.load_state_dict(_bck["criterion_state_dict"], strict=False)
        print("[baseline] loaded v5 criterion state (Kendall log_vars)")
    del _bck

    baseline_rows = run_interpolation_baseline(
        example_dataset=example_dataset, cfg=cfg, device=device,
        checkpoint_path=_CKPT_IN,
        test_loader=test_loader, criterion=criterion,
        target_norm=norms["target_norm"],
        target_names=example_dataset.target_feature_names,
    )
    print()
    print("BASELINE complete. superres_grid_baseline.csv is the bar the query "
          "decoder has to clear.")

# =============================================================================
# MODE: HPO_* / FINAL_* — train the query decoder
# =============================================================================
# =============================================================================
# MODE: EVAL_NOISE -- no training. Score the SHIPPED Stage-2 model over a
# motion-sensor noise sweep, and emit the per-window statistics that the
# wave-train repeat split needs.
# =============================================================================
elif _IS_NOISE:
    from pathlib import Path as _P
    _P(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)
    assert cfg.sensor_noise_enabled, "EVAL_NOISE requires sensor_noise_enabled"

    # ---- the shipped Stage-2 model, loaded WHOLE (this is not a warm start) --
    model = build_model_from_config(
        example_dataset=example_dataset, cfg=cfg, device=device
    )
    assert model.query_decoder is not None, "Stage-2 model has no query decoder"
    try:
        _nck = torch.load(_CKPT_IN, map_location=device, weights_only=False)
    except TypeError:
        _nck = torch.load(_CKPT_IN, map_location=device)
    _nsd = _nck["model_state_dict"]
    assert any(k.startswith("query_decoder.") for k in _nsd), (
        f"{_CKPT_IN} carries no query_decoder.* keys -- that is a STAGE-1 "
        f"checkpoint, not the Stage-2 model this mode scores"
    )
    model.load_state_dict(_nsd, strict=True)
    model.eval()
    if _nck.get("criterion_state_dict"):
        criterion.load_state_dict(_nck["criterion_state_dict"], strict=False)
    print(f"[noise] Stage-2 checkpoint epoch {_nck.get('epoch')} loaded strict=True")
    del _nck, _nsd

    def _pin_test_sampler(_ep=0):
        """Pin the bucketed sampler so EVERY level scores the SAME windows.

        BucketedNodeCountBatchSampler.__iter__ seeds rng = Random(seed + epoch)
        and then does epoch += 1, so consecutive passes reshuffle -- two
        unpinned passes were measured sharing only 35 of 160 windows. Without
        this the noise levels would sit on different draws and could not be
        compared. dump_test_timeseries pins it internally for the same reason.
        """
        _bs = getattr(test_loader, "batch_sampler", None)
        if _bs is not None and hasattr(_bs, "epoch"):
            _bs.epoch = int(_ep)

    # ---- the levels ---------------------------------------------------------
    # ARM P  independent channels, ABSOLUTE sigma. grafe2024virtual Table 6
    #        quotes 0.05 m rms heave and 0.05 m/s rms velocity for a commercial
    #        GNSS-IMU, so 0.05 is the instrument anchor; 0.01 is RTK-class.
    # ARM S  per-channel SNR, xie2026sparse's own levels, so the robustness
    #        table can be read against the closest prior work.
    # ARM C  derivative-coupled bound: velocity is NOT reported by the sensor
    #        but differentiated from the noisy position, which at dt = 0.1 s
    #        amplifies ~7x and much more in the quadratic drag features.
    # ARM B  constant per-station bias. grafe2024virtual 5 and williams2024shred
    #        both explicitly disclaim this failure mode; it is also the one that
    #        probes this model's known weak spot, the loc10 level offset.
    _SN_SIGMAS        = (0.01, 0.05, 0.10, 0.25)
    _SN_SNRS_DB       = (20.0, 10.0, 5.0, 0.0)
    _SN_COUPLED_SIGMA = 0.05
    _SN_BIAS_SIGMA    = 0.05

    _nn = norms["node_norm"]
    _LEVELS = [("", None)]
    for _k, _s in enumerate(_SN_SIGMAS):
        _LEVELS.append(("_p%03d" % round(_s * 1000),
                        sensor_noise_spec_absolute(10 + _k, "abs_%.3f" % _s,
                                                   sigma_p=_s, sigma_v=_s)))
    for _k, _db in enumerate(_SN_SNRS_DB):
        _LEVELS.append(("_snr%02d" % int(_db),
                        sensor_noise_spec_snr(20 + _k, _db, _nn, cfg)))
    _LEVELS.append(("_coupled%03d" % round(_SN_COUPLED_SIGMA * 1000),
                    sensor_noise_spec_absolute(30, "coupled", sigma_p=_SN_COUPLED_SIGMA,
                                               sigma_v=0.0)))
    _LEVELS.append(("_bias%03d" % round(_SN_BIAS_SIGMA * 1000),
                    sensor_noise_spec_absolute(31, "bias", sigma_b=_SN_BIAS_SIGMA)))
    print(f"[noise] {len(_LEVELS)} levels: {[t or '(clean)' for t, _ in _LEVELS]}")

    # ---- the conversion table: sigma <-> SNR <-> normalised sigma ------------
    # Emitted once so the robustness table can be read against xie2026sparse
    # (dB), yan2026dacla (normalised sigma) or grafe2024virtual (absolute rms)
    # without re-deriving anything offline.
    import csv as _csv_sn
    with open(_P(cfg.checkpoint_dir) / "sensor_noise_levels.csv", "w",
              newline="", encoding="utf-8") as _f_sn:
        _w_sn = _csv_sn.writer(_f_sn)
        _w_sn.writerow(["tag", "name", "axis", "channel", "sigma_physical",
                        "snr_dB", "sigma_normalised", "channel_train_std"])
        for _tg, _sp in _LEVELS:
            if _sp is None:
                _w_sn.writerow([_tg or "clean", "clean", "none", "", 0.0,
                                "inf", 0.0, ""])
                continue
            for _cn, _sd, _snr, _rel, _std in sensor_noise_report(_sp, _nn):
                _w_sn.writerow([_tg, _sp["name"], _sp.get("axis", ""), _cn,
                                "%.6g" % _sd, "%.4g" % _snr, "%.6g" % _rel,
                                "%.6g" % _std])
    print("[noise] wrote sensor_noise_levels.csv (the sigma / dB / normalised "
          "conversion table)")

    # ---- the sweep ----------------------------------------------------------
    for _tg, _spec in _LEVELS:
        _lbl = _tg or "(clean)"
        print()
        print("=" * 70)
        print(f"[noise] LEVEL {_lbl}")
        print("=" * 70)
        set_sensor_noise(test_loader, _spec)

        _pin_test_sampler(0)
        test_model(
            model=model, test_loader=test_loader, criterion=criterion,
            device=device, target_norm=norms["target_norm"],
            target_names=example_dataset.target_feature_names, cfg=cfg,
            tag=_tg,
        )
        try:
            _pin_test_sampler(0)
            dump_test_timeseries(
                model=model, test_loader=test_loader, device=device,
                target_norm=norms["target_norm"],
                target_names=example_dataset.target_feature_names, cfg=cfg,
                tag=_tg, make_plots=False,
            )
        except Exception as _tse:
            print(f"[ts-dump] level {_lbl} FAILED (metrics unaffected): {_tse}")
        try:
            # The SAME dataset feeds the baseline, so it sees the SAME noise
            # realisation -- which is what makes the decoder-minus-baseline
            # comparison valid at every level. Restricted to N_out = 21: that
            # is the paper's claim, and it costs one pass instead of ten.
            _pin_test_sampler(0)
            run_interpolation_baseline(
                example_dataset=example_dataset, cfg=cfg, device=device,
                checkpoint_path=cfg.superres_baseline_checkpoint,
                test_loader=test_loader, criterion=criterion,
                target_norm=norms["target_norm"],
                target_names=example_dataset.target_feature_names,
                out_N_values=(21,), tag="_baseline" + _tg,
            )
        except Exception as _be:
            print(f"[baseline] level {_lbl} FAILED (decoder results unaffected): {_be}")

    set_sensor_noise(test_loader, None)

    # ---- THE REPRODUCTION GATE ----------------------------------------------
    # The mask, the split seed and the window budgets are untouched and the
    # clean level injects no noise at all (sensor_noise_is_off short-circuits
    # before the generator is built), so the clean pass MUST reproduce the
    # shipped Stage-2 numbers. If it does not, the noise plumbing perturbed
    # something it should not have and NOTHING in this job is interpretable.
    _SR19_REF = {
        "test_global_R2_tension": 0.9986449491362092,
        "test_MAE_tension": 34.35475408737659,
        "test_MAPE_tension": 2.536523470958695,
        "test_temporal_diff_skill_tension": 0.3321352845488848,
        "test_peak_tension_MAE": 78.24673563661575,
        "test_peak_tension_underpred_rate_p90": 36.013333333333335,
        "test_peak_tension_underpred_rate_p99": 76.99047619047619,
        "test_contact_touchdown_R2_median": 0.9901615977287292
    }
    try:
        import csv as _csv_chk
        with open(_P(cfg.checkpoint_dir) / "test_metrics.csv", encoding="utf-8") as _f_chk:
            _got = {r[0]: r[1] for r in _csv_chk.reader(_f_chk) if len(r) >= 2}
        _n_diff = 0
        print()
        for _k, _want in _SR19_REF.items():
            if _k not in _got:
                print(f"[check] {_k}: NOT IN test_metrics.csv")
                _n_diff += 1
                continue
            _v = float(_got[_k])
            _rel = abs(_v - _want) / max(abs(_want), 1e-9)
            _st = "OK  " if _rel < 1e-3 else "DIFF"
            if _rel >= 1e-3:
                _n_diff += 1
            print(f"[check] {_st} {_k}: got {_v:.6f}, shipped {_want:.6f}  "
                  f"({100.0 * _rel:.4f} %)")
        if _n_diff:
            print("*" * 70)
            print(f"*** {_n_diff} METRIC(S) MOVED AT sigma = 0 -- the clean pass does")
            print("*** NOT reproduce the shipped Stage-2 model. The noise plumbing has")
            print("*** perturbed something it should not have. Do NOT use this sweep.")
            print("*" * 70)
        else:
            print("[check] clean level reproduces the shipped Stage-2 model -- the "
                  "only thing that changed is the injected sensor noise.")
    except Exception as _ce:
        print(f"[check] reproduction check could not run ({_ce}); compare by hand")

    print()
    print("EVAL_NOISE complete. Per level: test_metrics{tag}.csv, "
          "test_metrics_by_node_count{tag}.csv, test_per_window_stats{tag}.csv, "
          "test_timeseries_dump{tag}.npz, superres_grid_baseline{tag}.csv.")

# =============================================================================
# MODE: EVAL_SNAPROWS (patch sr-20) -- no training. Re-score the two SHIPPED
# models on their own published test draws and store, for every catalogued snap
# event, the true and predicted tension at ALL output stations over +-2 samples
# (dump keys snap_nbhd_*). The FINAL dumps hold only the LINE MAXIMUM at the
# event, so capture at the snap's OWN station was measurable only on the ~136
# windows stored with full traces.
#   PASS A  Stage 2, FINAL_WARM ep76, paired Stage-2 draw, N_out = 21
#           -> checkpoints_P2_eval_snaprows/
#   PASS B  Stage 1, v5 ep32, ORIGINAL unpaired Stage-1 draw, matched
#           resolution (the draw of eval_v5_snapfix, job 10652819)
#           -> checkpoints_P2_eval_snaprows/stage1/
# Each pass must reproduce its published test metrics AND its published
# snap-event rows exactly, or the new rows describe OTHER events.
# =============================================================================
elif _IS_SNAPROWS:
    import csv as _csv20
    import dataclasses as _dc20
    import hashlib as _hl20
    import traceback as _tb20
    import numpy as _np20
    from pathlib import Path as _P
    _P(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)

    # published test_metrics.csv of each shipped model (generated at patch time)
    _SR20_REF_S2 = {
        "test_global_R2_tension": 0.9986449491362092,
        "test_MAE_tension": 34.35475408737659,
        "test_MAPE_tension": 2.536523470958695,
        "test_temporal_diff_skill_tension": 0.3321352845488848,
        "test_peak_tension_MAE": 78.24673563661575,
        "test_peak_tension_underpred_rate_p90": 36.013333333333335,
        "test_peak_tension_underpred_rate_p99": 76.99047619047619,
        "test_contact_touchdown_R2_median": 0.9901615977287292,
    }
    _SR20_REF_S1 = {
        "test_global_R2_tension": 0.998690946534352,
        "test_MAE_tension": 45.23321028131485,
        "test_MAPE_tension": 1.8002109384909273,
        "test_temporal_diff_skill_tension": 0.3683136906736695,
        "test_peak_tension_MAE": 86.14339745024205,
        "test_peak_tension_underpred_rate_p90": 8.419069153947616,
        "test_peak_tension_underpred_rate_p99": 38.552854720180896,
        "test_contact_touchdown_R2_median": 0.9907784461975098,
    }
    # published snap-event rows: (rows, distinct events, sha of the sorted rows)
    _SR20_EV_S2 = (1130, 89, '93e52a7979859fda')
    _SR20_EV_S1 = (1161, 272, '24d319ff6803c4e0')
    # v5's own target normalizer (criterion t_mean / t_std of the checkpoint)
    _SR20_V5_NORM = (2791.087646484375, 2727.0)

    def _sr20_pin(_loader, _ep=0):
        """Pin the bucketed sampler: it reshuffles on every pass otherwise."""
        _bs = getattr(_loader, "batch_sampler", None)
        if _bs is not None and hasattr(_bs, "epoch"):
            _bs.epoch = int(_ep)

    def _sr20_check_metrics(_dir, _ref, _label):
        with open(_P(_dir) / "test_metrics.csv", encoding="utf-8") as _f:
            _got = {r[0]: r[1] for r in _csv20.reader(_f) if len(r) >= 2}
        _n = 0
        for _k, _want in _ref.items():
            if _k not in _got:
                print(f"[check {_label}] DIFF {_k}: not in test_metrics.csv")
                _n += 1
                continue
            _v = float(_got[_k])
            _rel = abs(_v - _want) / max(abs(_want), 1e-9)
            _ok = _rel < 1e-3
            _n += 0 if _ok else 1
            print(f"[check {_label}] {'OK  ' if _ok else 'DIFF'} {_k}: got {_v:.6f}, "
                  f"published {_want:.6f}  ({100.0 * _rel:.4f} %)")
        return _n

    def _sr20_check_events(_dir, _want, _label):
        _nrow, _nev, _sha = _want
        _z = _np20.load(_P(_dir) / "test_timeseries_dump.npz")
        _ev = _np20.asarray(_z["snap_ev"], dtype=_np20.int64).reshape(-1, 5)
        _srt = _ev[_np20.lexsort(_ev.T[::-1])]
        _h = _hl20.sha256(_np20.ascontiguousarray(_srt).tobytes()).hexdigest()[:16]
        _ne = len({(int(r[0]), int(r[1]), int(r[4])) for r in _ev})
        _n = 0
        for _what, _g, _w in (("rows", len(_ev), _nrow),
                              ("distinct events", _ne, _nev),
                              ("sorted-row sha", _h, _sha)):
            _ok = (_g == _w)
            _n += 0 if _ok else 1
            print(f"[check {_label}] {'OK  ' if _ok else 'DIFF'} snap {_what}: "
                  f"got {_g}, published {_w}")
        # the stored event row must give back the line maxima exactly
        _nt, _npr = _z["snap_nbhd_true"], _z["snap_nbhd_pred"]
        _i0 = list(_z["snap_nbhd_offsets"]).index(0)
        _ok = (len(_ev) > 0 and len(_nt) == len(_ev)
               and _np20.allclose(_np20.nanmax(_nt[:, _i0, :], axis=1),
                                  _z["snap_true"], rtol=1e-6)
               and _np20.allclose(_np20.nanmax(_npr[:, _i0, :], axis=1),
                                  _z["snap_pred"], rtol=1e-6))
        _n += 0 if _ok else 1
        print(f"[check {_label}] {'OK  ' if _ok else 'DIFF'} stored event rows give "
              f"back snap_true / snap_pred")
        return _n

    _n_bad = 0

    # ---- PASS A: Stage 2 on the paired draw built above -------------------
    print()
    print("=" * 70)
    print("[snaprows] PASS A -- Stage 2 (FINAL_WARM), paired draw, N_out = 21")
    print("=" * 70)
    model = None
    try:
        model = build_model_from_config(
            example_dataset=example_dataset, cfg=cfg, device=device
        )
        assert model.query_decoder is not None, "Stage-2 model has no query decoder"
        try:
            _ck20 = torch.load(_CKPT_IN, map_location=device, weights_only=False)
        except TypeError:
            _ck20 = torch.load(_CKPT_IN, map_location=device)
        assert any(k.startswith("query_decoder.") for k in _ck20["model_state_dict"]), (
            f"{_CKPT_IN} carries no query_decoder.* keys -- that is a STAGE-1 "
            f"checkpoint, not the Stage-2 model")
        model.load_state_dict(_ck20["model_state_dict"], strict=True)
        model.eval()
        if _ck20.get("criterion_state_dict"):
            criterion.load_state_dict(_ck20["criterion_state_dict"], strict=False)
        print(f"[snaprows] Stage-2 checkpoint epoch {_ck20.get('epoch')} loaded strict=True")
        del _ck20
        _sr20_pin(test_loader)
        test_model(
            model=model, test_loader=test_loader, criterion=criterion,
            device=device, target_norm=norms["target_norm"],
            target_names=example_dataset.target_feature_names, cfg=cfg,
        )
        _sr20_pin(test_loader)
        dump_test_timeseries(
            model=model, test_loader=test_loader, device=device,
            target_norm=norms["target_norm"],
            target_names=example_dataset.target_feature_names, cfg=cfg,
            make_plots=False,
        )
        _n_bad += _sr20_check_metrics(cfg.checkpoint_dir, _SR20_REF_S2, "S2")
        _n_bad += _sr20_check_events(cfg.checkpoint_dir, _SR20_EV_S2, "S2")
    except Exception as _ea:
        _tb20.print_exc()
        print(f"[snaprows] PASS A FAILED: {_ea!r}")
        _n_bad += 1
    model = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---- PASS B: Stage 1 on ITS OWN draw, rebuilt -------------------------
    print()
    print("=" * 70)
    print("[snaprows] PASS B -- Stage 1 (v5), ORIGINAL unpaired draw, matched")
    print("=" * 70)
    try:
        cfg_s1 = _dc20.replace(
            cfg, superres_mode=False, pair_windows_across_node_counts=False,
            use_query_decoder=False,
            checkpoint_dir=str(_P(cfg.checkpoint_dir) / "stage1"),
        )
        _P(cfg_s1.checkpoint_dir).mkdir(parents=True, exist_ok=True)
        set_seed(cfg_s1.seed_list[0])
        _tr1, _va1, test_loader_s1, norms_s1, _sp1 = build_dataloaders(
            load_case_datasets=load_case_datasets, cfg=cfg_s1,
        )
        del _tr1, _va1, _sp1
        _tn1 = norms_s1["target_norm"]
        _m1, _s1 = float(_tn1.mean[0]), float(_tn1.std[0])
        _ok = (abs(_m1 - _SR20_V5_NORM[0]) / _SR20_V5_NORM[0] < 1e-6
               and abs(_s1 - _SR20_V5_NORM[1]) / _SR20_V5_NORM[1] < 1e-6)
        _n_bad += 0 if _ok else 1
        print(f"[check S1] {'OK  ' if _ok else 'DIFF'} target normalizer "
              f"{_m1:.6f} / {_s1:.6f} N, v5 {_SR20_V5_NORM[0]:.6f} / "
              f"{_SR20_V5_NORM[1]:.6f} N")
        criterion_s1 = ReconstructionKendallLoss(
            node_norm=norms_s1["node_norm"], target_norm=_tn1, cfg=cfg_s1,
        ).to(device)
        model_s1 = build_model_from_config(
            example_dataset=example_dataset, cfg=cfg_s1, device=device
        )
        assert getattr(model_s1, "query_decoder", None) is None, (
            "the Stage-1 pass must build the matched model, with no decoder")
        try:
            _v520 = torch.load(cfg.superres_baseline_checkpoint, map_location=device,
                               weights_only=False)
        except TypeError:
            _v520 = torch.load(cfg.superres_baseline_checkpoint, map_location=device)
        assert int(_v520.get("epoch", -1)) == 32, "the Stage-1 checkpoint is not v5 ep32"
        assert not any(k.startswith("query_decoder.") for k in _v520["model_state_dict"]), (
            "the Stage-1 checkpoint carries query_decoder.* keys -- it is a Stage-2 one")
        model_s1.load_state_dict(_v520["model_state_dict"], strict=True)
        model_s1.eval()
        if _v520.get("criterion_state_dict"):
            criterion_s1.load_state_dict(_v520["criterion_state_dict"], strict=False)
        print(f"[snaprows] v5 checkpoint epoch {_v520.get('epoch')} loaded strict=True")
        del _v520
        _sr20_pin(test_loader_s1)
        test_model(
            model=model_s1, test_loader=test_loader_s1, criterion=criterion_s1,
            device=device, target_norm=_tn1,
            target_names=example_dataset.target_feature_names, cfg=cfg_s1,
        )
        _sr20_pin(test_loader_s1)
        dump_test_timeseries(
            model=model_s1, test_loader=test_loader_s1, device=device,
            target_norm=_tn1,
            target_names=example_dataset.target_feature_names, cfg=cfg_s1,
            make_plots=False,
        )
        _n_bad += _sr20_check_metrics(cfg_s1.checkpoint_dir, _SR20_REF_S1, "S1")
        _n_bad += _sr20_check_events(cfg_s1.checkpoint_dir, _SR20_EV_S1, "S1")
    except Exception as _eb:
        _tb20.print_exc()
        print(f"[snaprows] PASS B FAILED: {_eb!r}")
        _n_bad += 1

    print()
    if _n_bad:
        print("*" * 70)
        print(f"*** {_n_bad} CHECK(S) FAILED -- a pass does NOT reproduce its published")
        print("*** test set, so its snap rows may describe OTHER events. Do NOT use")
        print("*** that pass's snap_nbhd_* arrays.")
        print("*" * 70)
    else:
        print("[check] both passes reproduce their published test metrics and "
              "snap-event rows -- the snap_nbhd_* arrays belong to the published events.")
    print("EVAL_SNAPROWS complete. Stage 2: test_timeseries_dump.npz; "
          "Stage 1: stage1/test_timeseries_dump.npz.")

# =============================================================================
# MODE: EVAL_CLEAN (patch sr-22) -- no training. Re-score the two SHIPPED models
# under the EVALUATION mask: the published test draws with every window dropped that
# covers a sample of the catenary rule (a station above 1 kN out-pulling the fairlead
# at the same instant, CLAUDE.md 12.14). The models were trained on the mask without
# that rule, so only the TEST population changes, and only by removal: each draw is
# built exactly as published, verified, and then thinned after the cap.
#   PASS A  Stage 2 (ep76): reproduce on the published draw (-> published/), then on
#           the clean draw: metrics + per-window stats + dump, the full grid and its
#           interpolation baseline (-> checkpoints_P2_eval_clean/).
#   PASS B  Stage 1 (v5 ep32), original unpaired draw: reproduce
#           (-> stage1/published/), then clean metrics + per-window stats + dump
#           (-> stage1/).
#   PASS C  the EVAL_NOISE levels on the clean Stage-2 draw.
# Every clean dump joins on SNAP_EVENT_TIMES_CAT, the catalogue under the evaluation
# mask; the published catalogue is restored after each dump.
# =============================================================================
elif _IS_CLEAN:
    import csv as _csv22
    import dataclasses as _dc22
    import hashlib as _hl22
    import traceback as _tb22
    import numpy as _np22
    from pathlib import Path as _P
    _P(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)
    assert cfg.sensor_noise_enabled, "EVAL_CLEAN runs the noise sweep: sensor_noise_enabled must be on"

    # published test metrics of each shipped model (generated at patch time)
    _SR22_REF_S2 = {
        'test_global_R2_tension': 0.9986449491362092,
        'test_MAE_tension': 34.35475408737659,
        'test_MAPE_tension': 2.536523470958695,
        'test_temporal_diff_skill_tension': 0.3321352845488848,
        'test_peak_tension_MAE': 78.24673563661575,
        'test_peak_tension_underpred_rate_p90': 36.013333333333335,
        'test_peak_tension_underpred_rate_p99': 76.99047619047619,
        'test_contact_touchdown_R2_median': 0.9901615977287292,
    }
    _SR22_REF_S1 = {
        'test_global_R2_tension': 0.998690946534352,
        'test_MAE_tension': 45.23321028131485,
        'test_MAPE_tension': 1.8002109384909273,
        'test_temporal_diff_skill_tension': 0.3683136906736695,
        'test_peak_tension_MAE': 86.14339745024205,
        'test_peak_tension_underpred_rate_p90': 8.419069153947616,
        'test_peak_tension_underpred_rate_p99': 38.552854720180896,
        'test_contact_touchdown_R2_median': 0.9907784461975098,
    }
    _SR22_V5_NORM = (2791.087646484375, 2727.0)
    # published test draws: (pairs, sha of the sorted (loc, case, N, start) rows)
    _SR22_DRAW_S2 = (50000, 'fb681c574549d8cb')
    _SR22_DRAW_S1 = (50000, 'cf2d297458b7d78b')
    # pairs of each published draw that cover a CATENARY_SAMPLES sample
    _SR22_DROP_S2 = 470
    _SR22_DROP_S1 = 487
    # clean-draw snap rows / distinct events when joined on SNAP_EVENT_TIMES_CAT
    _SR22_EV_S2 = (500, 45)
    _SR22_EV_S1 = (523, 154)
    _SR22_SNAP_PUB = SNAP_EVENT_TIMES     # published catalogue; restored after every clean dump

    def _sr22_pin(_loader, _ep=0):
        """Pin the bucketed sampler: it reshuffles on every pass otherwise."""
        _bs = getattr(_loader, "batch_sampler", None)
        if _bs is not None and hasattr(_bs, "epoch"):
            _bs.epoch = int(_ep)

    def _sr22_draw(_split):
        """(pairs, sha, rows) of a test split, rows = sorted (loc, case, N, start)."""
        _a = _np22.array(sorted((int(k[0]), int(k[1]), int(k[2]), int(w))
                                for k, w in _split["test"]), dtype=_np22.int64).reshape(-1, 4)
        return (len(_a), _hl22.sha256(_np22.ascontiguousarray(_a).tobytes()).hexdigest()[:16], _a)

    def _sr22_check(_label, _what, _got, _want):
        _ok = (_got == _want)
        print(f"[check {_label}] {'OK  ' if _ok else 'DIFF'} {_what}: got {_got}, "
              f"expected {_want}")
        return 0 if _ok else 1

    def _sr22_check_metrics(_dir, _ref, _label):
        with open(_P(_dir) / "test_metrics.csv", encoding="utf-8") as _f:
            _got = {r[0]: r[1] for r in _csv22.reader(_f) if len(r) >= 2}
        _n = 0
        for _k, _want in _ref.items():
            if _k not in _got:
                print(f"[check {_label}] DIFF {_k}: not in test_metrics.csv")
                _n += 1
                continue
            _v = float(_got[_k])
            _rel = abs(_v - _want) / max(abs(_want), 1e-9)
            _ok = _rel < 1e-3
            _n += 0 if _ok else 1
            print(f"[check {_label}] {'OK  ' if _ok else 'DIFF'} {_k}: got {_v:.6f}, "
                  f"published {_want:.6f}  ({100.0 * _rel:.4f} %)")
        return _n

    def _sr22_check_clean_draw(_label, _pub_rows, _clean_rows, _want_drop):
        _pub = set(map(tuple, _pub_rows.tolist()))
        _cln = set(map(tuple, _clean_rows.tolist()))
        _n = _sr22_check(_label, "clean draw is a subset of the published draw", _cln <= _pub, True)
        _n += _sr22_check(_label, "pairs dropped by the evaluation mask", len(_pub) - len(_cln), _want_drop)
        return _n

    def _sr22_check_events(_dir, _want, _label):
        # the dump never raises (it returns None on failure), so a missing file must
        # count as a failed check here rather than abort the pass and lose the grid
        _f = _P(_dir) / "test_timeseries_dump.npz"
        if not _f.exists():
            print(f"[check {_label}] DIFF clean dump missing: {_f}")
            return 1
        _z = _np22.load(_f)
        _ev = _np22.asarray(_z["snap_ev"], dtype=_np22.int64).reshape(-1, 5)
        _ne = len({(int(r[0]), int(r[1]), int(r[4])) for r in _ev})
        _n = _sr22_check(_label, "clean snap rows (evaluation catalogue)", len(_ev), _want[0])
        _n += _sr22_check(_label, "clean distinct snap events (evaluation catalogue)", _ne, _want[1])
        return _n

    def _sr22_same_norm(_a, _b):
        return (float(_a["target_norm"].mean[0]) == float(_b["target_norm"].mean[0])
                and float(_a["target_norm"].std[0]) == float(_b["target_norm"].std[0]))

    _n_bad = 0
    model = None
    test_loader_c = None

    # ---- PASS A: Stage 2 on its paired draw ------------------------------------
    print()
    print("=" * 70)
    print("[clean] PASS A -- Stage 2 (FINAL_WARM), paired draw, N_out = 21")
    print("=" * 70)
    try:
        model = build_model_from_config(
            example_dataset=example_dataset, cfg=cfg, device=device
        )
        assert model.query_decoder is not None, "Stage-2 model has no query decoder"
        try:
            _ck22 = torch.load(_CKPT_IN, map_location=device, weights_only=False)
        except TypeError:
            _ck22 = torch.load(_CKPT_IN, map_location=device)
        assert any(k.startswith("query_decoder.") for k in _ck22["model_state_dict"]), (
            f"{_CKPT_IN} carries no query_decoder.* keys -- that is a STAGE-1 "
            f"checkpoint, not the Stage-2 model")
        model.load_state_dict(_ck22["model_state_dict"], strict=True)
        model.eval()
        if _ck22.get("criterion_state_dict"):
            criterion.load_state_dict(_ck22["criterion_state_dict"], strict=False)
        print(f"[clean] Stage-2 checkpoint epoch {_ck22.get('epoch')} loaded strict=True")
        del _ck22

        # A1 -- the published draw, built above: identity + reproduction
        _np2, _sha2, _rows2 = _sr22_draw(split_indices)
        _n_bad += _sr22_check("S2", "published draw size", _np2, _SR22_DRAW_S2[0])
        _n_bad += _sr22_check("S2", "published draw sha", _sha2, _SR22_DRAW_S2[1])
        _cfg_pub2 = _dc22.replace(cfg, checkpoint_dir=str(_P(cfg.checkpoint_dir) / "published"))
        _P(_cfg_pub2.checkpoint_dir).mkdir(parents=True, exist_ok=True)
        _sr22_pin(test_loader)
        test_model(
            model=model, test_loader=test_loader, criterion=criterion,
            device=device, target_norm=norms["target_norm"],
            target_names=example_dataset.target_feature_names, cfg=_cfg_pub2,
        )
        _n_bad += _sr22_check_metrics(_cfg_pub2.checkpoint_dir, _SR22_REF_S2, "S2")

        # A2 -- the clean draw: the SAME build, then thinned by the evaluation mask
        set_seed(cfg.seed_list[0])
        _tr22, _va22, test_loader_c, norms_c, split_c = build_dataloaders(
            load_case_datasets=load_case_datasets, cfg=cfg,
            test_window_drop=CATENARY_SAMPLES,
        )
        del _tr22, _va22
        _n_bad += _sr22_check("S2", "clean-draw normalizer equals the published one",
                              _sr22_same_norm(norms, norms_c), True)
        assert_windows_are_paired(split_c, cfg.node_counts)
        _nc2, _shac2, _rowsc2 = _sr22_draw(split_c)
        _n_bad += _sr22_check_clean_draw("S2", _rows2, _rowsc2, _SR22_DROP_S2)
        print(f"[clean] Stage-2 clean draw: {_nc2} layout-window pairs, sha {_shac2}")

        set_sensor_noise(test_loader_c, None)
        _sr22_pin(test_loader_c)
        test_model(
            model=model, test_loader=test_loader_c, criterion=criterion,
            device=device, target_norm=norms["target_norm"],
            target_names=example_dataset.target_feature_names, cfg=cfg,
        )
        SNAP_EVENT_TIMES = SNAP_EVENT_TIMES_CAT
        try:
            _sr22_pin(test_loader_c)
            dump_test_timeseries(
                model=model, test_loader=test_loader_c, device=device,
                target_norm=norms["target_norm"],
                target_names=example_dataset.target_feature_names, cfg=cfg,
                make_plots=False,
            )
        finally:
            SNAP_EVENT_TIMES = _SR22_SNAP_PUB
        _n_bad += _sr22_check_events(cfg.checkpoint_dir, _SR22_EV_S2, "S2")

        # A3 -- the grid and its interpolation baseline, on the clean draw
        try:
            _sr22_pin(test_loader_c)
            grid_rows = evaluate_superres_grid(
                model=model, test_loader=test_loader_c, criterion=criterion,
                device=device, target_norm=norms["target_norm"],
                target_names=example_dataset.target_feature_names, cfg=cfg,
            )
            _sr22_pin(test_loader_c)
            baseline_rows = run_interpolation_baseline(
                example_dataset=example_dataset, cfg=cfg, device=device,
                checkpoint_path=cfg.superres_baseline_checkpoint,
                test_loader=test_loader_c, criterion=criterion,
                target_norm=norms["target_norm"],
                target_names=example_dataset.target_feature_names,
            )
            for _m, _hi in (("global_R2_tension", True),
                            ("MAE_tension", False),
                            ("peak_tension_MAE", False),
                            ("temporal_diff_skill_tension", False)):
                compare_superres_grids(grid_rows, baseline_rows, cfg=cfg,
                                       metric=_m, higher_is_better=_hi)
        except Exception as _ge:
            _tb22.print_exc()
            print(f"[clean] grid / baseline FAILED: {_ge!r}")
            _n_bad += 1
    except Exception as _ea:
        _tb22.print_exc()
        print(f"[clean] PASS A FAILED: {_ea!r}")
        _n_bad += 1
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---- PASS B: Stage 1 on ITS OWN draw, rebuilt -------------------------------
    print()
    print("=" * 70)
    print("[clean] PASS B -- Stage 1 (v5), ORIGINAL unpaired draw, matched")
    print("=" * 70)
    model_s1 = None
    try:
        cfg_s1 = _dc22.replace(
            cfg, superres_mode=False, use_query_decoder=False,
            pair_windows_across_node_counts=False, sensor_noise_enabled=False,
            checkpoint_dir=str(_P(cfg.checkpoint_dir) / "stage1"),
        )
        _P(cfg_s1.checkpoint_dir).mkdir(parents=True, exist_ok=True)
        set_seed(cfg_s1.seed_list[0])
        _tr1, _va1, test_loader_s1, norms_s1, split_s1 = build_dataloaders(
            load_case_datasets=load_case_datasets, cfg=cfg_s1,
        )
        del _tr1, _va1
        _tn1 = norms_s1["target_norm"]
        _m1, _s1 = float(_tn1.mean[0]), float(_tn1.std[0])
        _ok = (abs(_m1 - _SR22_V5_NORM[0]) / _SR22_V5_NORM[0] < 1e-6
               and abs(_s1 - _SR22_V5_NORM[1]) / _SR22_V5_NORM[1] < 1e-6)
        _n_bad += _sr22_check("S1", "target normalizer is v5's", _ok, True)
        _np1, _sha1, _rows1 = _sr22_draw(split_s1)
        _n_bad += _sr22_check("S1", "published draw size", _np1, _SR22_DRAW_S1[0])
        _n_bad += _sr22_check("S1", "published draw sha", _sha1, _SR22_DRAW_S1[1])

        criterion_s1 = ReconstructionKendallLoss(
            node_norm=norms_s1["node_norm"], target_norm=_tn1, cfg=cfg_s1,
        ).to(device)
        model_s1 = build_model_from_config(
            example_dataset=example_dataset, cfg=cfg_s1, device=device
        )
        assert getattr(model_s1, "query_decoder", None) is None, (
            "the Stage-1 pass must build the matched model, with no decoder")
        try:
            _v522 = torch.load(cfg.superres_baseline_checkpoint, map_location=device,
                               weights_only=False)
        except TypeError:
            _v522 = torch.load(cfg.superres_baseline_checkpoint, map_location=device)
        assert int(_v522.get("epoch", -1)) == 32, "the Stage-1 checkpoint is not v5 ep32"
        assert not any(k.startswith("query_decoder.") for k in _v522["model_state_dict"]), (
            "the Stage-1 checkpoint carries query_decoder.* keys -- it is a Stage-2 one")
        model_s1.load_state_dict(_v522["model_state_dict"], strict=True)
        model_s1.eval()
        if _v522.get("criterion_state_dict"):
            criterion_s1.load_state_dict(_v522["criterion_state_dict"], strict=False)
        print(f"[clean] v5 checkpoint epoch {_v522.get('epoch')} loaded strict=True")
        del _v522

        # B1 -- reproduction on the published draw
        _cfg_pub1 = _dc22.replace(cfg_s1, checkpoint_dir=str(_P(cfg_s1.checkpoint_dir) / "published"))
        _P(_cfg_pub1.checkpoint_dir).mkdir(parents=True, exist_ok=True)
        _sr22_pin(test_loader_s1)
        test_model(
            model=model_s1, test_loader=test_loader_s1, criterion=criterion_s1,
            device=device, target_norm=_tn1,
            target_names=example_dataset.target_feature_names, cfg=_cfg_pub1,
        )
        _n_bad += _sr22_check_metrics(_cfg_pub1.checkpoint_dir, _SR22_REF_S1, "S1")

        # B2 -- the clean draw
        set_seed(cfg_s1.seed_list[0])
        _tr1, _va1, test_loader_s1c, norms_s1c, split_s1c = build_dataloaders(
            load_case_datasets=load_case_datasets, cfg=cfg_s1,
            test_window_drop=CATENARY_SAMPLES,
        )
        del _tr1, _va1
        _n_bad += _sr22_check("S1", "clean-draw normalizer equals the published one",
                              _sr22_same_norm(norms_s1, norms_s1c), True)
        _nc1, _shac1, _rowsc1 = _sr22_draw(split_s1c)
        _n_bad += _sr22_check_clean_draw("S1", _rows1, _rowsc1, _SR22_DROP_S1)
        print(f"[clean] Stage-1 clean draw: {_nc1} windows, sha {_shac1}")
        _sr22_pin(test_loader_s1c)
        test_model(
            model=model_s1, test_loader=test_loader_s1c, criterion=criterion_s1,
            device=device, target_norm=_tn1,
            target_names=example_dataset.target_feature_names, cfg=cfg_s1,
        )
        SNAP_EVENT_TIMES = SNAP_EVENT_TIMES_CAT
        try:
            _sr22_pin(test_loader_s1c)
            dump_test_timeseries(
                model=model_s1, test_loader=test_loader_s1c, device=device,
                target_norm=_tn1,
                target_names=example_dataset.target_feature_names, cfg=cfg_s1,
                make_plots=False,
            )
        finally:
            SNAP_EVENT_TIMES = _SR22_SNAP_PUB
        _n_bad += _sr22_check_events(cfg_s1.checkpoint_dir, _SR22_EV_S1, "S1")
    except Exception as _eb:
        _tb22.print_exc()
        print(f"[clean] PASS B FAILED: {_eb!r}")
        _n_bad += 1
    model_s1 = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---- PASS C: the noise sweep on the clean Stage-2 draw ----------------------
    # Same level ids and specs as EVAL_NOISE (job 236311), so every window that
    # survives the evaluation mask carries exactly the noise realisation it had there.
    print()
    print("=" * 70)
    print("[clean] PASS C -- Stage 2 noise sweep on the clean draw")
    print("=" * 70)
    if model is None or test_loader_c is None:
        print("[clean] PASS C skipped -- PASS A did not produce the clean model/draw")
        _n_bad += 1
    else:
        _SN_SIGMAS        = (0.01, 0.05, 0.10, 0.25)
        _SN_SNRS_DB       = (20.0, 10.0, 5.0, 0.0)
        _SN_COUPLED_SIGMA = 0.05
        _SN_BIAS_SIGMA    = 0.05
        _nn = norms["node_norm"]
        _LEVELS = [("", None)]
        for _k, _s in enumerate(_SN_SIGMAS):
            _LEVELS.append(("_p%03d" % round(_s * 1000),
                            sensor_noise_spec_absolute(10 + _k, "abs_%.3f" % _s,
                                                       sigma_p=_s, sigma_v=_s)))
        for _k, _db in enumerate(_SN_SNRS_DB):
            _LEVELS.append(("_snr%02d" % int(_db),
                            sensor_noise_spec_snr(20 + _k, _db, _nn, cfg)))
        _LEVELS.append(("_coupled%03d" % round(_SN_COUPLED_SIGMA * 1000),
                        sensor_noise_spec_absolute(30, "coupled", sigma_p=_SN_COUPLED_SIGMA,
                                                   sigma_v=0.0)))
        _LEVELS.append(("_bias%03d" % round(_SN_BIAS_SIGMA * 1000),
                        sensor_noise_spec_absolute(31, "bias", sigma_b=_SN_BIAS_SIGMA)))
        with open(_P(cfg.checkpoint_dir) / "sensor_noise_levels.csv", "w",
                  newline="", encoding="utf-8") as _f_sn:
            _w_sn = _csv22.writer(_f_sn)
            _w_sn.writerow(["tag", "name", "axis", "channel", "sigma_physical",
                            "snr_dB", "sigma_normalised", "channel_train_std"])
            for _tg, _sp in _LEVELS:
                if _sp is None:
                    _w_sn.writerow([_tg or "clean", "clean", "none", "", 0.0, "inf", 0.0, ""])
                    continue
                for _cn, _sd, _snr, _rel, _std in sensor_noise_report(_sp, _nn):
                    _w_sn.writerow([_tg, _sp["name"], _sp.get("axis", ""), _cn,
                                    "%.6g" % _sd, "%.4g" % _snr, "%.6g" % _rel,
                                    "%.6g" % _std])
        # the clean level ("") was scored in PASS A; only the noisy levels here
        for _tg, _spec in _LEVELS[1:]:
            print()
            print(f"[clean] noise LEVEL {_tg}")
            set_sensor_noise(test_loader_c, _spec)
            try:
                _sr22_pin(test_loader_c)
                test_model(
                    model=model, test_loader=test_loader_c, criterion=criterion,
                    device=device, target_norm=norms["target_norm"],
                    target_names=example_dataset.target_feature_names, cfg=cfg,
                    tag=_tg,
                )
            except Exception as _te:
                _tb22.print_exc()
                print(f"[clean] level {_tg} test FAILED: {_te!r}")
                _n_bad += 1
                continue
            SNAP_EVENT_TIMES = SNAP_EVENT_TIMES_CAT
            try:
                _sr22_pin(test_loader_c)
                dump_test_timeseries(
                    model=model, test_loader=test_loader_c, device=device,
                    target_norm=norms["target_norm"],
                    target_names=example_dataset.target_feature_names, cfg=cfg,
                    tag=_tg, make_plots=False,
                )
            except Exception as _tse:
                print(f"[ts-dump] level {_tg} FAILED (metrics unaffected): {_tse}")
            finally:
                SNAP_EVENT_TIMES = _SR22_SNAP_PUB
            try:
                _sr22_pin(test_loader_c)
                run_interpolation_baseline(
                    example_dataset=example_dataset, cfg=cfg, device=device,
                    checkpoint_path=cfg.superres_baseline_checkpoint,
                    test_loader=test_loader_c, criterion=criterion,
                    target_norm=norms["target_norm"],
                    target_names=example_dataset.target_feature_names,
                    out_N_values=(21,), tag="_baseline" + _tg,
                )
            except Exception as _be:
                print(f"[baseline] level {_tg} FAILED (decoder results unaffected): {_be}")
        set_sensor_noise(test_loader_c, None)

    print()
    if _n_bad:
        print("*" * 70)
        print(f"*** {_n_bad} CHECK(S) FAILED -- see the [check ...] / FAILED lines above.")
        print("*** A failed reproduction means that pass does NOT score the published")
        print("*** population; do NOT use its clean outputs.")
        print("*" * 70)
    else:
        print("[check] every pass reproduced its published draw and metrics, and every "
              "clean draw is exactly the published draw minus the evaluation-mask windows.")
    print("EVAL_CLEAN complete. Stage 2: test_metrics*.csv, test_per_window_stats*.csv, "
          "test_timeseries_dump*.npz, superres_grid*.csv; Stage 1: stage1/.")

else:
    model = build_model_from_config(
        example_dataset=example_dataset, cfg=cfg, device=device
    )
    assert model.query_decoder is not None, "Stage-2 model has no query decoder"
    print(f"[model] parameters: {sum(p.numel() for p in model.parameters()):,} "
          f"(decoder "
          f"{sum(p.numel() for p in model.query_decoder.parameters()):,})")

    optimizer, scheduler = build_optimizer_and_scheduler(model, cfg, criterion=criterion)

    # physics-term conditioning on ground truth, now at the QUERY stations:
    # the residual is expected to GROW as N_in falls, because the geometry fed
    # to the terms is progressively smoother than the truth. Kendall absorbs it,
    # and sigma vs N_in is itself a measurement.
    run_physics_sanity_check(val_loader, criterion, norms["target_norm"], max_batches=10)

    criterion.calibrate_log_vars(model, val_loader, device, max_batches=5)

    t0 = time.time()
    model, history, model_path = train_model(
        model=model, train_loader=train_loader, val_loader=val_loader,
        optimizer=optimizer, scheduler=scheduler, criterion=criterion,
        device=device, target_norm=norms["target_norm"],
        target_names=example_dataset.target_feature_names,
        cfg=cfg, run_seed=cfg.seed_list[0],
    )
    print(f"training took {time.time() - t0:.1f} s")
    if torch.cuda.is_available():
        print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

    if cfg.run_test_evaluation:
        test_results = test_model(
            model=model, test_loader=test_loader, criterion=criterion,
            device=device, target_norm=norms["target_norm"],
            target_names=example_dataset.target_feature_names, cfg=cfg,
        )
    else:
        test_results = None
        print("Test evaluation skipped (HPO run -- test set untouched).")

    # -- predicted-vs-true time series (node_count in the dump meta is N_in) --
    if cfg.run_test_evaluation:
        try:
            dump_test_timeseries(
                model=model, test_loader=test_loader, device=device,
                target_norm=norms["target_norm"],
                target_names=example_dataset.target_feature_names, cfg=cfg,
            )
        except Exception as _tse:
            print(f"[ts-dump] FAILED (run results are unaffected): {_tse}")

    # -- THE Stage-2 deliverable: the N_in x N_out surface and its baseline ---
    if cfg.run_test_evaluation:
        try:
            grid_rows = evaluate_superres_grid(
                model=model, test_loader=test_loader, criterion=criterion,
                device=device, target_norm=norms["target_norm"],
                target_names=example_dataset.target_feature_names, cfg=cfg,
            )
            baseline_rows = run_interpolation_baseline(
                example_dataset=example_dataset, cfg=cfg, device=device,
                checkpoint_path=cfg.superres_baseline_checkpoint,
                test_loader=test_loader, criterion=criterion,
                target_norm=norms["target_norm"],
                target_names=example_dataset.target_feature_names,
            )
            for _m, _hi in (("global_R2_tension", True),
                            ("MAE_tension", False),
                            ("peak_tension_MAE", False),
                            ("temporal_diff_skill_tension", False)):
                compare_superres_grids(grid_rows, baseline_rows, cfg=cfg,
                                       metric=_m, higher_is_better=_hi)
        except Exception as _ge:
            print(f"[grid] FAILED (training results unaffected): {_ge}")

    # -- one .npz per feasible checkpoint, pinned to identical test windows ---
    if cfg.run_test_evaluation and getattr(cfg, "ts_dump_all_checkpoints", False):
        try:
            import copy as _copy
            import re as _re
            from pathlib import Path as _P

            _ck_dir = _P(cfg.checkpoint_dir)
            _ck_files = sorted(_ck_dir.glob("checkpoint_epoch*.pt"))
            _selected_state = _copy.deepcopy(model.state_dict())
            print()
            print(f"[ts-dump] per-checkpoint dumps: {len(_ck_files)} feasible "
                  f"checkpoint(s) x {cfg.ts_dump_max_windows_per_ckpt} test windows")
            for _cf in _ck_files:
                _m = _re.search(r"checkpoint_epoch(\d+)\.pt$", _cf.name)
                if _m is None:
                    continue
                _ep = int(_m.group(1))
                try:
                    _sd = torch.load(_cf, map_location=device, weights_only=False)
                except TypeError:
                    _sd = torch.load(_cf, map_location=device)
                model.load_state_dict(_sd["model_state_dict"], strict=True)
                model.eval()
                dump_test_timeseries(
                    model=model, test_loader=test_loader, device=device,
                    target_norm=norms["target_norm"],
                    target_names=example_dataset.target_feature_names, cfg=cfg,
                    tag=f"_epoch{_ep:04d}", make_plots=False,
                    max_windows=cfg.ts_dump_max_windows_per_ckpt,
                )
                del _sd
            model.load_state_dict(_selected_state, strict=True)
            model.eval()
            print("[ts-dump] per-checkpoint dumps done; model restored to the "
                  "selected state")
        except Exception as _pce:
            print(f"[ts-dump] per-checkpoint dumps FAILED "
                  f"(run results unaffected): {_pce}")
            try:
                model.load_state_dict(_selected_state, strict=True)
                model.eval()
            except Exception:
                pass

    print()
    print("Run complete.")
    print(f"Best checkpoint: {model_path}")
    print(f"Final train loss: {history['train_loss'][-1]:.6f}")
    print(f"Final val loss:   {history['val_loss'][-1]:.6f}"
          if history['val_loss'] else "Final val loss:   (no validation completed)")


In [23]:
# -------------------------------------------------------------
# CPU/GPU device-isolation smoke test
# -------------------------------------------------------------

def _smoke_test_device_isolation(ds):
    s0 = ds[0]
    assert s0["x_seq"].device.type == "cpu", "x_seq must be on CPU after __getitem__"
    assert s0["graph"].x.device.type == "cpu", "graph.x must be on CPU after __getitem__"

    if torch.cuda.is_available():
        device = torch.device("cuda")
        moved = move_sample_to_device(s0, device)
        assert moved["x_seq"].device.type == "cuda", "moved x_seq must be on CUDA"
        assert moved["graph"].x.device.type == "cuda", "moved graph.x must be on CUDA"

        # Fetch again — dataset graph must still be on CPU
        s1 = ds[1]
        assert s1["x_seq"].device.type == "cpu", "x_seq must remain on CPU after GPU move"
        assert s1["graph"].x.device.type == "cpu", "graph.x must remain on CPU after GPU move"
        print("smoke test passed (CPU + CUDA checks)")
    else:
        print("smoke test passed (CPU-only: no CUDA available)")


# Run against the first available lazy dataset
_first_ds = next(iter(load_case_datasets.values()))
_smoke_test_device_isolation(_first_ds)

smoke test passed (CPU + CUDA checks)
